# Vaani SED V5 — complete pipeline

**What V5 changes** (everything else is your V3 network and `best_stage2.pt`):
1. **7 category frame tracks + a MIL clip head.** The single noise/no-noise output becomes one track per category. The new heads are copied from your trained head, so training starts from your model. A clip-level head pools those same tracks, so Silver and Bronze category labels teach category identity without trusting their timestamps.
2. **Corrected BEATs alignment.** Every window starts on the 0.16 s BEATs patch grid, and the cached rows are sliced so each row sits where the interpolation puts it (residual 7.5 ms, was up to −228 ms).
3. **Synthetic mixtures with exact labels.** Clean-speech stretches from Gold-train clips + pasted Gold-train noise events, so the model trains on the "clean speech + added noise" type that makes up ~38% of the test set. Built only from your training data; test audio is never used.
4. **Selection on natural + synthetic validation.** Checkpoints and the decoder are chosen with 0.62·Val-N + 0.38·Val-S and must hold on both district halves.

## How to run
1. Edit **V5 USER CONFIG** (next cell): `BASE_DIR`, `TEST_DIR`. Leave `ESC50_DIR = None` unless external data is allowed.
2. Recommended first: set `SMOKE = True`, switch every `RUN_*` flag on, Run All (minutes; writes only to `*_smoke` folders and `*_SMOKE.zip`), check there are no errors. Then set `SMOKE = False` and switch the flags back off.
3. The default config runs **E0 only**. Read its verdict in the log, then switch on the next stage and Run All again; finished stages are skipped or reloaded from disk:

| Order | Flag | What it does | Time (RTX, estimate) | Output |
|---|---|---|---|---|
| 1 | `RUN_E0_RECONCILE` | best_stage2 through 3 inference paths on all Gold val | 20–40 min | `v5_work/e0_tracks.pkl`, verdict in log |
| 2 | `RUN_BUILD_SYNTH` | 24,000 train mixtures + 1,000 Val-S, with BEATs; every file verified | 1–2 h, resumable | `synth_v5/` |
| 3 | `RUN_TRAIN_V5` | V5-A fine-tune, 8,000 updates, eval every 1,000 | ~2.5–3 h | `v5_work/v5a_ema.pt` / `_student.pt` |
| 4 | `RUN_DECODER_SELECTION` | decoder selection + control comparison | 15–30 min | `v5_work/v5a_decoder.json`, verdict in log |
| 5 | `RUN_TEST_SUBMISSION` | test BEATs cache (explicit) + V5 zip | ~15 min | `submission/submission_v5a.zip` |
| 6 | `RUN_FINAL_RETRAIN` | retrain on all Gold for the same update count | ~2.5 h | `submission/submission_v5a_final.zip` |

Everything is logged to `v5_work/v5_log.txt`; `v5_work/progress_live.txt` shows the current training step.

**Follow-up experiments (after V5-A has an LB score):** in the training cell set ONE of `per_cat_boundary=True` (7 onset + 7 offset channels) or `short_mel=True` (32 ms mel branch), change `RUN_TAG` (e.g. `"v5b_bnd"`), and run stages 3–5 again. Both start exactly at the V5-A warm start (copied / zero-initialised weights), so each LB change stays attributable.

**Scorer:** the local scorer (Part 1) is the organisers' published Track 1 code (50 ms tolerance floor, 10 ms raster with inclusive right frame). It is kept as is on purpose so local numbers track the leaderboard.

**Fallback:** `RUN_CONTROL_ZIP = True` re-creates the 0.9965 submission (cell-29 path) as `submission_0p9965_control.zip`. Submit V5 only if the E3 log line says it beats the control on both halves, and change one thing per leaderboard submission.


In [28]:
# =============================================================================
# V5 USER CONFIG -- the only cell you should need to edit.
# Every stage below checks its RUN_* flag; a stage that is switched off is
# skipped cleanly, and stages that depend on it load its saved outputs from
# disk instead (so you can run the notebook in several sittings).
# =============================================================================
BASE_DIR = "C:/Users/231b258"                 # same folder as before (caches, best_stage2.pt)
TEST_DIR = f"{BASE_DIR}/test"
TEST_AUDIO_DIR = f"{TEST_DIR}/audio"
EXPECTED_TEST_CLIPS = 5517                    # hard check on the submission line count

# Optional external foregrounds for the synthetic mixtures. Leave None unless the
# competition rules allow external data. If set, point it at the unzipped ESC-50
# folder (the one containing audio/ and meta/esc50.csv).
ESC50_DIR = None

# ---- stages: turn them on ONE AT A TIME, in this order (see the run guide) -----
# Default = E0 only, so Run All cannot start hours of work before E0 is read.
RUN_E0_RECONCILE      = False    # ~20-40 min: which inference path reproduces 0.9965?
RUN_CONTROL_ZIP       = False   # re-create the 0.9965 control submission (cell-29 path)
RUN_E0_TEST_PROBE     = False   # best_stage2 + corrected cached BEATs -> 1 LB probe zip
RUN_BUILD_SYNTH       = False    # synthetic train mixtures + Val-S (resumable)
RUN_TRAIN_V5          = True    # V5-A fine-tune from best_stage2.pt (~2.5-3 h)
RUN_DECODER_SELECTION = True    # CPU-only once val tracks are cached
RUN_TEST_SUBMISSION   = True    # builds the test BEATs cache (explicit) + V5 zip
RUN_FINAL_RETRAIN     = False   # E7: retrain on all Gold (train+val) at a fixed update count

SMOKE = False   # True = tiny end-to-end check (minutes). Never submit a SMOKE output.
RUN_TAG = "v5c_smel"
SEED = 42


## Part 1 — your V3 definitions (unchanged except: `BASE_DIR` now comes from the config cell)


In [29]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


Python: C:\Users\231b258\torch_gpu\Scripts\python.exe
PyTorch: 2.14.0+cu132
CUDA: 13.2
CUDA available: True
GPU: NVIDIA RTX PRO 2000 Blackwell
VRAM: 15.93 GB


In [30]:
# =============================================================================
# CELL: Setup  (LOCAL / WINDOWS -- native Windows, no WSL2, confirmed via
# `!uname -a` failing with "not recognized". No bash available, so every
# step below uses plain Python/subprocess instead of `!`-shell commands --
# these are the exact same operations, just OS-independent. Confirmed
# working stack: torch 2.14.0+cu132, CUDA 13.2, compute capability (12, 0)
# -- install with:
#   pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu130
# =============================================================================
import os, sys, random, json, re, csv, io, difflib, shutil, subprocess
from collections import Counter, defaultdict
from copy import deepcopy

# 387GB free on C:, so the home directory is fine -- no need to relocate
# anything. unilm/, beats_cache_full/, wav_cache_16k_flac/ will be created
# fresh here on first run.
# BASE_DIR is set in the V5 USER CONFIG cell
os.makedirs(BASE_DIR, exist_ok=True)

# ---- unilm/BEATs source: clone once, keep on re-runs (was `!rm -rf` +
# `!git clone` unconditionally every run -- that's wasteful even on Linux;
# now only clones if missing).
UNILM_DIR = f"{BASE_DIR}/unilm"
if not os.path.isdir(os.path.join(UNILM_DIR, "beats")):
    shutil.rmtree(UNILM_DIR, ignore_errors=True)
    if shutil.which("git") is None:
        raise FileNotFoundError(
            "git not found on PATH. Install Git for Windows "
            "(https://git-scm.com/download/win) and restart the kernel.")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/microsoft/unilm.git", UNILM_DIR], check=True)
else:
    print(f"unilm already present at {UNILM_DIR}, skipping clone")

# %pip (not !pip) targets THIS kernel's environment specifically -- on
# Windows especially, `!pip` can silently resolve to a different Python
# install than the one Jupyter is actually running.
%pip install -q scipy scikit-learn soundfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from sklearn.model_selection import GroupShuffleSplit
from scipy.signal import medfilt
import soundfile as sf
from datasets import load_from_disk, disable_caching

disable_caching()

BEATS_DIR = f"{UNILM_DIR}/beats"
if not os.path.isdir(BEATS_DIR):
    raise FileNotFoundError(f"BEATs source not found: {BEATS_DIR}")
sys.path.insert(0, BEATS_DIR)
from BEATs import BEATs, BEATsConfig

_dataset_root = f"{BASE_DIR}/archive/vaani_noise_dataset"
# Try the Kaggle-matching layout (.../vaani_noise_dataset/train) first;
# fall back to the root itself if this archive wasn't split into a `train`
# subfolder -- either way, DATASET_PATH ends up pointing at whichever
# directory actually has the load_from_disk arrow/json files in it.
DATASET_PATH = f"{_dataset_root}/train" if os.path.isdir(f"{_dataset_root}/train") else _dataset_root
BEATS_CHECKPOINT = f"{BASE_DIR}/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"
if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f"DATASET_PATH not found: {DATASET_PATH} -- check the actual folder "
        f"name under {_dataset_root} (`dir \"{_dataset_root}\"`) and adjust.")
if not os.path.isfile(BEATS_CHECKPOINT):
    raise FileNotFoundError(f"BEATS_CHECKPOINT not found: {BEATS_CHECKPOINT} -- update this path")
print(f"DATASET_PATH resolved to: {DATASET_PATH}")
# BEATs cache: pure /kaggle/working, no packaged Kaggle Dataset involved.
# beats-cache-full-3 (a prior packaged upload) was confirmed 100% all-zero
# (every cached array read back as exact 0.0 -- almost certainly the old
# fp16-autocast NaN bug getting silently laundered into zeros by
# nan_to_num during a build that predates the fp32 fix). Rather than
# re-publish a Dataset, this rebuilds directly into the working directory
# each session and trains straight from that. KAGGLE_DATASET_CACHE_DIR is
# kept as a hook (None = unused) in case you want a packaged cache again
# later -- every downstream cell already reads BEATS_CACHE_DIRS as a list
# and degrades to a single directory when this is None, so nothing else
# needs to change if you do.
KAGGLE_DATASET_CACHE_DIR = None
BEATS_CACHE_BUILD_DIR = f"{BASE_DIR}/beats_cache_full"      # writable -- the only cache tier now
BEATS_CACHE_DIRS = ([KAGGLE_DATASET_CACHE_DIR] if KAGGLE_DATASET_CACHE_DIR else []) + [BEATS_CACHE_BUILD_DIR]
BEATS_CACHE_DIR = BEATS_CACHE_BUILD_DIR  # kept as an alias for any old single-dir usage

print("BEATs source:", BEATS_DIR)
print("Cache (read, priority order):", BEATS_CACHE_DIRS)
print("Cache (build/writable):", BEATS_CACHE_BUILD_DIR)
if KAGGLE_DATASET_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_CACHE_DIR):
    _n_packaged = len([f for f in os.listdir(KAGGLE_DATASET_CACHE_DIR) if f.endswith(".npy")])
    print(f"  packaged dataset cache: {_n_packaged} .npy files found at {KAGGLE_DATASET_CACHE_DIR}")
else:
    print("  no packaged dataset configured -- reading/writing BEATS_CACHE_BUILD_DIR only")

# Same pure-working-directory approach for the waveform cache (raw-audio
# decode+resample cache, separate from the BEATs feature cache above) --
# no zip/dataset publishing, just build once per session into
# WAV_CACHE_BUILD_DIR and read straight from it. KAGGLE_DATASET_WAV_CACHE_DIR
# is kept as an unused hook (None) for the same reason as its BEATs
# counterpart above.
KAGGLE_DATASET_WAV_CACHE_DIR = None
WAV_CACHE_BUILD_DIR = f"{BASE_DIR}/wav_cache_16k_flac"
WAV_CACHE_DIR = WAV_CACHE_BUILD_DIR  # alias, mirrors BEATS_CACHE_DIR
WAV_CACHE_DIRS = ([KAGGLE_DATASET_WAV_CACHE_DIR] if KAGGLE_DATASET_WAV_CACHE_DIR else []) + [WAV_CACHE_BUILD_DIR]
print("Waveform cache (read, priority order):", WAV_CACHE_DIRS)
if KAGGLE_DATASET_WAV_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_WAV_CACHE_DIR):
    _n_wav_packaged = len([f for f in os.listdir(KAGGLE_DATASET_WAV_CACHE_DIR) if f.endswith(".flac")])
    print(f"  packaged waveform cache: {_n_wav_packaged} .flac files found at {KAGGLE_DATASET_WAV_CACHE_DIR}")
else:
    print("  no packaged waveform dataset configured -- reading/writing WAV_CACHE_BUILD_DIR only")

# BEATs fbank framing is 25 ms window / 10 ms hop, and patch_embedding has
# stride = input_patch_size = 16. So each BEATs patch advances 16 * 10 ms
# = 160 ms, with the first patch centered ~75 ms into the clip. These constants
# replace the previous `duration / T_full` approximation.
BEATS_FBANK_HOP_SEC    = 0.010
BEATS_PATCH_SIZE       = 16
BEATS_PATCH_HOP_SEC    = BEATS_FBANK_HOP_SEC * BEATS_PATCH_SIZE          # 0.16 s
BEATS_PATCH_OFFSET_SEC = (BEATS_PATCH_SIZE - 1) * BEATS_FBANK_HOP_SEC / 2  # 0.075 s
print(f"BEATs patch hop={BEATS_PATCH_HOP_SEC}s offset={BEATS_PATCH_OFFSET_SEC}s")


unilm already present at C:/Users/231b258/unilm, skipping clone
Note: you may need to restart the kernel to use updated packages.
DATASET_PATH resolved to: C:/Users/231b258/archive/vaani_noise_dataset/train
BEATs source: C:/Users/231b258/unilm/beats
Cache (read, priority order): ['C:/Users/231b258/beats_cache_full']
Cache (build/writable): C:/Users/231b258/beats_cache_full
  no packaged dataset configured -- reading/writing BEATS_CACHE_BUILD_DIR only
Waveform cache (read, priority order): ['C:/Users/231b258/wav_cache_16k_flac']
  no packaged waveform dataset configured -- reading/writing WAV_CACHE_BUILD_DIR only
BEATs patch hop=0.16s offset=0.075s


In [31]:
# =============================================================================
# CELL: Tag canonicalization (unchanged)
# =============================================================================
_BRACKET_RE = re.compile(r"^[\[]?<(.+)>[\]]?$")
_SEED_SYNONYMS = {
    "crow cawaing": "crow cawing", "lip_smacking": "lip smacking",
    "cawaing": "cawing", "yawing": "yawning", "sniffling": "sniffing",
    "hiccups": "hiccup", "nose sneezing": "sneezing",
    "nose clearing": "throat clearing", "cleaning throat": "throat clearing",
    "throat cleaning": "throat clearing",
}
_INTERCHANGEABLE_SUFFIXES = {"noise", "sound", "voice", "playing", "tone", "song"}
_PLURAL_MAP = {"children": "child", "kids": "kid", "birds": "bird", "bugs": "bug",
               "insects": "insect", "vehicles": "vehicle", "songs": "song", "crickets": "cricket"}

def normalize_tag(raw_tag):
    m = _BRACKET_RE.match(raw_tag.strip())
    inner = m.group(1) if m else raw_tag.strip()
    inner = inner.replace("_", " ").lower().strip()
    inner = re.sub(r"\s+", " ", inner)
    return _SEED_SYNONYMS.get(inner, inner)

def _canonical_tokens(norm_tag):
    tokens = [_PLURAL_MAP.get(t, t) for t in norm_tag.split(" ")]
    core = [t for t in tokens if t not in _INTERCHANGEABLE_SUFFIXES]
    return tuple(core) if core else tuple(tokens)

def build_tag_clusters(tag_counts, similarity_threshold=0.82):
    normalized = {raw: normalize_tag(raw) for raw in tag_counts}
    token_key_to_raws = defaultdict(list)
    for raw, norm in normalized.items():
        token_key_to_raws[_canonical_tokens(norm)].append(raw)
    clusters = list(token_key_to_raws.values())
    keys = list(token_key_to_raws.keys())
    merged, used = [], [False] * len(clusters)
    for i in range(len(clusters)):
        if used[i]: continue
        group = list(clusters[i]); used[i] = True
        key_i = " ".join(keys[i])
        for j in range(i + 1, len(clusters)):
            if used[j]: continue
            if difflib.SequenceMatcher(None, key_i, " ".join(keys[j])).ratio() >= similarity_threshold:
                group.extend(clusters[j]); used[j] = True
        merged.append(group)
    raw_to_canonical, cluster_report = {}, []
    for group in merged:
        rep = max(group, key=lambda r: tag_counts.get(r, 0))
        canonical = normalize_tag(rep)
        total = sum(tag_counts.get(r, 0) for r in group)
        for r in group: raw_to_canonical[r] = canonical
        cluster_report.append({"canonical_tag": canonical, "total_count": total,
                               "n_raw_variants": len(group),
                               "raw_variants": sorted(group, key=lambda r: -tag_counts.get(r, 0))})
    cluster_report.sort(key=lambda r: -r["total_count"])
    return raw_to_canonical, cluster_report

def write_cluster_report_csv(cluster_report, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["canonical_tag", "total_count", "n_raw_variants", "raw_variants"])
        for row in cluster_report:
            w.writerow([row["canonical_tag"], row["total_count"], row["n_raw_variants"],
                        " | ".join(row["raw_variants"])])


In [32]:
# =============================================================================
# CELL: Constants and helpers
# =============================================================================
COARSE_CATEGORIES = ["animal", "appliance_machine", "baby_child", "human_non_speech",
                     "phone_signal_alarm", "singing_music", "vehicle_traffic"]
COARSE_TO_IDX = {c: i for i, c in enumerate(COARSE_CATEGORIES)}

TIER_VERIFIED, TIER_UNVERIFIED, TIER_WEAK_ONLY, TIER_PSEUDO_BRONZE = 0, 1, 2, 3
_QUALITY_TO_TIER = {"verified_timestamps": TIER_VERIFIED,
                    "unverified_timestamps": TIER_UNVERIFIED,
                    "no_timestamps": TIER_WEAK_ONLY}

def split_indices_by_tier(dataset, indices=None):
    """Returns tier -> list of GLOBAL indices (positions into `dataset`)."""
    quality_col = dataset["annotationQuality"]
    idxs = range(len(dataset)) if indices is None else list(indices)
    gold, silver, bronze = [], [], []
    for i in idxs:
        q = quality_col[i]
        if q == "verified_timestamps": gold.append(i)
        elif q == "unverified_timestamps": silver.append(i)
        elif q == "no_timestamps": bronze.append(i)
    return {"gold": gold, "silver": silver, "bronze": bronze}

def _load_audio(audio_field):
    if audio_field.get("array") is not None:
        wav = np.asarray(audio_field["array"], dtype=np.float32)
        return wav, int(audio_field["sampling_rate"])
    if audio_field.get("bytes") is not None:
        wav, sr = sf.read(io.BytesIO(audio_field["bytes"]), dtype="float32")
    elif audio_field.get("path"):
        wav, sr = sf.read(audio_field["path"], dtype="float32")
    else:
        raise ValueError(f"Unknown audio field: {list(audio_field.keys())}")
    if wav.ndim > 1: wav = wav.mean(axis=1)
    return wav.astype(np.float32), int(sr)

def resolve_wav_cache_path(full_idx, cache_dir_or_dirs):
    """Same lookup pattern as resolve_cache_path (checks each dir in
    priority order, first match wins), but for the FLAC waveform cache --
    kept as a separate function rather than overloading resolve_cache_path
    so every existing BEATs-cache call site (.npy) is untouched."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.flac")
        if os.path.exists(p):
            return p
    return None

def _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir=None):
    """Loads a resampled mono waveform for full_idx. Uses the waveform
    cache when available (no audio decode, no row access at all); falls
    back to a live decode+resample otherwise. wav_cache_dir may be a
    single directory (old behavior) or a list -- read priority order,
    mirroring BEATS_CACHE_DIRS: e.g. a read-only packaged Kaggle Dataset
    checked first, the writable working-dir cache second. Self-healing
    writes always go to the LAST directory in the list, by convention the
    writable build dir (any earlier ones are read-only packaged data)."""
    dirs = _normalize_cache_dirs(wav_cache_dir)
    if dirs:
        p = resolve_wav_cache_path(full_idx, dirs)
        if p is not None:
            wav_np, _ = sf.read(p, dtype="float32")
            return torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    wav_np, sr = _load_audio(parent_hf[int(full_idx)]["audio"])
    wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    if wav.ndim > 1: wav = wav.mean(dim=-1)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
    wav = wav.contiguous()
    if dirs:
        write_dir = dirs[-1]
        try:
            os.makedirs(write_dir, exist_ok=True)
            sf.write(os.path.join(write_dir, f"{int(full_idx):06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception:
            pass
    return wav

def build_label_table(hf_dataset):
    """Pulls ONLY the columns needed for labels via columnar access, once.
    This is what actually makes the waveform cache pay off: HF datasets
    decode the audio column as a side effect of ANY row access
    (hf_dataset[i]), even when you never touch that field -- so without
    this, every __getitem__ would still fully decode audio just to read
    NoiseCategory/NoiseSubCategoryTimeStamp/annotationQuality off the same
    row, then discard the decoded audio in favor of the cached waveform.
    Column access (hf_dataset["col"]) does not trigger that decode."""
    cols = hf_dataset.column_names
    return {
        "NoiseCategory": hf_dataset["NoiseCategory"],
        "NoiseSubCategoryTimeStamp": hf_dataset["NoiseSubCategoryTimeStamp"],
        "annotationQuality": hf_dataset["annotationQuality"],
        "id": hf_dataset["id"] if "id" in cols else [None] * len(hf_dataset),
    }

def _ref_events_for_full_idx(full_idx, label_table):
    evs = []
    for e in (label_table["NoiseSubCategoryTimeStamp"][full_idx] or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def _normalize_cache_dirs(cache_dir_or_dirs):
    if cache_dir_or_dirs is None: return []
    if isinstance(cache_dir_or_dirs, (list, tuple)): return list(cache_dir_or_dirs)
    return [cache_dir_or_dirs]

def resolve_cache_path(full_idx, cache_dir_or_dirs):
    """Checks each directory in priority order (e.g. a read-only packaged
    Kaggle Dataset first, the writable working-dir cache second) and
    returns the first existing path, or None if not found in any of them.
    Accepts either a single directory string (old behavior, unchanged) or
    a list of directories (new two-tier behavior)."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.npy")
        if os.path.exists(p):
            return p
    return None

def _validate_cache_exhaustive(all_required_indices, cache_dir, label=""):
    """Verify EVERY required index has a cache file in AT LEAST ONE of the
    given directories (str or list). Raises if any are missing everywhere.
    This is the only acceptable pre-training check -- sampling leaves room
    for thousands of silent mismatches."""
    all_required_indices = sorted({int(i) for i in all_required_indices})
    missing = [fi for fi in all_required_indices if resolve_cache_path(fi, cache_dir) is None]
    tag = f"[{label}] " if label else ""
    if missing:
        raise RuntimeError(
            f"{tag}BEATs cache incomplete: {len(missing)} of "
            f"{len(all_required_indices)} required indices missing from "
            f"{_normalize_cache_dirs(cache_dir)}. First 10: {missing[:10]}. "
            f"Rebuild the cache before training -- the notebook does not "
            f"fall back to live BEATs.")
    print(f"  {tag}[cache-validate] all {len(all_required_indices)} required "
          f"indices present across {_normalize_cache_dirs(cache_dir)}")
    return True


In [33]:
# =============================================================================
# CELL: Mel frontend (unchanged)
# =============================================================================
class MelFrontend(nn.Module):
    def __init__(self, n_mels=128, sample_rate=16000, n_fft=2048,
                 hop_length=160, max_seconds=8.0):
        super().__init__()
        self.sample_rate = int(sample_rate); self.hop_length = int(hop_length)
        self.max_seconds = float(max_seconds); self.n_mels = int(n_mels)
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=n_fft, hop_length=self.hop_length,
            n_mels=self.n_mels, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
    def forward(self, waveform):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        max_len = int(round(self.max_seconds * self.sample_rate))
        if waveform.shape[1] > max_len: waveform = waveform[:, :max_len]
        elif waveform.shape[1] < max_len: waveform = F.pad(waveform, (0, max_len - waveform.shape[1]))
        mel = self.to_db(self.mel_spec(waveform))
        mean = mel.mean(dim=(1, 2), keepdim=True)
        std = mel.std(dim=(1, 2), keepdim=True).clamp_min(1e-5)
        return (mel - mean) / std

def apply_mel_specaugment(mel, time_masks=2, freq_masks=2, max_time_frac=0.08, max_freq_bins=12):
    if mel is None: return None
    out = mel.clone(); B, Freq, Time = out.shape
    max_t = max(1, int(Time * max_time_frac))
    for b in range(B):
        for _ in range(time_masks):
            w = random.randint(0, max_t)
            if w:
                s = random.randint(0, max(0, Time - w)); out[b, :, s:s+w] = 0
        for _ in range(freq_masks):
            w = random.randint(0, min(max_freq_bins, Freq))
            if w:
                s = random.randint(0, max(0, Freq - w)); out[b, s:s+w, :] = 0
    return out


In [34]:
# =============================================================================
# CELL: DFD-LKA encoder
# =============================================================================
class DFDConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, n_kernels=4,
                 dilations=(1, 2, 3, 4)):
        super().__init__()
        self.kernel_size = kernel_size
        self.dilations = list(dilations)[:n_kernels]
        self.kernels = nn.ParameterList([
            nn.Parameter(torch.randn(out_ch, in_ch, kernel_size, kernel_size) * 0.02)
            for _ in self.dilations])
        self.biases = nn.ParameterList([nn.Parameter(torch.zeros(out_ch)) for _ in self.dilations])
        self.attn = nn.Sequential(nn.AdaptiveAvgPool2d((None, 1)),
                                  nn.Conv2d(in_ch, n_kernels, kernel_size=1))
    def forward(self, x):
        logits = self.attn(x).squeeze(-1)
        weights = F.softmax(logits, dim=1)
        H, W = x.shape[-2], x.shape[-1]
        out = 0.0
        for k, (kernel, bias, d) in enumerate(zip(self.kernels, self.biases, self.dilations)):
            pad = (self.kernel_size - 1) * d // 2
            y = F.conv2d(x, kernel, bias, padding=pad, dilation=d)
            if y.shape[-2] != H or y.shape[-1] != W: y = y[..., :H, :W]
            w = weights[:, k, :].unsqueeze(1).unsqueeze(-1)
            out = out + y * w
        return out

class LKAAttention(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.dw = nn.Conv2d(channels, channels, kernel_size, padding=pad,
                            dilation=dilation, groups=channels)
        self.pw = nn.Conv2d(channels, channels, 1)
        self.gate = nn.Sigmoid()
    def forward(self, x):
        return x * self.gate(self.pw(self.dw(x))) + x

class DFDLKABlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_kernels=4, lka_kernel=7):
        super().__init__()
        self.dfd = DFDConv2d(in_ch, out_ch, n_kernels=n_kernels)
        self.bn = nn.BatchNorm2d(out_ch); self.act = nn.GELU()
        self.lka = LKAAttention(out_ch, kernel_size=lka_kernel)
    def forward(self, x):
        return self.lka(self.act(self.bn(self.dfd(x))))

class DFDLKAEncoder(nn.Module):
    def __init__(self, in_ch=1, channels=(32, 64, 128, 192),
                 freq_reduction=(2, 2, 2, 1), n_kernels=4, lka_kernel=7):
        super().__init__()
        self.blocks = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = in_ch
        for ch, fr in zip(channels, freq_reduction):
            self.blocks.append(DFDLKABlock(prev, ch, n_kernels, lka_kernel))
            self.pools.append(nn.MaxPool2d((fr, 1)) if fr > 1 else nn.Identity())
            prev = ch
        self.final_freq_pool = nn.AdaptiveAvgPool2d((1, None))
    def forward(self, x):
        for block, pool in zip(self.blocks, self.pools): x = pool(block(x))
        x = self.final_freq_pool(x).squeeze(2)
        return x.transpose(1, 2)

class TemporalRefinement(nn.Module):
    def __init__(self, dim, kernel_size=5):
        super().__init__()
        self.dw = nn.Conv1d(dim, dim, kernel_size, padding=kernel_size//2, groups=dim)
        self.norm = nn.LayerNorm(dim); self.act = nn.GELU()
    def forward(self, x):
        y = self.dw(x.transpose(1, 2)).transpose(1, 2)
        return self.act(self.norm(x + y))


In [35]:
# =============================================================================
# CELL: BEATs temporal extractor + loader + cache builder
# =============================================================================
def beats_extract_temporal(beats_model, source, padding_mask=None,
                           fbank_mean=15.41663, fbank_std=6.55582):
    fbank = beats_model.preprocess(source, fbank_mean=fbank_mean, fbank_std=fbank_std)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(fbank, padding_mask)
    fbank = fbank.unsqueeze(1)
    feat2d = beats_model.patch_embedding(fbank)
    T_patches, F_patches = feat2d.shape[2], feat2d.shape[3]
    feat_seq = feat2d.reshape(feat2d.shape[0], feat2d.shape[1], -1).transpose(1, 2)
    feat_seq = beats_model.layer_norm(feat_seq)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(feat_seq, padding_mask)
    if beats_model.post_extract_proj is not None:
        feat_seq = beats_model.post_extract_proj(feat_seq)
    x = beats_model.dropout_input(feat_seq)
    x, _ = beats_model.encoder(x, padding_mask=padding_mask)
    x = x.reshape(x.shape[0], T_patches, F_patches, x.shape[-1])
    x_time = x.mean(dim=2)
    return x_time, T_patches, None

def load_beats_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    beats_cfg = BEATsConfig(checkpoint["cfg"])
    print("BEATs cfg: layers=", beats_cfg.encoder_layers,
          " dim=", beats_cfg.encoder_embed_dim,
          " patch=", getattr(beats_cfg, "input_patch_size", "n/a"))
    beats_model = BEATs(beats_cfg)
    beats_model.load_state_dict(checkpoint["model"], strict=True)
    beats_model.predictor = None
    beats_model.to(device).eval()
    for p in beats_model.parameters(): p.requires_grad_(False)
    print("BEATs loaded and frozen.")
    return beats_model, beats_cfg.encoder_embed_dim

@torch.no_grad()
def extract_beats_full(beats_model, waveform_1d, device):
    wav = waveform_1d.to(device).unsqueeze(0)
    # FIXED: was enabled=(device.type == "cuda"). Running a real, deep,
    # pretrained transformer under fp16 autocast produced all-NaN output
    # here (confirmed empirically -- fp16 gave NaN=True, fp32 gave a clean
    # min/max of -1.58/2.29 on the same clip). BEATs is frozen and this
    # only runs once per clip during caching, so there is no speed reason
    # to keep it in fp16 -- autocast is for things you backprop through.
    with autocast(device_type="cuda", enabled=False):
        feat, T_patches, _ = beats_extract_temporal(beats_model, wav)
    return feat[0].float().cpu().numpy(), T_patches

def build_beats_cache(hf_dataset, beats_model, device, cache_dir, sample_rate=16000,
                      max_clips=None, log_every=200, also_check_dirs=None):
    """Cache keyed by FULL-dataset index (i in range(len(hf_dataset))).
    Always WRITES to cache_dir (must be writable). also_check_dirs, if
    given, is a read-only-dirs-first check -- an index already present
    there (e.g. in a packaged Kaggle Dataset) is skipped entirely, so this
    never wastes time recomputing something you already have."""
    os.makedirs(cache_dir, exist_ok=True)
    n = len(hf_dataset) if max_clips is None else min(max_clips, len(hf_dataset))
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    manifest = {}
    n_skipped_from_other_dirs = 0
    for i in range(n):
        path = os.path.join(cache_dir, f"{i:06d}.npy")
        if resolve_cache_path(i, read_dirs) is not None:
            if not os.path.exists(path):
                n_skipped_from_other_dirs += 1
            continue
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, sr, sample_rate)
            feat, T_patches = extract_beats_full(beats_model, wav, device)

            # NEW: loud, unmissable warning if raw BEATs output is ever bad
            # again -- this is what should have existed the first time
            # instead of letting nan_to_num silently launder NaN into
            # deceptively clean-looking zeros with no trace left behind.
            if not np.isfinite(feat).all():
                print(f"  [cache] WARNING idx={i}: raw BEATs output has NaN/Inf before "
                      f"sanitizing ({np.isnan(feat).mean()*100:.1f}% NaN, "
                      f"{np.isinf(feat).mean()*100:.1f}% Inf)")

            # Kept as a backstop for genuinely corrupt individual clips only
            # -- with the fp32 fix above, this should now rarely if ever
            # actually trigger.
            feat32 = np.nan_to_num(feat.astype(np.float32), nan=0.0, posinf=6.0e4, neginf=-6.0e4)
            feat16 = np.clip(feat32, -6.0e4, 6.0e4).astype(np.float16)
            np.save(path, feat16)
            manifest[i] = {"T_patches": int(T_patches),
                           "duration": float(wav.numel() / sample_rate)}
        except Exception as e:
            print(f"  [cache] failed idx={i}: {e}")
        if (i + 1) % log_every == 0: print(f"  [cache] {i+1}/{n}")
    manifest_path = os.path.join(cache_dir, "manifest.json")
    existing = {}
    if os.path.exists(manifest_path):
        with open(manifest_path) as f: existing = json.load(f)
    existing.update({str(k): v for k, v in manifest.items()})
    with open(manifest_path, "w") as f: json.dump(existing, f)
    print(f"[cache] done. newly built here: {len(existing)}, "
          f"skipped ({n_skipped_from_other_dirs} already in another configured dir, "
          f"rest already in {cache_dir})")
    return manifest

# =============================================================================
# Waveform cache -- decode+resample once, reuse every epoch.
# Stored as FLAC (lossless, ~40-60% the size of raw float16 PCM) rather
# than .npy: the raw-array cache filled /kaggle/working's disk quota
# mid-build once already (16kHz float16 mono is 32KB/sec uncompressed --
# roughly 23GB across the full dataset), and FLAC keeps the same speed
# win (still skip decode+resample every epoch) at roughly half the disk
# footprint. Mirrors build_beats_cache's two-tier pattern exactly: skips
# indices already present in any read-only packaged dir (also_check_dirs),
# only writes to the given writable cache_dir. Unlike the BEATs cache, a
# missing entry here falls back gracefully to a live decode (see
# _load_waveform) rather than raising -- this cache is purely a speed
# optimization, not a correctness dependency.
# =============================================================================
def build_waveform_cache(hf_dataset, indices, cache_dir, sample_rate=16000,
                         log_every=2000, also_check_dirs=None):
    os.makedirs(cache_dir, exist_ok=True)
    indices = sorted({int(i) for i in indices})
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    todo = [i for i in indices if resolve_wav_cache_path(i, read_dirs) is None]
    n_skipped = len(indices) - len(todo)
    print(f"  [wav-cache] {len(todo)} of {len(indices)} indices need caching "
          f"({n_skipped} already present in {read_dirs})")
    n_failed = 0
    for n, i in enumerate(todo):
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
            sf.write(os.path.join(cache_dir, f"{i:06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception as e:
            n_failed += 1
            print(f"  [wav-cache] failed idx={i}: {e}")
        if (n + 1) % log_every == 0:
            print(f"    [wav-cache] {n + 1}/{len(todo)} cached ({n_failed} failed so far)")
    print(f"  [wav-cache] done: {cache_dir} ({n_failed} failed, {n_skipped} skipped)")


In [36]:
# =============================================================================
# CELL: VaaniNoiseDataset + collate_fn
#
# - Global-index invariant: cache + HF row indexed by same integer.
# - Short-clip BEATs padded to canonical 8s length; per-sample valid ratio
#   propagates so fusion can gate padded frames.
# - Patch-time slicing uses BEATs' fixed hop/offset, not duration/T_full.
# - STRICT lookup: missing cache raises, never silent zero-fill.
# =============================================================================
class VaaniNoiseDataset(Dataset):
    def __init__(self, parent_hf, full_indices, raw_to_canonical, canonical_tags,
                 fine_to_coarse_idx, beats_cache_dir=None, wav_cache_dir=None,
                 label_table=None, pseudo_overrides=None, crop_seconds=8.0, sample_rate=16000,
                 training=True, background_crop_prob=0.3, event_jitter_seconds=1.5):
        self.parent = parent_hf
        self.full_indices = np.asarray(full_indices, dtype=np.int64)
        assert self.full_indices.ndim == 1 and len(self.full_indices) > 0
        assert int(self.full_indices.max()) < len(parent_hf), \
            f"full_indices out of range: max={int(self.full_indices.max())} parent={len(parent_hf)}"
        self.raw_to_canonical = raw_to_canonical
        self.canonical_tags = canonical_tags
        self.tag_to_idx = {t: i for i, t in enumerate(canonical_tags)}
        self.fine_to_coarse_idx = fine_to_coarse_idx
        self.pseudo_overrides = pseudo_overrides or {}
        self.crop_seconds = float(crop_seconds); self.sample_rate = int(sample_rate)
        self.crop_samples = int(round(self.crop_seconds * self.sample_rate)) if self.crop_seconds > 0 else 0
        self.training = bool(training)
        self.background_crop_prob = float(background_crop_prob)
        self.event_jitter_seconds = float(event_jitter_seconds)
        self.beats_cache_dir = beats_cache_dir
        # NEW: wav_cache_dir avoids re-decoding audio every epoch;
        # label_table avoids the row-access that would otherwise decode
        # audio anyway just to read labels off the same row. Both optional
        # and independently gracefully-degrading -- omit either and this
        # dataset behaves exactly as before, just slower.
        self.wav_cache_dir = wav_cache_dir
        self.label_table = label_table
        self.canonical_T = (max(1, int(round(self.crop_seconds / BEATS_PATCH_HOP_SEC)))
                            if self.crop_seconds > 0 else 1)
        self._debug_prints_left = 3

    def __len__(self):
        return len(self.full_indices)

    def _full_idx(self, local_idx):
        return int(self.full_indices[local_idx])

    def _cache_path(self, full_idx):
        """Display/debug only -- returns the resolved path if found in any
        configured cache dir, else the join against the first dir (so a
        MISSING-file print still shows a sensible path, not None)."""
        found = resolve_cache_path(full_idx, self.beats_cache_dir)
        if found is not None:
            return found
        first_dir = _normalize_cache_dirs(self.beats_cache_dir)
        return os.path.join(first_dir[0], f"{int(full_idx):06d}.npy") if first_dir else "n/a"

    def _fine_idx(self, raw_tag):
        canon = self.raw_to_canonical.get(raw_tag, normalize_tag(raw_tag))
        return self.tag_to_idx.get(canon)

    def _resample_mono(self, wav_np, sr):
        wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
        if wav.ndim > 1: wav = wav.mean(dim=-1)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, int(sr), self.sample_rate)
        return wav.contiguous()

    def _load_beats_full(self, full_idx):
        if self.beats_cache_dir is None:
            raise RuntimeError("beats_cache_dir is required for this dataset.")
        path = resolve_cache_path(full_idx, self.beats_cache_dir)
        if path is None:
            raise RuntimeError(
                f"Missing BEATs cache entry for full_idx={full_idx} in "
                f"{_normalize_cache_dirs(self.beats_cache_dir)}. "
                f"Run build_beats_cache before training.")
        return np.load(path).astype(np.float32)

    def _slice_beats_crop(self, beats_full, start_s, end_s):
        """Patch-time slicing using BEATs' fixed hop/offset."""
        T_full = beats_full.shape[0]
        p0 = int(np.floor((start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
        p1 = int(np.ceil((end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
        p0 = max(0, min(p0, T_full - 1))
        p1 = max(p0 + 1, min(p1, T_full))
        return beats_full[p0:p1]

    def _pad_or_truncate_to_canonical(self, bw):
        if bw is None: return None
        if bw.shape[0] < self.canonical_T:
            pad = np.zeros((self.canonical_T - bw.shape[0], bw.shape[1]), dtype=bw.dtype)
            return np.concatenate([bw, pad], axis=0)
        if bw.shape[0] > self.canonical_T:
            return bw[:self.canonical_T]
        return bw

    def _select_crop(self, wav, event_spans, beats_full):
        n = wav.numel(); full_duration = n / self.sample_rate
        if self.crop_seconds <= 0:
            return wav, event_spans, full_duration, beats_full, 1.0
        target = self.crop_samples

        # -------- Short clip: pad waveform AND pad BEATs to canonical length --
        if n <= target:
            valid_seconds = n / self.sample_rate
            wav_p = F.pad(wav, (0, target - n))
            if beats_full is not None:
                bw = self._pad_or_truncate_to_canonical(beats_full)
                valid_ratio = min(1.0, valid_seconds / self.crop_seconds)
            else:
                bw = None; valid_ratio = 0.0
            return wav_p, event_spans, valid_seconds, bw, valid_ratio

        # -------- Normal crop inside a longer recording ----------------------
        take_bg = (not event_spans) or (random.random() < self.background_crop_prob)
        if take_bg:
            start = random.randint(0, n - target)
        else:
            s0, e0 = random.choice(event_spans)
            center = (s0 + e0) * 0.5 + random.uniform(-self.event_jitter_seconds, self.event_jitter_seconds)
            start = int(round(center * self.sample_rate - target / 2))
            start = max(0, min(start, n - target))
        end = start + target
        crop = wav[start:end]
        cs = start / self.sample_rate; ce = end / self.sample_rate
        cropped = []
        for s, e in event_spans:
            ns = max(0.0, float(s) - cs); ne = min(self.crop_seconds, float(e) - cs)
            if ne > ns: cropped.append((ns, ne))
        if beats_full is not None:
            bw = self._slice_beats_crop(beats_full, cs, ce)
            bw = self._pad_or_truncate_to_canonical(bw)
            valid_ratio = 1.0
        else:
            bw = None; valid_ratio = 0.0
        return crop, cropped, self.crop_seconds, bw, valid_ratio

    def __getitem__(self, idx, _retries=0):
        full_idx = self._full_idx(idx)
        if self._debug_prints_left > 0:
            print(f"    [dataset] local={idx} full_idx={full_idx} "
                  f"cache={self._cache_path(full_idx) if self.beats_cache_dir else 'n/a'}")
            self._debug_prints_left -= 1
        try:
            # CHANGED: was `self.parent[full_idx]` + `_load_audio` +
            # `_resample_mono` every single call. Uses the waveform cache
            # when available (no decode at all); falls back to a live
            # decode only for indices the cache doesn't have yet.
            wav = _load_waveform(full_idx, self.parent, self.sample_rate, self.wav_cache_dir)
        except Exception as exc:
            if _retries >= 5: raise RuntimeError(f"Failed to load audio full_idx={full_idx}") from exc
            return self.__getitem__(random.randrange(len(self)), _retries + 1)

        # CHANGED: was `sample = self.parent[full_idx]` then read fields off
        # it -- that row access decodes the audio column as a side effect
        # regardless of whether it's used, which would have made the
        # waveform cache above pointless. label_table (columnar, built
        # once) avoids that entirely; parent_hf row access is now only a
        # fallback for the rare case label_table wasn't provided.
        if self.label_table is not None:
            cats = self.label_table["NoiseCategory"][full_idx] or []
            raw_events = self.label_table["NoiseSubCategoryTimeStamp"][full_idx] or []
            quality = self.label_table["annotationQuality"][full_idx]
        else:
            sample = self.parent[full_idx]
            cats = sample.get("NoiseCategory") or []
            raw_events = sample.get("NoiseSubCategoryTimeStamp") or []
            quality = sample.get("annotationQuality")

        coarse_target = torch.zeros(len(COARSE_CATEGORIES), dtype=torch.float32)
        for cat in cats:
            if cat in COARSE_TO_IDX: coarse_target[COARSE_TO_IDX[cat]] = 1.0
        tier = _QUALITY_TO_TIER.get(quality, TIER_WEAK_ONLY)
        override = self.pseudo_overrides.get(full_idx)
        fine_clip_target = torch.zeros(len(self.canonical_tags), dtype=torch.float32)
        event_spans = []
        if override is not None:
            tier = TIER_PSEUDO_BRONZE
            for ev in override.get("events", []):
                try: s = float(ev["start"]); e = float(ev["end"])
                except (TypeError, ValueError, KeyError): continue
                if e > s: event_spans.append((s, e))
        else:
            for e in raw_events:
                try: s = float(e["start"]); en = float(e["end"])
                except (TypeError, ValueError, KeyError): continue
                if en <= s: continue
                event_spans.append((s, en))
                fi = self._fine_idx(e.get("tag", ""))
                if fi is not None: fine_clip_target[fi] = 1.0

        beats_full = self._load_beats_full(full_idx)   # STRICT

        if self.training and self.crop_seconds > 0:
            waveform, event_spans, valid_seconds, beats_crop, beats_valid_ratio = \
                self._select_crop(wav, event_spans, beats_full)
        else:
            waveform = wav
            valid_seconds = wav.numel() / self.sample_rate
            beats_crop = self._pad_or_truncate_to_canonical(beats_full)
            beats_valid_ratio = 1.0

        event_clip_target = torch.tensor(
            1.0 if (len(event_spans) > 0 or coarse_target.sum() > 0 or fine_clip_target.sum() > 0) else 0.0,
            dtype=torch.float32)

        return {
            "waveform": waveform.float().unsqueeze(0),
            "beats_crop": torch.from_numpy(np.asarray(beats_crop, dtype=np.float32)),
            "beats_valid_ratio": float(beats_valid_ratio),
            "duration": float(valid_seconds),
            "coarse_target": coarse_target,
            "fine_clip_target": fine_clip_target,
            "event_clip_target": event_clip_target,
            "event_spans": event_spans,
            "tier": int(tier),
            "src_idx": int(idx),
            "full_idx": int(full_idx),
        }

    def verify_cache_alignment(self, n_samples=5):
        if self.beats_cache_dir is None:
            print("  [cache-verify] no cache dir"); return
        k = min(n_samples, len(self))
        check = random.sample(range(len(self)), k) if len(self) > k else list(range(len(self)))
        print(f"  [cache-verify] checking {k}/{len(self)} samples "
              f"(expected canonical_T={self.canonical_T})")
        for li in check:
            fi = self._full_idx(li)
            cp = resolve_cache_path(fi, self.beats_cache_dir)
            if cp is None:
                print(f"    MISSING: local={li} full_idx={fi}"); continue
            feat = np.load(cp, mmap_mode="r")
            wav_np, sr = _load_audio(self.parent[fi]["audio"])
            dur = wav_np.shape[0] / sr
            expected_T = max(1, int(round(dur / BEATS_PATCH_HOP_SEC)))
            ratio = feat.shape[0] / expected_T
            flag = "OK" if 0.85 < ratio < 1.15 else "SUSPECT"
            print(f"    {flag}: local={li} full_idx={fi} dur={dur:.2f}s "
                  f"T_cached={feat.shape[0]} expected~{expected_T} ratio={ratio:.2f}")


def collate_fn(batch):
    waveforms = torch.cat([b["waveform"] for b in batch], dim=0)
    beats_list = [b["beats_crop"] for b in batch]
    T_max = max(f.shape[0] for f in beats_list)
    D = beats_list[0].shape[1]
    beats_padded = torch.zeros(len(batch), T_max, D, dtype=torch.float32)
    beats_mask = torch.zeros(len(batch), T_max, dtype=torch.float32)
    for i, f in enumerate(beats_list):
        beats_padded[i, :f.shape[0]] = f
        beats_mask[i, :f.shape[0]] = 1.0
    return {
        "waveform": waveforms,
        "beats_padded": beats_padded,
        "beats_mask": beats_mask,
        "beats_valid_ratio": torch.tensor([b["beats_valid_ratio"] for b in batch], dtype=torch.float32),
        "duration": torch.tensor([b["duration"] for b in batch], dtype=torch.float32),
        "coarse_target": torch.stack([b["coarse_target"] for b in batch]),
        "fine_clip_target": torch.stack([b["fine_clip_target"] for b in batch]),
        "event_clip_target": torch.stack([b["event_clip_target"] for b in batch]),
        "event_spans": [b["event_spans"] for b in batch],
        "tier": torch.tensor([b["tier"] for b in batch], dtype=torch.long),
        "src_idx": torch.tensor([b["src_idx"] for b in batch], dtype=torch.long),
        "full_idx": torch.tensor([b["full_idx"] for b in batch], dtype=torch.long),
    }


In [37]:
# =============================================================================
# CELL: Model -- accepts beats_valid_ratio and gates fusion on valid region
# =============================================================================
class VaaniLiteSED(nn.Module):
    def __init__(self, beats_model=None, n_fine=1, n_coarse=7,
                 beats_dim=768, proj_dim=192, gru_hidden=128, gru_layers=2,
                 mel_frontend=None):
        super().__init__()
        self.beats = beats_model; self.mel_frontend = mel_frontend
        self.beats_dim = beats_dim; self.proj_dim = proj_dim
        self.dfd_lka = DFDLKAEncoder(channels=(32, 64, 128, 192))
        self.beats_proj = nn.Linear(beats_dim, proj_dim)
        self.fdy_proj = nn.Linear(192, proj_dim)
        self.fuse_gate = nn.Sequential(nn.Linear(proj_dim * 2, proj_dim), nn.Sigmoid())
        self.fuse_norm = nn.LayerNorm(proj_dim)
        self.gru = nn.GRU(proj_dim, gru_hidden, num_layers=gru_layers,
                          batch_first=True, bidirectional=True, dropout=0.10)
        gru_out = 2 * gru_hidden
        self.gru_norm = nn.LayerNorm(gru_out)
        self.temporal_refine = TemporalRefinement(gru_out)
        self.event_frame_head = nn.Linear(gru_out, 1)
        self.event_onset_head = nn.Linear(gru_out, 1)
        self.event_offset_head = nn.Linear(gru_out, 1)
        self.event_attn = nn.Conv1d(gru_out, 1, kernel_size=1)
        self.event_clip_proj = nn.Linear(gru_out, 1)
        self.fine_attn = nn.Conv1d(gru_out, n_fine, kernel_size=1)
        self.fine_clip_proj = nn.Linear(gru_out, n_fine)
        self.coarse_attn = nn.Conv1d(gru_out, n_coarse, kernel_size=1)
        self.coarse_proj = nn.Linear(gru_out, n_coarse)

    def _beats_features(self, waveform, beats_cached):
        if beats_cached is not None: return beats_cached
        if self.beats is None: raise RuntimeError("No cached BEATs and no live BEATs model.")
        with torch.no_grad():
            feat, _, _ = beats_extract_temporal(self.beats, waveform)
        return feat

    def forward(self, waveform, beats_cached=None, beats_mask=None,
                beats_valid_ratio=None, mel=None):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        beats_feat = self._beats_features(waveform, beats_cached)
        beats_feat = self.beats_proj(beats_feat)
        if mel is None:
            if self.mel_frontend is None: raise RuntimeError("mel_frontend required when mel=None")
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        if beats_feat.shape[1] != T:
            beats_feat = F.interpolate(beats_feat.transpose(1, 2), size=T,
                                       mode="linear", align_corners=False).transpose(1, 2)
        gate = self.fuse_gate(torch.cat([beats_feat, fdy], dim=-1))
        # Gate fusion on the BEATs-valid region (short clips).
        if beats_valid_ratio is not None:
            arange = torch.arange(T, device=waveform.device, dtype=torch.float32).unsqueeze(0)
            valid_T = beats_valid_ratio.unsqueeze(1) * T
            frame_valid = (arange < valid_T).float()
            gate = gate * frame_valid.unsqueeze(-1)
        x = self.fuse_norm(gate * beats_feat + (1.0 - gate) * fdy)
        x, _ = self.gru(x); x = self.gru_norm(x); x = self.temporal_refine(x)
        event_frame_logits = self.event_frame_head(x)
        event_onset_logits = self.event_onset_head(x)
        event_offset_logits = self.event_offset_head(x)
        xf = x.transpose(1, 2)
        event_attn = F.softmax(self.event_attn(xf), dim=-1)
        event_pool = torch.einsum("bct,btd->bcd", event_attn, x)
        event_clip_logits = (event_pool * self.event_clip_proj.weight).sum(-1) + self.event_clip_proj.bias
        fine_attn = F.softmax(self.fine_attn(xf), dim=-1)
        coarse_attn = F.softmax(self.coarse_attn(xf), dim=-1)
        fine_pool = torch.einsum("bct,btd->bcd", fine_attn, x)
        coarse_pool = torch.einsum("bct,btd->bcd", coarse_attn, x)
        fine_clip_logits = torch.einsum("bcd,cd->bc", fine_pool, self.fine_clip_proj.weight) + self.fine_clip_proj.bias
        coarse_logits = torch.einsum("bcd,cd->bc", coarse_pool, self.coarse_proj.weight) + self.coarse_proj.bias
        return {"event_frame_logits": event_frame_logits,
                "event_onset_logits": event_onset_logits,
                "event_offset_logits": event_offset_logits,
                "event_clip_logits": event_clip_logits,
                "fine_clip_logits": fine_clip_logits,
                "coarse_logits": coarse_logits,
                "encoder_time_steps": T}


In [38]:
# =============================================================================
# CELL: Losses (FIXED: AsymmetricFocalLoss pt was double-complemented for
# target=0, which inverted the focal weighting for negative frames --
# see explanation above. xs_neg is now used directly as pt for negatives,
# matching the canonical Asymmetric Loss formulation, instead of taking
# (1 - xs_neg) again.)
# =============================================================================
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg; self.gamma_pos = gamma_pos; self.clip = clip
    def forward(self, logits, targets, mask=None):
        probs = torch.sigmoid(logits)
        xs_neg = 1.0 - probs
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        pt = probs * targets + xs_neg * (1.0 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * (1.0 - pt).pow(gamma)
        if mask is not None:
            m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
            return (loss * m).sum() / m.sum().clamp_min(1.0)
        return loss.mean()

def masked_bce(logits, targets, mask=None):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    if mask is None: return loss.mean()
    m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
    return (loss * m).sum() / m.sum().clamp_min(1.0)

def classwise_attention_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)

def build_event_targets(event_spans_batch, batch_size, T, durations, device,
                        window_seconds=8.0, safe_margin_seconds=None):
    frame_target = torch.zeros(batch_size, T, 1, device=device)
    onset_target = torch.zeros(batch_size, T, 1, device=device)
    offset_target = torch.zeros(batch_size, T, 1, device=device)
    positive_mask = torch.zeros(batch_size, T, device=device)
    valid_mask = torch.zeros(batch_size, T, device=device)
    safe_negative_mask = torch.zeros(batch_size, T, device=device)
    centers = (torch.arange(T, device=device, dtype=torch.float32) + 0.5) * (window_seconds / T)
    for b in range(batch_size):
        dur = max(float(durations[b]), 1e-6)
        valid_mask[b] = (centers < float(durations[b])).float()
        spans = event_spans_batch[b]
        for start, end in spans:
            start = float(start); end = float(end)
            active = (centers >= start) & (centers < end)
            frame_target[b, active, 0] = 1.0; positive_mask[b, active] = 1.0
            sigma = max(dur / T, 0.03)
            on = torch.exp(-0.5 * ((centers - start) / sigma).pow(2))
            off = torch.exp(-0.5 * ((centers - end) / sigma).pow(2))
            onset_target[b, :, 0] = torch.maximum(onset_target[b, :, 0], on)
            offset_target[b, :, 0] = torch.maximum(offset_target[b, :, 0], off)
        if safe_margin_seconds is not None:
            if not spans:
                safe_negative_mask[b] = valid_mask[b]
            else:
                md = torch.full((T,), float("inf"), device=device)
                for start, end in spans:
                    start = float(start); end = float(end)
                    inside = (centers >= start) & (centers < end)
                    d = torch.minimum((centers - start).abs(), (centers - end).abs())
                    d = torch.where(inside, torch.zeros_like(d), d)
                    md = torch.minimum(md, d)
                safe_negative_mask[b] = valid_mask[b] * (md > safe_margin_seconds).float()
    return frame_target, onset_target, offset_target, valid_mask, positive_mask, safe_negative_mask

def consistency_loss(student_logits, teacher_logits, valid_mask, confidence_floor=0.60):
    s = torch.sigmoid(student_logits); t = torch.sigmoid(teacher_logits).detach()
    conf = torch.maximum(t, 1.0 - t)
    conf = ((conf - confidence_floor) / max(1.0 - confidence_floor, 1e-6)).clamp(0, 1)
    w = valid_mask.unsqueeze(-1) * conf
    return ((s - t).pow(2) * w).sum() / w.sum().clamp_min(1.0)

def hierarchical_consistency(fine_clip_logits, coarse_logits, fine_to_coarse_idx):
    fine_p = torch.sigmoid(fine_clip_logits); coarse_p = torch.sigmoid(coarse_logits)
    penalties = []
    for fi, ci in enumerate(fine_to_coarse_idx.tolist()):
        penalties.append(F.relu(fine_p[:, fi] - coarse_p[:, ci]).mean())
    return torch.stack(penalties).mean() if penalties else fine_clip_logits.new_tensor(0.0)


In [39]:
# =============================================================================
# CELL: Mean teacher + decoder + scorer
# =============================================================================
@torch.no_grad()
def update_teacher(student, teacher, decay=0.999):
    for sp, tp in zip(student.parameters(), teacher.parameters()):
        if tp is sp: continue
        tp.data.mul_(decay).add_(sp.data, alpha=1.0 - decay)
    for sb, tb in zip(student.buffers(), teacher.buffers()):
        if tb is sb: continue
        tb.copy_(sb)

def freeze_model(model):
    for p in model.parameters(): p.requires_grad_(False)
    model.eval()

def smooth_probs(probs_1d, window):
    if window is None or window <= 1: return probs_1d
    k = int(window)
    if k % 2 == 0: k += 1
    if k >= len(probs_1d): k = len(probs_1d) - (1 - len(probs_1d) % 2)
    if k < 3: return probs_1d
    return medfilt(probs_1d, kernel_size=k)

def snap_to_peak(probs, idx, radius):
    if probs is None or radius <= 0: return idx
    lo = max(0, idx - radius); hi = min(len(probs), idx + radius + 1)
    if hi <= lo: return idx
    return lo + int(np.argmax(probs[lo:hi]))

def decode_events_single(frame_probs, fps, threshold=0.5, smooth_window=5,
                         min_duration=0.15, merge_gap=0.10,
                         onset_probs=None, offset_probs=None,
                         snap_radius_seconds=0.15):
    probs = smooth_probs(np.asarray(frame_probs, dtype=np.float32), smooth_window)
    binary = (probs >= threshold).astype(np.int32)
    raw = []; in_ev, s_idx = False, 0
    for i, v in enumerate(binary):
        if v and not in_ev: in_ev, s_idx = True, i
        elif not v and in_ev: in_ev = False; raw.append((s_idx, i))
    if in_ev: raw.append((s_idx, len(binary)))
    radius = max(1, int(round(snap_radius_seconds * fps)))
    snapped = []
    for s, e in raw:
        ss = snap_to_peak(onset_probs, s, radius)
        es = snap_to_peak(offset_probs, e, radius)
        if es <= ss: ss, es = s, e
        snapped.append((ss, es))
    merged = []
    for s, e in snapped:
        if merged and (s - merged[-1][1]) / fps <= merge_gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else: merged.append((s, e))
    out = []
    for s, e in merged:
        onset, offset = s / fps, e / fps
        if offset - onset >= min_duration: out.append((float(onset), float(offset)))
    return out

def match_events(ref_events, pred_events, tolerance_frac=0.20, min_tol=0.05):
    mr, mp = set(), set(); cand = []
    for ri, (ro, rf) in enumerate(ref_events):
        tol = max(tolerance_frac * (rf - ro), min_tol)
        for pi, (po, pf) in enumerate(pred_events):
            if abs(po - ro) <= tol and abs(pf - rf) <= tol:
                cand.append((abs(po - ro) + abs(pf - rf), ri, pi))
    for _, ri, pi in sorted(cand):
        if ri not in mr and pi not in mp: mr.add(ri); mp.add(pi)
    tp = len(mr)
    return tp, len(pred_events) - tp, len(ref_events) - tp

def event_based_f1(ref_data, pred_data):
    TP = FP = FN = 0
    for cid, re in ref_data.items():
        tp, fp, fn = match_events(re, pred_data.get(cid, []))
        TP += tp; FP += fp; FN += fn
    for cid in pred_data:
        if cid not in ref_data: FP += len(pred_data[cid])
    p = TP / (TP + FP) if (TP + FP) else 0.0
    r = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return f1, p, r

def events_to_frames(events, max_time, frame_len=0.01):
    n = int(max_time / frame_len) + 1
    m = np.zeros(n, dtype=np.int32)
    for on, off in events:
        i0 = int(on / frame_len); i1 = min(int(off / frame_len) + 1, n)
        if i1 > i0: m[i0:i1] = 1
    return m

def segment_dice(ref_data, pred_data):
    scores = []
    for cid, re in ref_data.items():
        pe = pred_data.get(cid, []); all_ev = re + pe
        if not all_ev: scores.append(1.0); continue
        mt = max(off for _, off in all_ev) + 0.5
        r = events_to_frames(re, mt); p = events_to_frames(pe, mt)
        i = int((r & p).sum()); t = int(r.sum() + p.sum())
        scores.append(1.0 if t == 0 else 2.0 * i / t)
    return sum(scores) / len(scores) if scores else 0.0

def official_combined_score(ref_data, pred_data):
    f1, p, r = event_based_f1(ref_data, pred_data)
    dice = segment_dice(ref_data, pred_data)
    return {"event_f1": f1, "precision": p, "recall": r,
            "segment_dice": dice, "combined": f1 + dice}


In [40]:
# =============================================================================
# CELL: Split / label space / priors (index-aware)
# =============================================================================
def split_train_val(dataset, val_fraction=0.10, seed=42, group_col="district"):
    groups = dataset[group_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    ia = np.arange(len(dataset))
    tr, va = next(gss.split(ia, groups=groups))
    return tr.tolist(), va.tolist()

def build_label_space(ds, indices=None, min_tag_count_to_keep=5):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    tag_counts = Counter(); tag_to_coarse_votes = defaultdict(Counter)
    try:
        subcat_col = ds["NoiseSubCategoryTimeStamp"]; cats_col = ds["NoiseCategory"]
        use_cols = True
    except Exception:
        use_cols = False
    for i in idxs:
        if use_cols:
            events = subcat_col[i] or []; cats = cats_col[i] or []
        else:
            s = ds[i]; events = s.get("NoiseSubCategoryTimeStamp") or []; cats = s.get("NoiseCategory") or []
        for e in events:
            raw = e.get("tag", "")
            if not raw: continue
            tag_counts[raw] += 1
            for c in cats:
                if c in COARSE_TO_IDX: tag_to_coarse_votes[raw][c] += 1
    raw_to_canonical, cluster_report = build_tag_clusters(tag_counts)
    canon_counts = Counter(); canon_coarse_votes = defaultdict(Counter)
    for raw, cnt in tag_counts.items():
        canon = raw_to_canonical[raw]; canon_counts[canon] += cnt
        for c, v in tag_to_coarse_votes[raw].items(): canon_coarse_votes[canon][c] += v
    canonical_tags = sorted(t for t, c in canon_counts.items() if c >= min_tag_count_to_keep)
    f2c = torch.zeros(len(canonical_tags), dtype=torch.long)
    for i, tag in enumerate(canonical_tags):
        v = canon_coarse_votes.get(tag)
        f2c[i] = COARSE_TO_IDX[v.most_common(1)[0][0]] if v else 0
    return canonical_tags, raw_to_canonical, f2c, cluster_report

def estimate_global_duration_prior(ds, indices=None):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    try:
        qcol = ds["annotationQuality"]; scol = ds["NoiseSubCategoryTimeStamp"]; use_cols = True
    except Exception:
        use_cols = False
    out = []
    for i in idxs:
        if use_cols:
            if qcol[i] != "verified_timestamps": continue
            events = scol[i] or []
        else:
            s = ds[i]
            if s.get("annotationQuality") != "verified_timestamps": continue
            events = s.get("NoiseSubCategoryTimeStamp") or []
        for e in events:
            try: d = float(e["end"]) - float(e["start"])
            except (TypeError, ValueError, KeyError): continue
            if d > 0: out.append(d)
    return float(np.median(out)) if out else 1.0

def compute_adaptive_window(median_duration, fps, min_window=3, max_window=25):
    w = int(round(median_duration * fps * 0.5))
    w = max(min_window, min(max_window, w))
    return w + 1 if w % 2 == 0 else w

def weak_augment(mel): return apply_mel_specaugment(mel, 1, 1, 0.05, 8)
def strong_augment(mel): return apply_mel_specaugment(mel, 2, 2, 0.12, 16)

# =============================================================================
# TIER_FRAME_SCALE -- per-tier scale applied ONLY to the frame-level losses
# (event_frame, onset, offset).
#
# Silver (unverified_timestamps) carries a trustworthy clip category but NOT
# trustworthy start/end times -- its spans tile ~68% of every clip, so as
# frame-level supervision it says "most of the audio is an event", which is
# false. The previous full 33-epoch run confirmed this: with Silver in the
# frame losses, frac_pos_frame pinned at ~0.60, exactly Silver's positive
# frame rate, and Event-F1 never left ~0.07 across all three stages.
#
# Fix: zero Silver out of the frame/onset/offset losses. Gold is the only
# tier whose timestamps can define per-frame positives and negatives.
# Silver still drives the clip-level losses (fine_clip, coarse, event_clip)
# below -- those only need the clip category, which Silver has.
#
# TIER_PSEUDO_BRONZE stays at 0.4 because Stage 3 pseudo-labels are generated
# by the model's own Gold-trained decoder.
# =============================================================================
TIER_FRAME_SCALE = {TIER_VERIFIED: 1.0, TIER_UNVERIFIED: 0.0,
                    TIER_WEAK_ONLY: 0.0, TIER_PSEUDO_BRONZE: 0.4}

# Alias kept so any external references to TIER_LOSS_SCALE still resolve.
TIER_LOSS_SCALE = TIER_FRAME_SCALE

def make_tier_scales(tier_tensor, device, scale_map=None):
    scale_map = TIER_FRAME_SCALE if scale_map is None else scale_map
    scale = torch.zeros(tier_tensor.shape[0], device=device)
    for t, s in scale_map.items(): scale[tier_tensor == t] = s
    return scale


In [41]:
# =============================================================================
# CELL: Inference utilities -- strict cache lookup
# =============================================================================
@torch.no_grad()
def get_model_time_grid(model, device, window_seconds, sample_rate):
    model.eval()
    dummy = torch.zeros(1, int(round(window_seconds * sample_rate)), device=device)
    out = model(dummy)
    T = out["encoder_time_steps"]
    return T, T / window_seconds

def _ref_events_for_sample(sample):
    evs = []
    for e in (sample.get("NoiseSubCategoryTimeStamp") or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def _window_starts(total_samples, win, stride):
    if total_samples <= win: return [0]
    starts = list(range(0, total_samples - win + 1, stride))
    if starts[-1] + win < total_samples: starts.append(total_samples - win)
    return starts

@torch.no_grad()
def _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                            window_seconds, stride_seconds,
                            beats_full_np=None, full_duration=None):
    total = wav.numel(); duration_sec = total / sample_rate
    win = int(round(window_seconds * sample_rate)); stride = int(round(stride_seconds * sample_rate))
    total_frames = int(round(duration_sec * fps)) + T_win + 1
    sums = {k: np.zeros(total_frames, dtype=np.float64) for k in ("frame", "onset", "offset")}
    wsum = np.zeros(total_frames, dtype=np.float64)
    for s0 in _window_starts(total, win, stride):
        chunk = wav[s0:s0+win]
        pad_len = win - chunk.numel()
        if pad_len > 0: chunk = F.pad(chunk, (0, pad_len))
        chunk = chunk.unsqueeze(0).to(device)
        beats_cached = None
        if beats_full_np is not None and full_duration is not None:
            T_full = beats_full_np.shape[0]
            win_start_s = s0 / sample_rate
            win_end_s   = (s0 + win) / sample_rate
            # Use the same patch-time mapping as the dataset.
            p0 = int(np.floor((win_start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
            p1 = int(np.ceil((win_end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
            p0 = max(0, min(p0, T_full - 1)); p1 = max(p0 + 1, min(p1, T_full))
            beats_cached = torch.from_numpy(beats_full_np[p0:p1]).float().unsqueeze(0).to(device)
        out = model(chunk, beats_cached=beats_cached)
        frame_l = out["event_frame_logits"][0, :, 0].detach().cpu().numpy()
        onset_l = out["event_onset_logits"][0, :, 0].detach().cpu().numpy()
        offset_l = out["event_offset_logits"][0, :, 0].detach().cpu().numpy()
        off = int(round((s0 / sample_rate) * fps))
        end = min(off + T_win, total_frames); L = end - off
        if L > 0:
            sums["frame"][off:end] += frame_l[:L]
            sums["onset"][off:end] += onset_l[:L]
            sums["offset"][off:end] += offset_l[:L]
            wsum[off:end] += 1.0
    valid_len = min(int(round(duration_sec * fps)) + 1, total_frames)
    w = np.maximum(wsum[:valid_len], 1e-6)
    merged = {k: sums[k][:valid_len] / w for k in sums}
    return {k: 1.0 / (1.0 + np.exp(-merged[k])) for k in merged}

@torch.no_grad()
def compute_val_probabilities(model, parent_hf, full_indices, device, sample_rate,
                              beats_cache_dir=None, wav_cache_dir=None, label_table=None,
                              window_seconds=8.0, stride_seconds=4.0, max_clips=None):
    model.eval()
    T_win, fps = get_model_time_grid(model, device, window_seconds, sample_rate)
    n = len(full_indices) if max_clips is None else min(max_clips, len(full_indices))
    ref_data, probs_by_clip = {}, {}
    for i in range(n):
        full_idx = int(full_indices[i])
        # CHANGED: cache-aware waveform load (see _load_waveform) instead of
        # a live decode+resample every epoch this runs.
        wav = _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        beats_full_np = None
        if beats_cache_dir is not None:
            p = resolve_cache_path(full_idx, beats_cache_dir)
            if p is None:
                raise RuntimeError(f"Missing cache for full_idx={full_idx} during val inference "
                                   f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
            beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        # CHANGED: label_table avoids the row access that would otherwise
        # decode audio just to read the clip id / ref events.
        if label_table is not None:
            cid = str(label_table["id"][full_idx] or full_idx)
            ref_data[cid] = _ref_events_for_full_idx(full_idx, label_table)
        else:
            sample = parent_hf[full_idx]
            cid = str(sample.get("id") or sample.get("clip_id") or full_idx)
            ref_data[cid] = _ref_events_for_sample(sample)
        probs_by_clip[cid] = merged
    return ref_data, probs_by_clip, fps

def _decode_clip(cached, fps, thr, sw, md, mg, sr):
    return decode_events_single(cached["frame"], fps, threshold=thr, smooth_window=sw,
                                min_duration=md, merge_gap=mg,
                                onset_probs=cached["onset"], offset_probs=cached["offset"],
                                snap_radius_seconds=sr)

def grid_search_decode_params(ref_data, probs_by_clip, fps, median_duration,
                              thresholds=(0.3, 0.4, 0.5, 0.6, 0.7),
                              smooth_windows=None,
                              min_duration_fracs=(0.1, 0.2, 0.3),
                              merge_gaps=(0.05, 0.10, 0.15, 0.20),
                              snap_radius_seconds_options=(0.10, 0.25)):
    if smooth_windows is None:
        base_w = compute_adaptive_window(median_duration, fps)
        smooth_windows = sorted(set([max(3, base_w // 2), base_w, base_w * 2]))
    best = {"combined": -1.0}
    for thr in thresholds:
        for sw in smooth_windows:
            for mdf in min_duration_fracs:
                md = max(0.05, mdf * median_duration)
                for g in merge_gaps:
                    for sr in snap_radius_seconds_options:
                        pred = {cid: _decode_clip(c, fps, thr, sw, md, g, sr)
                                for cid, c in probs_by_clip.items()}
                        score = official_combined_score(ref_data, pred)
                        if score["combined"] > best["combined"]:
                            best = {**score, "threshold": thr, "smooth_window": sw,
                                    "min_duration": md, "merge_gap": g,
                                    "snap_radius_seconds": sr}
    return best

@torch.no_grad()
def generate_pseudo_labels(teacher_model, parent_hf, bronze_full_indices, sample_rate, device,
                           decode_params, beats_cache_dir=None, wav_cache_dir=None,
                           window_seconds=8.0, stride_seconds=4.0,
                           min_confidence=0.65, max_clips=None):
    teacher_model.eval()
    T_win, fps = get_model_time_grid(teacher_model, device, window_seconds, sample_rate)
    indices = [int(i) for i in bronze_full_indices]
    if max_clips is not None: indices = indices[:max_clips]
    overrides = {}
    for fi in indices:
        # CHANGED: cache-aware waveform load instead of a live decode --
        # this only runs once per Bronze clip per stage, so lower total
        # impact than the training loader, but still free to fix the same way.
        wav = _load_waveform(fi, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        p = resolve_cache_path(fi, beats_cache_dir)
        if p is None:
            raise RuntimeError(f"Missing cache for full_idx={fi} during pseudo-labeling "
                               f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
        beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(teacher_model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        spans = decode_events_single(
            merged["frame"], fps, threshold=decode_params["threshold"],
            smooth_window=decode_params["smooth_window"],
            min_duration=decode_params["min_duration"], merge_gap=decode_params["merge_gap"],
            onset_probs=merged["onset"], offset_probs=merged["offset"],
            snap_radius_seconds=decode_params.get("snap_radius_seconds", 0.15))
        accepted = []
        for s, e in spans:
            si = int(round(s * fps)); ei = min(int(round(e * fps)), len(merged["frame"]) - 1)
            peak = float(merged["frame"][si:ei+1].max()) if ei >= si else 0.0
            if peak >= min_confidence: accepted.append({"start": s, "end": e})
        if accepted: overrides[fi] = {"events": accepted}
    return overrides


In [42]:
# =============================================================================
# CELL: Config
# =============================================================================
CONFIG = dict(
    seed=42, val_fraction=0.10, sample_rate=16000, crop_seconds=8.0, n_mels=128,
    min_tag_count_to_keep=5, batch_size=8, grad_accum_steps=2,
    # num_workers=0: confirmed via a live AttributeError traceback that Windows'
    # spawn-based multiprocessing can't unpickle VaaniNoiseDataset in a worker
    # process, since it's defined in a notebook cell, not an importable .py
    # module -- num_workers>0 deadlocks silently on this setup.
    num_workers=0,
    stage1_epochs=15, stage2_epochs=10, stage3_epochs=8,
    lr_stage1=3e-4, lr_stage2=8e-5, lr_stage3=5e-5,
    lr_warmup_steps=500, teacher_decay=0.999,
    # silver_negative_weight is INERT for Silver now: TIER_FRAME_SCALE
    # [TIER_UNVERIFIED] = 0 zeros Silver out of the frame-level losses
    # entirely before this weight is ever applied. Kept at 0.75 so that if
    # you later re-admit Silver partially, the existing negative-frame
    # weighting still applies unchanged.
    silver_negative_weight=0.75,
    # gamma_neg_frame: passed into AsymmetricFocalLoss in run_epoch.
    # Was the class default 4.0. With Silver removed from the frame losses,
    # Gold is the only source of frame-level negatives left -- and
    # gamma_neg=4.0 was down-weighting those too aggressively on a class
    # balance that isn't actually extreme. 2.0 keeps a mild positive/
    # negative asymmetry without starving the head of "not event" gradient.
    # Watch frac_pos_frame in progress_live.txt: it should settle around
    # 0.15-0.35. If it drifts above 0.55, drop this to 1.0 and leave Silver
    # at 0 -- do NOT re-enable Silver to compensate.
    gamma_neg_frame=2.0,
    # gold_target_fraction: expected share of Gold rows per training batch
    # under make_gold_aware_sampler (see train_one_stage). Gold is only
    # ~9-12% of any stage's subset, so plain uniform shuffling at
    # batch_size=8 leaves 36-46% of batches with ZERO Gold rows -- and
    # since frame/onset/offset losses only get gradient from Gold rows
    # (TIER_FRAME_SCALE), those heads get no update at all on that
    # fraction of steps. 0.35 drops that to roughly 3%.
    gold_target_fraction=0.35,
    max_eval_clips=300,
)


## Part 2 — V5 core definitions


In [43]:
# =============================================================================
# V5 CORE DEFINITIONS (no work is done in this cell; it only defines things)
#
#  1. time alignment: BEATs rows <-> 8 s windows on the 0.16 s patch grid
#  2. labels: per-event category, per-crop weak targets with "unsure" masks
#  3. VaaniSEDV5: your V3 network + 7 category frame tracks + MIL weak head
#  4. dataset (natural + synthetic rows), tier sampler, collate
#  5. targets + losses
#  6. windowed inference -> per-category tracks
#  7. per-category decoder + flatten
# =============================================================================
import math, time, json, pickle, hashlib, zipfile, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast, GradScaler

SR = 16000
W_SEC = 8.0
WIN_SAMPLES = int(W_SEC * SR)                      # 128000
PATCH_SAMPLES = int(round(BEATS_PATCH_HOP_SEC * SR))  # 2560 = one BEATs patch (0.16 s)
STRIDE_SAMPLES = WIN_SAMPLES // 2                  # 4 s = exactly 25 patches
CANON_T = int(round(W_SEC / BEATS_PATCH_HOP_SEC))  # 50 BEATs rows per window
assert STRIDE_SAMPLES % PATCH_SAMPLES == 0 and WIN_SAMPLES % PATCH_SAMPLES == 0

CATS = list(COARSE_CATEGORIES)
NC = len(CATS)
CAT_IDX = {c: i for i, c in enumerate(CATS)}
SHORT_CATS = {"human_non_speech", "phone_signal_alarm"}     # decoder parameter groups
CAT_GROUP = ["short" if c in SHORT_CATS else "long" for c in CATS]

T_GOLD, T_SILVER, T_BRONZE, T_SYNTH = 0, 1, 2, 4
TIER_NAME = {T_GOLD: "gold", T_SILVER: "silver", T_BRONZE: "bronze", T_SYNTH: "synth"}
BAD_INDICES = {3561, 48940}                        # known corrupt clips, no cache entry

# BEATs patch k covers fbank frames 16k..16k+15 (25 ms window, 10 ms hop), so its
# centre is at 0.0125 + 0.01*(16k+7.5) = 0.0875 + 0.16k s. Linear interpolation of
# 50 rows onto the 801 mel frames puts row j at w + 0.08 + 0.16j for a window that
# starts at w. Matching the two gives k0 = (w - 0.0075)/0.16; with every window start
# on the 2560-sample grid this is exactly k0 = w/0.16 with a 7.5 ms residual.
PATCH_CENTRE_OFFSET = 0.0875
ALIGN_TOL_SEC = 0.0076


def beats_T_expected(n_samples):
    """Rows BEATs produces for an n-sample clip (kaldi fbank, snip_edges, 16-frame patches)."""
    frames = 1 + (int(n_samples) - 400) // 160 if n_samples >= 400 else 0
    return max(0, frames // 16)


def beats_rows_for_window(beats_full, start_sample):
    """BEATs rows for the window starting at start_sample (must be on the patch grid).
    Returns (rows[CANON_T, D], n_real_rows)."""
    assert start_sample % PATCH_SAMPLES == 0, f"window start {start_sample} is off the patch grid"
    w = start_sample / SR
    k0 = int(round((w - (PATCH_CENTRE_OFFSET - 0.08)) / BEATS_PATCH_HOP_SEC))
    k0 = max(0, k0)
    # assertion: first row centre lands where interpolation will put it
    assert abs((PATCH_CENTRE_OFFSET + BEATS_PATCH_HOP_SEC * k0) - (w + 0.08)) <= ALIGN_TOL_SEC + 1e-9
    rows = beats_full[k0:k0 + CANON_T]
    n_real = rows.shape[0]
    if n_real < CANON_T:
        rows = np.concatenate([rows, np.zeros((CANON_T - n_real, beats_full.shape[1]), rows.dtype)], 0)
    return rows, n_real


def grid_window_starts(total_samples):
    """8 s windows, 4 s stride, every start on the 0.16 s patch grid. The last window
    may run past the clip end by < 0.16 s (zero-padded, gated off)."""
    if total_samples <= WIN_SAMPLES:
        return [0]
    starts = list(range(0, total_samples - WIN_SAMPLES + 1, STRIDE_SAMPLES))
    last = int(math.ceil((total_samples - WIN_SAMPLES) / PATCH_SAMPLES)) * PATCH_SAMPLES
    if starts[-1] < last:
        starts.append(last)
    return starts


# ---------------------------------------------------------------------------
# 2. labels
# ---------------------------------------------------------------------------
_TAG_TO_CAT = {}   # filled by build_tag_to_cat (fallback only; events carry 'category')


def build_tag_to_cat(label_table, indices):
    votes = defaultdict(Counter)
    for i in indices:
        for e in (label_table["NoiseSubCategoryTimeStamp"][i] or []):
            c = e.get("category") if isinstance(e, dict) else None
            if c in CAT_IDX:
                votes[e.get("tag", "")][c] += 1
    _TAG_TO_CAT.clear()
    _TAG_TO_CAT.update({t: v.most_common(1)[0][0] for t, v in votes.items()})
    return _TAG_TO_CAT


def parse_events(raw_events, clip_cats=None):
    """-> list of (start, end, cat_idx). Category from the event, else its tag, else
    the clip's only category. Events whose category cannot be resolved are dropped
    (counted by the caller)."""
    out = []
    for e in (raw_events or []):
        try:
            s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError):
            continue
        if not en > s:
            continue
        c = e.get("category")
        if c not in CAT_IDX:
            c = _TAG_TO_CAT.get(e.get("tag", ""))
        if c not in CAT_IDX and clip_cats and len(clip_cats) == 1:
            c = clip_cats[0]
        if c in CAT_IDX:
            out.append((s, en, CAT_IDX[c]))
    return out


# ---------------------------------------------------------------------------
# 3. model
# ---------------------------------------------------------------------------
UNUSED_V3_HEADS = ("event_frame_head", "event_attn", "event_clip_proj", "fine_attn",
                   "fine_clip_proj", "coarse_attn", "coarse_proj")


class ShortMelBranch(nn.Module):
    """Optional high-time-resolution evidence: 32 ms analysis window (vs 128 ms in the
    main mel frontend), 64 mel bands, two temporal convs, projected to the fused width.
    The output projection is ZERO-initialised, so adding the branch does not change the
    warm-started model at step 0."""
    def __init__(self, out_dim, n_mels=64, n_fft=512, hidden=64):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=n_fft, win_length=n_fft,
                                                        hop_length=160, n_mels=n_mels, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.conv = nn.Sequential(nn.Conv1d(n_mels, hidden, 5, padding=2), nn.GELU(),
                                  nn.Conv1d(hidden, hidden, 5, padding=2, groups=hidden), nn.GELU(),
                                  nn.Conv1d(hidden, hidden, 1), nn.GELU())
        self.proj = nn.Linear(hidden, out_dim)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, waveform, T):
        with autocast(device_type="cuda", enabled=False):
            m = self.to_db(self.mel(waveform.float()))
            m = (m - m.mean(dim=(1, 2), keepdim=True)) / m.std(dim=(1, 2), keepdim=True).clamp_min(1e-5)
        h = self.conv(m)[..., :T]
        if h.shape[-1] < T:
            h = F.pad(h, (0, T - h.shape[-1]))
        return self.proj(h.transpose(1, 2))


class VaaniSEDV5(VaaniLiteSED):
    """Your V3 network (same mel branch, BEATs projection, gated fusion, BiGRU,
    TemporalRefinement, onset/offset heads) with
      * cat_frame_head: Linear(256 -> 7), one frame track per category
      * cat_attn: Conv1d(256 -> 7) attention that pools the *frame-track
        probabilities* into clip-level category probabilities (MIL), so weak labels
        train the same tracks the decoder reads.
    Optional (off in V5-A, each a separate experiment):
      * per_cat_boundary: 7 onset + 7 offset channels (copied from the shared heads)
      * short_mel: ShortMelBranch added to the mel features (zero-initialised)
    BEATs is never run inside this module (beats_model=None): features always come
    from a cache, so a missing cache entry can never silently become live BEATs."""

    def __init__(self, n_fine, beats_dim=768, mel_frontend=None, feat_dropout=0.2,
                 per_cat_boundary=False, short_mel=False):
        super().__init__(beats_model=None, n_fine=n_fine, n_coarse=len(COARSE_CATEGORIES),
                         beats_dim=beats_dim, mel_frontend=mel_frontend)
        gru_out = self.event_frame_head.in_features
        self.per_cat_boundary = bool(per_cat_boundary)
        self.feat_drop = nn.Dropout(feat_dropout)
        self.cat_frame_head = nn.Linear(gru_out, NC)
        self.cat_attn = nn.Conv1d(gru_out, NC, kernel_size=1)
        if self.per_cat_boundary:
            self.cat_onset_head = nn.Linear(gru_out, NC)
            self.cat_offset_head = nn.Linear(gru_out, NC)
        self.short_mel = ShortMelBranch(self.proj_dim) if short_mel else None

    def new_keys(self):
        k = {"cat_frame_head.weight", "cat_frame_head.bias", "cat_attn.weight", "cat_attn.bias"}
        if self.per_cat_boundary:
            k |= {f"cat_{w}_head.{p}" for w in ("onset", "offset") for p in ("weight", "bias")}
        if self.short_mel is not None:
            k |= {n for n in self.state_dict() if n.startswith("short_mel.")}
        return k

    def init_from_v3(self, v3_state):
        sd = {k: v for k, v in v3_state.items() if not k.startswith("beats.")}
        missing, unexpected = self.load_state_dict(sd, strict=False)
        assert set(missing) == self.new_keys(), f"unexpected missing keys {set(missing) ^ self.new_keys()}"
        assert not unexpected, f"unexpected keys {unexpected}"
        with torch.no_grad():
            self.cat_frame_head.weight.copy_(self.event_frame_head.weight.expand(NC, -1))
            self.cat_frame_head.bias.copy_(self.event_frame_head.bias.expand(NC))
            self.cat_attn.weight.copy_(self.coarse_attn.weight)
            self.cat_attn.bias.copy_(self.coarse_attn.bias)
            if self.per_cat_boundary:
                for w in ("onset", "offset"):
                    src = getattr(self, f"event_{w}_head"); dst = getattr(self, f"cat_{w}_head")
                    dst.weight.copy_(src.weight.expand(NC, -1)); dst.bias.copy_(src.bias.expand(NC))

    def trainable_parameters(self):
        unused = UNUSED_V3_HEADS + (("event_onset_head", "event_offset_head") if self.per_cat_boundary else ())
        return [p for n, p in self.named_parameters()
                if p.requires_grad and not n.split(".")[0] in unused]

    def forward(self, waveform, beats, gate_span, mel=None):
        """waveform (B, 128000); beats (B, 50, 768); gate_span (B, 2) = [lo, hi) in
        seconds inside the window where BEATs features are real."""
        if waveform.ndim == 3:
            waveform = waveform.squeeze(1)
        bf = self.beats_proj(beats)
        if mel is None:
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        if self.short_mel is not None:
            fdy = fdy + self.short_mel(waveform, T).to(fdy.dtype)
        bf = F.interpolate(bf.transpose(1, 2), size=T, mode="linear", align_corners=False).transpose(1, 2)
        bf = self.feat_drop(bf)
        fdy = self.feat_drop(fdy)
        gate = self.fuse_gate(torch.cat([bf, fdy], dim=-1))
        centres = (torch.arange(T, device=waveform.device, dtype=torch.float32) + 0.5) * (W_SEC / T)
        gspan = gate_span.float()
        bvalid = (centres[None] >= gspan[:, :1]) & (centres[None] < gspan[:, 1:2])
        gate = gate * bvalid.to(gate.dtype).unsqueeze(-1)
        x = self.fuse_norm(gate * bf + (1.0 - gate) * fdy)
        x, _ = self.gru(x)
        x = self.temporal_refine(self.gru_norm(x))
        if self.per_cat_boundary:
            on, off = self.cat_onset_head(x), self.cat_offset_head(x)          # (B,T,7)
        else:
            on, off = self.event_onset_head(x)[..., 0], self.event_offset_head(x)[..., 0]  # (B,T)
        return {"cat_logits": self.cat_frame_head(x),                       # (B,T,7)
                "onset_logits": on, "offset_logits": off,
                "attn_logits": self.cat_attn(x.transpose(1, 2)).transpose(1, 2),  # (B,T,7)
                "T": T}


def mil_pool(cat_logits, attn_logits, valid):
    """Clip-level category probability = attention-weighted mean of frame
    probabilities over valid frames."""
    p = torch.sigmoid(cat_logits.float())
    a = attn_logits.float().masked_fill(~valid.bool().unsqueeze(-1), -1e4)
    a = torch.softmax(a, dim=1)
    return (a * p).sum(1).clamp(1e-6, 1 - 1e-6)


def state_hash(sd):
    h = hashlib.sha1()
    for k in sorted(sd):
        h.update(k.encode()); h.update(sd[k].detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()[:16]     # full tensor contents


# ---------------------------------------------------------------------------
# 4. data
# ---------------------------------------------------------------------------
class SynthStore:
    """Synthetic mixtures: <root>/wav/<id>.flac, <root>/beats/<id>.npy,
    <root>/manifest.jsonl. Separate ID namespace (s000000, v000000) -- never touches
    the global-index BEATs cache."""
    def __init__(self, root):
        self.root = root
        self.items = {}
        mp = os.path.join(root, "manifest.jsonl")
        if os.path.exists(mp):
            with open(mp, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        r = json.loads(line); self.items[r["id"]] = r

    def wav(self, sid):
        x, sr = sf.read(os.path.join(self.root, "wav", f"{sid}.flac"), dtype="float32")
        assert sr == SR
        return torch.from_numpy(np.ascontiguousarray(x))

    def beats(self, sid):
        p = os.path.join(self.root, "beats", f"{sid}.npy")
        if not os.path.exists(p):
            raise RuntimeError(f"missing synthetic BEATs entry {p} -- rebuild the synthetic cache")
        return np.load(p).astype(np.float32)


def load_nat_beats(gidx):
    p = resolve_cache_path(gidx, BEATS_CACHE_DIRS)
    if p is None:
        raise RuntimeError(f"Missing BEATs cache entry for global index {gidx} "
                           f"(checked {BEATS_CACHE_DIRS}). No live fallback by design.")
    assert os.path.basename(p) == f"{int(gidx):06d}.npy", (p, gidx)   # global_index == cache_index
    return np.load(p).astype(np.float32)


def check_beats_len(beats_full, n_samples, key):
    exp = beats_T_expected(n_samples)
    if abs(beats_full.shape[0] - exp) > 1:
        raise RuntimeError(f"BEATs rows {beats_full.shape[0]} != expected {exp} for {key} "
                           f"({n_samples / SR:.2f}s audio) -- cache/index misalignment?")


class V5Dataset(Dataset):
    """Rows are ("nat", global_index, tier) or ("syn", synth_id, T_SYNTH)."""

    def __init__(self, rows, full_dataset, label_table, synth_store=None, training=True,
                 shift_prob=0.8, beats_mask_prob=0.5, beats_mask_max=3,
                 bg_crop_prob=0.3, jitter=1.5):
        self.rows = list(rows)
        self.fd = full_dataset; self.lt = label_table; self.syn = synth_store
        self.training = training
        self.shift_prob = shift_prob; self.beats_mask_prob = beats_mask_prob
        self.beats_mask_max = beats_mask_max; self.bg_crop_prob = bg_crop_prob; self.jitter = jitter

    def __len__(self):
        return len(self.rows)

    def _load(self, kind, key):
        if kind == "nat":
            wav = _load_waveform(key, self.fd, SR, WAV_CACHE_DIRS)
            bf = load_nat_beats(key)
            cats = [c for c in (self.lt["NoiseCategory"][key] or []) if c in CAT_IDX]
            evs = parse_events(self.lt["NoiseSubCategoryTimeStamp"][key], cats)
        else:
            r = self.syn.items[key]
            wav = self.syn.wav(key); bf = self.syn.beats(key)
            evs = [(float(s), float(e), CAT_IDX[c]) for s, e, c in r["events"]]
            cats = sorted({c for _, _, c in r["events"]})
        check_beats_len(bf, wav.numel(), f"{kind}:{key}")
        return wav, bf, evs, cats

    def _mask_beats(self, rows):
        if random.random() >= self.beats_mask_prob:
            return rows
        rows = rows.copy()
        for _ in range(random.randint(1, 2)):
            w = random.randint(1, self.beats_mask_max)
            s = random.randint(0, max(0, rows.shape[0] - w))
            rows[s:s + w] = 0
        return rows

    def __getitem__(self, i):
        kind, key, tier = self.rows[i]
        wav, bf, evs, cats = self._load(kind, key)
        n = wav.numel()
        if n <= WIN_SAMPLES:
            k_max = (WIN_SAMPLES - n) // PATCH_SAMPLES
            k = random.randint(0, k_max) if (self.training and k_max > 0 and random.random() < self.shift_prob) else 0
            shift = k * PATCH_SAMPLES
            chunk = F.pad(wav, (shift, WIN_SAMPLES - n - shift))
            rows = bf[:CANON_T - k]
            n_real = rows.shape[0]
            rows = np.concatenate([np.zeros((k, bf.shape[1]), np.float32), rows,
                                   np.zeros((CANON_T - k - n_real, bf.shape[1]), np.float32)], 0)
            vs = shift / SR; ve = (shift + n) / SR
            ghi = min(ve, (k + n_real) * BEATS_PATCH_HOP_SEC)
            crop_evs = [(s + vs, e + vs, c) for s, e, c in evs]
            whole = True
        else:
            k_max = (n - WIN_SAMPLES) // PATCH_SAMPLES
            if evs and random.random() >= self.bg_crop_prob:
                s0, e0, _ = random.choice(evs)
                centre = 0.5 * (s0 + e0) + random.uniform(-self.jitter, self.jitter)
                k = int(round((centre * SR - WIN_SAMPLES / 2) / PATCH_SAMPLES))
            else:
                k = random.randint(0, k_max)
            k = max(0, min(k_max, k))
            start = k * PATCH_SAMPLES
            chunk = wav[start:start + WIN_SAMPLES]
            rows, n_real = beats_rows_for_window(bf, start)
            cs = start / SR
            vs, ve = 0.0, W_SEC
            ghi = min(ve, n_real * BEATS_PATCH_HOP_SEC)
            crop_evs = []
            for s, e, c in evs:
                ns, ne = max(0.0, s - cs), min(W_SEC, e - cs)
                if ne > ns:
                    crop_evs.append((ns, ne, c))
            whole = False
        if self.training:
            rows = self._mask_beats(rows)

        # weak (clip-level) targets z and per-category confidence mask m
        z = np.zeros(NC, np.float32); m = np.ones(NC, np.float32)
        in_crop = {c for s, e, c in crop_evs if (e - s) >= 0.05}
        if tier in (T_GOLD, T_SYNTH):
            for c in in_crop: z[c] = 1.0
        elif tier == T_SILVER:
            clip_c = {CAT_IDX[c] for c in cats} | {c for _, _, c in evs}
            if whole:
                for c in clip_c: z[c] = 1.0
            else:
                strong_in = {c for s, e, c in crop_evs if (e - s) >= 0.3}
                for c in clip_c:
                    if c in strong_in: z[c] = 1.0
                    else: m[c] = 0.0             # present in clip, unsure in this crop
        else:  # bronze: clip categories only
            clip_c = {CAT_IDX[c] for c in cats}
            for c in clip_c:
                if whole: z[c] = 1.0
                else: m[c] = 0.0
        return {"waveform": chunk.float(), "beats": torch.from_numpy(np.ascontiguousarray(rows, np.float32)),
                "vspan": (vs, ve), "gspan": (vs, ghi), "events": crop_evs,
                "z": torch.from_numpy(z), "zmask": torch.from_numpy(m), "tier": tier, "key": f"{kind}:{key}"}


def collate_v5(batch):
    return {"waveform": torch.stack([b["waveform"] for b in batch]),
            "beats": torch.stack([b["beats"] for b in batch]),
            "vspan": torch.tensor([b["vspan"] for b in batch], dtype=torch.float32),
            "gspan": torch.tensor([b["gspan"] for b in batch], dtype=torch.float32),
            "events": [b["events"] for b in batch],
            "z": torch.stack([b["z"] for b in batch]), "zmask": torch.stack([b["zmask"] for b in batch]),
            "tier": torch.tensor([b["tier"] for b in batch], dtype=torch.long),
            "key": [b["key"] for b in batch]}


def make_tier_sampler(rows, shares, num_samples):
    counts = Counter(r[2] for r in rows)
    live = {t: s for t, s in shares.items() if counts.get(t, 0) > 0 and s > 0}
    tot = sum(live.values())
    print("  sampler target shares:", {TIER_NAME[t]: round(s / tot, 3) for t, s in live.items()},
          "| rows:", {TIER_NAME[t]: counts.get(t, 0) for t in shares})
    w = torch.tensor([live.get(r[2], 0.0) / tot / counts[r[2]] if r[2] in live else 0.0 for r in rows],
                     dtype=torch.double)
    return WeightedRandomSampler(w, num_samples=int(num_samples), replacement=True)


# ---------------------------------------------------------------------------
# 5. targets + losses
# ---------------------------------------------------------------------------
BOUNDARY_SIGMA = 0.03     # same gaussian boundary targets best_stage2 was trained with


def build_targets_v5(events, vspan, T, device, per_cat=False):
    """Boundary targets are (B,T) class-agnostic, or (B,T,7) when per_cat."""
    B = len(events)
    centres = (torch.arange(T, dtype=torch.float32) + 0.5) * (W_SEC / T)
    vs = vspan[:, :1]; ve = vspan[:, 1:2]
    valid = ((centres[None] >= vs) & (centres[None] < ve)).float()
    frame = torch.zeros(B, T, NC)
    on = torch.zeros(B, T, NC) if per_cat else torch.zeros(B, T)
    off = torch.zeros_like(on)
    bmask = valid.clone()
    for b in range(B):
        lo, hi = float(vspan[b, 0]), float(vspan[b, 1])
        for s, e, c in events[b]:
            frame[b, (centres >= s) & (centres < e), c] = 1.0
            for edge, tgt in ((s, on), (e, off)):
                if edge <= lo + 1e-3 or edge >= hi - 1e-3:
                    # boundary created by the crop/clip edge: no target, and do not
                    # punish the head near it either
                    bmask[b, (centres - edge).abs() < 0.1] = 0.0
                    continue
                g_ = torch.exp(-0.5 * ((centres - edge) / BOUNDARY_SIGMA) ** 2)
                if per_cat:
                    tgt[b, :, c] = torch.maximum(tgt[b, :, c], g_)
                else:
                    tgt[b] = torch.maximum(tgt[b], g_)
    return frame.to(device), on.to(device), off.to(device), valid.to(device), bmask.to(device)


_ASL = AsymmetricFocalLoss(gamma_neg=2.0, gamma_pos=1.0, clip=0.05)


def v5_losses(out, batch, cfg, device):
    T = out["T"]
    per_cat = out["onset_logits"].ndim == 3
    frame_t, on_t, off_t, valid, bmask = build_targets_v5(batch["events"], batch["vspan"], T, device, per_cat)
    tier = batch["tier"].to(device)
    strong = ((tier == T_GOLD) | (tier == T_SYNTH)).float()
    sm = valid * strong[:, None]
    cl = out["cat_logits"].float()
    parts = {}
    parts["frame"] = _ASL(cl, frame_t, mask=sm)
    bm = bmask * strong[:, None]
    parts["onset"] = masked_bce(out["onset_logits"].float(), on_t, mask=bm)
    parts["offset"] = masked_bce(out["offset_logits"].float(), off_t, mask=bm)
    p = torch.sigmoid(cl)
    pu = (1.0 - torch.prod(1.0 - p, dim=-1)) * valid
    yu = frame_t.max(-1).values * valid
    d = 1.0 - (2.0 * (pu * yu).sum(1) + 1.0) / (pu.sum(1) + yu.sum(1) + 1.0)
    parts["dice"] = (d * strong).sum() / strong.sum().clamp_min(1.0)
    q = mil_pool(cl, out["attn_logits"], valid)
    z = batch["z"].to(device); zm = batch["zmask"].to(device)
    roww = torch.ones_like(strong); roww[tier == T_BRONZE] = cfg["bronze_weak_w"]
    wz = zm * roww[:, None]
    bce = -(z * torch.log(q) + (1 - z) * torch.log(1 - q))
    parts["weak"] = (bce * wz).sum() / wz.sum().clamp_min(1.0)
    loss = (cfg["w_frame"] * parts["frame"] + cfg["w_on"] * parts["onset"] + cfg["w_off"] * parts["offset"]
            + cfg["w_dice"] * parts["dice"] + cfg["w_weak"] * parts["weak"])
    return loss, parts


# ---------------------------------------------------------------------------
# 6. inference: per-category tracks for a whole clip
# ---------------------------------------------------------------------------
def model_time_grid(net, device):
    net.eval()
    with torch.no_grad():
        o = net(torch.zeros(1, WIN_SAMPLES, device=device), torch.zeros(1, CANON_T, 768, device=device),
                torch.tensor([[0.0, W_SEC]], device=device))
    return o["T"], o["T"] / W_SEC


@torch.no_grad()
def tracks_v5(net, wav, beats_full, device, T_win, fps, max_batch=8):
    """Average logits of overlapping grid windows; returns probabilities."""
    net.eval()
    n = wav.numel(); dur = n / SR
    starts = grid_window_starts(n)
    total_frames = int(round(dur * fps)) + T_win + 1
    nb_ = NC if getattr(net, "per_cat_boundary", False) else 1      # boundary channels
    acc = np.zeros((total_frames, NC + 2 * nb_), np.float64); cnt = np.zeros(total_frames)
    for b0 in range(0, len(starts), max_batch):
        bs = starts[b0:b0 + max_batch]
        chunks, rows, gsp = [], [], []
        for s0 in bs:
            ch = wav[s0:s0 + WIN_SAMPLES]
            real = ch.numel()
            if real < WIN_SAMPLES:
                ch = F.pad(ch, (0, WIN_SAMPLES - real))
            r, n_real = beats_rows_for_window(beats_full, s0)
            chunks.append(ch); rows.append(torch.from_numpy(np.ascontiguousarray(r, np.float32)))
            gsp.append((0.0, min(real / SR, n_real * BEATS_PATCH_HOP_SEC)))
        out = net(torch.stack(chunks).to(device), torch.stack(rows).to(device),
                  torch.tensor(gsp, dtype=torch.float32, device=device))
        on_, of_ = out["onset_logits"].float(), out["offset_logits"].float()
        if on_.ndim == 2:
            on_, of_ = on_[..., None], of_[..., None]
        assert on_.shape[-1] == nb_
        lg = torch.cat([out["cat_logits"].float(), on_, of_], -1).cpu().numpy()
        for j, s0 in enumerate(bs):
            w = s0 / SR
            off = int(round(w * fps))
            assert abs(off / fps - w) <= 0.5 / fps + 1e-9
            end = min(off + T_win, total_frames); L = end - off
            acc[off:end] += lg[j, :L]; cnt[off:end] += 1
    vl = min(int(round(dur * fps)) + 1, total_frames)
    m = acc[:vl] / np.maximum(cnt[:vl, None], 1e-6)
    pr = 1.0 / (1.0 + np.exp(-m))
    on_p, of_p = pr[:, NC:NC + nb_], pr[:, NC + nb_:NC + 2 * nb_]
    if nb_ == 1:
        on_p, of_p = on_p[:, 0], of_p[:, 0]
    return {"cat": pr[:, :NC].astype(np.float32), "onset": on_p.astype(np.float32),
            "offset": of_p.astype(np.float32), "duration": float(dur)}


def ensemble_tracks(list_of_tracks, weights=None):
    weights = weights or [1.0 / len(list_of_tracks)] * len(list_of_tracks)
    out = {k: sum(w * t[k] for w, t in zip(weights, list_of_tracks)) for k in ("cat", "onset", "offset")}
    out["duration"] = list_of_tracks[0]["duration"]
    return out


# ---------------------------------------------------------------------------
# 7. decoder: per category -> flatten
# ---------------------------------------------------------------------------
# nms_iou: an event is dropped only if it overlaps an already-kept event (any category)
# with IoU >= nms_iou. Scoring is class-agnostic, so two categories firing on the same
# sound would otherwise count as 1 TP + 1 FP. On Gold, only 10 cross-category event
# pairs out of 15,585 events have IoU >= 0.95 (51 at >= 0.7), so 0.95 costs almost
# nothing on true overlaps. 1.01 = off; the decoder search may choose it.
DEC_DEFAULT = dict(thr_short=0.5, thr_long=0.5, hyst=0.1, sw_short=3, sw_long=9,
                   mindur_short=0.06, mindur_long=0.15, gap_short=0.05, gap_long=0.15,
                   snap=0.10, snap_min=0.2, nms_iou=0.95, max_events=99)


def _runs(mask):
    d = np.diff(np.concatenate([[0], mask.astype(np.int8), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def _snap(probs, idx, radius, pmin):
    lo = max(0, idx - radius); hi = min(len(probs), idx + radius + 1)
    if hi <= lo:
        return idx
    j = lo + int(np.argmax(probs[lo:hi]))
    return j if probs[j] >= pmin else idx


def smooth_track(x, k):
    k = int(k)
    if k <= 1 or len(x) < 3:
        return x
    if k % 2 == 0:
        k += 1
    if k > len(x):
        k = len(x) if len(x) % 2 else len(x) - 1
    return medfilt(x, kernel_size=k) if k >= 3 else x


def decode_v5(tr, p, fps, smoothed=None):
    """smoothed: optional {(c, sw): array} precomputed for speed."""
    dur = tr["duration"]
    radius = int(round(p["snap"] * fps))
    cands = []
    for c in range(NC):
        g = CAT_GROUP[c]
        sw = p[f"sw_{g}"]
        pr = smoothed[(c, sw)] if smoothed is not None else smooth_track(tr["cat"][:, c], sw)
        hi = p[f"thr_{g}"]; lo = max(0.01, hi - p["hyst"])
        evs = []
        for s, e in _runs(pr >= lo):
            if pr[s:e].max() < hi:
                continue
            if radius > 0:
                on_tr = tr["onset"][:, c] if tr["onset"].ndim == 2 else tr["onset"]
                of_tr = tr["offset"][:, c] if tr["offset"].ndim == 2 else tr["offset"]
                ss = _snap(on_tr, s, radius, p["snap_min"])
                es = _snap(of_tr, e, radius, p["snap_min"])
                if es <= ss:
                    ss, es = s, e
            else:
                ss, es = s, e
            evs.append([ss, es])
        evs.sort()
        merged = []
        for s, e in evs:
            if merged and (s - merged[-1][1]) / fps <= p[f"gap_{g}"]:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])
        for s, e in merged:
            if (e - s) / fps >= p[f"mindur_{g}"]:
                cands.append((float(pr[s:max(e, s + 1)].mean()), s, e, c))
    cands.sort(key=lambda x: -x[0])
    kept = []
    for sc, s, e, c in cands:
        ok = True
        for _, ks, ke, _ in kept:
            inter = min(e, ke) - max(s, ks)
            if inter > 0 and inter / (max(e, ke) - min(s, ks)) >= p["nms_iou"]:
                ok = False; break
        if ok:
            kept.append((sc, s, e, c))
        if len(kept) >= p["max_events"]:
            break
    out = []
    for _, s, e, _ in kept:
        on, of = min(s / fps, dur), min(e / fps, dur)
        if of > on:
            out.append((round(on, 3), round(of, 3)))
    return sorted(out)


def precompute_smoothed(tracks_by_key, sws_short, sws_long):
    out = {}
    for k, tr in tracks_by_key.items():
        d = {}
        for c in range(NC):
            for sw in (sws_short if CAT_GROUP[c] == "short" else sws_long):
                d[(c, sw)] = smooth_track(tr["cat"][:, c], sw).astype(np.float32)
        out[k] = d
    return out


def score_sets(dec, sets, fps, smoothed=None, keys=None):
    """sets: {"N": (tracks, refs), "S": (tracks, refs)} -> per-set official scores."""
    res = {}
    for name, (tracks, refs) in sets.items():
        ks = keys[name] if keys is not None else list(tracks)
        pred = {k: decode_v5(tracks[k], dec, fps, smoothed[k] if smoothed is not None else None) for k in ks}
        res[name] = official_combined_score({k: refs[k] for k in ks}, pred)
    return res


W_NAT, W_SYN = 0.62, 0.38            # test mix: 3,437 natural / 2,080 synthetic


def joint(res):
    if "S" not in res:
        return res["N"]["combined"]
    return W_NAT * res["N"]["combined"] + W_SYN * res["S"]["combined"]


def write_submission(pred_by_clip, out_dir, zip_name, durations, expected=None):
    """Writes predictions.jsonl + zip after hard format checks."""
    os.makedirs(out_dir, exist_ok=True)
    if expected is not None and len(pred_by_clip) != expected:
        raise RuntimeError(f"{len(pred_by_clip)} clips predicted, expected {expected}")
    for cid, evs in pred_by_clip.items():
        for on, of in evs:
            assert 0.0 <= on < of <= durations[cid] + 1e-3, (cid, on, of, durations[cid])
    jl = os.path.join(out_dir, zip_name.replace(".zip", ".jsonl"))
    with open(jl, "w", encoding="utf-8") as f:
        for cid in sorted(pred_by_clip):
            f.write(json.dumps({"clip_id": cid, "events": [{"onset": round(float(on), 3), "offset": round(float(of), 3)}
                                                          for on, of in pred_by_clip[cid]]}) + "\n")
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jl, "predictions.jsonl")
    n_ev = sum(len(v) for v in pred_by_clip.values())
    n_empty = sum(1 for v in pred_by_clip.values() if not v)
    print(f"  wrote {zp}: {len(pred_by_clip)} clips, {n_ev} events, {n_empty} empty")
    return zp

print("V5 core definitions loaded.")


V5 core definitions loaded.


## Part 3 — data setup and exhaustive cache validation


In [44]:
# =============================================================================
# V5 DATA SETUP: dataset, split (same as V3), tiers, labels, exhaustive cache
# validation, district halves. Every later cell uses these objects.
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
set_seed(SEED)
full_dataset = load_from_disk(DATASET_PATH)
N_ROWS = len(full_dataset)
print("rows:", N_ROWS)

train_idx_all, val_idx_all = split_train_val(full_dataset, val_fraction=CONFIG["val_fraction"], seed=CONFIG["seed"])

# ---- exhaustive cache validation: every index except the 2 known-bad ones -----
_missing = [i for i in range(N_ROWS) if resolve_cache_path(i, BEATS_CACHE_DIRS) is None]
_unexpected = sorted(set(_missing) - BAD_INDICES)
print(f"[cache] {N_ROWS - len(_missing)}/{N_ROWS} entries present; missing: {_missing[:10]}")
if _unexpected:
    raise RuntimeError(f"BEATs cache is missing {len(_unexpected)} unexpected entries, first: {_unexpected[:10]}. "
                       f"Fix the cache before running V5 (no live fallback, no substitution).")
train_idx = [i for i in train_idx_all if i not in BAD_INDICES and i not in _missing]
val_idx = [i for i in val_idx_all if i not in BAD_INDICES and i not in _missing]
print(f"split: train {len(train_idx)} | val {len(val_idx)} (excluded {sorted(set(_missing))})")

label_table = build_label_table(full_dataset)
DISTRICT = full_dataset["district"]
DURATION_COL = full_dataset["duration"] if "duration" in full_dataset.column_names else None
build_tag_to_cat(label_table, train_idx)
tiers = split_indices_by_tier(full_dataset, indices=train_idx)
q = label_table["annotationQuality"]
VAL_GOLD = [i for i in val_idx if q[i] == "verified_timestamps"]
print("train tiers:", {k: len(v) for k, v in tiers.items()}, "| val Gold:", len(VAL_GOLD))

# per-event category coverage (should be ~100%: every event carries 'category')
_n_raw = _n_ok = 0
for i in tiers["gold"] + VAL_GOLD:
    raw = label_table["NoiseSubCategoryTimeStamp"][i] or []
    _n_raw += len(raw); _n_ok += len(parse_events(raw, label_table["NoiseCategory"][i]))
print(f"Gold events with a resolved category: {_n_ok}/{_n_raw}")
assert _n_ok >= 0.98 * _n_raw, "category field missing on many Gold events -- check the dataset"

# label space of the old auxiliary heads (needed to load best_stage2.pt unchanged)
canonical_tags, raw_to_canonical, f2c, _ = build_label_space(
    full_dataset, indices=train_idx, min_tag_count_to_keep=CONFIG["min_tag_count_to_keep"])
median_duration = estimate_global_duration_prior(full_dataset, indices=train_idx)
print(f"aux fine tags: {len(canonical_tags)} | median Gold event duration {median_duration:.3f}s")

# district halves of val (same rule as the earlier decoder search: alternate sorted districts)
_val_districts = sorted({DISTRICT[i] for i in VAL_GOLD})
HALF_OF_DISTRICT = {d: ("A" if j % 2 == 0 else "B") for j, d in enumerate(_val_districts)}
print(f"val districts: {len(_val_districts)} -> halves "
      f"A={sum(1 for i in VAL_GOLD if HALF_OF_DISTRICT[DISTRICT[i]] == 'A')} "
      f"B={sum(1 for i in VAL_GOLD if HALF_OF_DISTRICT[DISTRICT[i]] == 'B')} clips")

# test files + ids (used by the leakage guard and the submission cells)
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a")
TEST_FILES = sorted(p for p in glob.glob(os.path.join(TEST_AUDIO_DIR, "*"))
                    if os.path.splitext(p)[1].lower() in AUDIO_EXTS)
TEST_IDS = {os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES}
print(f"test clips found: {len(TEST_FILES)} (expected {EXPECTED_TEST_CLIPS})")
# ---- leakage guard on a real key -----------------------------------------------
# The local dataset has no `id` column, so the key is the audio file stem
# (audio.path, unique per row), read from the Arrow column without decoding audio.
# Row position stays the BEATs-cache global index.
def _audio_stems(ds):
    col = ds.data.column("audio")
    paths = []
    for ch in col.chunks:
        paths.extend(ch.field("path").to_pylist())
    assert len(paths) == len(ds), "audio column length != dataset length (indices mapping?)"
    return [os.path.splitext(os.path.basename(str(p)))[0] for p in paths]


AUDIO_STEM = _audio_stems(full_dataset)
assert len(set(AUDIO_STEM)) == N_ROWS, "audio.path stems are not unique -- cannot use them as the clip key"


def _rec_and_span(stem):
    """'<recording>_<start_ms>_<end_ms>' -> (recording, start_ms, end_ms) or (stem, None, None)."""
    parts = stem.rsplit("_", 2)
    if len(parts) == 3 and parts[1].isdigit() and parts[2].isdigit():
        return parts[0], int(parts[1]), int(parts[2])
    return stem, None, None


_overlap = TEST_IDS & set(AUDIO_STEM)
assert not _overlap, f"{len(_overlap)} test clips are also training rows: {sorted(_overlap)[:5]}"
_rec_train = defaultdict(list)
for i, st in enumerate(AUDIO_STEM):
    r, a, b = _rec_and_span(st)
    _rec_train[r].append((a, b, i))
_shared_rec, _time_overlap = 0, []
for t in TEST_IDS:
    r, a, b = _rec_and_span(t)
    if r in _rec_train:
        _shared_rec += 1
        for a2, b2, i in _rec_train[r]:
            if None not in (a, b, a2, b2) and min(b, b2) > max(a, a2):
                _time_overlap.append((t, AUDIO_STEM[i]))
print(f"leakage guard: 0 identical clips; {_shared_rec} test clips share a recording with training rows; "
      f"{len(_time_overlap)} overlap in time")
if _time_overlap:
    # organiser-provided training rows, so not excluded automatically -- but reported
    print(f"  WARNING: {len(_time_overlap)} test clips overlap provided training audio in time, e.g. {_time_overlap[:3]}")

V5_DIR = f"{BASE_DIR}/v5_work" + ("_smoke" if SMOKE else "")
_SYN_ROOT = f"{BASE_DIR}/synth_v5" + ("_smoke" if SMOKE else "")
SYNTH_TRAIN_DIR = f"{_SYN_ROOT}/train"
SYNTH_VAL_DIR = f"{_SYN_ROOT}/val"
TEST_BEATS_DIR = f"{BASE_DIR}/beats_cache_test"
SUB_DIR = f"{BASE_DIR}/submission"
for d in (V5_DIR, SUB_DIR):
    os.makedirs(d, exist_ok=True)
LOG_PATH = f"{V5_DIR}/v5_log.txt"
PROGRESS_LIVE_PATH = f"{V5_DIR}/progress_live.txt"


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

log(f"setup done: train {len(train_idx)} val {len(val_idx)} valGold {len(VAL_GOLD)} test {len(TEST_FILES)}")


device: cuda
rows: 88447
[cache] 88445/88447 entries present; missing: [3561, 48940]
split: train 80164 | val 8281 (excluded [3561, 48940])
train tiers: {'gold': 7554, 'silver': 55482, 'bronze': 17128} | val Gold: 1367
Gold events with a resolved category: 15585/15585
aux fine tags: 97 | median Gold event duration 0.487s
val districts: 12 -> halves A=695 B=672 clips
test clips found: 5517 (expected 5517)
leakage guard: 0 identical clips; 56 test clips share a recording with training rows; 0 overlap in time
[2026-10-09 15:27:31] setup done: train 80164 val 8281 valGold 1367 test 5517


## Val-N (all Gold val clips) in RAM


In [45]:
# =============================================================================
# Val-N in RAM: all Gold val clips (waveform + cached BEATs + reference events).
# Used by E0, by every training evaluation and by decoder selection.
# (~1,367 clips, ~1 GB; loading takes a few minutes the first time.)
# =============================================================================
_valn_keys = VAL_GOLD[:120] if SMOKE else VAL_GOLD
VALN = {}
t0 = time.time()
for gi in _valn_keys:
    w = _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS)
    bf = load_nat_beats(gi)
    check_beats_len(bf, w.numel(), f"val:{gi}")
    VALN[str(gi)] = {"wav": w, "beats": bf.astype(np.float16),
                     "ref": [(s, e) for s, e, _ in parse_events(label_table["NoiseSubCategoryTimeStamp"][gi],
                                                               label_table["NoiseCategory"][gi])],
                     "ref_all": _ref_events_for_full_idx(gi, label_table),
                     "half": HALF_OF_DISTRICT[DISTRICT[gi]], "district": DISTRICT[gi]}
# the scorer must see exactly the events the organisers would score: use every valid
# event, even the (rare) ones whose category could not be resolved
VALN_REFS = {k: v["ref_all"] for k, v in VALN.items()}
VALN_HALF = {h: [k for k, v in VALN.items() if v["half"] == h] for h in ("A", "B")}
log(f"Val-N loaded: {len(VALN)} clips in {time.time() - t0:.0f}s "
    f"(A={len(VALN_HALF['A'])}, B={len(VALN_HALF['B'])}); all BEATs lengths match audio")


[2026-10-09 15:28:05] Val-N loaded: 1367 clips in 34s (A=695, B=672); all BEATs lengths match audio


## E0 — reconcile the inference paths on best_stage2 (+ control / probe zips)


In [46]:
# =============================================================================
# E0 -- reconcile the three inference paths on best_stage2.pt (no training).
#
#   A        live BEATs per 8 s window (the 0.9965 submission, cell 29)
#   B-floor  cached BEATs, floor() slicing (one patch early on windows >= 2)
#   B-fixed  cached BEATs, windows on the 0.16 s grid, corrected slicing (V5 path)
#
# Every path is decoded with the cell-29 decoder and its exact parameters, on all
# Gold val clips and on each district half. Also defines the control-model helpers
# and the explicit test-BEATs cache builder used by later cells.
# =============================================================================

# ---- cell-29 decoder, verbatim --------------------------------------------
DECODE_C29 = dict(base_thr=0.5, mean_target=0.30, alpha=0.6, thr_min=0.15, thr_max=0.85,
                  clip_gate=0.20, max_p_gate=0.25, smooth_window=3,
                  min_duration=max(0.05, 0.15 * median_duration), merge_gap=0.05,
                  snap_radius_seconds=0.10)


def decode_adaptive(tracks, fps, cfg):
    frame = tracks["frame"]
    max_p = float(frame.max())
    clip_prob = tracks.get("clip_prob", 1.0)
    if clip_prob < cfg["clip_gate"] and max_p < cfg["max_p_gate"]:
        return []
    if max_p < 0.15:
        return []
    mean_p = float(frame.mean())
    thr = cfg["base_thr"] + cfg["alpha"] * (mean_p - cfg["mean_target"])
    thr = max(cfg["thr_min"], min(cfg["thr_max"], thr))
    return decode_events_single(frame, fps, threshold=thr, smooth_window=cfg["smooth_window"],
                                min_duration=cfg["min_duration"], merge_gap=cfg["merge_gap"],
                                onset_probs=tracks["onset"], offset_probs=tracks["offset"],
                                snap_radius_seconds=cfg["snap_radius_seconds"])


_beats_live = None


def get_live_beats():
    global _beats_live
    if _beats_live is None:
        _beats_live, _ = load_beats_model(BEATS_CHECKPOINT, device)
        _beats_live.eval()
    return _beats_live


def load_v3_control():
    ck = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    assert ck["canonical_tags"] == canonical_tags, "label space drifted vs best_stage2.pt"
    mel = MelFrontend(n_mels=CONFIG["n_mels"], sample_rate=SR, max_seconds=W_SEC).to(device)
    m = VaaniLiteSED(get_live_beats(), n_fine=len(canonical_tags), n_coarse=len(COARSE_CATEGORIES),
                     beats_dim=768, mel_frontend=mel).to(device)
    m.load_state_dict(ck["model"])
    m.eval()
    return m, ck


def _merge_windows(outs, starts, T_win, fps, dur):
    total_frames = int(round(dur * fps)) + T_win + 1
    acc = np.zeros((total_frames, 3)); cnt = np.zeros(total_frames); clip = []
    for (fr, on, of, cl), s0 in zip(outs, starts):
        off = int(round((s0 / SR) * fps)); end = min(off + T_win, total_frames); L = end - off
        if L > 0:
            acc[off:end, 0] += fr[:L]; acc[off:end, 1] += on[:L]; acc[off:end, 2] += of[:L]; cnt[off:end] += 1
        clip.append(cl)
    vl = min(int(round(dur * fps)) + 1, total_frames)
    m = acc[:vl] / np.maximum(cnt[:vl, None], 1e-6)
    pr = 1.0 / (1.0 + np.exp(-m))
    return {"frame": pr[:, 0], "onset": pr[:, 1], "offset": pr[:, 2],
            "clip_prob": float(1.0 / (1.0 + np.exp(-np.mean(clip)))), "duration": dur}


def _v3_out(out):
    return (out["event_frame_logits"][0, :, 0].float().cpu().numpy(),
            out["event_onset_logits"][0, :, 0].float().cpu().numpy(),
            out["event_offset_logits"][0, :, 0].float().cpu().numpy(),
            float(out["event_clip_logits"][0, 0]))


@torch.no_grad()
def v3_tracks(model, wav, beats_full, path, T_win, fps):
    n = wav.numel(); dur = n / SR; outs = []
    if path == "B-fixed":
        starts = grid_window_starts(n)
    else:
        starts = _window_starts(n, WIN_SAMPLES, STRIDE_SAMPLES)
    for s0 in starts:
        ch = wav[s0:s0 + WIN_SAMPLES]; real = ch.numel()
        if real < WIN_SAMPLES:
            ch = F.pad(ch, (0, WIN_SAMPLES - real))
        ch = ch.unsqueeze(0).to(device)
        if path == "A":
            vr = torch.tensor([real / WIN_SAMPLES], device=device)
            out = model(ch, beats_cached=None, beats_valid_ratio=vr)
        elif path == "B-floor":       # V4 tracks_core slicing
            if n <= WIN_SAMPLES:
                bw = beats_full
            else:
                T_full = beats_full.shape[0]
                p0 = int(np.floor((s0 / SR - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
                p1 = int(np.ceil(((s0 + WIN_SAMPLES) / SR - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
                p0 = max(0, min(p0, T_full - 1)); p1 = max(p0 + 1, min(p1, T_full))
                bw = beats_full[p0:p1]
            if bw.shape[0] < CANON_T:
                bw = np.concatenate([bw, np.zeros((CANON_T - bw.shape[0], bw.shape[1]), bw.dtype)], 0)
            bw = torch.from_numpy(np.ascontiguousarray(bw[:CANON_T], np.float32)).unsqueeze(0).to(device)
            out = model(ch, beats_cached=bw, beats_valid_ratio=torch.tensor([real / WIN_SAMPLES], device=device))
        else:                          # B-fixed
            rows, n_real = beats_rows_for_window(beats_full, s0)
            bw = torch.from_numpy(np.ascontiguousarray(rows, np.float32)).unsqueeze(0).to(device)
            vr = min(real / SR, n_real * BEATS_PATCH_HOP_SEC) / W_SEC
            out = model(ch, beats_cached=bw, beats_valid_ratio=torch.tensor([vr], device=device))
        outs.append(_v3_out(out))
    return _merge_windows(outs, starts, T_win, fps, dur)


def score_c29(tracks_by_key, refs, fps, keys=None):
    ks = keys if keys is not None else list(tracks_by_key)
    pred = {k: decode_adaptive(tracks_by_key[k], fps, DECODE_C29) for k in ks}
    return official_combined_score({k: refs[k] for k in ks}, pred)


def build_test_beats_cache():
    """EXPLICIT live extraction for the test clips only (same fp32 extractor and
    fp16 storage as the training cache). Resumable; hard-fails on bad output."""
    os.makedirs(TEST_BEATS_DIR, exist_ok=True)
    bm = get_live_beats(); n_new = 0
    for j, pth in enumerate(TEST_FILES):
        cid = os.path.splitext(os.path.basename(pth))[0]
        out = os.path.join(TEST_BEATS_DIR, f"{cid}.npy")
        if os.path.exists(out):
            continue
        w = load_test_wav(pth)
        if w.numel() < SR:     # BEATs needs >= ~0.2 s; pad very short clips to 1 s for extraction only
            print(f"  [test-beats] {cid}: {w.numel() / SR:.2f}s clip, padded to 1 s for BEATs")
            w = F.pad(w, (0, SR - w.numel()))
        feat, _ = extract_beats_full(bm, w, device)
        if not np.isfinite(feat).all():
            raise RuntimeError(f"non-finite BEATs output for test clip {cid}")
        check_beats_len(feat, w.numel(), f"test:{cid}")
        np.save(out, feat.astype(np.float16)); n_new += 1
        if (j + 1) % 500 == 0:
            print(f"  [test-beats] {j + 1}/{len(TEST_FILES)}")
    log(f"test BEATs cache: {n_new} new, {len(TEST_FILES)} total in {TEST_BEATS_DIR}")


def load_test_wav(pth):
    x, sr0 = sf.read(pth, dtype="float32")
    if x.ndim > 1:
        x = x.mean(axis=1)
    w = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))
    if sr0 != SR:
        w = torchaudio.functional.resample(w, int(sr0), SR)
    return w.contiguous()


def load_test_beats(cid, n_samples):
    p = os.path.join(TEST_BEATS_DIR, f"{cid}.npy")
    if not os.path.exists(p):
        raise RuntimeError(f"missing test BEATs entry {p}: run build_test_beats_cache() first")
    bf = np.load(p).astype(np.float32)
    check_beats_len(bf, max(n_samples, SR), f"test:{cid}")
    return bf


E0_PATH = f"{V5_DIR}/e0_tracks.pkl"
if RUN_E0_RECONCILE:
    v3, _ck = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    log(f"E0: best_stage2 stored combined={_ck['combined']:.4f}; time grid T={T3} fps={fps3:.3f}")
    E0 = {}
    for path in ("A", "B-floor", "B-fixed"):
        t0 = time.time()
        E0[path] = {k: v3_tracks(v3, v["wav"], v["beats"].astype(np.float32), path, T3, fps3)
                    for k, v in VALN.items()}
        r_all = score_c29(E0[path], VALN_REFS, fps3)
        r_a = score_c29(E0[path], VALN_REFS, fps3, VALN_HALF["A"])
        r_b = score_c29(E0[path], VALN_REFS, fps3, VALN_HALF["B"])
        log(f"E0 {path:8s}: all={r_all['combined']:.4f} (F1 {r_all['event_f1']:.4f} Dice {r_all['segment_dice']:.4f}) "
            f"A={r_a['combined']:.4f} B={r_b['combined']:.4f} [{time.time() - t0:.0f}s]")
    with open(E0_PATH, "wb") as f:
        pickle.dump({"fps": fps3, "tracks": E0}, f)
    a = score_c29(E0["A"], VALN_REFS, fps3)["combined"]
    bf_ = score_c29(E0["B-fixed"], VALN_REFS, fps3)["combined"]
    bl = score_c29(E0["B-floor"], VALN_REFS, fps3)["combined"]
    log(f"E0 verdict: B-fixed - A = {bf_ - a:+.4f}; B-fixed - B-floor = {bf_ - bl:+.4f}. "
        + ("Cached+fixed inference is at least as good as the live path -> safe for test."
           if bf_ >= a - 0.005 else
           "Cached+fixed is worse than live on val: keep the live path for the control; "
           "V5 is trained on the fixed cache so it uses B-fixed regardless."))
    del v3; torch.cuda.empty_cache() if torch.cuda.is_available() else None

if RUN_CONTROL_ZIP or RUN_E0_TEST_PROBE:
    v3, _ = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    if RUN_E0_TEST_PROBE:
        build_test_beats_cache()
    for path, flag, name in (("A", RUN_CONTROL_ZIP, "submission_0p9965_control.zip"),
                             ("B-fixed", RUN_E0_TEST_PROBE, "submission_E0_bfixed.zip")):
        if not flag:
            continue
        preds, durs = {}, {}
        for pth in TEST_FILES:
            cid = os.path.splitext(os.path.basename(pth))[0]
            w = load_test_wav(pth)
            bf = load_test_beats(cid, w.numel()) if path == "B-fixed" else None
            tr = v3_tracks(v3, w, bf, path, T3, fps3)
            preds[cid] = decode_adaptive(tr, fps3, DECODE_C29); durs[cid] = tr["duration"]
        write_submission(preds, SUB_DIR, name, durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    del v3


## E1 — synthetic mixtures and Val-S


In [47]:
# =============================================================================
# E1 -- synthetic mixtures ("clean speech + added noise") with exact labels.
#
# Sources are ONLY training-dataset clips (plus ESC-50 if you enable it):
#   bed  70%: clean speech = event-free stretches (>=1 s, 0.1 s margin) of Gold clips
#        30%: a whole Gold clip, keeping its own labelled events
#   fg   Gold event cuts (category from the annotation), category-balanced (p ~ n^0.5)
#   K = 1/2/3 events (p .55/.30/.15), SNR U[-5,+15] dB vs the bed's speech RMS,
#   5 ms fades, peak-limited to 0.99. Labels: onset = paste position, offset =
#   paste position + pasted length (exact by construction).
# Train mixtures come from TRAIN-district Gold clips only; Val-S comes from
# VAL-district Gold clips only, so no audio is shared between the two.
# Output: <root>/wav/<id>.flac, <root>/beats/<id>.npy (fp16), manifest.jsonl.
# Resumable: items already in the manifest are skipped.
# =============================================================================
N_SYNTH_TRAIN = 300 if SMOKE else 24000
N_SYNTH_VAL = 80 if SMOKE else 1000
SNR_RANGE = (-5.0, 15.0)
K_PROBS = (0.55, 0.30, 0.15)
FG_DUR_RANGE = (0.08, 6.0)
CLEAN_BED_PROB = 0.7
ESC_MAP = {  # ESC-50 class -> Vaani category (used only if ESC50_DIR is set)
    "dog": "animal", "rooster": "animal", "pig": "animal", "cow": "animal", "frog": "animal",
    "cat": "animal", "hen": "animal", "insects": "animal", "sheep": "animal", "crow": "animal",
    "chirping_birds": "animal",
    "crying_baby": "baby_child",
    "sneezing": "human_non_speech", "clapping": "human_non_speech", "breathing": "human_non_speech",
    "coughing": "human_non_speech", "laughing": "human_non_speech", "snoring": "human_non_speech",
    "footsteps": "human_non_speech", "drinking_sipping": "human_non_speech", "brushing_teeth": "human_non_speech",
    "car_horn": "vehicle_traffic", "engine": "vehicle_traffic", "train": "vehicle_traffic",
    "airplane": "vehicle_traffic", "helicopter": "vehicle_traffic",
    "vacuum_cleaner": "appliance_machine", "washing_machine": "appliance_machine",
    "hand_saw": "appliance_machine", "chainsaw": "appliance_machine", "keyboard_typing": "appliance_machine",
    "mouse_click": "appliance_machine",
    "clock_alarm": "phone_signal_alarm", "siren": "phone_signal_alarm", "church_bells": "phone_signal_alarm",
}


class _WavLRU:
    def __init__(self, cap=256):
        self.cap = cap; self.d = {}
    def get(self, gi):
        if gi in self.d:
            return self.d[gi]
        w = _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS).numpy()
        if len(self.d) >= self.cap:
            self.d.pop(next(iter(self.d)))
        self.d[gi] = w
        return w


def _active_rms(x):
    if len(x) < 400:
        return float(np.sqrt(np.mean(x ** 2)) + 1e-8)
    fr = x[: len(x) // 160 * 160].reshape(-1, 160)
    e = np.sqrt((fr ** 2).mean(1) + 1e-12)
    act = e[e > e.max() * 10 ** (-40 / 20)]
    return float(np.sqrt(np.mean(act ** 2)) + 1e-8)


def _fade(x, n=80):
    n = min(n, len(x) // 2)
    if n > 0:
        r = np.linspace(0, 1, n, dtype=np.float32)
        x = x.copy(); x[:n] *= r; x[-n:] *= r[::-1]
    return x


def _esc_bank():
    if not ESC50_DIR:
        return []
    import csv as _csv
    meta = os.path.join(ESC50_DIR, "meta", "esc50.csv")
    bank = []
    with open(meta) as f:
        for r in _csv.DictReader(f):
            if r["category"] in ESC_MAP:
                bank.append({"path": os.path.join(ESC50_DIR, "audio", r["filename"]),
                             "cat": ESC_MAP[r["category"]], "fold": int(r["fold"])})
    print(f"  ESC-50 foregrounds available: {len(bank)}")
    return bank


def _esc_cut(path):
    x, sr0 = sf.read(path, dtype="float32")
    if x.ndim > 1:
        x = x.mean(1)
    if sr0 != SR:
        x = torchaudio.functional.resample(torch.from_numpy(x), sr0, SR).numpy()
    fr = x[: len(x) // 160 * 160].reshape(-1, 160)
    e = 20 * np.log10(np.sqrt((fr ** 2).mean(1)) + 1e-9)
    act = np.where(e > e.max() - 35)[0]
    if len(act) == 0:
        return None
    return x[act[0] * 160:(act[-1] + 1) * 160]


def build_synth(root, prefix, n_items, gold_indices, seed, esc_folds):
    os.makedirs(os.path.join(root, "wav"), exist_ok=True)
    os.makedirs(os.path.join(root, "beats"), exist_ok=True)
    store = SynthStore(root)
    done = set(store.items)
    # assertion: no test clip and no clip from the other split can be a source
    for gi in gold_indices:
        assert AUDIO_STEM[gi] not in TEST_IDS
    lru = _WavLRU()
    # clean-speech beds and foreground bank
    beds, fg, gold_clips = [], defaultdict(list), []
    for gi in gold_indices:
        raw = label_table["NoiseSubCategoryTimeStamp"][gi] or []
        evs = parse_events(raw, label_table["NoiseCategory"][gi])
        d = float(DURATION_COL[gi]) if DURATION_COL is not None else len(lru.get(gi)) / SR
        if len(evs) == len([1 for e in raw if e]):           # every event resolved
            t = 0.0
            for s, e, _ in sorted(evs):
                if s - 0.1 - t >= 1.0:
                    beds.append((gi, t, s - 0.1))
                t = max(t, e + 0.1)
            if d - t >= 1.0:
                beds.append((gi, t, d))
            if 2.0 <= d <= 14.0:
                gold_clips.append((gi, d, evs))
        for s, e, c in evs:
            if FG_DUR_RANGE[0] <= e - s <= FG_DUR_RANGE[1]:
                fg[c].append(("gold", gi, s, e))
    esc = [b for b in _esc_bank() if b["fold"] in esc_folds]
    for b in esc:
        fg[CAT_IDX[b["cat"]]].append(("esc", b["path"], 0.0, 0.0))
    cats_avail = [c for c in range(NC) if fg[c]]
    pc = np.array([len(fg[c]) ** 0.5 for c in cats_avail]); pc /= pc.sum()
    durs = np.array([d for _, d, _ in gold_clips]) if gold_clips else np.array([6.0])
    print(f"  [{prefix}] beds={len(beds)} fg per cat={ {CATS[c]: len(fg[c]) for c in range(NC)} } "
          f"whole-clip beds={len(gold_clips)} | already built {len(done)}/{n_items}")
    bm = None
    t0 = time.time(); n_new = 0
    mf = open(os.path.join(root, "manifest.jsonl"), "a", encoding="utf-8")
    try:
        for n in range(n_items):
            sid = f"{prefix}{n:06d}"
            if sid in done:
                continue
            rng = np.random.default_rng([seed, n])
            # ---- bed ----
            if rng.random() < CLEAN_BED_PROB or not gold_clips:
                D = float(np.clip(rng.choice(durs), 2.0, 14.0))
                parts, src, L = [], [], 0
                while L < D * SR:
                    gi, a, b = beds[int(rng.integers(len(beds)))]
                    w = lru.get(gi)
                    seg = _fade(w[int(a * SR):int(b * SR)])
                    parts.append(seg); src.append([int(gi), round(a, 3), round(b, 3)]); L += len(seg)
                bed = np.concatenate(parts)[: int(D * SR)].astype(np.float32)
                bed_events, mode = [], "clean"
            else:
                gi, D, evs = gold_clips[int(rng.integers(len(gold_clips)))]
                bed = lru.get(gi).astype(np.float32).copy()
                D = len(bed) / SR
                bed_events = [(s, min(e, D), CATS[c]) for s, e, c in evs if min(e, D) > s]
                src, mode = [[int(gi), 0.0, round(D, 3)]], "gold"
            D = len(bed) / SR
            speech_rms = _active_rms(bed)
            # ---- foregrounds ----
            K = int(rng.choice([1, 2, 3], p=K_PROBS))
            pasted, mix = [], bed.copy()
            for _ in range(K):
                c = cats_avail[int(rng.choice(len(cats_avail), p=pc))]
                kind, a, s, e = fg[c][int(rng.integers(len(fg[c])))]
                if kind == "gold":
                    seg = lru.get(a)[int(s * SR):int(e * SR)]
                else:
                    seg = _esc_cut(a)
                    if seg is None:
                        continue
                if len(seg) < int(0.05 * SR):
                    continue
                max_len = int(0.8 * D * SR)
                if len(seg) > max_len:
                    o = int(rng.integers(0, len(seg) - max_len + 1)); seg = seg[o:o + max_len]
                pos = int(rng.integers(0, len(mix) - len(seg) + 1))
                snr = float(rng.uniform(*SNR_RANGE))
                gain = speech_rms / (_active_rms(seg) * 10 ** (snr / 20))
                mix[pos:pos + len(seg)] += _fade(seg.astype(np.float32)) * gain
                pasted.append({"cat": CATS[c], "src": [kind, a if kind == "esc" else int(a), round(s, 4), round(e, 4)],
                               "pos": pos, "len": int(len(seg)), "snr": round(snr, 2), "gain": round(float(gain), 5)})
            if not pasted:
                continue
            pk = float(np.abs(mix).max())
            if pk > 0.99:
                mix *= 0.99 / pk
            events = [(round(s, 4), round(e, 4), c) for s, e, c in bed_events]
            events += [(round(p["pos"] / SR, 4), round((p["pos"] + p["len"]) / SR, 4), p["cat"]) for p in pasted]
            sf.write(os.path.join(root, "wav", f"{sid}.flac"), mix, SR, format="FLAC", subtype="PCM_16")
            w_back, _ = sf.read(os.path.join(root, "wav", f"{sid}.flac"), dtype="float32")
            if bm is None:
                bm = get_live_beats()
            feat, _ = extract_beats_full(bm, torch.from_numpy(w_back), device)
            if not np.isfinite(feat).all():
                raise RuntimeError(f"non-finite BEATs output for {sid}")
            np.save(os.path.join(root, "beats", f"{sid}.npy"), feat.astype(np.float16))
            rec = {"id": sid, "duration": round(len(w_back) / SR, 4), "mode": mode, "bed_src": src,
                   "district": DISTRICT[src[0][0]], "events": events, "pasted": pasted}
            mf.write(json.dumps(rec) + "\n"); mf.flush()
            n_new += 1
            if n_new % 1000 == 0:
                print(f"    [{prefix}] {n_new} new ({time.time() - t0:.0f}s)")
    finally:
        mf.close()
    log(f"synthetic {prefix}: {n_new} new mixtures in {time.time() - t0:.0f}s -> {root}")


def verify_synth(root, gold_set):
    """Labels re-derived from the stored recipe must equal the stored labels; sources
    must belong to the intended split; files must exist with the stated length."""
    st = SynthStore(root)
    for sid, r in st.items.items():
        exp = [(round(p["pos"] / SR, 4), round((p["pos"] + p["len"]) / SR, 4), p["cat"]) for p in r["pasted"]]
        got = [tuple(e) for e in r["events"]][-len(exp):]
        assert [tuple(x) for x in exp] == [(a, b, c) for a, b, c in got], f"label mismatch {sid}"
        for g in r["bed_src"]:
            assert g[0] in gold_set, f"{sid} uses a bed outside its split"
        for p in r["pasted"]:
            if p["src"][0] == "gold":
                assert p["src"][1] in gold_set, f"{sid} uses a foreground outside its split"
        for s, e, c in r["events"]:
            assert 0 <= s < e <= r["duration"] + 1e-3 and c in CAT_IDX, (sid, s, e, c)
        # artifacts: waveform and BEATs entry exist, are readable and consistent
        wp = os.path.join(root, "wav", f"{sid}.flac"); bp = os.path.join(root, "beats", f"{sid}.npy")
        assert os.path.exists(wp) and os.path.exists(bp), f"{sid}: missing wav or BEATs file"
        info = sf.info(wp)
        assert info.samplerate == SR and info.channels == 1, f"{sid}: {info.samplerate} Hz, {info.channels} ch"
        assert abs(info.frames / SR - r["duration"]) <= 1e-3, f"{sid}: {info.frames / SR:.4f}s vs manifest {r['duration']}"
        feat = np.load(bp)
        assert feat.ndim == 2 and feat.shape[1] == 768, f"{sid}: BEATs shape {feat.shape}"
        assert feat.shape[0] == beats_T_expected(info.frames), f"{sid}: {feat.shape[0]} BEATs rows, expected {beats_T_expected(info.frames)}"
        assert np.isfinite(feat).all(), f"{sid}: non-finite BEATs values"
    print(f"  verified {len(st.items)} items in {root} (labels, sources, audio, BEATs)")
    return st


if RUN_BUILD_SYNTH:
    _tr_gold = [i for i in tiers["gold"]]
    build_synth(SYNTH_TRAIN_DIR, "s", N_SYNTH_TRAIN, _tr_gold, seed=SEED, esc_folds={1, 2, 3, 4})
    build_synth(SYNTH_VAL_DIR, "v", N_SYNTH_VAL, VAL_GOLD, seed=SEED + 1, esc_folds={5})
SYNTH_TRAIN = verify_synth(SYNTH_TRAIN_DIR, set(tiers["gold"])) if os.path.exists(SYNTH_TRAIN_DIR) else None
SYNTH_VAL = verify_synth(SYNTH_VAL_DIR, set(VAL_GOLD)) if os.path.exists(SYNTH_VAL_DIR) else None


  verified 24000 items in C:/Users/231b258/synth_v5/train (labels, sources, audio, BEATs)
  verified 1000 items in C:/Users/231b258/synth_v5/val (labels, sources, audio, BEATs)


## E2 — train V5-A


In [48]:
# =============================================================================
# E2 -- train V5-A (warm start from best_stage2.pt).
#
# Model selection uses ALL Gold val clips (Val-N) and all Val-S mixtures:
#   J = 0.62 * Combined(Val-N) + 0.38 * Combined(Val-S)
# with a light threshold sweep (the submission decoder is selected later, E3).
# Real optimiser steps are counted only when AMP did not skip the step; the LR
# schedule and EMA advance only on real steps.
# =============================================================================
VALS = {}
if SYNTH_VAL is not None:
    for sid in sorted(SYNTH_VAL.items)[: (60 if SMOKE else None)]:
        r = SYNTH_VAL.items[sid]
        w = SYNTH_VAL.wav(sid); bf = SYNTH_VAL.beats(sid)
        check_beats_len(bf, w.numel(), f"valS:{sid}")
        VALS[sid] = {"wav": w, "beats": bf.astype(np.float16), "ref": [(s, e) for s, e, _ in r["events"]],
                     "half": HALF_OF_DISTRICT.get(r["district"], "A")}
VALS_REFS = {k: v["ref"] for k, v in VALS.items()}
VALS_HALF = {h: [k for k, v in VALS.items() if v["half"] == h] for h in ("A", "B")}
log(f"Val-S loaded: {len(VALS)} mixtures (A={len(VALS_HALF['A'])}, B={len(VALS_HALF['B'])})")
if not VALS:
    print("WARNING: no Val-S -- selection will use Val-N only")

CFG_V5 = dict(lr=1e-4, min_lr=1e-6, weight_decay=1e-2, warmup_updates=300, total_updates=8000,
              eval_every=1000, batch_size=CONFIG["batch_size"], grad_accum_steps=CONFIG["grad_accum_steps"],
              ema_decay=0.999, early_stop_evals=4,
              w_frame=3.0, w_on=0.5, w_off=0.5, w_dice=0.5, w_weak=0.5, bronze_weak_w=0.5,
              shares={T_GOLD: 0.35, T_SYNTH: 0.30, T_SILVER: 0.25, T_BRONZE: 0.10},
              shift_prob=0.8, beats_mask_prob=0.5, feat_dropout=0.2,
              # follow-up experiments -- keep False for V5-A; turn ONE on per new RUN_TAG
              per_cat_boundary=False,   # 7 onset + 7 offset channels (copied from the shared heads)
              short_mel=True)          # 32 ms / 64-band mel branch, zero-initialised
if SMOKE:
    CFG_V5.update(total_updates=30, eval_every=15, warmup_updates=5)

mel_v5 = MelFrontend(n_mels=CONFIG["n_mels"], sample_rate=SR, max_seconds=W_SEC).to(device)


def new_v5(cfg=None):
    cfg = cfg or CFG_V5
    return VaaniSEDV5(n_fine=len(canonical_tags), mel_frontend=mel_v5, feat_dropout=cfg["feat_dropout"],
                      per_cat_boundary=cfg.get("per_cat_boundary", False),
                      short_mel=cfg.get("short_mel", False)).to(device)


def v5_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = new_v5(ck["cfg"]); net.load_state_dict(ck["model"]); net.eval()
    assert state_hash(net.state_dict()) == ck["hash"], f"checkpoint {path} does not match its stored hash"
    return net, ck


def eval_tracks(net, valset, T_win, fps):
    return {k: tracks_v5(net, v["wav"], v["beats"].astype(np.float32), device, T_win, fps)
            for k, v in valset.items()}


SEL_DEC = dict(DEC_DEFAULT)


def selection_J(tracks_n, tracks_s, fps):
    """Light, threshold-robust selection score (same threshold for every category)."""
    best = None
    for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
        dec = dict(SEL_DEC, thr_short=thr, thr_long=thr)
        sets = {"N": (tracks_n, VALN_REFS)}
        if tracks_s:
            sets["S"] = (tracks_s, VALS_REFS)
        res = score_sets(dec, sets, fps)
        J = joint(res)
        if best is None or J > best["J"]:
            best = {"J": J, "thr": thr, "N": res["N"]["combined"], "S": res["S"]["combined"] if "S" in res else None,
                    "N_f1": res["N"]["event_f1"], "N_dice": res["N"]["segment_dice"]}
    return best


def train_v5(rows, run_tag, total_updates, do_eval=True, init_state=None):
    set_seed(SEED)
    model = new_v5()
    if init_state is not None:
        model.init_from_v3(init_state)
    ema = new_v5(); ema.load_state_dict(model.state_dict()); freeze_model(ema)
    T_win, fps = model_time_grid(model, device)
    assert T_win == 801, f"unexpected time grid {T_win}"
    ds = V5Dataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True,
                   shift_prob=CFG_V5["shift_prob"], beats_mask_prob=CFG_V5["beats_mask_prob"])
    n_draw = (total_updates + 50) * CFG_V5["batch_size"] * CFG_V5["grad_accum_steps"]
    loader = DataLoader(ds, batch_size=CFG_V5["batch_size"], sampler=make_tier_sampler(rows, CFG_V5["shares"], n_draw),
                        num_workers=0, collate_fn=collate_v5, drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    opt = torch.optim.AdamW(params, lr=CFG_V5["lr"], weight_decay=CFG_V5["weight_decay"])
    floor = CFG_V5["min_lr"] / CFG_V5["lr"]

    def lr_at(u):
        wu = CFG_V5["warmup_updates"]
        if u < wu:
            return 0.1 + 0.9 * u / wu
        pr = min(1.0, (u - wu) / max(1, total_updates - wu))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = CFG_V5["lr"] * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda"))
    accum = CFG_V5["grad_accum_steps"]
    paths = {"student": f"{V5_DIR}/{run_tag}_student.pt", "ema": f"{V5_DIR}/{run_tag}_ema.pt"}
    best = {"student": -1.0, "ema": -1.0}; best_info = {}
    hist = []
    if do_eval:
        b0 = selection_J(eval_tracks(model, VALN, T_win, fps), eval_tracks(model, VALS, T_win, fps), fps)
        log(f"{run_tag} start (warm start from best_stage2 via 7 copied heads): J={b0['J']:.4f} "
            f"N={b0['N']:.4f} S={b0['S'] if b0['S'] is None else round(b0['S'], 4)} thr={b0['thr']}")
        hist.append({"update": 0, "ema": b0})
    upd = micro = skipped = 0; run = defaultdict(float); evals_since = 0; t0 = time.time()
    model.train()
    opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = model.mel_frontend(wav.float())
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device),
                        mel=weak_augment(mel))
        loss, parts = v5_losses(out, batch, CFG_V5, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss {({k: float(v) for k, v in parts.items()})} keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            run[k] += float(v.detach())
        run["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:          # AMP skipped this step (inf/nan grads)
            skipped += 1
            continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = CFG_V5["lr"] * lr_at(upd)
        update_teacher(model, ema, decay=CFG_V5["ema_decay"])
        if upd % 100 == 0:
            with open(PROGRESS_LIVE_PATH, "w") as f:
                f.write(f"run={run_tag}\nupdate={upd}/{total_updates}\nskipped={skipped}\n"
                        f"lr={opt.param_groups[0]['lr']:.2e}\nloss={float(loss):.4f}\n"
                        f"elapsed_min={(time.time() - t0) / 60:.1f}\n")
        last = upd >= total_updates
        if (upd % CFG_V5["eval_every"] == 0) or last:
            tr = " ".join(f"{k}={v / run['n']:.4f}" for k, v in run.items() if k != "n"); run = defaultdict(float)
            line = f"{run_tag} upd={upd} skipped={skipped} lr={opt.param_groups[0]['lr']:.2e} {tr}"
            if do_eval:
                improved = False
                for name, net in (("student", model), ("ema", ema)):
                    sc = selection_J(eval_tracks(net, VALN, T_win, fps), eval_tracks(net, VALS, T_win, fps), fps)
                    line += f" | {name}: J={sc['J']:.4f} N={sc['N']:.4f} S={sc['S'] if sc['S'] is None else round(sc['S'], 4)}"
                    if sc["J"] > best[name]:
                        best[name] = sc["J"]; improved = True; best_info[name] = {"update": upd, **sc}
                        sd = net.state_dict()
                        torch.save({"model": sd, "update": upd, "score": sc, "hash": state_hash(sd),
                                    "canonical_tags": canonical_tags, "cfg": CFG_V5}, paths[name])
                        line += " *saved*"
                    hist.append({"update": upd, name: sc})
                evals_since = 0 if improved else evals_since + 1
            model.train()
            log(line)
            if do_eval and evals_since >= CFG_V5["early_stop_evals"]:
                log(f"{run_tag}: early stop, no improvement in {evals_since} evaluations"); break
        if upd >= total_updates:
            break
    if upd < total_updates and evals_since < CFG_V5["early_stop_evals"]:
        raise RuntimeError(f"loader exhausted early at {upd}/{total_updates} updates")
    if not do_eval:
        for name, net in (("student", model), ("ema", ema)):
            sd = net.state_dict()
            torch.save({"model": sd, "update": upd, "hash": state_hash(sd), "canonical_tags": canonical_tags,
                        "cfg": CFG_V5}, paths[name])
    with open(f"{V5_DIR}/{run_tag}_history.json", "w") as f:
        json.dump({"best": best_info, "hist": hist, "updates": upd, "skipped": skipped}, f, default=float)
    log(f"{run_tag} done: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min; best {best_info}")
    return paths, best_info


TRAIN_ROWS = ([("nat", i, T_GOLD) for i in tiers["gold"]] + [("nat", i, T_SILVER) for i in tiers["silver"]]
              + [("nat", i, T_BRONZE) for i in tiers["bronze"]]
              + ([("syn", s, T_SYNTH) for s in sorted(SYNTH_TRAIN.items)] if SYNTH_TRAIN is not None else []))
_val_set = set(VAL_GOLD)
assert not any(r[0] == "nat" and r[1] in _val_set for r in TRAIN_ROWS), "val clip leaked into training rows"

if RUN_TRAIN_V5:
    _v3 = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    assert _v3["canonical_tags"] == canonical_tags
    V5_PATHS, V5_BEST = train_v5(TRAIN_ROWS, RUN_TAG, CFG_V5["total_updates"], do_eval=True, init_state=_v3["model"])
    del _v3


[2026-10-09 15:32:53] Val-S loaded: 1000 mixtures (A=515, B=485)
  sampler target shares: {'gold': 0.35, 'synth': 0.3, 'silver': 0.25, 'bronze': 0.1} | rows: {'gold': 7554, 'synth': 24000, 'silver': 55482, 'bronze': 17128}
[2026-10-09 15:33:41] v5c_smel start (warm start from best_stage2 via 7 copied heads): J=0.7799 N=0.8778 S=0.6203 thr=0.6


C:\Users\231b258\AppData\Local\Temp\ipykernel_10252\2108717081.py:146: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  f"lr={opt.param_groups[0]['lr']:.2e}\nloss={float(loss):.4f}\n"


[2026-10-09 15:49:21] v5c_smel upd=1000 skipped=0 lr=9.80e-05 frame=0.0584 onset=0.0796 offset=0.0834 dice=0.5303 weak=0.3873 | student: J=0.8480 N=0.8520 S=0.8415 *saved* | ema: J=0.7807 N=0.8362 S=0.6902 *saved*
[2026-10-09 16:04:41] v5c_smel upd=2000 skipped=0 lr=8.86e-05 frame=0.0442 onset=0.0752 offset=0.0780 dice=0.4190 weak=0.2688 | student: J=0.9075 N=0.8735 S=0.963 *saved* | ema: J=0.8721 N=0.8744 S=0.8682 *saved*
[2026-10-09 16:20:05] v5c_smel upd=3000 skipped=0 lr=7.29e-05 frame=0.0452 onset=0.0735 offset=0.0765 dice=0.3856 weak=0.2562 | student: J=0.9330 N=0.8856 S=1.0103 *saved* | ema: J=0.9168 N=0.8826 S=0.9725 *saved*
[2026-10-09 16:35:52] v5c_smel upd=4000 skipped=0 lr=5.35e-05 frame=0.0437 onset=0.0712 offset=0.0744 dice=0.3626 weak=0.2474 | student: J=0.9531 N=0.9095 S=1.0243 *saved* | ema: J=0.9355 N=0.8822 S=1.0225 *saved*
[2026-10-09 16:51:24] v5c_smel upd=5000 skipped=2 lr=3.37e-05 frame=0.0422 onset=0.0710 offset=0.0736 dice=0.3518 weak=0.2420 | student: J=0.9549

## E3 — decoder selection and control comparison


In [49]:
# =============================================================================
# E3 -- decoder selection for the chosen V5 checkpoint, and the control comparison.
#
# Rule: objective J = 0.62*Comb(Val-N) + 0.38*Comb(Val-S).
#   1. random search (N_CFG configs) scored on district half A only
#   2. the top 15 on A are re-scored on half B
#   3. accepted only if it beats DEC_DEFAULT on BOTH halves; else DEC_DEFAULT
# The control (best_stage2 + cell-29 decoder, live path A) is scored on the same
# halves. V5 should only replace the control on the LB if it beats it on both.
# =============================================================================
N_CFG = 40 if SMOKE else 300
SW_SHORT = [1, 3, 5, 9]
SW_LONG = [3, 5, 9, 15, 25]


def pick_v5_checkpoint(run_tag):
    cands = []
    for name in ("ema", "student"):
        p = f"{V5_DIR}/{run_tag}_{name}.pt"
        if os.path.exists(p):
            ck = torch.load(p, map_location="cpu", weights_only=False)
            cands.append((ck.get("score", {}).get("J", -1), name, p))
    if not cands:
        raise RuntimeError(f"no V5 checkpoint for {run_tag} in {V5_DIR} -- run the training cell")
    cands.sort(reverse=True)
    print("  checkpoints:", [(n, round(j, 4)) for j, n, _ in cands])
    return cands[0][2]


def sample_dec(r):
    return dict(thr_short=float(r.uniform(0.2, 0.8)), thr_long=float(r.uniform(0.2, 0.8)),
                hyst=float(r.choice([0.0, 0.05, 0.1, 0.2])),
                sw_short=int(r.choice(SW_SHORT)), sw_long=int(r.choice(SW_LONG)),
                mindur_short=float(r.uniform(0.02, 0.15)), mindur_long=float(r.uniform(0.05, 0.4)),
                gap_short=float(r.choice([0.0, 0.03, 0.05, 0.1])), gap_long=float(r.choice([0.0, 0.05, 0.1, 0.2, 0.3])),
                snap=float(r.choice([0.0, 0.05, 0.1, 0.15, 0.25])), snap_min=float(r.choice([0.1, 0.2, 0.3])),
                nms_iou=float(r.choice([0.7, 0.95, 1.01])), max_events=int(r.choice([2, 3, 5, 99])))


def half_scores(dec, TN, TS, fps, SMN, SMS):
    out = {}
    for h in ("A", "B"):
        sets = {"N": (TN, VALN_REFS)}; keys = {"N": VALN_HALF[h]}; sm = dict(SMN)
        if TS:
            sets["S"] = (TS, VALS_REFS); keys["S"] = VALS_HALF[h]; sm.update(SMS)
        out[h] = joint(score_sets(dec, sets, fps, sm, keys))
    return out


def select_decoder(TN, TS, fps, tag):
    SMN = precompute_smoothed(TN, SW_SHORT, SW_LONG)
    SMS = precompute_smoothed(TS, SW_SHORT, SW_LONG) if TS else {}
    base = half_scores(DEC_DEFAULT, TN, TS, fps, SMN, SMS)
    log(f"{tag} DEC_DEFAULT: A={base['A']:.4f} B={base['B']:.4f}")
    r = np.random.default_rng(SEED)
    scored = []
    t0 = time.time()
    for i in range(N_CFG):
        d = sample_dec(r)
        sets = {"N": (TN, VALN_REFS)}; keys = {"N": VALN_HALF["A"]}; sm = dict(SMN)
        if TS:
            sets["S"] = (TS, VALS_REFS); keys["S"] = VALS_HALF["A"]; sm.update(SMS)
        scored.append((joint(score_sets(d, sets, fps, sm, keys)), i, d))
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{N_CFG} configs ({time.time() - t0:.0f}s), best A so far {max(scored, key=lambda x: x[0])[0]:.4f}")
    scored.sort(key=lambda x: -x[0])
    chosen, chosen_s, searched = dict(DEC_DEFAULT), base, False
    for ja, _, d in scored[:15]:
        hs = half_scores(d, TN, TS, fps, SMN, SMS)
        if hs["A"] > base["A"] and hs["B"] > base["B"] and hs["B"] > chosen_s["B"]:
            chosen, chosen_s, searched = d, hs, True
    log(f"{tag} selected decoder: A={chosen_s['A']:.4f} B={chosen_s['B']:.4f} "
        f"({'searched config' if searched else 'DEC_DEFAULT kept: no searched config beat it on both halves'}) {chosen}")
    return chosen, chosen_s


def control_half_scores(fps3, trA_N, trA_S):
    out = {}
    for h in ("A", "B"):
        rn = score_c29(trA_N, VALN_REFS, fps3, VALN_HALF[h])["combined"]
        rs = score_c29(trA_S, VALS_REFS, fps3, VALS_HALF[h])["combined"] if trA_S else None
        out[h] = rn if rs is None else W_NAT * rn + W_SYN * rs
    return out


if RUN_DECODER_SELECTION:
    ck_path = pick_v5_checkpoint(RUN_TAG)
    net, ck = v5_from_ckpt(ck_path)
    T_win, fps = model_time_grid(net, device)
    TN = eval_tracks(net, VALN, T_win, fps); TS = eval_tracks(net, VALS, T_win, fps)
    DEC_V5, DEC_V5_S = select_decoder(TN, TS, fps, RUN_TAG)
    full = joint(score_sets(DEC_V5, {"N": (TN, VALN_REFS), **({"S": (TS, VALS_REFS)} if TS else {})}, fps))
    with open(f"{V5_DIR}/{RUN_TAG}_decoder.json", "w") as f:
        json.dump({"decoder": DEC_V5, "halves": DEC_V5_S, "J_full": full, "checkpoint": ck_path,
                   "hash": ck["hash"]}, f, indent=1)
    with open(f"{V5_DIR}/{RUN_TAG}_val_tracks.pkl", "wb") as f:
        pickle.dump({"fps": fps, "N": TN, "S": TS, "hash": ck["hash"]}, f)
    del net

    # ---- control: best_stage2 + cell-29 decoder, live path A ----------------
    v3, _ = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    if os.path.exists(E0_PATH):
        with open(E0_PATH, "rb") as f:
            trA_N = pickle.load(f)["tracks"]["A"]
        trA_N = {k: trA_N[k] for k in VALN}
    else:
        trA_N = {k: v3_tracks(v3, v["wav"], None, "A", T3, fps3) for k, v in VALN.items()}
    trA_S = {k: v3_tracks(v3, v["wav"], None, "A", T3, fps3) for k, v in VALS.items()}
    del v3
    CTRL_S = control_half_scores(fps3, trA_N, trA_S)
    v5_wins = DEC_V5_S["A"] > CTRL_S["A"] and DEC_V5_S["B"] > CTRL_S["B"]
    log(f"CONTROL (0.9965 path) on val: A={CTRL_S['A']:.4f} B={CTRL_S['B']:.4f} | "
        f"V5 {RUN_TAG}: A={DEC_V5_S['A']:.4f} B={DEC_V5_S['B']:.4f} -> "
        + ("V5 beats the control on both halves: submit V5." if v5_wins else
           "V5 does NOT beat the control on both halves: keep the control as your main submission."))


  checkpoints: [('student', 0.9765), ('ema', 0.9759)]
[2026-10-09 17:38:38] v5c_smel DEC_DEFAULT: A=1.0030 B=0.9507
    100/300 configs (18s), best A so far 1.0132
    200/300 configs (37s), best A so far 1.0188
    300/300 configs (55s), best A so far 1.0188
[2026-10-09 17:39:38] v5c_smel selected decoder: A=1.0140 B=0.9521 (searched config) {'thr_short': 0.28104787533051445, 'thr_long': 0.6161242817520318, 'hyst': 0.1, 'sw_short': 5, 'sw_long': 9, 'mindur_short': 0.029629171856031825, 'mindur_long': 0.20639359484495345, 'gap_short': 0.05, 'gap_long': 0.0, 'snap': 0.1, 'snap_min': 0.2, 'nms_iou': 0.7, 'max_events': 99}
BEATs cfg: layers= 12  dim= 768  patch= 16


C:\Users\231b258\torch_gpu\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATs loaded and frozen.
[2026-10-09 17:40:22] CONTROL (0.9965 path) on val: A=0.7935 B=0.7394 | V5 v5c_smel: A=1.0140 B=0.9521 -> V5 beats the control on both halves: submit V5.


## Test submission (V5)


In [ ]:
# =============================================================================
# Test submission for V5 (path B-fixed: cached test BEATs + corrected slicing).
# 1. builds the test BEATs cache EXPLICITLY (resumable, hard-fails on bad output)
# 2. runs the selected checkpoint, caches test tracks (re-decode later on CPU)
# 3. decodes with the decoder selected for THAT checkpoint (hash-checked)
# 4. writes submission/submission_<RUN_TAG>.zip after format checks
# =============================================================================
def run_test_submission(run_tag, ck_path, dec_json_path, zip_name):
    with open(dec_json_path) as f:
        dj = json.load(f)
    net, ck = v5_from_ckpt(ck_path)
    if dj["hash"] != ck["hash"]:
        print(f"  NOTE: decoder was selected on checkpoint {dj['hash']}, applying it to {ck['hash']} "
              f"({'expected for the final retrain' if 'final' in run_tag else 'UNEXPECTED'})")
        assert "final" in run_tag, "decoder/checkpoint mismatch"
    T_win, fps = model_time_grid(net, device)
    build_test_beats_cache()
    preds, durs, tracks = {}, {}, {}
    t0 = time.time()
    for j, pth in enumerate(TEST_FILES):
        cid = os.path.splitext(os.path.basename(pth))[0]
        w = load_test_wav(pth)
        bf = load_test_beats(cid, w.numel())
        tr = tracks_v5(net, w, bf, device, T_win, fps)
        tracks[cid] = {k: (v.astype(np.float16) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}
        preds[cid] = decode_v5(tr, dj["decoder"], fps); durs[cid] = tr["duration"]
        if (j + 1) % 500 == 0:
            print(f"  [test] {j + 1}/{len(TEST_FILES)} ({time.time() - t0:.0f}s)")
    with open(f"{V5_DIR}/{run_tag}_test_tracks.pkl", "wb") as f:
        pickle.dump({"fps": fps, "tracks": tracks, "hash": ck["hash"]}, f)
    zp = write_submission(preds, SUB_DIR, zip_name, durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    n_ev = np.array([len(v) for v in preds.values()])
    log(f"{run_tag} test submission {zp}: events/clip mean {n_ev.mean():.2f}, empty {int((n_ev == 0).sum())}")
    return zp


if RUN_TEST_SUBMISSION:
    _dj = f"{V5_DIR}/{RUN_TAG}_decoder.json"
    if not os.path.exists(_dj):
        raise RuntimeError("run the decoder-selection cell first (it writes the decoder for this checkpoint)")
    with open(_dj) as f:
        _ck_path = json.load(f)["checkpoint"]
    run_test_submission(RUN_TAG, _ck_path, _dj, f"submission_{RUN_TAG}{'_SMOKE' if SMOKE else ''}.zip")


[2026-10-09 17:40:22] test BEATs cache: 0 new, 5517 total in C:/Users/231b258/beats_cache_test
  [test] 500/5517 (14s)
  [test] 1000/5517 (31s)
  [test] 1500/5517 (47s)
  [test] 2000/5517 (64s)
  [test] 2500/5517 (83s)
  [test] 3000/5517 (101s)
  [test] 3500/5517 (119s)


## E7 — optional final retrain on all Gold


In [ ]:
# =============================================================================
# E7 (optional) -- final retrain on ALL Gold (train + val) + Silver/Bronze + synthetic,
# warm-started from best_stage2.pt, for the SAME number of updates at which the E2
# run's selected checkpoint peaked. No selection happens here (there is no held-out
# data left), so the decoder chosen in E3 is reused. Submit this as a separate LB
# entry and keep whichever of the two scores better.
# =============================================================================
if RUN_FINAL_RETRAIN:
    with open(f"{V5_DIR}/{RUN_TAG}_decoder.json") as f:
        _dj = json.load(f)
    _sel = torch.load(_dj["checkpoint"], map_location="cpu", weights_only=False)
    _n_upd = int(_sel["update"])
    assert _n_upd > 0, "selected checkpoint is the untrained start -- nothing to retrain"
    _which = "ema" if _dj["checkpoint"].endswith("_ema.pt") else "student"
    log(f"E7: retraining for {_n_upd} updates (selected {_which} checkpoint peaked there)")
    _rows = TRAIN_ROWS + [("nat", i, T_GOLD) for i in VAL_GOLD]
    _v3 = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    final_tag = f"{RUN_TAG}_final"
    _paths, _ = train_v5(_rows, final_tag, _n_upd, do_eval=False, init_state=_v3["model"])
    _final_dec = f"{V5_DIR}/{final_tag}_decoder.json"
    with open(_final_dec, "w") as f:
        json.dump(dict(_dj, checkpoint=_paths[_which]), f, indent=1)
    run_test_submission(final_tag, _paths[_which], _final_dec,
                        f"submission_{final_tag}{'_SMOKE' if SMOKE else ''}.zip")


In [ ]:
# =============================================================================
# DIAGNOSE -- where are the missing points? (CPU only, uses cached val tracks)
#
# For each RUN_TAG in DIAG_TAGS it loads <tag>_val_tracks.pkl + <tag>_decoder.json,
# decodes Val-N and Val-S, and reports:
#   1. F1 / precision / recall / Dice, predicted vs reference events per clip
#   2. anatomy of every reference event: matched / boundary-miss (a prediction
#      overlaps it but onset or offset is outside tolerance) / merged (one
#      prediction covers >=2 refs) / missed (nothing overlaps)
#      and of every prediction: TP / boundary-FP (overlaps a ref) / false alarm
#   3. recall by reference-duration bin and by category
#   4. onset / offset error (ms) of overlapping pairs, vs the tolerance
#   5. ORACLES on the same detections:
#        boundary oracle -- every prediction that overlaps a ref (1-1, by IoU)
#                           takes that ref's exact boundaries
#        split oracle    -- merged predictions are split into the refs they cover
#      The gap between the actual score and each oracle says which lever is worth
#      the remaining days: boundaries, event cardinality, or detection itself.
# =============================================================================
RUN_DIAGNOSE = True
DIAG_TAGS = ["v5a", "v5c_smel"]          # e.g. ["v5a", "v5b_bnd"] to compare runs side by side
DUR_BINS = [(0, 0.3), (0.3, 1.0), (1.0, 3.0), (3.0, 1e9)]


def _tol(s, e):
    return max(0.2 * (e - s), 0.05)


def _iou(a, b):
    i = min(a[1], b[1]) - max(a[0], b[0])
    return i / (max(a[1], b[1]) - min(a[0], b[0])) if i > 0 else 0.0


def _match(ref, pred):
    """organiser matching (closest-first, both edges within tolerance) -> set of (ri, pi)"""
    cand = []
    for ri, (rs, re_) in enumerate(ref):
        t = _tol(rs, re_)
        for pi, (ps, pe) in enumerate(pred):
            if abs(ps - rs) <= t and abs(pe - re_) <= t:
                cand.append((abs(ps - rs) + abs(pe - re_), ri, pi))
    mr, mp, pairs = set(), set(), []
    for _, ri, pi in sorted(cand):
        if ri not in mr and pi not in mp:
            mr.add(ri); mp.add(pi); pairs.append((ri, pi))
    return pairs


def anatomy(refs_cat, preds):
    """refs_cat: {clip: [(s,e,cat_idx)]}, preds: {clip: [(s,e)]}"""
    R = Counter(); P = Counter(); dur_tot = Counter(); dur_hit = Counter()
    cat_tot = Counter(); cat_hit = Counter(); on_err = []; off_err = []; tol_l = []
    oracle_b, oracle_s = {}, {}
    for k, rc in refs_cat.items():
        ref = [(s, e) for s, e, _ in rc]; pred = list(preds.get(k, []))
        pairs = _match(ref, pred); mref = {r for r, _ in pairs}; mpred = {p for _, p in pairs}
        cover = defaultdict(list)                       # pred -> refs it overlaps
        for pi, p in enumerate(pred):
            for ri, r in enumerate(ref):
                if _iou(p, r) > 0:
                    cover[pi].append(ri)
        for ri, (s, e, c) in enumerate(rc):
            b = next(i for i, (lo, hi) in enumerate(DUR_BINS) if lo <= e - s < hi)
            dur_tot[b] += 1; cat_tot[c] += 1
            if ri in mref:
                R["matched"] += 1; dur_hit[b] += 1; cat_hit[c] += 1; continue
            over = [pi for pi in range(len(pred)) if ri in cover[pi]]
            if not over:
                R["missed"] += 1
            elif any(len(cover[pi]) >= 2 for pi in over):
                R["merged"] += 1
            else:
                R["boundary-miss"] += 1
        for pi in range(len(pred)):
            P["TP" if pi in mpred else ("boundary-FP" if cover[pi] else "false-alarm")] += 1
        # 1-1 overlap assignment by IoU for error stats and the boundary oracle
        used_r, used_p, ob = set(), set(), []
        for v, ri, pi in sorted(((_iou(pred[pi], ref[ri]), ri, pi) for pi in cover for ri in cover[pi]), reverse=True):
            if ri in used_r or pi in used_p:
                continue
            used_r.add(ri); used_p.add(pi)
            on_err.append(pred[pi][0] - ref[ri][0]); off_err.append(pred[pi][1] - ref[ri][1])
            tol_l.append(_tol(*ref[ri])); ob.append(ref[ri])
        ob += [p for pi, p in enumerate(pred) if pi not in used_p]
        oracle_b[k] = sorted(ob)
        os_ = []
        for pi, p in enumerate(pred):
            os_ += [ref[ri] for ri in cover[pi]] if len(cover[pi]) >= 2 else [p]
        oracle_s[k] = sorted(set(os_))
    return dict(R=R, P=P, dur=(dur_tot, dur_hit), cat=(cat_tot, cat_hit),
                on=np.array(on_err), off=np.array(off_err), tol=np.array(tol_l),
                oracle_b=oracle_b, oracle_s=oracle_s)


def _report(name, refs_cat, preds):
    refs = {k: [(s, e) for s, e, _ in v] for k, v in refs_cat.items()}
    sc = official_combined_score(refs, preds)
    a = anatomy(refs_cat, preds)
    nr = sum(len(v) for v in refs.values()); npd = sum(len(v) for v in preds.values())
    log(f"  [{name}] Comb {sc['combined']:.4f} = F1 {sc['event_f1']:.4f} (P {sc['precision']:.3f} R {sc['recall']:.3f}) "
        f"+ Dice {sc['segment_dice']:.4f} | events/clip ref {nr / len(refs):.2f} pred {npd / len(refs):.2f}")
    log("     refs : " + ", ".join(f"{k} {v / nr:.1%}" for k, v in a["R"].most_common()))
    log("     preds: " + ", ".join(f"{k} {v / max(npd, 1):.1%}" for k, v in a["P"].most_common()))
    dt, dh = a["dur"]
    log("     recall by ref duration: " + ", ".join(
        f"{DUR_BINS[b][0]:g}-{DUR_BINS[b][1]:g}s {dh[b] / max(dt[b], 1):.2f} (n={dt[b]})" for b in range(len(DUR_BINS))))
    ct, ch = a["cat"]
    log("     recall by category: " + ", ".join(f"{CATS[c]} {ch[c] / max(ct[c], 1):.2f} (n={ct[c]})" for c in sorted(ct)))
    if len(a["on"]):
        ae_on, ae_off = np.abs(a["on"]) * 1000, np.abs(a["off"]) * 1000
        log(f"     overlapping pairs n={len(a['on'])}: |onset err| median {np.median(ae_on):.0f} ms p75 {np.percentile(ae_on, 75):.0f}; "
            f"|offset err| median {np.median(ae_off):.0f} ms p75 {np.percentile(ae_off, 75):.0f}; "
            f"mean signed onset {a['on'].mean() * 1000:+.0f} ms offset {a['off'].mean() * 1000:+.0f} ms; "
            f"onset within tol {np.mean(np.abs(a['on']) <= a['tol']):.2f}, offset within tol {np.mean(np.abs(a['off']) <= a['tol']):.2f}")
    for oname in ("oracle_b", "oracle_s"):
        so = official_combined_score(refs, a[oname])
        log(f"     {'boundary' if oname == 'oracle_b' else 'split'} oracle: Comb {so['combined']:.4f} "
            f"(F1 {so['event_f1']:.4f}, Dice {so['segment_dice']:.4f})  gain {so['combined'] - sc['combined']:+.4f}")
    return sc, a


if RUN_DIAGNOSE:
    refsN = {str(gi): parse_events(label_table["NoiseSubCategoryTimeStamp"][gi], label_table["NoiseCategory"][gi])
             for gi in (int(k) for k in VALN)}
    refsS = ({sid: [(s, e, CAT_IDX[c]) for s, e, c in r["events"]] for sid, r in SYNTH_VAL.items.items()}
             if SYNTH_VAL is not None else {})
    for tag in DIAG_TAGS:
        tp, dp = f"{V5_DIR}/{tag}_val_tracks.pkl", f"{V5_DIR}/{tag}_decoder.json"
        if not (os.path.exists(tp) and os.path.exists(dp)):
            print(f"skip {tag}: run decoder selection for it first ({tp})"); continue
        with open(tp, "rb") as f:
            vt = pickle.load(f)
        with open(dp) as f:
            dj = json.load(f)
        assert vt["hash"] == dj["hash"], "val tracks and decoder come from different checkpoints"
        log(f"DIAGNOSE {tag} (decoder {dj['decoder']})")
        for name, TR, RC in (("Val-N", vt["N"], refsN), ("Val-S", vt["S"], refsS)):
            RC = {k: v for k, v in RC.items() if k in TR}
            if not RC:
                continue
            preds = {k: decode_v5(TR[k], dj["decoder"], vt["fps"]) for k in RC}
            _report(name, RC, preds)


In [ ]:
# =============================================================================
# DECODER-2 -- CPU only, no training. Re-decodes the CACHED tracks of an existing
# run (default v5a) with three new decoder options aimed at what the diagnosis
# found (boundary misses and merged events):
#   split_{g}  split a run at a valley deeper than ratio * run peak  (merges)
#   edge_{g}   place each edge where the track crosses rel * run peak (boundaries)
#   shon_{g} / shoff_{g}  constant onset / offset shift            (bias: offsets late)
# Same rule as E3: search on district half A (0.62 Val-N + 0.38 Val-S), confirm on
# half B, accept only if it beats the run's current decoder on BOTH halves.
# If accepted, it re-decodes the cached TEST tracks (no GPU, no BEATs) and writes
# submission/submission_<tag>_dec2.zip -- one leaderboard probe, one change.
# Run this single cell after setup + Val-N + training cell (all RUN_* flags off).
# =============================================================================
# ---- paste-in version for an EXISTING kernel: redefines decode_v5 with the new
# ---- options (old decoder configs decode exactly as before), then runs Decoder-2.
def _split_valleys(pr, s, e, ratio, hi, min_len):
    """Split run [s,e) at local minima deeper than ratio * run peak, keeping only pieces
    that still reach hi. ratio <= 0 disables."""
    if ratio <= 0 or e - s < 2 * min_len + 1:
        return [(s, e)]
    peak = float(pr[s:e].max()); cut = ratio * peak
    pieces, cur = [], s
    i = s + min_len
    while i < e - min_len:
        if pr[i] < cut and pr[i] <= pr[i - 1] and pr[i] <= pr[i + 1] \
                and pr[cur:i].max() >= hi and pr[i + 1:e].max() >= hi:
            pieces.append((cur, i)); cur = i + 1; i = cur + min_len
            continue
        i += 1
    pieces.append((cur, e))
    return pieces


def _rel_edges(pr, s, e, rel, max_ext):
    """Move edges to where pr crosses rel * (run peak): shrink inside the run, or extend
    outward by at most max_ext frames. rel <= 0 disables."""
    if rel <= 0:
        return s, e
    t = rel * float(pr[s:e].max()); n = len(pr)
    if pr[s] >= t:
        lim = max(0, s - max_ext)
        while s > lim and pr[s - 1] >= t:
            s -= 1
    else:
        while s < e - 1 and pr[s] < t:
            s += 1
    if pr[e - 1] >= t:
        lim = min(n, e + max_ext)
        while e < lim and pr[e] >= t:
            e += 1
    else:
        while e > s + 1 and pr[e - 1] < t:
            e -= 1
    return s, e


def decode_v5(tr, p, fps, smoothed=None):
    """smoothed: optional {(c, sw): array} precomputed for speed.
    Decoder-2 keys (all optional; missing = off, so older configs decode identically):
      split_{g}: valley-split ratio, edge_{g}: relative-edge level,
      shon_{g} / shoff_{g}: constant onset / offset shift in seconds."""
    dur = tr["duration"]
    radius = int(round(p["snap"] * fps))
    max_ext = int(round(0.3 * fps)); min_piece = max(2, int(round(0.05 * fps)))
    n_fr = tr["cat"].shape[0]
    cands = []
    for c in range(NC):
        g = CAT_GROUP[c]
        sw = p[f"sw_{g}"]
        pr = smoothed[(c, sw)] if smoothed is not None else smooth_track(tr["cat"][:, c], sw)
        hi = p[f"thr_{g}"]; lo = max(0.01, hi - p["hyst"])
        split_r = p.get(f"split_{g}", 0.0); edge_r = p.get(f"edge_{g}", 0.0)
        sh_on = int(round(p.get(f"shon_{g}", 0.0) * fps)); sh_off = int(round(p.get(f"shoff_{g}", 0.0) * fps))
        evs = []
        for s0, e0 in _runs(pr >= lo):
            if pr[s0:e0].max() < hi:
                continue
            for s, e in _split_valleys(pr, s0, e0, split_r, hi, min_piece):
                s, e = _rel_edges(pr, s, e, edge_r, max_ext)
                if radius > 0:
                    on_tr = tr["onset"][:, c] if tr["onset"].ndim == 2 else tr["onset"]
                    of_tr = tr["offset"][:, c] if tr["offset"].ndim == 2 else tr["offset"]
                    ss = _snap(on_tr, s, radius, p["snap_min"])
                    es = _snap(of_tr, e, radius, p["snap_min"])
                    if es <= ss:
                        ss, es = s, e
                else:
                    ss, es = s, e
                ss = min(max(0, ss + sh_on), n_fr); es = min(max(0, es + sh_off), n_fr)
                if es > ss:
                    evs.append([ss, es])
        evs.sort()
        merged = []
        for s, e in evs:
            if merged and (s - merged[-1][1]) / fps <= p[f"gap_{g}"]:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])
        for s, e in merged:
            if (e - s) / fps >= p[f"mindur_{g}"]:
                cands.append((float(pr[s:max(e, s + 1)].mean()), s, e, c))
    cands.sort(key=lambda x: -x[0])
    kept = []
    for sc, s, e, c in cands:
        ok = True
        for _, ks, ke, _ in kept:
            inter = min(e, ke) - max(s, ks)
            if inter > 0 and inter / (max(e, ke) - min(s, ks)) >= p["nms_iou"]:
                ok = False; break
        if ok:
            kept.append((sc, s, e, c))
        if len(kept) >= p["max_events"]:
            break
    out = []
    for _, s, e, _ in kept:
        on, of = min(s / fps, dur), min(e / fps, dur)
        if of > on:
            out.append((round(on, 3), round(of, 3)))
    return sorted(out)


RUN_DECODER2 = False
DEC2_TAG = "v5a"
N_CFG2 = 40 if SMOKE else 400

if RUN_DECODER2:
    with open(f"{V5_DIR}/{DEC2_TAG}_val_tracks.pkl", "rb") as f:
        vt = pickle.load(f)
    with open(f"{V5_DIR}/{DEC2_TAG}_decoder.json") as f:
        dj = json.load(f)
    assert vt["hash"] == dj["hash"], "val tracks and decoder come from different checkpoints"
    fps2, TN2, TS2 = vt["fps"], vt["N"], vt["S"]
    TN2 = {k: v for k, v in TN2.items() if k in VALN_REFS}
    TS2 = {k: v for k, v in TS2.items() if k in VALS_REFS}
    SMN2 = precompute_smoothed(TN2, SW_SHORT, SW_LONG); SMS2 = precompute_smoothed(TS2, SW_SHORT, SW_LONG)
    old = dict(dj["decoder"])
    base2 = half_scores(old, TN2, TS2, fps2, SMN2, SMS2)
    log(f"DECODER-2 {DEC2_TAG}: current decoder A={base2['A']:.4f} B={base2['B']:.4f}")

    def sample_dec2(r):
        d = sample_dec(r) if r.random() < 0.5 else dict(old)       # half fresh, half around the current one
        if d is old or d == old:
            d = dict(old)
            for k in ("thr_short", "thr_long"):
                d[k] = float(np.clip(d[k] + r.normal(0, 0.05), 0.1, 0.9))
        for g in ("short", "long"):
            d[f"split_{g}"] = float(r.choice([0.0, 0.0, 0.5, 0.6, 0.7, 0.8]))
            d[f"edge_{g}"] = float(r.choice([0.0, 0.0, 0.3, 0.4, 0.5, 0.6]))
            d[f"shon_{g}"] = float(r.choice([0.0, 0.0, -0.04, -0.02, 0.02, 0.04]))
            d[f"shoff_{g}"] = float(r.choice([0.0, 0.0, -0.08, -0.05, -0.03, 0.03]))
        return d

    r2 = np.random.default_rng(SEED + 7)
    scored2 = []; t0 = time.time()
    for i in range(N_CFG2):
        d = sample_dec2(r2)
        res = score_sets(d, {"N": (TN2, VALN_REFS), "S": (TS2, VALS_REFS)}, fps2,
                         {**SMN2, **SMS2}, {"N": VALN_HALF["A"], "S": VALS_HALF["A"]})
        scored2.append((joint(res), i, d))
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{N_CFG2} ({time.time() - t0:.0f}s) best A {max(x[0] for x in scored2):.4f}")
    scored2.sort(key=lambda x: -x[0])
    best2, best2_s = None, base2
    for ja, _, d in scored2[:20]:
        hs = half_scores(d, TN2, TS2, fps2, SMN2, SMS2)
        if hs["A"] > base2["A"] and hs["B"] > base2["B"] and hs["B"] > best2_s["B"]:
            best2, best2_s = d, hs
    if best2 is None:
        log("DECODER-2: no config beat the current decoder on both halves -> nothing to submit")
    else:
        log(f"DECODER-2 selected: A={best2_s['A']:.4f} (+{best2_s['A'] - base2['A']:.4f}) "
            f"B={best2_s['B']:.4f} (+{best2_s['B'] - base2['B']:.4f}) {best2}")
        refsN2 = {k: parse_events(label_table["NoiseSubCategoryTimeStamp"][int(k)], label_table["NoiseCategory"][int(k)])
                  for k in TN2}
        refsS2 = {sid: [(s, e, CAT_IDX[c]) for s, e, c in SYNTH_VAL.items[sid]["events"]] for sid in TS2}
        for name, TR, RC in (("Val-N", TN2, refsN2), ("Val-S", TS2, refsS2)):
            _report(f"{name} dec2", RC, {k: decode_v5(TR[k], best2, fps2) for k in RC})`
        with open(f"{V5_DIR}/{DEC2_TAG}_decoder2.json", "w") as f:
            json.dump(dict(dj, decoder=best2, halves=best2_s, previous=old), f, indent=1)
        tp = f"{V5_DIR}/{DEC2_TAG}_test_tracks.pkl"
        with open(tp, "rb") as f:
            tt = pickle.load(f)
        assert tt["hash"] == dj["hash"], "cached test tracks come from a different checkpoint"
        preds2, durs2 = {}, {}
        for cid, tr in tt["tracks"].items():
            tr = {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}
            preds2[cid] = decode_v5(tr, best2, tt["fps"]); durs2[cid] = tr["duration"]
        zp = write_submission(preds2, SUB_DIR, f"submission_{DEC2_TAG}_dec2{'_SMOKE' if SMOKE else ''}.zip",
                              durs2, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
        ne = np.array([len(v) for v in preds2.values()])
        log(f"DECODER-2 test zip {zp}: events/clip {ne.mean():.2f}, empty {int((ne == 0).sum())}")


In [54]:
# =============================================================================
# FIXED-DECODER MODEL COMPARISON (v2) -- CPU only, no training, no decoder search.
#
# Judges a candidate model against V5-A with the decoder held FIXED at the V5-A
# decoder that produced 1.2047 on the leaderboard. FAILS CLOSED: any missing
# input, schema mismatch or coverage mismatch raises instead of producing a verdict.
#
# Predeclared accept rule (set before any candidate result was seen):
#   Val-N Combined gain >= MIN_GAIN on district half A AND on half B,
#   Val-N aggregate gain >= MIN_GAIN, and Val-S drop no worse than -MAX_S_DROP.
# Provenance: the fixed decoder must reproduce submission_v5a.zip exactly from the
# cached v5a test tracks, so it is provably the 1.2047 decoder.
# Output if accepted: submission/submission_<CAND>_v5adec.zip
# Needs: the diagnose cell (for _match) to have been run in this kernel.
# =============================================================================
RUN_FIXED_COMPARE = True
BASE_TAG = "v5a"
CAND_TAG = "v5c_smel"
MIN_GAIN = 0.005        # per natural half and aggregate
MAX_S_DROP = 0.02


def _load_pkl(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} missing: run training + decoder selection + test submission for that tag")
    with open(p, "rb") as f:
        return pickle.load(f)


def _check_val_schema(v, tag):
    for k in ("fps", "N", "S", "hash"):
        assert k in v, f"{tag} val pickle lacks '{k}' (keys: {list(v)})"
    tr = next(iter(v["N"].values()))
    assert {"cat", "onset", "offset", "duration"} <= set(tr), f"{tag} val track keys {list(tr)}"
    assert tr["cat"].ndim == 2 and tr["cat"].shape[1] == NC, f"{tag} cat shape {tr['cat'].shape}"


def _check_test_schema(t, tag):
    for k in ("fps", "tracks", "hash"):
        assert k in t, f"{tag} test pickle lacks '{k}' (keys: {list(t)})"
    tr = next(iter(t["tracks"].values()))
    assert {"cat", "onset", "offset", "duration"} <= set(tr), f"{tag} test track keys {list(tr)}"


def _test_tracks32(tag):
    """Test tracks exactly as decoded for the zip. Older runs cached them as float16,
    which shifts onset/offset snapping (argmax near-ties) -> regenerate in float32 once
    from the same checkpoint (GPU, ~3 min) and keep as <tag>_test_tracks32.pkl."""
    p32 = f"{V5_DIR}/{tag}_test_tracks32.pkl"
    if os.path.exists(p32):
        return _load_pkl(p32)
    t = _load_pkl(f"{V5_DIR}/{tag}_test_tracks.pkl")
    _check_test_schema(t, tag)
    if next(iter(t["tracks"].values()))["cat"].dtype == np.float32:
        return t
    with open(f"{V5_DIR}/{tag}_decoder.json") as f:
        ck_path = json.load(f)["checkpoint"]
    net, ck = v5_from_ckpt(ck_path)
    assert ck["hash"] == t["hash"], f"{tag}: checkpoint {ck_path} is not the one that produced the cached test tracks"
    T_win, fps = model_time_grid(net, device)
    log(f"FIXED-DEC regenerating float32 test tracks for {tag} from {os.path.basename(ck_path)} ...")
    out = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        w = load_test_wav(pth)
        out[cid] = tracks_v5(net, w, load_test_beats(cid, w.numel()), device, T_win, fps)
    res = {"fps": fps, "tracks": out, "hash": ck["hash"]}
    with open(p32, "wb") as f:
        pickle.dump(res, f)
    del net
    return res


def _f32(tr):
    return {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}


def _short_recall(tracks, refs_cat, dec, fps):
    missing = [k for k in refs_cat if k not in tracks]
    if missing:
        raise RuntimeError(f"short-event recall: {len(missing)} reference clips have no track, e.g. {missing[:3]}")
    tot = hit = 0
    for k, rc in refs_cat.items():
        pairs = _match([(s, e) for s, e, _ in rc], decode_v5(tracks[k], dec, fps))
        matched = {r for r, _ in pairs}
        for ri, (s, e, _) in enumerate(rc):
            if e - s < 0.3:
                tot += 1; hit += ri in matched
    return hit / max(tot, 1), tot


if RUN_FIXED_COMPARE:
    assert "_match" in globals(), "run the diagnose cell first (it defines _match)"
    with open(f"{V5_DIR}/{BASE_TAG}_decoder.json") as f:
        _bj = json.load(f)
    DEC_FIXED = dict(_bj["decoder"])
    dec_sha = hashlib.sha1(json.dumps(DEC_FIXED, sort_keys=True).encode()).hexdigest()[:12]
    V = {t: _load_pkl(f"{V5_DIR}/{t}_val_tracks.pkl") for t in (BASE_TAG, CAND_TAG)}
    TT = {t: _test_tracks32(t) for t in (BASE_TAG, CAND_TAG)}
    for t in V:
        _check_val_schema(V[t], t); _check_test_schema(TT[t], t)
    assert V[BASE_TAG]["hash"] == _bj["hash"] == TT[BASE_TAG]["hash"], "v5a val/test tracks and decoder disagree"
    assert abs(V[BASE_TAG]["fps"] - V[CAND_TAG]["fps"]) < 1e-9
    fpsx = V[BASE_TAG]["fps"]

    # ---- provenance: fixed decoder + cached v5a test tracks == submission_v5a.zip
    _bzip = f"{SUB_DIR}/submission_{BASE_TAG}{'_SMOKE' if SMOKE else ''}.zip"
    with zipfile.ZipFile(_bzip) as z:
        _sub = {json.loads(l)["clip_id"]: [(e["onset"], e["offset"]) for e in json.loads(l)["events"]]
                for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()}
    _redo = {cid: decode_v5(_f32(tr), DEC_FIXED, TT[BASE_TAG]["fps"]) for cid, tr in TT[BASE_TAG]["tracks"].items()}
    assert set(_sub) == set(_redo), f"{_bzip} and the cached v5a test tracks cover different clips"
    _same = lambda a, b: len(a) == len(b) and all(abs(x[0] - y[0]) < 1e-6 and abs(x[1] - y[1]) < 1e-6 for x, y in zip(a, b))
    _diff = [c for c in _sub if not _same(_sub[c], _redo[c])]
    if len(_diff) > 0.01 * len(_sub):
        raise RuntimeError(f"fixed decoder does not reproduce {_bzip}: {len(_diff)}/{len(_sub)} clips differ")
    log(f"FIXED-DEC decoder sha {dec_sha} reproduces {os.path.basename(_bzip)}: {len(_sub) - len(_diff)}/{len(_sub)} "
        f"clips identical" + (f" ({len(_diff)} differ: GPU non-determinism in regenerated tracks, <1% tolerated)" if _diff else ""))

    # ---- identical coverage, fail closed
    if not VALS_REFS:
        raise RuntimeError("Val-S references not loaded -- synthetic comparison would be skipped; refusing")
    for t in V:
        for name, refs in (("N", VALN_REFS), ("S", VALS_REFS)):
            miss = [k for k in refs if k not in V[t][name]]
            if miss:
                raise RuntimeError(f"{t}: {len(miss)} {name} validation clips missing from cached tracks, e.g. {miss[:3]}")
    TT_ids = {t: set(TT[t]["tracks"]) for t in TT}
    assert TT_ids[BASE_TAG] == TT_ids[CAND_TAG] and len(TT_ids[CAND_TAG]) == (len(TEST_FILES) if not SMOKE else len(TT_ids[CAND_TAG]))
    refsN = {k: parse_events(label_table["NoiseSubCategoryTimeStamp"][int(k)], label_table["NoiseCategory"][int(k)])
             for k in VALN_REFS}
    keys = {h: {"N": list(VALN_HALF[h]), "S": list(VALS_HALF[h])} for h in ("A", "B")}
    keys["all"] = {"N": list(VALN_REFS), "S": list(VALS_REFS)}
    for h in keys:
        assert keys[h]["N"] and keys[h]["S"], f"empty subset for {h}"
    log(f"FIXED-DEC coverage: Val-N {len(VALN_REFS)} (A {len(keys['A']['N'])}, B {len(keys['B']['N'])}), "
        f"Val-S {len(VALS_REFS)} (A {len(keys['A']['S'])}, B {len(keys['B']['S'])}), test {len(TT_ids[CAND_TAG])} -- identical for both")

    rows = {}
    for t in (BASE_TAG, CAND_TAG):
        r = {h: score_sets(DEC_FIXED, {"N": (V[t]["N"], VALN_REFS), "S": (V[t]["S"], VALS_REFS)}, fpsx, keys=keys[h])
             for h in keys}
        r["epc"] = np.mean([len(decode_v5(V[t]["N"][k], DEC_FIXED, fpsx)) for k in VALN_REFS])
        r["short"], n_short = _short_recall(V[t]["N"], refsN, DEC_FIXED, fpsx)
        rows[t] = r
        a = r["all"]
        log(f"FIXED-DEC [{t}] Val-N {a['N']['combined']:.4f} (F1 {a['N']['event_f1']:.4f} P {a['N']['precision']:.3f} "
            f"R {a['N']['recall']:.3f} Dice {a['N']['segment_dice']:.4f}) A {r['A']['N']['combined']:.4f} "
            f"B {r['B']['N']['combined']:.4f} | Val-S {a['S']['combined']:.4f} (F1 {a['S']['event_f1']:.4f} "
            f"Dice {a['S']['segment_dice']:.4f}) | pred/clip {r['epc']:.2f} | short<0.3s recall {r['short']:.3f} (n={n_short})")
    b, c = rows[BASE_TAG], rows[CAND_TAG]
    d = {h: c[h]["N"]["combined"] - b[h]["N"]["combined"] for h in ("A", "B", "all")}
    dS = c["all"]["S"]["combined"] - b["all"]["S"]["combined"]
    ok = d["A"] >= MIN_GAIN and d["B"] >= MIN_GAIN and d["all"] >= MIN_GAIN and dS >= -MAX_S_DROP
    log(f"FIXED-DEC verdict {CAND_TAG} vs {BASE_TAG} (decoder {dec_sha}, MIN_GAIN {MIN_GAIN}): dValN A {d['A']:+.4f} "
        f"B {d['B']:+.4f} all {d['all']:+.4f}, dValS {dS:+.4f} -> "
        + ("ACCEPT: submit the zip below." if ok else "REJECT: keep v5a; do not submit."))
    if ok:
        preds, durs = {}, {}
        for cid, tr in TT[CAND_TAG]["tracks"].items():
            tr = _f32(tr)
            preds[cid] = decode_v5(tr, DEC_FIXED, TT[CAND_TAG]["fps"]); durs[cid] = tr["duration"]
        zp = write_submission(preds, SUB_DIR, f"submission_{CAND_TAG}_v5adec{'_SMOKE' if SMOKE else ''}.zip",
                              durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
        ne = np.array([len(v) for v in preds.values()])
        log(f"FIXED-DEC zip {zp} (decoder {dec_sha}): test events/clip {ne.mean():.2f}, empty {int((ne == 0).sum())}")


AssertionError: checkpoint C:/Users/231b258/v5_work/v5a_student.pt does not match its stored hash

In [55]:
# =============================================================================
# CROSS-DECODE -- is the short-mel MODEL better, or just its DECODER? (CPU only)
#
# Scores every (tracks from run X) x (decoder from run Y) pair on the cached
# validation tracks, so the effect of swapping the model and the effect of
# swapping the decoder can be read separately.
# Run it in the same session where the DIAGNOSE cell works (cells 1-26 executed,
# so VALN, SYNTH_VAL, label_table, parse_events and decode_v5 all exist).
# =============================================================================
XD_TAGS = ("v5a", "v5c_smel")      # (reference run, challenger run)
W_N, W_S = 0.62, 0.38              # same selection formula as the notebook: J = 0.62*Val-N + 0.38*Val-S


def _load_run(tag):
    tp, dp = f"{V5_DIR}/{tag}_val_tracks.pkl", f"{V5_DIR}/{tag}_decoder.json"
    assert os.path.exists(tp) and os.path.exists(dp), f"{tag}: run its decoder selection first ({tp})"
    with open(tp, "rb") as f:
        vt = pickle.load(f)
    with open(dp) as f:
        dj = json.load(f)
    assert vt["hash"] == dj["hash"], f"{tag}: val tracks and decoder come from different checkpoints"
    return vt, dj["decoder"]


runs = {t: _load_run(t) for t in XD_TAGS}

refsN = {str(gi): parse_events(label_table["NoiseSubCategoryTimeStamp"][gi], label_table["NoiseCategory"][gi])
         for gi in (int(k) for k in VALN)}
refsS = ({sid: [(s, e, CAT_IDX[c]) for s, e, c in r["events"]] for sid, r in SYNTH_VAL.items.items()}
         if SYNTH_VAL is not None else {})


def _common(refs, split):
    ks = set(refs)
    for t in XD_TAGS:
        ks &= set(runs[t][0][split])
    return sorted(ks)


keysN, keysS = _common(refsN, "N"), _common(refsS, "S")
assert keysN, "no common Val-N clips across the two runs"
assert keysS, "no common Val-S clips -- make sure the E1 cell has loaded SYNTH_VAL"
plain = {"N": {k: [(s, e) for s, e, _ in refsN[k]] for k in keysN},
         "S": {k: [(s, e) for s, e, _ in refsS[k]] for k in keysS}}
keys = {"N": keysN, "S": keysS}


def _score(model_tag, dec_tag):
    vt, _ = runs[model_tag]
    dec = runs[dec_tag][1]
    out = {}
    for sp in ("N", "S"):
        pred = {k: decode_v5(vt[sp][k], dec, vt["fps"]) for k in keys[sp]}
        out[sp] = official_combined_score(plain[sp], pred)
        out[sp + "_epc"] = sum(len(v) for v in pred.values()) / len(keys[sp])
    out["J"] = W_N * out["N"]["combined"] + W_S * out["S"]["combined"]
    return out


grid = {(m, d): _score(m, d) for m in XD_TAGS for d in XD_TAGS}

log(f"CROSS-DECODE on {len(keysN)} Val-N + {len(keysS)} Val-S clips")
for (m, d), r in grid.items():
    log(f"  tracks {m:<9} + decoder {d:<9}: J {r['J']:.4f} | "
        f"Val-N {r['N']['combined']:.4f} (F1 {r['N']['event_f1']:.4f} Dice {r['N']['segment_dice']:.4f}, "
        f"{r['N_epc']:.2f} ev/clip) | "
        f"Val-S {r['S']['combined']:.4f} (F1 {r['S']['event_f1']:.4f} Dice {r['S']['segment_dice']:.4f}, "
        f"{r['S_epc']:.2f} ev/clip)")

a, b = XD_TAGS


def _effects(get):
    # model effect: swap a->b tracks, averaged over both decoders
    model_eff = 0.5 * ((get(grid[(b, a)]) - get(grid[(a, a)])) + (get(grid[(b, b)]) - get(grid[(a, b)])))
    # decoder effect: swap a->b decoder, averaged over both track sets
    dec_eff = 0.5 * ((get(grid[(a, b)]) - get(grid[(a, a)])) + (get(grid[(b, b)]) - get(grid[(b, a)])))
    return model_eff, dec_eff


log(f"EFFECT of swapping {a} -> {b}  (positive = {b} is better):")
for label, get in (("J    ", lambda r: r["J"]),
                   ("Val-N", lambda r: r["N"]["combined"]),
                   ("Val-S", lambda r: r["S"]["combined"])):
    me, de = _effects(get)
    log(f"  {label}: MODEL {me:+.4f} | DECODER {de:+.4f}")

[2026-10-10 09:44:23] CROSS-DECODE on 1367 Val-N + 1000 Val-S clips
[2026-10-10 09:44:23]   tracks v5a       + decoder v5a      : J 0.9782 | Val-N 0.9208 (F1 0.2771 Dice 0.6436, 1.50 ev/clip) | Val-S 1.0718 (F1 0.3724 Dice 0.6994, 1.51 ev/clip)
[2026-10-10 09:44:23]   tracks v5a       + decoder v5c_smel : J 0.9727 | Val-N 0.9082 (F1 0.2639 Dice 0.6443, 1.84 ev/clip) | Val-S 1.0780 (F1 0.3550 Dice 0.7230, 1.83 ev/clip)
[2026-10-10 09:44:23]   tracks v5c_smel  + decoder v5a      : J 0.9741 | Val-N 0.9135 (F1 0.2693 Dice 0.6442, 1.53 ev/clip) | Val-S 1.0729 (F1 0.3781 Dice 0.6948, 1.52 ev/clip)
[2026-10-10 09:44:23]   tracks v5c_smel  + decoder v5c_smel : J 0.9824 | Val-N 0.9106 (F1 0.2622 Dice 0.6484, 1.89 ev/clip) | Val-S 1.0994 (F1 0.3652 Dice 0.7342, 1.88 ev/clip)
[2026-10-10 09:44:23] EFFECT of swapping v5a -> v5c_smel  (positive = v5c_smel is better):
[2026-10-10 09:44:23]   J    : MODEL +0.0028 | DECODER +0.0014
[2026-10-10 09:44:23]   Val-N: MODEL -0.0024 | DECODER -0.0077
[2026-1

In [56]:
# =============================================================================
# EXPERIMENT A -- Track-level ensemble of V5-A and V5-C.
# Writes submission_ens_v5a_v5c_v1.zip only if the ensemble beats both singles
# on Val-N (both district halves) and does not drop Val-S by more than 0.02.
# =============================================================================
import pickle, json, os
import numpy as np

ENS_TAG = "ens_v5a_v5c_v1"
W_NAT, W_SYN = 0.62, 0.38

def _load_run(tag):
    tp = f"{V5_DIR}/{tag}_val_tracks.pkl"
    dp = f"{V5_DIR}/{tag}_decoder.json"
    assert os.path.exists(tp), f"missing {tp}"
    assert os.path.exists(dp), f"missing {dp}"
    with open(tp, "rb") as f: vt = pickle.load(f)
    with open(dp) as f:      dj = json.load(f)
    assert vt["hash"] == dj["hash"], f"{tag}: tracks/decoder hash mismatch"
    return vt, dj

vtA, djA = _load_run("v5a")
vtC, djC = _load_run("v5c_smel")

assert abs(vtA["fps"] - vtC["fps"]) < 1e-9, "fps mismatch"
fps = float(vtA["fps"])
print(f"fps={fps:.4f}  (T_win={int(round(fps*8.0))})")

keysN = sorted(set(vtA["N"]) & set(vtC["N"]))
keysS = sorted(set(vtA["S"]) & set(vtC["S"]))
print(f"common Val-N clips: {len(keysN)}  (A={sum(1 for k in keysN if k in VALN_HALF['A'])}, "
      f"B={sum(1 for k in keysN if k in VALN_HALF['B'])})")
print(f"common Val-S clips: {len(keysS)}")

def _as1d(x):
    x = np.asarray(x, dtype=np.float32)
    return x[:, 0] if x.ndim == 2 and x.shape[1] == 1 else x

def blend_tracks(tA, tC, w_a):
    """w_a = weight on V5-A, 1-w_a = weight on V5-C."""
    out = {}
    for k in ("cat", "onset", "offset"):
        a = _as1d(tA[k]); c = _as1d(tC[k])
        assert a.shape == c.shape, f"{k}: {a.shape} vs {c.shape}"
        out[k] = (w_a * a + (1.0 - w_a) * c).astype(np.float32)
    out["duration"] = float(tA["duration"])
    return out

def _score_halves(predsN, predsS):
    """Return dict of {A, B, N, S} combined scores."""
    res = {}
    for h in ("A", "B"):
        kN = [k for k in predsN if k in VALN_HALF[h]]
        kS = [k for k in predsS if k in VALS_HALF[h]]
        rN = official_combined_score({k: VALN_REFS[k] for k in kN},
                                     {k: predsN[k] for k in kN})
        rS = official_combined_score({k: VALS_REFS[k] for k in kS},
                                     {k: predsS[k] for k in kS})
        res[h] = W_NAT * rN["combined"] + W_SYN * rS["combined"]
        res[f"N_{h}"] = rN["combined"]; res[f"S_{h}"] = rS["combined"]
    res["N"] = official_combined_score(
        {k: VALN_REFS[k] for k in predsN}, predsN)["combined"]
    res["S"] = official_combined_score(
        {k: VALS_REFS[k] for k in predsS}, predsS)["combined"]
    res["J"] = W_NAT * res["N"] + W_SYN * res["S"]
    return res

def eval_blend(w_a, decoder):
    predsN = {k: decode_v5(blend_tracks(vtA["N"][k], vtC["N"][k], w_a), decoder, fps)
              for k in keysN}
    predsS = {k: decode_v5(blend_tracks(vtA["S"][k], vtC["S"][k], w_a), decoder, fps)
              for k in keysS}
    return _score_halves(predsN, predsS)

# --- Baseline singles (each with its OWN decoder) ---
scA = _score_halves(
    {k: decode_v5(vtA["N"][k], djA["decoder"], fps) for k in keysN},
    {k: decode_v5(vtA["S"][k], djA["decoder"], fps) for k in keysS})
scC = _score_halves(
    {k: decode_v5(vtC["N"][k], djC["decoder"], fps) for k in keysN},
    {k: decode_v5(vtC["S"][k], djC["decoder"], fps) for k in keysS})
print(f"V5-A      decoder=v5a      : J={scA['J']:.4f}  N_A={scA['N_A']:.4f} N_B={scA['N_B']:.4f} "
      f"| N={scA['N']:.4f} S={scA['S']:.4f}")
print(f"V5-C smel decoder=v5c_smel : J={scC['J']:.4f}  N_A={scC['N_A']:.4f} N_B={scC['N_B']:.4f} "
      f"| N={scC['N']:.4f} S={scC['S']:.4f}")

# --- Blend grid ---
GRID = [0.30, 0.40, 0.50, 0.60, 0.70]
for w_a in GRID:
    sc = eval_blend(w_a, djA["decoder"])   # use V5-A decoder (matches the 1.2047 LB probe)
    print(f"  blend w_a={w_a:.2f} w_c={1-w_a:.2f} dec=v5a : J={sc['J']:.4f}  "
          f"N_A={sc['N_A']:.4f} N_B={sc['N_B']:.4f} | N={sc['N']:.4f} S={sc['S']:.4f}")

# --- Pick best on half A, accept only if it wins on half B too ---
best_w, best_sc = None, None
for w_a in GRID:
    sc = eval_blend(w_a, djA["decoder"])
    if best_sc is None or sc["N_A"] > best_sc["N_A"]:
        best_w, best_sc = w_a, sc

ACC_N_MIN = 0.005   # per-half Val-N gain required
ACC_S_DROP_MAX = 0.02
wins = (best_sc["N_A"] > scA["N_A"] + ACC_N_MIN
        and best_sc["N_B"] > scA["N_B"] + ACC_N_MIN
        and best_sc["S"]   > scA["S"]   - ACC_S_DROP_MAX)
print(f"best blend w_a={best_w:.2f}: N_A={best_sc['N_A']:.4f} N_B={best_sc['N_B']:.4f} "
      f"S={best_sc['S']:.4f}   wins={wins}")

if not wins:
    print("EXPERIMENT A VERDICT: REJECT -- ensemble does not beat V5-A on both halves. "
          "Keep v5c_smel as your submission, skip to Experiment B.")
else:
    print(f"EXPERIMENT A VERDICT: ACCEPT (w_a={best_w:.2f}, decoder=v5a). Writing zip...")

    # --- Load test tracks (float32 needed for accurate snapping) ---
    with open(f"{V5_DIR}/v5a_test_tracks.pkl", "rb") as f: ttA = pickle.load(f)
    with open(f"{V5_DIR}/v5c_smel_test_tracks.pkl", "rb") as f: ttC = pickle.load(f)
    assert ttA["hash"] == djA["hash"], "v5a test tracks / decoder mismatch"
    assert ttC["hash"] == djC["hash"], "v5c_smel test tracks / decoder mismatch"
    common = sorted(set(ttA["tracks"]) & set(ttC["tracks"]))
    assert len(common) == EXPECTED_TEST_CLIPS, \
        f"test clips: {len(common)} vs expected {EXPECTED_TEST_CLIPS}"

    preds, durs = {}, {}
    for cid in common:
        tA = {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v)
              for k, v in ttA["tracks"][cid].items()}
        tC = {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v)
              for k, v in ttC["tracks"][cid].items()}
        blended = blend_tracks(tA, tC, best_w)
        preds[cid] = decode_v5(blended, djA["decoder"], fps)
        durs[cid] = blended["duration"]
    zp = write_submission(preds, SUB_DIR,
                          f"submission_{ENS_TAG}{'_SMOKE' if SMOKE else ''}.zip",
                          durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    print(f"WROTE {zp}")

    # --- Archive verification ---
    import zipfile
    with zipfile.ZipFile(zp) as z:
        lines = z.read("predictions.jsonl").decode().splitlines()
    assert len(lines) == EXPECTED_TEST_CLIPS, f"zip has {len(lines)} lines"
    ev_per_clip = [json.loads(l)["events"] for l in lines]
    n_empty = sum(1 for e in ev_per_clip if not e)
    print(f"zip verified: {len(lines)} clips, {n_empty} empty, "
          f"{sum(len(e) for e in ev_per_clip)} events")

fps=100.1250  (T_win=801)
common Val-N clips: 1367  (A=695, B=672)
common Val-S clips: 1000
V5-A      decoder=v5a      : J=0.9782  N_A=0.9576 N_B=0.8850 | N=0.9208 S=1.0718
V5-C smel decoder=v5c_smel : J=0.9824  N_A=0.9537 N_B=0.8701 | N=0.9106 S=1.0994
  blend w_a=0.30 w_c=0.70 dec=v5a : J=0.9773  N_A=0.9559 N_B=0.8816 | N=0.9181 S=1.0740
  blend w_a=0.40 w_c=0.60 dec=v5a : J=0.9774  N_A=0.9603 N_B=0.8819 | N=0.9204 S=1.0703
  blend w_a=0.50 w_c=0.50 dec=v5a : J=0.9770  N_A=0.9636 N_B=0.8752 | N=0.9186 S=1.0721
  blend w_a=0.60 w_c=0.40 dec=v5a : J=0.9772  N_A=0.9639 N_B=0.8780 | N=0.9202 S=1.0701
  blend w_a=0.70 w_c=0.30 dec=v5a : J=0.9791  N_A=0.9644 N_B=0.8810 | N=0.9220 S=1.0724
best blend w_a=0.70: N_A=0.9644 N_B=0.8810 S=1.0724   wins=False
EXPERIMENT A VERDICT: REJECT -- ensemble does not beat V5-A on both halves. Keep v5c_smel as your submission, skip to Experiment B.


In [60]:
import os, sys, importlib, glob

print("cwd         :", os.getcwd())
print("sys.exec   :", sys.executable)

CANDIDATES = [
    "C:/Users/231b258/atst_common.py",
    "C:/Users/231b258/Atst common.py",
    "C:/Users/231b258/atst_common (1).py",
    "C:/Users/231b258/Downloads/atst_common.py",
    os.path.expanduser("~/Downloads/atst_common.py"),
    os.path.expanduser("~/Desktop/atst_common.py"),
    os.path.expanduser("~/atst_common.py"),
    os.path.join(os.getcwd(), "atst_common.py"),
]
found = None
for p in CANDIDATES:
    ok = os.path.isfile(p)
    print(f"  {'FOUND   ' if ok else 'missing '} {p}")
    if ok and found is None:
        found = p

# Also glob just in case the name has odd characters
for pat in ["C:/Users/231b258/*common*.py",
            os.path.expanduser("~/Downloads/*common*.py"),
            os.path.expanduser("~/Desktop/*common*.py")]:
    hits = glob.glob(pat)
    for h in hits:
        print(f"  glob    {h}")
        if found is None:
            found = h

print()
print("RESULT:", "found at " + found if found else "NOT FOUND anywhere")

cwd         : C:\Users\231b258
sys.exec   : C:\Users\231b258\torch_gpu\Scripts\python.exe
  missing  C:/Users/231b258/atst_common.py
  missing  C:/Users/231b258/Atst common.py
  missing  C:/Users/231b258/atst_common (1).py
  FOUND    C:/Users/231b258/Downloads/atst_common.py
  FOUND    C:\Users\231b258/Downloads/atst_common.py
  missing  C:\Users\231b258/Desktop/atst_common.py
  missing  C:\Users\231b258/atst_common.py
  missing  C:\Users\231b258\atst_common.py
  glob    C:\Users\231b258/Downloads\atst_common.py

RESULT: found at C:/Users/231b258/Downloads/atst_common.py


In [61]:
import os, shutil, sys, importlib

SRC = "C:/Users/231b258/Downloads/atst_common.py"
DST = "C:/Users/231b258/atst_common.py"

assert os.path.isfile(SRC), f"source missing: {SRC}"
if not os.path.isfile(DST):
    shutil.copy2(SRC, DST)
    print(f"copied {SRC} -> {DST}")
else:
    print(f"already present: {DST}")

print("src size:", os.path.getsize(SRC), "bytes")
print("dst size:", os.path.getsize(DST), "bytes")
assert os.path.getsize(DST) > 10000, "copied file looks too small -- check it"

# Force a clean import
for m in list(sys.modules):
    if m == "atst_common" or m.startswith("atst_common."):
        del sys.modules[m]
if "C:/Users/231b258" not in sys.path:
    sys.path.insert(0, "C:/Users/231b258")
importlib.invalidate_caches()

from atst_common import (load_atst_encoder, ATSTFeatureExtractor, ATSTCache,
                         build_cache, verify_cache, atst_T_expected,
                         atst_rows_for_window, atst_rows_for_shifted_clip,
                         TOKEN_SAMPLES, TOKEN_SEC, MAX_TOKENS, EMBED_DIM,
                         sha256_file)
print("[ok] atst_common imported from:", sys.modules["atst_common"].__file__)
print(f"  TOKEN_SEC={TOKEN_SEC}  MAX_TOKENS={MAX_TOKENS}  EMBED_DIM={EMBED_DIM}")

copied C:/Users/231b258/Downloads/atst_common.py -> C:/Users/231b258/atst_common.py
src size: 28530 bytes
dst size: 28530 bytes
[ok] atst_common imported from: C:\Users/231b258\atst_common.py
  TOKEN_SEC=0.04  MAX_TOKENS=250  EMBED_DIM=768


In [62]:
# =============================================================================
# CELL B1 -- load ATST encoder, verify checkpoint hash, smoke-test the front-end.
# Requires: atst_common.py on sys.path, atst_as2M.ckpt on disk.
# =============================================================================
import sys, os, hashlib, time
sys.path.insert(0, "C:/Users/231b258")
from atst_common import (load_atst_encoder, ATSTFeatureExtractor, ATSTCache,
                         build_cache, verify_cache, atst_T_expected,
                         atst_rows_for_window, atst_rows_for_shifted_clip,
                         TOKEN_SAMPLES, TOKEN_SEC, MAX_TOKENS, EMBED_DIM,
                         sha256_file)

ATST_CKPT = "C:/Users/231b258/atst_as2M.ckpt"
ATST_CACHE_ROOT = f"{BASE_DIR}/atst_cache_v1"
ATST_TAG = "atst_cache_v1"

assert os.path.isfile(ATST_CKPT), ATST_CKPT
print(f"[ATST] ckpt sha256 = {sha256_file(ATST_CKPT)[:16]}...  size={os.path.getsize(ATST_CKPT)/1e6:.1f} MB")

# Load encoder (strict; raises on any missing keys, no silent fallback)
enc = load_atst_encoder(ATST_CKPT, device=device, freeze=True)
ext = ATSTFeatureExtractor(enc, device, n_last=1)
print(f"[ATST] loaded: dim={ext.dim}  token_sec={TOKEN_SEC}  max_tokens={MAX_TOKENS}")

# Smoke test: 1 clip from Val-N
_test_key = next(iter(VALN))
_w = VALN[_test_key]["wav"]
_feat = ext(_w)
print(f"[ATST] smoke: clip={_test_key} len={len(_w)/SR:.2f}s  feat.shape={_feat.shape}  "
      f"mean={_feat.mean():.3f} std={_feat.std():.3f} finite={np.isfinite(_feat).all()}")
_exp_T = atst_T_expected(ext.padded_len(_w.numel()))
assert _feat.shape == (_exp_T, EMBED_DIM), f"shape {_feat.shape} != {(_exp_T, EMBED_DIM)}"
assert np.isfinite(_feat).all()
print(f"[ATST] smoke OK (expected T={_exp_T})")

# Sanity: verify alignment constant used by V5
# V5 8s window = 200 ATST tokens; V5 4s stride = 100 ATST tokens.
assert int(round(8.0 / TOKEN_SEC)) == 200, "8 s / 40 ms != 200"
assert int(round(4.0 / TOKEN_SEC)) == 100, "4 s / 40 ms != 100"
print("[ATST] alignment constants verified: 200 tokens/window, 100 tokens/stride")

[ATST] ckpt sha256 = 8739265cdc6f5990...  size=686.7 MB
[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstream_finetuned': 138, 'pretrainedsed_wrapper': 0}
   hyper_parameters: {'learning_rate': 3.125, 'max_epochs': 11, 'niter_per_epoch': 1208, 'warmup_epochs': 2, 'num_labels': 527, 'multi

In [64]:
# =============================================================================
# CELL B2 -- build ATST feature caches. Resumable.
#   cache kinds: 'nat' (global dataset index), 'synth' (synthetic id), 'test' (clip id)
# Run this in the background; ~2-4 h for the full training set on one GPU.
# =============================================================================
ATST_META = {
    "sha256": enc.meta["sha256"],
    "mel_variant": "pretrain",
    "n_last": 1,
    "chunk": MAX_TOKENS,
    "hop": 125,
    "embed_dim": ext.dim,
}
cache = ATSTCache(ATST_CACHE_ROOT, ATST_TAG, ATST_META)

# --- Storage estimate (do this BEFORE a long extraction) ---
def _est_gb(n_clips, avg_sec):
    return n_clips * avg_sec / TOKEN_SEC * ext.dim * 2 / 1e9

total_train_sec = sum(
    (len(VALN[k]["wav"]) / SR) for k in VALN)   # just to sanity-check the format
print(f"[ATST] Val-N total: {total_train_sec:.1f}s")
print(f"[ATST] storage estimate: train ~{_est_gb(len(train_idx), 6.0):.1f} GB fp16, "
      f"val ~{_est_gb(len(VAL_GOLD), 6.0):.1f} GB, test ~{_est_gb(5517, 8.0):.1f} GB")

BUILD_VAL_N    = True    # ~200 MB, 1-2 min
BUILD_VAL_S    = True    # ~150 MB, 1-2 min
BUILD_TRAIN    = False    # ~20 GB, 2-4 h  -- set False if short on time
BUILD_TEST     = False    # ~1 GB, 5-15 min

if BUILD_VAL_N:
    build_cache(
        items=VAL_GOLD,
        get_wav=lambda gi: _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS),
        cache=cache, kind="nat", extractor=ext,
        key_of=lambda gi: gi, log=print, max_new=None, log_every=100)
    verify_cache(cache, "nat", VAL_GOLD,
                 n_samples_of=lambda gi: len(_load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS)),
                 full=True)

if BUILD_VAL_S:
    build_cache(
        items=sorted(VALS),
        get_wav=lambda sid: VALS[sid]["wav"],
        cache=cache, kind="synth", extractor=ext,
        key_of=lambda sid: sid, log=print, max_new=None, log_every=100)
    verify_cache(cache, "synth", sorted(VALS),
                 n_samples_of=lambda sid: len(VALS[sid]["wav"]),
                 full=True)

if BUILD_TRAIN:
    build_cache(
        items=train_idx,
        get_wav=lambda gi: _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS),
        cache=cache, kind="nat", extractor=ext,
        key_of=lambda gi: gi, log=print, max_new=None, log_every=500)
    print("[ATST] train cache build finished")

if BUILD_TEST:
    # Test wavs are loaded from disk; keys are clip ids.
    def _test_wav_by_cid(cid):
        pth = next(p for p in TEST_FILES if os.path.splitext(os.path.basename(p))[0] == cid)
        return load_test_wav(pth)   # defined in the V5 E0 cell
    test_ids = sorted(os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES)
    build_cache(
        items=test_ids,
        get_wav=_test_wav_by_cid,
        cache=cache, kind="test", extractor=ext,
        key_of=lambda cid: cid, log=print, max_new=None, log_every=500)
    verify_cache(cache, "test", test_ids,
                 n_samples_of=lambda cid: max(len(_test_wav_by_cid(cid)), SR),
                 full=True)

print("[ATST] cache ready:", ATST_CACHE_ROOT)

[ATST] Val-N total: 8091.4s
[ATST] storage estimate: train ~18.5 GB fp16, val ~0.3 GB, test ~1.7 GB
  [atst-cache:nat] done {'kind': 'nat', 'new': 0, 'skipped': 1367, 'seconds': 0.0, 'fp16_min_cosine': 1.0, 'fp16_max_abs_err': 0.0}
  [atst-cache:nat] verified 1367 entries (shape+finite)
  [atst-cache:synth] done {'kind': 'synth', 'new': 0, 'skipped': 1000, 'seconds': 0.0, 'fp16_min_cosine': 1.0, 'fp16_max_abs_err': 0.0}
  [atst-cache:synth] verified 1000 entries (shape+finite)
[ATST] cache ready: C:/Users/231b258/atst_cache_v1


In [65]:
# =============================================================================
# CELL B2b -- build the remaining ATST caches: 'nat' for train_idx, 'test' for
# the 5517 test clips. Val-N ('nat') and Val-S ('synth') are already built.
# =============================================================================
import os, time

# The cache object from Cell B2 is already bound. Confirm meta compatibility.
assert os.path.isdir(ATST_CACHE_ROOT), f"missing cache root {ATST_CACHE_ROOT}"
with open(f"{ATST_CACHE_ROOT}/meta.json") as f:
    _m = json.load(f)
assert _m["sha256"] == enc.meta["sha256"], "cache was built with a different ATST ckpt"
print(f"[ATST] cache meta OK: tag={_m['tag']} embed_dim={_m['embed_dim']}")

# ---- Train cache: global indices in train_idx ----
BUILD_TRAIN = True
if BUILD_TRAIN:
    _n_missing = sum(1 for gi in train_idx if not cache.has("nat", int(gi)))
    print(f"[ATST] train cache: {len(train_idx) - _n_missing}/{len(train_idx)} present, "
          f"building {_n_missing} ...")
    if _n_missing > 0:
        build_cache(
            items=train_idx,
            get_wav=lambda gi: _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS),
            cache=cache, kind="nat", extractor=ext,
            key_of=lambda gi: gi, log=print, max_new=None, log_every=500)

# ---- Test cache: keys are clip IDs (strings) ----
BUILD_TEST = True
if BUILD_TEST:
    def _test_wav_by_cid(cid):
        pth = next(p for p in TEST_FILES
                   if os.path.splitext(os.path.basename(p))[0] == cid)
        return load_test_wav(pth)   # defined in the V5 E0 cell
    test_ids = sorted(os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES)
    _n_missing = sum(1 for cid in test_ids if not cache.has("test", cid))
    print(f"[ATST] test cache: {len(test_ids) - _n_missing}/{len(test_ids)} present, "
          f"building {_n_missing} ...")
    if _n_missing > 0:
        build_cache(
            items=test_ids, get_wav=_test_wav_by_cid,
            cache=cache, kind="test", extractor=ext,
            key_of=lambda cid: cid, log=print, max_new=None, log_every=500)

# ---- Exhaustive verify ----
verify_cache(cache, "nat", train_idx,
             n_samples_of=lambda gi: len(_load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS)),
             full=False)   # presence-only (fast); set full=True if you want shape re-check
verify_cache(cache, "test", sorted(os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES),
             n_samples_of=lambda cid: max(len(_test_wav_by_cid(cid)), SR),
             full=False)
print("[ATST] all caches ready")

[ATST] cache meta OK: tag=atst_cache_v1 embed_dim=768
[ATST] train cache: 30551/80164 present, building 49613 ...
  [atst-cache:nat] 500 new, 30551 skipped, 7s
  [atst-cache:nat] 1000 new, 30551 skipped, 15s
  [atst-cache:nat] 1500 new, 30551 skipped, 22s
  [atst-cache:nat] 2000 new, 30551 skipped, 30s
  [atst-cache:nat] 2500 new, 30551 skipped, 38s
  [atst-cache:nat] 3000 new, 30551 skipped, 46s
  [atst-cache:nat] 3500 new, 30551 skipped, 53s
  [atst-cache:nat] 4000 new, 30551 skipped, 61s
  [atst-cache:nat] 4500 new, 30551 skipped, 68s
  [atst-cache:nat] 5000 new, 30551 skipped, 76s
  [atst-cache:nat] 5500 new, 30551 skipped, 83s
  [atst-cache:nat] 6000 new, 30551 skipped, 91s
  [atst-cache:nat] 6500 new, 30551 skipped, 98s
  [atst-cache:nat] 7000 new, 30551 skipped, 106s
  [atst-cache:nat] 7500 new, 30551 skipped, 115s
  [atst-cache:nat] 8000 new, 30551 skipped, 123s
  [atst-cache:nat] 8500 new, 30551 skipped, 131s
  [atst-cache:nat] 9000 new, 30551 skipped, 139s
  [atst-cache:nat] 

In [66]:
# =============================================================================
# CELL B3 -- V5DatasetATST: same rows as V5Dataset, plus per-sample ATST rows
# aligned to the SAME 8 s window the parent picked for BEATs.
# Requires the V5Dataset.__getitem__ patch to be active (from the earlier cell).
# =============================================================================
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import DataLoader

CANON_ATST_T = int(round(W_SEC / TOKEN_SEC))       # 200
assert CANON_ATST_T == 200, f"unexpected: {CANON_ATST_T}"
assert PATCH_SAMPLES == 4 * TOKEN_SAMPLES, (PATCH_SAMPLES, TOKEN_SAMPLES)


class V5DatasetATST(V5Dataset):
    """Adds an 'atst' float32 tensor (CANON_ATST_T, 768) sliced at the same
    8 s window the parent chose for BEATs, plus 'atst_valid' (int) = number of
    real (non-pad) ATST rows in that slice."""

    def __init__(self, *args, atst_cache=None, **kwargs):
        super().__init__(*args, **kwargs)
        assert atst_cache is not None, "atst_cache required"
        self.atst_cache = atst_cache

    def _atst_kind_key(self, kind, key):
        # 'nat' rows are keyed by int(global index); 'syn' rows by string sid.
        if kind == "nat":
            return "nat", int(key)
        if kind == "syn":
            return "synth", key
        raise ValueError(f"unknown row kind {kind!r}")

    def __getitem__(self, i):
        d = super().__getitem__(i)
        kind, key, _tier = self.rows[i]
        ckind, ckey = self._atst_kind_key(kind, key)
        atst_full = self.atst_cache.load(ckind, ckey).astype(np.float32)   # (T_full, 768)

        shift_samples = int(d["shift_samples"])
        whole_clip = bool(d["whole_clip"])
        assert shift_samples % TOKEN_SAMPLES == 0, \
            f"shift {shift_samples} not on 40 ms grid"

        if whole_clip:
            # short clip: full track placed at k = shift_samples / TOKEN_SAMPLES
            k = shift_samples // TOKEN_SAMPLES
            rows, n_real = atst_rows_for_shifted_clip(atst_full, k, CANON_ATST_T)
        else:
            # long clip: window slides along the source clip
            rows, n_real = atst_rows_for_window(atst_full, shift_samples, CANON_ATST_T)

        d["atst"] = torch.from_numpy(np.ascontiguousarray(rows, np.float32))
        d["atst_valid"] = int(n_real)
        return d


def collate_v5_atst(batch):
    base = collate_v5(batch)
    base["atst"] = torch.stack([b["atst"] for b in batch])
    base["atst_valid"] = torch.tensor([b["atst_valid"] for b in batch], dtype=torch.long)
    return base

print("V5DatasetATST + collate_v5_atst defined")

V5DatasetATST + collate_v5_atst defined


In [68]:
# =============================================================================
# ENSURE V5Dataset.__getitem__ PATCH -- safe to re-run any time.
# If the V5 core cell is ever re-run, re-run this cell immediately after.
# =============================================================================
import inspect as _inspect, random as _random
import numpy as _np, torch as _torch, torch.nn.functional as _F


def _v5dataset_getitem_with_shift(self, i):
    kind, key, tier = self.rows[i]
    wav, bf, evs, cats = self._load(kind, key)
    n = wav.numel()
    if n <= WIN_SAMPLES:
        k_max = (WIN_SAMPLES - n) // PATCH_SAMPLES
        k = (_random.randint(0, k_max)
             if (self.training and k_max > 0 and _random.random() < self.shift_prob)
             else 0)
        shift = k * PATCH_SAMPLES
        chunk = _F.pad(wav, (shift, WIN_SAMPLES - n - shift))
        rows = bf[:CANON_T - k]
        n_real = rows.shape[0]
        rows = _np.concatenate(
            [_np.zeros((k, bf.shape[1]), _np.float32),
             rows,
             _np.zeros((CANON_T - k - n_real, bf.shape[1]), _np.float32)],
            0,
        )
        vs = shift / SR
        ve = (shift + n) / SR
        ghi = min(ve, (k + n_real) * BEATS_PATCH_HOP_SEC)
        crop_evs = [(s + vs, e + vs, c) for s, e, c in evs]
        whole = True
        shift_samples = int(shift)
    else:
        k_max = (n - WIN_SAMPLES) // PATCH_SAMPLES
        if evs and _random.random() >= self.bg_crop_prob:
            s0, e0, _ = _random.choice(evs)
            centre = 0.5 * (s0 + e0) + _random.uniform(-self.jitter, self.jitter)
            k = int(round((centre * SR - WIN_SAMPLES / 2) / PATCH_SAMPLES))
        else:
            k = _random.randint(0, k_max)
        k = max(0, min(k_max, k))
        start = k * PATCH_SAMPLES
        chunk = wav[start:start + WIN_SAMPLES]
        rows, n_real = beats_rows_for_window(bf, start)
        cs = start / SR
        vs, ve = 0.0, W_SEC
        ghi = min(ve, n_real * BEATS_PATCH_HOP_SEC)
        crop_evs = []
        for s, e, c in evs:
            ns, ne = max(0.0, s - cs), min(W_SEC, e - cs)
            if ne > ns:
                crop_evs.append((ns, ne, c))
        whole = False
        shift_samples = int(start)

    if self.training:
        rows = self._mask_beats(rows)

    z = _np.zeros(NC, _np.float32)
    m = _np.ones(NC, _np.float32)
    in_crop = {c for s, e, c in crop_evs if (e - s) >= 0.05}
    if tier in (T_GOLD, T_SYNTH):
        for c in in_crop:
            z[c] = 1.0
    elif tier == T_SILVER:
        clip_c = {CAT_IDX[c] for c in cats} | {c for _, _, c in evs}
        if whole:
            for c in clip_c:
                z[c] = 1.0
        else:
            strong_in = {c for s, e, c in crop_evs if (e - s) >= 0.3}
            for c in clip_c:
                if c in strong_in:
                    z[c] = 1.0
                else:
                    m[c] = 0.0
    else:
        clip_c = {CAT_IDX[c] for c in cats}
        for c in clip_c:
            if whole:
                z[c] = 1.0
            else:
                m[c] = 0.0

    return {
        "waveform": chunk.float(),
        "beats": _torch.from_numpy(_np.ascontiguousarray(rows, _np.float32)),
        "vspan": (vs, ve),
        "gspan": (vs, ghi),
        "events": crop_evs,
        "z": _torch.from_numpy(z),
        "zmask": _torch.from_numpy(m),
        "tier": tier,
        "key": f"{kind}:{key}",
        "shift_samples": shift_samples,   # NEW
        "whole_clip": whole,              # NEW
    }


# --- install only if not already installed ---
_try_src = None
try:
    _try_src = _inspect.getsource(V5Dataset.__getitem__)
except (TypeError, OSError):
    pass

_already = (_try_src is not None
            and "shift_samples" in _try_src
            and "whole_clip" in _try_src)

if _already:
    print("[patch] already active on V5Dataset.__getitem__")
else:
    V5Dataset.__getitem__ = _v5dataset_getitem_with_shift
    print("[patch] INSTALLED on V5Dataset.__getitem__")

# --- verify now ---
_chk = _inspect.getsource(V5Dataset.__getitem__)
assert "shift_samples" in _chk, "patch did not stick"
assert "whole_clip" in _chk, "patch did not stick"
print("[patch] verified: source contains shift_samples and whole_clip")

[patch] INSTALLED on V5Dataset.__getitem__
[patch] verified: source contains shift_samples and whole_clip


In [73]:
# =============================================================================
# RESUME synthetic-train ATST cache build. Skips existing files.
# =============================================================================
import os, time
import numpy as np

assert SYNTH_TRAIN is not None, "SYNTH_TRAIN missing -- re-run the V5 E1 cell"

_all_sids = sorted(SYNTH_TRAIN.items)
_missing = [sid for sid in _all_sids if not cache.has("synth", sid)]
print(f"[atst-cache:synth] SYNTH_TRAIN total={len(_all_sids)}  "
      f"present={len(_all_sids)-len(_missing)}  missing={len(_missing)}")

if _missing:
    # Quick storage estimate on a sample of the missing ones
    _T_sum = 0
    for sid in _missing[:200]:
        _r = SYNTH_TRAIN.items[sid]
        _T_sum += int(np.ceil(_r["duration"] / TOKEN_SEC))
    _T_avg = _T_sum / max(1, min(200, len(_missing)))
    _est_gb = _T_avg * len(_missing) * EMBED_DIM * 2 / 1e9
    print(f"[atst-cache:synth] estimated additional disk: ~{_est_gb:.2f} GB fp16 "
          f"(avg {_T_avg:.1f} tokens/clip)")

    def _get_wav_synth(sid):
        return SYNTH_TRAIN.wav(sid)

    build_cache(
        items=_missing, get_wav=_get_wav_synth,
        cache=cache, kind="synth", extractor=ext,
        key_of=lambda sid: sid, log=print, max_new=None, log_every=500,
    )

# Presence check for the full SYNTH_TRAIN set (fast; no shape re-check)
verify_cache(cache, "synth", _all_sids,
             n_samples_of=lambda sid: max(len(SYNTH_TRAIN.wav(sid)), SR),
             full=False)
print(f"[atst-cache:synth] ready: {len(_all_sids)} synthetic train entries")

[atst-cache:synth] SYNTH_TRAIN total=24000  present=6706  missing=17294
[atst-cache:synth] estimated additional disk: ~4.28 GB fp16 (avg 161.0 tokens/clip)
  [atst-cache:synth] 500 new, 0 skipped, 8s
  [atst-cache:synth] 1000 new, 0 skipped, 17s
  [atst-cache:synth] 1500 new, 0 skipped, 25s
  [atst-cache:synth] 2000 new, 0 skipped, 33s
  [atst-cache:synth] 2500 new, 0 skipped, 41s
  [atst-cache:synth] 3000 new, 0 skipped, 50s
  [atst-cache:synth] 3500 new, 0 skipped, 58s
  [atst-cache:synth] 4000 new, 0 skipped, 66s
  [atst-cache:synth] 4500 new, 0 skipped, 74s
  [atst-cache:synth] 5000 new, 0 skipped, 82s
  [atst-cache:synth] 5500 new, 0 skipped, 90s
  [atst-cache:synth] 6000 new, 0 skipped, 98s
  [atst-cache:synth] 6500 new, 0 skipped, 106s
  [atst-cache:synth] 7000 new, 0 skipped, 114s
  [atst-cache:synth] 7500 new, 0 skipped, 123s
  [atst-cache:synth] 8000 new, 0 skipped, 131s
  [atst-cache:synth] 8500 new, 0 skipped, 139s
  [atst-cache:synth] 9000 new, 0 skipped, 147s
  [atst-cach

In [74]:
# =============================================================================
# PREFLIGHT -- confirm every global this experiment needs is in scope.
# =============================================================================
import os, json, math, time, pickle, hashlib
from collections import defaultdict
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast

_required = [
    "V5_DIR", "SUB_DIR", "EXPECTED_TEST_CLIPS", "SMOKE",
    "VALN", "VALN_REFS", "VALN_HALF",
    "VALS", "VALS_REFS", "VALS_HALF",
    "VAL_GOLD", "full_dataset", "label_table", "TEST_FILES",
    "SYNTH_TRAIN", "SYNTH_VAL",
    "cache", "enc", "ext",
    "SR", "W_SEC", "WIN_SAMPLES", "CANON_T", "PATCH_SAMPLES",
    "TOKEN_SAMPLES", "TOKEN_SEC", "EMBED_DIM", "MAX_TOKENS",
    "CATS", "CAT_IDX", "NC",
    "T_GOLD", "T_SYNTH",
    "CFG_V5", "TRAIN_ROWS", "canonical_tags",
    "make_tier_sampler", "collate_v5", "V5Dataset",
    "V5DatasetATST", "collate_v5_atst",
    "atst_rows_for_window", "atst_rows_for_shifted_clip",
    "mel_v5", "VaaniSEDV5", "new_v5", "v5_from_ckpt",
    "v5_losses", "weak_augment", "update_teacher", "freeze_model",
    "select_decoder", "selection_J", "score_sets",
    "official_combined_score", "decode_v5", "write_submission",
    "model_time_grid", "tracks_v5", "state_hash",
    "grid_window_starts", "beats_rows_for_window",
    "load_test_wav", "load_test_beats", "build_test_beats_cache",
    "set_seed", "SEED", "log",
]
_missing = [n for n in _required if n not in globals()]
assert not _missing, "Missing globals:\n  " + "\n  ".join(_missing)

# Confirm the V5Dataset patch is still active (required by V5DatasetATST).
import inspect
_src = inspect.getsource(V5Dataset.__getitem__)
assert "shift_samples" in _src and "whole_clip" in _src, \
    "V5Dataset.__getitem__ patch is not active -- re-run the patch cell first"

CANON_ATST_T = int(round(W_SEC / TOKEN_SEC))
assert CANON_ATST_T == 200, CANON_ATST_T

print(f"[preflight] OK  T_win=CANON_ATST_T={CANON_ATST_T}  "
      f"cache={cache.root}  detector targets v5c_smel")

[preflight] OK  T_win=CANON_ATST_T=200  cache=C:/Users/231b258/atst_cache_v1  detector targets v5c_smel


In [75]:
# =============================================================================
# B3-VERIFY -- one natural + one synthetic sample through V5DatasetATST.
# =============================================================================
_ds_check = V5DatasetATST(
    rows=[("nat", int(VAL_GOLD[0]), T_GOLD)],
    full_dataset=full_dataset, label_table=label_table,
    synth_store=SYNTH_TRAIN, training=True, shift_prob=0.8, atst_cache=cache,
)
_s = _ds_check[0]
print("nat:", "beats", tuple(_s["beats"].shape),
      "atst", tuple(_s["atst"].shape),
      "valid", _s["atst_valid"], "shift", _s["shift_samples"],
      "whole", _s["whole_clip"])
assert _s["beats"].shape == (CANON_T, 768)
assert _s["atst"].shape == (CANON_ATST_T, EMBED_DIM)
assert 0 <= _s["atst_valid"] <= CANON_ATST_T
if _s["atst_valid"] > 1:
    assert _s["atst"][:_s["atst_valid"]].std() > 1e-3, "degenerate ATST real region"

_sid = next(iter(SYNTH_TRAIN.items))
_ds_syn = V5DatasetATST(
    rows=[("syn", _sid, T_SYNTH)],
    full_dataset=full_dataset, label_table=label_table,
    synth_store=SYNTH_TRAIN, training=True, shift_prob=0.8, atst_cache=cache,
)
_ss = _ds_syn[0]
print("syn:", "atst", tuple(_ss["atst"].shape),
      "valid", _ss["atst_valid"], "shift", _ss["shift_samples"],
      "whole", _ss["whole_clip"])
assert _ss["atst"].shape == (CANON_ATST_T, EMBED_DIM)
assert _ss["whole_clip"] is True
print("B3 verify OK")

nat: beats (50, 768) atst (200, 768) valid 95 shift 28160 whole True
syn: atst (200, 768) valid 149 shift 10240 whole True
B3 verify OK


In [76]:
# =============================================================================
# B4 -- VaaniSEDV5_ATST: V5 + zero-init gated ATST residual.
# =============================================================================
class VaaniSEDV5_ATST(VaaniSEDV5):
    def __init__(self, n_fine, atst_dim=768, **kwargs):
        super().__init__(n_fine, **kwargs)
        self.atst_proj = nn.Linear(atst_dim, self.proj_dim)
        self.atst_gate = nn.Linear(self.proj_dim * 2, self.proj_dim)
        nn.init.zeros_(self.atst_proj.weight); nn.init.zeros_(self.atst_proj.bias)
        nn.init.zeros_(self.atst_gate.weight); nn.init.zeros_(self.atst_gate.bias)

    def new_keys(self):
        k = super().new_keys()
        k |= {n for n in self.state_dict() if n.startswith("atst_")}
        return k

    def forward(self, waveform, beats, gate_span, mel=None, atst=None):
        if waveform.ndim == 3:
            waveform = waveform.squeeze(1)
        bf = self.beats_proj(beats)
        if mel is None:
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        if self.short_mel is not None:
            fdy = fdy + self.short_mel(waveform, T).to(fdy.dtype)
        bf = F.interpolate(bf.transpose(1, 2), size=T, mode="linear",
                           align_corners=False).transpose(1, 2)
        bf = self.feat_drop(bf); fdy = self.feat_drop(fdy)
        if atst is not None:
            af = self.atst_proj(atst)
            af = F.interpolate(af.transpose(1, 2), size=T, mode="linear",
                               align_corners=False).transpose(1, 2)
            af = self.feat_drop(af)
            gate_a = torch.sigmoid(self.atst_gate(torch.cat([bf, af], dim=-1)))
            bf = bf + gate_a * af
        gate = self.fuse_gate(torch.cat([bf, fdy], dim=-1))
        centres = (torch.arange(T, device=waveform.device, dtype=torch.float32) + 0.5) * (W_SEC / T)
        gspan = gate_span.float()
        bvalid = (centres[None] >= gspan[:, :1]) & (centres[None] < gspan[:, 1:2])
        gate = gate * bvalid.to(gate.dtype).unsqueeze(-1)
        x = self.fuse_norm(gate * bf + (1.0 - gate) * fdy)
        x, _ = self.gru(x)
        x = self.temporal_refine(self.gru_norm(x))
        if self.per_cat_boundary:
            on, off = self.cat_onset_head(x), self.cat_offset_head(x)
        else:
            on, off = self.event_onset_head(x)[..., 0], self.event_offset_head(x)[..., 0]
        return {"cat_logits": self.cat_frame_head(x),
                "onset_logits": on, "offset_logits": off,
                "attn_logits": self.cat_attn(x.transpose(1, 2)).transpose(1, 2),
                "T": T}
print("VaaniSEDV5_ATST defined")

VaaniSEDV5_ATST defined


In [77]:
# =============================================================================
# tracks_v5_atst -- identical to V5's tracks_v5 but slices ATST per window.
# Must be defined before B5 (training uses it for periodic eval).
# =============================================================================
@torch.no_grad()
def tracks_v5_atst(net, wav, beats_full, atst_full_t, device, T_win, fps, max_batch=8):
    net.eval()
    n = wav.numel(); dur = n / SR
    starts = grid_window_starts(n)
    total_frames = int(round(dur * fps)) + T_win + 1
    nb_ = NC if getattr(net, "per_cat_boundary", False) else 1
    acc = np.zeros((total_frames, NC + 2 * nb_), np.float64)
    cnt = np.zeros(total_frames)
    for b0 in range(0, len(starts), max_batch):
        bs = starts[b0:b0 + max_batch]
        chunks, rows_b, gsp, atst_b = [], [], [], []
        for s0 in bs:
            ch = wav[s0:s0 + WIN_SAMPLES]
            real = ch.numel()
            if real < WIN_SAMPLES:
                ch = F.pad(ch, (0, WIN_SAMPLES - real))
            r, n_real = beats_rows_for_window(beats_full, s0)
            k0 = s0 // TOKEN_SAMPLES
            ar = atst_full_t[0, k0:k0 + CANON_ATST_T]
            if ar.shape[0] < CANON_ATST_T:
                ar = F.pad(ar, (0, 0, 0, CANON_ATST_T - ar.shape[0]))
            chunks.append(ch)
            rows_b.append(torch.from_numpy(np.ascontiguousarray(r, np.float32)))
            gsp.append((0.0, min(real / SR, n_real * BEATS_PATCH_HOP_SEC)))
            atst_b.append(ar.cpu())
        out = net(torch.stack(chunks).to(device),
                  torch.stack(rows_b).to(device),
                  torch.tensor(gsp, dtype=torch.float32, device=device),
                  atst=torch.stack(atst_b).to(device))
        on_, of_ = out["onset_logits"].float(), out["offset_logits"].float()
        if on_.ndim == 2:
            on_, of_ = on_[..., None], of_[..., None]
        assert on_.shape[-1] == nb_
        lg = torch.cat([out["cat_logits"].float(), on_, of_], -1).cpu().numpy()
        for j, s0 in enumerate(bs):
            w = s0 / SR
            off = int(round(w * fps))
            end = min(off + T_win, total_frames); L = end - off
            acc[off:end] += lg[j, :L]
            cnt[off:end] += 1
    vl = min(int(round(dur * fps)) + 1, total_frames)
    m = acc[:vl] / np.maximum(cnt[:vl, None], 1e-6)
    pr = 1.0 / (1.0 + np.exp(-m))
    on_p, of_p = pr[:, NC:NC + nb_], pr[:, NC + nb_:NC + 2 * nb_]
    if nb_ == 1:
        on_p, of_p = on_p[:, 0], of_p[:, 0]
    return {"cat": pr[:, :NC].astype(np.float32),
            "onset": on_p.astype(np.float32),
            "offset": of_p.astype(np.float32),
            "duration": float(dur)}
print("tracks_v5_atst defined")

tracks_v5_atst defined


In [79]:
# =============================================================================
# B4-VERIFY (fixed) -- call .eval() before the two forward passes.
# =============================================================================
import os, torch

_c5 = None
for _n in ("ema", "student"):
    _p = f"{V5_DIR}/v5c_smel_{_n}.pt"
    if os.path.exists(_p):
        _c5 = torch.load(_p, map_location="cpu", weights_only=False)
        print(f"warm start: {_p} (J={_c5.get('score',{}).get('J', float('nan')):.4f})")
        break
assert _c5 is not None

_m = VaaniSEDV5_ATST(
    n_fine=len(canonical_tags), mel_frontend=mel_v5,
    feat_dropout=_c5["cfg"]["feat_dropout"],
    per_cat_boundary=_c5["cfg"].get("per_cat_boundary", False),
    short_mel=_c5["cfg"].get("short_mel", True),
).to(device)

_sd = {k: v for k, v in _c5["model"].items() if not k.startswith("atst_")}
missing, unexpected = _m.load_state_dict(_sd, strict=False)
assert all(k.startswith("atst_") for k in missing), missing
assert not unexpected, unexpected

# *** the fix ***
_m.eval()

# Extra safety: confirm the two ATST layers are truly zero.
with torch.no_grad():
    assert _m.atst_proj.weight.abs().max().item() == 0.0
    assert _m.atst_proj.bias.abs().max().item() == 0.0
    assert _m.atst_gate.weight.abs().max().item() == 0.0
    assert _m.atst_gate.bias.abs().max().item() == 0.0
print("[verify] atst_proj and atst_gate are exactly zero")

torch.manual_seed(0)
_dw = torch.zeros(1, WIN_SAMPLES, device=device)
_db = torch.zeros(1, CANON_T, 768, device=device)
_da = torch.zeros(1, CANON_ATST_T, 768, device=device)
_dg = torch.tensor([[0.0, W_SEC]], device=device)
with torch.no_grad():
    _o1 = _m(_dw, _db, _dg, atst=_da)
    _o2 = _m(_dw, _db, _dg, atst=None)
d = float((_o1["cat_logits"] - _o2["cat_logits"]).abs().max())
print(f"residual-at-init diff (eval mode): {d:.3e}")
assert d < 1e-6, "ATST residual is not zero at init"

del _m
torch.cuda.empty_cache()
print("B4 verify OK")

warm start: C:/Users/231b258/v5_work/v5c_smel_ema.pt (J=0.9759)
[verify] atst_proj and atst_gate are exactly zero
residual-at-init diff (eval mode): 0.000e+00
B4 verify OK


In [81]:
# =============================================================================
# B5 (fixed) -- train V5-C + ATST residual for 2000 updates.
# =============================================================================
set_seed(SEED)

ATST_FUSION_TAG     = "v5c_atst_fusion_e1"
ATST_FUSION_UPDATES = 2000
ATST_LR             = 5e-5
ATST_WARMUP         = 200

# --- warm-start checkpoint ---
_c5 = None
for _n in ("ema", "student"):
    _p = f"{V5_DIR}/v5c_smel_{_n}.pt"
    if os.path.exists(_p):
        _c5 = torch.load(_p, map_location="cpu", weights_only=False)
        log(f"{ATST_FUSION_TAG}: warm start from {_p} "
            f"(J={_c5.get('score',{}).get('J', float('nan')):.4f})")
        break
assert _c5 is not None
_cfg_c5 = _c5["cfg"]

model_atst = VaaniSEDV5_ATST(
    n_fine=len(canonical_tags), mel_frontend=mel_v5,
    feat_dropout=_cfg_c5["feat_dropout"],
    per_cat_boundary=_cfg_c5.get("per_cat_boundary", False),
    short_mel=_cfg_c5.get("short_mel", True),
).to(device)
_sd = {k: v for k, v in _c5["model"].items() if not k.startswith("atst_")}
missing, unexpected = model_atst.load_state_dict(_sd, strict=False)
assert all(k.startswith("atst_") for k in missing) and not unexpected

# *** FIX *** : eval() to disable dropout for the residual zero-check
model_atst.eval()
with torch.no_grad():
    _z1 = model_atst(torch.zeros(1, WIN_SAMPLES, device=device),
                     torch.zeros(1, CANON_T, 768, device=device),
                     torch.tensor([[0.0, W_SEC]], device=device),
                     atst=torch.zeros(1, CANON_ATST_T, 768, device=device))
    _z2 = model_atst(torch.zeros(1, WIN_SAMPLES, device=device),
                     torch.zeros(1, CANON_T, 768, device=device),
                     torch.tensor([[0.0, W_SEC]], device=device), atst=None)
    _diff = float((_z1["cat_logits"] - _z2["cat_logits"]).abs().max())
    assert _diff < 1e-6, f"residual not zero at init: {_diff:.3e}"
    print(f"[B5] residual-at-init diff (eval): {_diff:.3e}")
model_atst.train()   # back to training mode

# --- dataset / loader ---
ds_atst = V5DatasetATST(
    rows=TRAIN_ROWS, full_dataset=full_dataset, label_table=label_table,
    synth_store=SYNTH_TRAIN, training=True,
    shift_prob=CFG_V5["shift_prob"], beats_mask_prob=CFG_V5["beats_mask_prob"],
    atst_cache=cache,
)
n_draw = (ATST_FUSION_UPDATES + 50) * CFG_V5["batch_size"] * CFG_V5["grad_accum_steps"]
sampler = make_tier_sampler(TRAIN_ROWS, CFG_V5["shares"], n_draw)
loader = DataLoader(ds_atst, batch_size=CFG_V5["batch_size"], sampler=sampler,
                    num_workers=0, collate_fn=collate_v5_atst, drop_last=True,
                    pin_memory=(device.type == "cuda"))

params = model_atst.trainable_parameters()
log(f"{ATST_FUSION_TAG}: trainable params = {sum(p.numel() for p in params)/1e6:.2f}M")

opt = torch.optim.AdamW(params, lr=ATST_LR, weight_decay=CFG_V5["weight_decay"])
def _lr_at(u):
    if u < ATST_WARMUP: return 0.1 + 0.9 * u / ATST_WARMUP
    p = min(1.0, (u - ATST_WARMUP) / max(1, ATST_FUSION_UPDATES - ATST_WARMUP))
    floor = 1e-6 / ATST_LR
    return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * p))
for g in opt.param_groups: g["lr"] = ATST_LR * _lr_at(0)

scaler = GradScaler(enabled=(device.type == "cuda"))
accum = CFG_V5["grad_accum_steps"]

# --- preload full ATST tracks for eval sets ---
VALN_ATST = {k: torch.from_numpy(cache.load("nat", int(k)).astype(np.float32)).unsqueeze(0)
             for k in VALN}
VALS_ATST = {k: torch.from_numpy(cache.load("synth", k).astype(np.float32)).unsqueeze(0)
             for k in VALS}

@torch.no_grad()
def _eval_tracks_atst(net, valset, atst_by_key, T_win, fps):
    was_training = net.training
    net.eval()
    out = {k: tracks_v5_atst(net, v["wav"], v["beats"].astype(np.float32),
                             atst_by_key[k], device, T_win, fps)
           for k, v in valset.items()}
    if was_training:
        net.train()
    return out

# --- EMA copy ---
ema = VaaniSEDV5_ATST(
    n_fine=len(canonical_tags), mel_frontend=mel_v5,
    feat_dropout=_cfg_c5["feat_dropout"],
    per_cat_boundary=_cfg_c5.get("per_cat_boundary", False),
    short_mel=_cfg_c5.get("short_mel", True),
).to(device)
ema.load_state_dict(model_atst.state_dict())
freeze_model(ema)

T_win, fps = model_time_grid(model_atst, device)
assert T_win == 801, T_win
log(f"{ATST_FUSION_TAG}: T_win={T_win} fps={fps:.4f}")

paths = {name: f"{V5_DIR}/{ATST_FUSION_TAG}_{name}.pt" for name in ("student", "ema")}
best = {"student": -1.0, "ema": -1.0}
best_info = {}
upd = micro = skipped = 0
run = defaultdict(float); t0 = time.time()
model_atst.train()
opt.zero_grad(set_to_none=True)

for batch in loader:
    wav = batch["waveform"].to(device, non_blocking=True)
    with autocast(device_type="cuda", enabled=False):
        mel = model_atst.mel_frontend(wav.float())
    with autocast(device_type="cuda", enabled=(device.type == "cuda")):
        out = model_atst(wav, batch["beats"].to(device, non_blocking=True),
                         batch["gspan"].to(device), mel=weak_augment(mel),
                         atst=batch["atst"].to(device, non_blocking=True))
    loss, parts = v5_losses(out, batch, CFG_V5, device)
    if not torch.isfinite(loss):
        raise RuntimeError(f"non-finite loss {({k: float(v) for k,v in parts.items()})}")
    scaler.scale(loss / accum).backward()
    for k, v in parts.items(): run[k] += float(v.detach())
    run["n"] += 1; micro += 1
    if micro % accum:
        continue
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 5.0)
    prev = scaler.get_scale()
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    if scaler.get_scale() < prev:
        skipped += 1; continue
    upd += 1
    for g in opt.param_groups: g["lr"] = ATST_LR * _lr_at(upd)
    update_teacher(model_atst, ema, decay=CFG_V5["ema_decay"])
    if upd % 250 == 0:
        tn = _eval_tracks_atst(model_atst, VALN, VALN_ATST, T_win, fps)
        ts = _eval_tracks_atst(model_atst, VALS, VALS_ATST, T_win, fps) if VALS else {}
        Js = selection_J(tn, ts, fps)
        tn_e = _eval_tracks_atst(ema, VALN, VALN_ATST, T_win, fps)
        ts_e = _eval_tracks_atst(ema, VALS, VALS_ATST, T_win, fps) if VALS else {}
        Je = selection_J(tn_e, ts_e, fps)
        line = f"{ATST_FUSION_TAG} upd={upd} skip={skipped} J_stu={Js['J']:.4f} J_ema={Je['J']:.4f}"
        for name, J, net in (("student", Js, model_atst), ("ema", Je, ema)):
            if J["J"] > best[name]:
                best[name] = J["J"]; best_info[name] = {"update": upd, **J}
                sd = net.state_dict()
                torch.save({"model": sd, "update": upd, "score": J,
                            "hash": state_hash(sd), "canonical_tags": canonical_tags,
                            "cfg": _cfg_c5}, paths[name])
                line += f" {name}*saved*"
        log(line)
        model_atst.train()

log(f"{ATST_FUSION_TAG} done: {upd} updates ({skipped} AMP-skipped) "
    f"in {(time.time()-t0)/60:.1f} min; best={best_info}")

[2026-10-10 13:03:58] v5c_atst_fusion_e1: warm start from C:/Users/231b258/v5_work/v5c_smel_ema.pt (J=0.9759)
[B5] residual-at-init diff (eval): 0.000e+00
  sampler target shares: {'gold': 0.35, 'synth': 0.3, 'silver': 0.25, 'bronze': 0.1} | rows: {'gold': 7554, 'synth': 24000, 'silver': 55482, 'bronze': 17128}
[2026-10-10 13:03:58] v5c_atst_fusion_e1: trainable params = 2.41M
[2026-10-10 13:04:16] v5c_atst_fusion_e1: T_win=801 fps=100.1250
[2026-10-10 13:10:07] v5c_atst_fusion_e1 upd=250 skip=0 J_stu=0.9819 J_ema=0.9779 student*saved* ema*saved*
[2026-10-10 13:16:03] v5c_atst_fusion_e1 upd=500 skip=0 J_stu=0.9655 J_ema=0.9707
[2026-10-10 13:21:53] v5c_atst_fusion_e1 upd=750 skip=0 J_stu=0.9901 J_ema=0.9779 student*saved*
[2026-10-10 13:27:26] v5c_atst_fusion_e1 upd=1000 skip=0 J_stu=0.9887 J_ema=0.9771
[2026-10-10 13:33:01] v5c_atst_fusion_e1 upd=1250 skip=0 J_stu=1.0004 J_ema=0.9858 student*saved* ema*saved*
[2026-10-10 13:38:34] v5c_atst_fusion_e1 upd=1500 skip=0 J_stu=0.9921 J_ema=

In [82]:
# =============================================================================
# B6b -- decoder selection + verdict + submission for the ATST fusion model.
#
# Rule (declared before seeing any result):
#   ACCEPT iff  fusion beats v5c_smel on Val-N half A AND half B by >= +0.005
#               AND Val-S does not drop by more than 0.02.
# On REJECT: nothing is written. Keep submission_v5c_smel.zip.
# =============================================================================
import os, json, pickle, time, zipfile
import numpy as np
import torch

ATST_FUSION_TAG = "v5c_atst_fusion_e1"

# ---- 1. Pick the fusion checkpoint (student or ema, whichever scored higher) ----
_cand = []
for _n in ("student", "ema"):
    _p = f"{V5_DIR}/{ATST_FUSION_TAG}_{_n}.pt"
    if os.path.exists(_p):
        _c = torch.load(_p, map_location="cpu", weights_only=False)
        _cand.append((_c.get("score", {}).get("J", -1.0), _n, _p, _c))
assert _cand, f"no fusion checkpoint in {V5_DIR} -- did B5 finish?"
_cand.sort(reverse=True, key=lambda x: x[0])
_best_J, _best_name, _fusion_path, _ck = _cand[0]
log(f"{ATST_FUSION_TAG}: using {_best_name} ({_fusion_path}) with J={_best_J:.4f}")

# ---- 2. Rebuild the fusion model exactly (hash check) ----
_net = VaaniSEDV5_ATST(
    n_fine=len(canonical_tags), mel_frontend=mel_v5,
    feat_dropout=_ck["cfg"]["feat_dropout"],
    per_cat_boundary=_ck["cfg"].get("per_cat_boundary", False),
    short_mel=_ck["cfg"].get("short_mel", True),
).to(device)
_net.load_state_dict(_ck["model"])
_net.eval()
_h = state_hash(_net.state_dict())
assert _h == _ck["hash"], f"fusion ckpt hash mismatch: {_h} != {_ck['hash']}"
log(f"{ATST_FUSION_TAG}: model loaded, hash={_h[:12]}")

T_win, fps = model_time_grid(_net, device)
assert T_win == 801, T_win

# ---- 3. Val-N and Val-S tracks for the fusion model ----
VALN_ATST = {k: torch.from_numpy(cache.load("nat", int(k)).astype(np.float32)).unsqueeze(0)
             for k in VALN}
VALS_ATST = ({k: torch.from_numpy(cache.load("synth", k).astype(np.float32)).unsqueeze(0)
              for k in VALS} if VALS else {})

@torch.no_grad()
def _eval_tracks_atst(net, valset, atst_by_key):
    net.eval()
    return {k: tracks_v5_atst(net, v["wav"], v["beats"].astype(np.float32),
                              atst_by_key[k], device, T_win, fps)
            for k, v in valset.items()}

log(f"{ATST_FUSION_TAG}: computing Val-N tracks ({len(VALN)} clips) ...")
t0 = time.time()
TN = _eval_tracks_atst(_net, VALN, VALN_ATST)
log(f"{ATST_FUSION_TAG}: Val-N done in {time.time()-t0:.0f}s")
TS = {}
if VALS:
    log(f"{ATST_FUSION_TAG}: computing Val-S tracks ({len(VALS)} clips) ...")
    t0 = time.time()
    TS = _eval_tracks_atst(_net, VALS, VALS_ATST)
    log(f"{ATST_FUSION_TAG}: Val-S done in {time.time()-t0:.0f}s")

# ---- 4. Decoder selection (same rule as E3: search on half A, confirm on B) ----
DEC_FUS, DEC_FUS_S = select_decoder(TN, TS, fps, ATST_FUSION_TAG)
with open(f"{V5_DIR}/{ATST_FUSION_TAG}_decoder.json", "w") as f:
    json.dump({"decoder": DEC_FUS, "halves": DEC_FUS_S,
               "checkpoint": _fusion_path, "hash": _ck["hash"]}, f, indent=1)
with open(f"{V5_DIR}/{ATST_FUSION_TAG}_val_tracks.pkl", "wb") as f:
    pickle.dump({"fps": fps, "N": TN, "S": TS, "hash": _ck["hash"]}, f)

# ---- 5. Load V5-C reference tracks + decoder ----
with open(f"{V5_DIR}/v5c_smel_decoder.json") as f:
    _djC = json.load(f)
with open(f"{V5_DIR}/v5c_smel_val_tracks.pkl", "rb") as f:
    _vtC = pickle.load(f)
assert _vtC["hash"] == _djC["hash"], "v5c_smel tracks / decoder mismatch"
fpsC = float(_vtC["fps"])
assert abs(fpsC - fps) < 1e-9, (fpsC, fps)
log(f"{ATST_FUSION_TAG}: v5c_smel reference loaded (hash={_djC['hash'][:12]})")

def _half_combined(tracks, decoder, fps_, half_keys, refs):
    pred = {k: decode_v5(tracks[k], decoder, fps_)
            for k in half_keys if k in tracks}
    return official_combined_score({k: refs[k] for k in pred}, pred)["combined"]

# ---- 6. Compare on Val-N halves A and B ----
log(f"{ATST_FUSION_TAG} vs v5c_smel on Val-N (both district halves):")
wins = []
for h in ("A", "B"):
    kN = list(VALN_HALF[h])
    rC = _half_combined(_vtC["N"], _djC["decoder"], fpsC, kN, VALN_REFS)
    rF = _half_combined(TN,        DEC_FUS,       fps,  kN, VALN_REFS)
    delta = rF - rC
    wins.append(delta > 0.005)
    log(f"  half {h}: v5c={rC:.4f}  fusion={rF:.4f}  delta={delta:+.4f}  "
        f"{'OK' if delta > 0.005 else 'below threshold'}")

# ---- 7. Val-S check (drop must be <= 0.02) ----
if VALS:
    kS = list(VALS_HALF["A"]) + list(VALS_HALF["B"])
    rS_C = _half_combined(_vtC["S"], _djC["decoder"], fpsC, kS, VALS_REFS)
    rS_F = _half_combined(TS,        DEC_FUS,       fps,  kS, VALS_REFS)
    dS = rS_F - rS_C
    log(f"  Val-S: v5c={rS_C:.4f}  fusion={rS_F:.4f}  delta={dS:+.4f}")
    if dS < -0.02:
        wins.append(False)
        log(f"  Val-S drop {dS:+.4f} exceeds -0.02 -> REJECT regardless of Val-N")

accept = all(wins)
log(f"{ATST_FUSION_TAG} verdict: {'ACCEPT' if accept else 'REJECT'}")

# ---- 8. If ACCEPT, write the test submission ----
if not accept:
    print("=" * 72)
    print("FUSION REJECTED")
    print("Do nothing new. Keep submission_v5c_smel.zip as your submission.")
    print(f"Fusion val tracks still saved at: {V5_DIR}/{ATST_FUSION_TAG}_val_tracks.pkl")
    print("=" * 72)
else:
    log(f"{ATST_FUSION_TAG}: ACCEPT -- building test submission")

    # Ensure the test BEATs cache is built (uses V5's explicit builder)
    build_test_beats_cache()

    test_ids = sorted(os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES)
    missing = [cid for cid in test_ids if not cache.has("test", cid)]
    assert not missing, f"missing test ATST cache entries: {missing[:5]}"

    preds, durs = {}, {}
    t0 = time.time()
    _net.eval()
    for j, cid in enumerate(test_ids):
        pth = next(p for p in TEST_FILES
                   if os.path.splitext(os.path.basename(p))[0] == cid)
        w = load_test_wav(pth)
        bf = load_test_beats(cid, w.numel())
        atst_full = cache.load("test", cid).astype(np.float32)
        atst_t = torch.from_numpy(atst_full).unsqueeze(0).to(device)
        tr = tracks_v5_atst(_net, w, bf, atst_t, device, T_win, fps)
        preds[cid] = decode_v5(tr, DEC_FUS, fps)
        durs[cid] = tr["duration"]
        if (j + 1) % 500 == 0:
            print(f"  [fusion-test] {j+1}/{len(test_ids)} ({time.time()-t0:.0f}s)")

    zp = write_submission(preds, SUB_DIR,
                          f"submission_{ATST_FUSION_TAG}{'_SMOKE' if SMOKE else ''}.zip",
                          durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)

    # ---- Archive verification ----
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        assert "predictions.jsonl" in names, names
        lines = z.read("predictions.jsonl").decode().splitlines()
    assert len(lines) == EXPECTED_TEST_CLIPS, \
        f"zip has {len(lines)} lines, expected {EXPECTED_TEST_CLIPS}"
    ev = [json.loads(l)["events"] for l in lines]
    n_empty = sum(1 for e in ev if not e)
    n_events = sum(len(e) for e in ev)
    log(f"zip OK: {len(lines)} clips, {n_empty} empty, {n_events} events -> {zp}")
    log(f"{ATST_FUSION_TAG}: ready to submit {os.path.basename(zp)}")

# ---- Cleanup ----
del _net
torch.cuda.empty_cache()

[2026-10-10 13:50:23] v5c_atst_fusion_e1: using student (C:/Users/231b258/v5_work/v5c_atst_fusion_e1_student.pt) with J=1.0031
[2026-10-10 13:50:23] v5c_atst_fusion_e1: model loaded, hash=6f651dc363d3
[2026-10-10 13:50:42] v5c_atst_fusion_e1: computing Val-N tracks (1367 clips) ...
[2026-10-10 13:51:07] v5c_atst_fusion_e1: Val-N done in 25s
[2026-10-10 13:51:07] v5c_atst_fusion_e1: computing Val-S tracks (1000 clips) ...
[2026-10-10 13:51:26] v5c_atst_fusion_e1: Val-S done in 18s
[2026-10-10 13:51:28] v5c_atst_fusion_e1 DEC_DEFAULT: A=1.0285 B=0.9781
    100/300 configs (16s), best A so far 1.0506
    200/300 configs (33s), best A so far 1.0506
    300/300 configs (49s), best A so far 1.0506
[2026-10-10 13:52:23] v5c_atst_fusion_e1 selected decoder: A=1.0294 B=0.9875 (searched config) {'thr_short': 0.48836457856538396, 'thr_long': 0.47387131670856303, 'hyst': 0.1, 'sw_short': 1, 'sw_long': 15, 'mindur_short': 0.05761161769330349, 'mindur_long': 0.14023294339582404, 'gap_short': 0.05, '

In [83]:
# =============================================================================
# DOMAIN ROUTING -- CPU only, no training.
#
# Leaderboard evidence: the short-mel submission (its own decoder, 2.78 events/clip)
# beat v5a overall (1.2604 vs 1.2047) ONLY because of the synthetic clips
# (F1 0.579 -> 0.640, Dice 0.784 -> 0.864); on the natural clips it was worse
# (F1 0.406 -> 0.374). So the best submission available today is:
#   natural test clips   <- events from submission_v5a.zip
#   synthetic test clips <- events from submission_v5c_smel.zip
# That needs to know which test clips are synthetic. This cell:
#   1. looks for an explicit cue (file-name pattern, sample rate, format, duration);
#   2. otherwise trains a small classifier on signal statistics -- our own Gold val
#      (natural) vs our own Val-S mixtures (clean speech + pasted noise) -- and
#      labels the TEST_SYN_COUNT most synthetic-looking test clips as synthetic;
#   3. writes submission_route_<NAT>_<SYN>.zip from the two existing zips (events are
#      copied verbatim -- nothing is re-decoded).
# The leaderboard's natural/synthetic breakdown then shows directly whether the
# routing worked: natural should move back toward v5a's, synthetic stay at smel's.
# =============================================================================
RUN_ROUTE = True
NAT_ZIP = "submission_v5a.zip"          # better on natural clips (LB nat F1 0.406 / Dice 0.663)
SYN_ZIP = "submission_v5c_smel.zip"     # better on synthetic clips (LB syn F1 0.640 / Dice 0.864)
TEST_SYN_COUNT = 2080                   # organisers' split: 3,437 natural + 2,080 synthetic
import re
from collections import Counter


def _sig_stats(x, sr):
    x = np.asarray(x, dtype=np.float32)
    n = len(x); hop = max(1, int(0.01 * sr))
    fr = x[: n // hop * hop].reshape(-1, hop) if n >= hop else x.reshape(1, -1)
    e = 10 * np.log10((fr ** 2).mean(1) + 1e-12)
    j = np.abs(np.diff(e)) if len(e) > 1 else np.zeros(1)
    nz = np.flatnonzero(np.abs(x) > 1e-4)
    return {"dur": n / sr,
            "zero_frac": float((x == 0).mean()),
            "lead_sil": (nz[0] / sr) if len(nz) else n / sr,
            "trail_sil": ((n - 1 - nz[-1]) / sr) if len(nz) else n / sr,
            "peak_db": float(20 * np.log10(np.abs(x).max() + 1e-9)),
            "rms_db": float(10 * np.log10((x ** 2).mean() + 1e-12)),
            "floor_db": float(np.percentile(e, 5)), "p50_db": float(np.percentile(e, 50)),
            "range_db": float(np.percentile(e, 95) - np.percentile(e, 5)),
            "jump_max": float(j.max()), "jump_p99": float(np.percentile(j, 99)),
            "clip_frac": float((np.abs(x) >= 0.999).mean())}


if RUN_ROUTE:
    # ---- 1. explicit cues --------------------------------------------------------
    info = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        i = sf.info(pth)
        info[cid] = {"pattern": re.sub(r"\d+", "#", cid), "sr": i.samplerate, "ch": i.channels,
                     "fmt": f"{i.format}/{i.subtype}", "ext": os.path.splitext(pth)[1].lower()}
    for key in ("pattern", "sr", "ch", "fmt", "ext"):
        cnt = Counter(v[key] for v in info.values())
        log(f"ROUTE test {key}: {cnt.most_common(8)}")

    # ---- 2. signal statistics: test, natural val, our synthetic val --------------
    t0 = time.time()
    S_test = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        x, sr0 = sf.read(pth, dtype="float32")
        if x.ndim > 1:
            x = x.mean(1)
        S_test[cid] = _sig_stats(x, sr0)
    S_nat = {k: _sig_stats(v["wav"].numpy(), SR) for k, v in VALN.items()}
    S_syn = {k: _sig_stats(v["wav"].numpy(), SR) for k, v in VALS.items()} if VALS else {}
    log(f"ROUTE stats computed for {len(S_test)} test / {len(S_nat)} natural / {len(S_syn)} synthetic clips "
        f"({time.time() - t0:.0f}s)")
    feats = list(next(iter(S_test.values())).keys())
    import pandas as pd
    df_t = pd.DataFrame.from_dict(S_test, orient="index")[feats]
    df_n = pd.DataFrame.from_dict(S_nat, orient="index")[feats]
    print("feature medians  test | natural val | our synthetic val")
    for f_ in feats:
        print(f"  {f_:10s} {df_t[f_].median():9.4f} | {df_n[f_].median():9.4f} | "
              f"{(pd.DataFrame.from_dict(S_syn, orient='index')[f_].median() if S_syn else float('nan')):9.4f}")
    df_t.assign(**{k: [info[c][k] for c in df_t.index] for k in ("pattern", "sr", "fmt")}).to_csv(
        f"{V5_DIR}/test_domain_stats.csv")

    # ---- classifier: natural val (0) vs our synthetic val (1) ----------------------
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import cross_val_score
    df_s = pd.DataFrame.from_dict(S_syn, orient="index")[feats]
    Xtr = np.vstack([df_n.values, df_s.values]); ytr = np.r_[np.zeros(len(df_n)), np.ones(len(df_s))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    acc = cross_val_score(clf, Xtr, ytr, cv=5).mean()
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(df_t.values)[:, 1]
    k = min(TEST_SYN_COUNT, len(p))
    order = np.argsort(-p)
    syn_ids = set(df_t.index[order[:k]])
    margin = p[order[k - 1]] - p[order[k]] if k < len(p) else float("nan")
    log(f"ROUTE classifier: 5-fold acc natural-vs-our-synthetic {acc:.3f}; test p(syn) quantiles "
        f"{np.percentile(p, [5, 25, 50, 75, 95]).round(3).tolist()}; share p>0.5 {np.mean(p > 0.5):.3f} "
        f"(expected {TEST_SYN_COUNT / len(p):.3f}); share in (0.2,0.8) {np.mean((p > 0.2) & (p < 0.8)):.3f}")
    log("ROUTE note: if a pattern/sr/format group above has ~2,080 clips, prefer that explicit cue over the "
        "classifier (set SYN_IDS_OVERRIDE below to that group).")
    SYN_IDS_OVERRIDE = None     # e.g. {c for c in info if info[c]["pattern"] == "<pattern>"}
    if SYN_IDS_OVERRIDE:
        syn_ids = set(SYN_IDS_OVERRIDE)
    with open(f"{V5_DIR}/test_syn_ids.json", "w") as f:
        json.dump(sorted(syn_ids), f)

    # ---- 3. routed submission from the two existing zips ---------------------------
    def _read_zip(name):
        with zipfile.ZipFile(f"{SUB_DIR}/{name}") as z:
            return {json.loads(l)["clip_id"]: [(e["onset"], e["offset"]) for e in json.loads(l)["events"]]
                    for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()}
    Z_nat, Z_syn = _read_zip(NAT_ZIP), _read_zip(SYN_ZIP)
    assert set(Z_nat) == set(Z_syn) == set(S_test), "the two zips and the test folder cover different clips"
    routed = {c: (Z_syn[c] if c in syn_ids else Z_nat[c]) for c in Z_nat}
    durs = {c: S_test[c]["dur"] for c in routed}
    tag = f"route_{NAT_ZIP.replace('submission_', '').replace('.zip', '')}_{SYN_ZIP.replace('submission_', '').replace('.zip', '')}"
    zp = write_submission(routed, SUB_DIR, f"{tag}{'_SMOKE' if SMOKE else ''}.zip".replace("route_", "submission_route_"),
                          durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    n_syn = sum(c in syn_ids for c in routed)
    log(f"ROUTE wrote {zp}: {n_syn} clips from {SYN_ZIP}, {len(routed) - n_syn} from {NAT_ZIP}; "
        f"events/clip {np.mean([len(v) for v in routed.values()]):.2f}")


[2026-10-12 16:46:25] ROUTE test pattern: [('IISc_VaaniProject_K_WestBengal_Darjeeling_Darjeeling#_#_Darjeeling-SPECIFIC_#_#_#', 216), ('IISc_VaaniProject_K_WestBengal_Darjeeling_Darjeeling#_#_GENERIC_#_#_#', 127), ('IISc_VaaniProject_K_Manipur_ImphalWest_ImphalWest#_#_GENERIC_#_#_#', 89), ('IISc_VaaniProject_K_UttarPradesh_Lalitpur_Lalitpur#_#_Lalitpur-SPECIFIC_#_#_#', 71), ('IISc_VaaniProject_S_AndhraPradesh_Anantpur_#_#_Anantpur-SPECIFIC_#_#_#', 71), ('IISc_VaaniProject_S_AndhraPradesh_Chittoor_#_#_Chittoor-SPECIFIC_#_#_#', 66), ('IISc_VaaniProject_S_AndhraPradesh_Anantpur_#_#_GENERIC_#_#_#', 63), ('IISc_VaaniProject_K_Karnataka_Bangalore_Bangalore#_#_Bangalore-SPECIFIC_#_#_#', 52)]
[2026-10-12 16:46:25] ROUTE test sr: [(16000, 5517)]
[2026-10-12 16:46:25] ROUTE test ch: [(1, 5517)]
[2026-10-12 16:46:25] ROUTE test fmt: [('WAV/PCM_16', 5517)]
[2026-10-12 16:46:25] ROUTE test ext: [('.wav', 5517)]
[2026-10-12 16:46:35] ROUTE stats computed for 5517 test / 1367 natural / 1000 syntheti

In [99]:
# ---- CELL 1: setup, guards, gates (the only cell with settings) ----------------------------------------------------
import os, sys, json, shutil, importlib, numpy as np, torch
assert all(n in globals() for n in ("VaaniSEDV5", "VALN", "VALS", "tracks_v5", "TRAIN_ROWS", "CFG_V5", "select_decoder", "V5_DIR")), \
    "V5 objects are not in this kernel. Run this inside the kernel where the V5 notebook ran."
HERE = f"{BASE_DIR}/atst_work"
CKPT = f"{BASE_DIR}/atst_as2M.ckpt"
AUDIOSSL = f"{BASE_DIR}/audiossl"          # optional clone of Audio-WestlakeU/audiossl (enables the reference-equivalence tests)
for f in ("atst_common.py", "atst_v5_integration.py", "vaani_track_ensemble.py"):
    assert os.path.isfile(f"{HERE}/{f}"), f"missing {HERE}/{f}"
assert os.path.isfile(CKPT), CKPT
if HERE not in sys.path: sys.path.insert(0, HERE)
_t = f"{HERE}/test_atst_common.py"
if os.path.exists(_t) and not os.path.exists(_t + ".bak"): shutil.copy(_t, _t + ".bak")

# experiment gates: turn ON one at a time, in this order
GO_PROBE_CACHE = True   # cell 7  : build probe cache (Gold + Val-S + 3000 synth)
GO_PROBE       = True   # cell 8  : BEATs vs ATST vs both frozen-feature probe (go/no-go)
GO_ENSEMBLE    = True   # cell 9  : Experiment A, CPU only
GO_FULL_CACHE  = True   # cell 10 : full ATST cache (only if the probe passed)
GO_TRAIN       = True   # cell 11 : B3 fusion training (~2.5 h)
GO_EVAL        = True   # cell 12 : B3 evaluation vs V5-C under the fixed V5-A decoder
GO_TEST_ZIP    = True   # cell 13 : B3 test tracks + zip (only if cell 12 said ACCEPT)
print("kernel SMOKE =", globals().get("SMOKE"), "| V5_DIR =", V5_DIR, "| device =", device)

kernel SMOKE = False | V5_DIR = C:/Users/231b258/v5_work | device = cuda


In [87]:
%%writefile C:/Users/231b258/atst_work/test_atst_common.py
"""test_atst_common.py (v2, corrected). Use from a notebook: import test_atst_common as T; T.test_...()"""
import os, sys, tempfile, types
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import atst_common as AC

torch.manual_seed(0)


def rand_encoder(seed=0):
    torch.manual_seed(seed)
    net = AC.FrameASTMin()
    for name, p in net.named_parameters():
        if p.ndim > 1:
            torch.nn.init.normal_(p, mean=0.0, std=0.02)
        elif name.endswith(".weight"):
            torch.nn.init.ones_(p)            # LayerNorm scales (zero would collapse every output to 0)
        else:
            torch.nn.init.zeros_(p)
    net.requires_grad_(False)
    net.eval()
    return net


def test_rand_encoder_nontrivial():
    net = rand_encoder()
    with torch.inference_mode():
        y = net(AC.ATSTMel()(torch.randn(1, 16000 * 4) * 0.1))
    assert torch.isfinite(y).all() and y.std().item() > 1e-3, float(y.std())
    print("ok rand_encoder: finite, std %.3f" % y.std().item())


def test_mel_per_example_floor():
    m = AC.ATSTMel("pretrain")
    a, b = torch.randn(1, 32000) * 0.3, torch.randn(1, 32000) * 0.003
    batch = m(torch.cat([a, b]))
    assert torch.allclose(batch[0], m(a)[0], atol=1e-5) and torch.allclose(batch[1], m(b)[0], atol=1e-5), \
        "top_db floor leaked across batch items"
    assert batch.shape == (2, 1, 64, 1 + 32000 // 160)
    print("ok mel: per-example floor, shape", tuple(batch.shape))


def test_zscore_shifts_levels():
    m = AC.ATSTMel("pretrain"); x = torch.randn(1, 64000) * 0.01
    z = (x - x.mean()) / (x.std() + 1e-5)
    d = float(m(z).mean() - m(x).mean())
    print(f"info: per-clip z-scoring moves the MinMax-normalised mel mean by {d:+.3f} (scale range is 2.0)")
    assert abs(d) > 0.1


def test_padding_mask_equivalence():
    net = rand_encoder()
    mel = AC.ATSTMel()(torch.randn(2, 16000 * 6) * 0.1)            # (2,1,64,601)
    mel = mel[..., : (mel.shape[-1] // 4) * 4]                      # 600 frames = 150 tokens (patch embed drops the rest)
    short = mel[:1, :, :, :4 * 100]
    with torch.inference_mode():
        alone = net(short, torch.tensor([100]))
        padded = torch.cat([torch.nn.functional.pad(short, (0, 4 * 50)), mel[1:]], dim=0)
        both = net(padded, torch.tensor([100, 150]))
    err = float((alone[0] - both[0, :100]).abs().max())
    assert err < 1e-4, err
    print("ok padded batch == unbatched (max err %.2e)" % err)


def test_chunk_plan_coverage():
    for n_tok in (1, 249, 250, 251, 375, 500, 501, 642, 1000):
        plan = AC.plan_chunks(n_tok)
        cov = np.zeros(n_tok, np.int32)
        for s, e, k0, k1 in plan:
            assert 0 <= s < e <= n_tok and s <= k0 < k1 <= e and e - s <= AC.MAX_TOKENS
            cov[k0:k1] += 1
        assert np.all(cov == 1), (n_tok, np.where(cov != 1)[0][:5])
        assert plan[0][2] == 0 and plan[-1][3] == n_tok
    print("ok chunk plan: no gaps, no duplicate kept tokens")


def test_extractor_shapes_and_chunking():
    enc = AC.ATSTEncoder(rand_encoder(), AC.ATSTMel(), {"embed_dim": 768})
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    for sec in (0.4, 1.0, 7.9, 8.0, 10.0, 10.3, 25.7):
        w = torch.randn(int(sec * AC.SR)) * 0.05
        f = ex(w)
        exp = AC.atst_T_expected(max(w.numel(), AC.SR))
        assert f.shape == (exp, 768) and np.isfinite(f).all(), (sec, f.shape, exp)
    w = torch.randn(int(9.0 * AC.SR)) * 0.05
    with torch.inference_mode():
        direct = enc.net(enc.mel(w[None]), torch.tensor([225]))[0].cpu().numpy()
    assert np.allclose(ex(w), direct, atol=1e-4)
    print("ok extractor: shapes for 0.4-25.7 s, single-chunk == direct")


_PREFIX = {"c2f_frame_encoder": "encoder.encoder.frame_encoder.", "lightning_teacher": "model.teacher.encoder.",
           "downstream_finetuned": "encoder.encoder.", "pretrainedsed_wrapper": "model.atst.", "bare": ""}


def _write_ckpt(path, layout, sd):
    pre = _PREFIX[layout]
    out = {}
    for k, v in sd.items():
        k2 = k
        if layout in ("lightning_teacher", "downstream_finetuned", "c2f_frame_encoder") and k.startswith("norm_frame."):
            k2 = "norm." + k[len("norm_frame."):]
        out[pre + k2] = v
    if layout == "lightning_teacher":
        out.update({"model.student.encoder." + k: v for k, v in sd.items()})
    torch.save({"state_dict": out, "hyper_parameters": {"pos_type": "cut"}}, path)


def test_loader_layouts_and_failures():
    net = rand_encoder(1); sd = net.state_dict(); x = AC.ATSTMel()(torch.randn(1, 32000) * 0.1)
    with torch.inference_mode():
        ref = net(x, torch.tensor([50]))
    with tempfile.TemporaryDirectory() as d:
        for layout in _PREFIX:
            p = os.path.join(d, layout + ".ckpt"); _write_ckpt(p, layout, sd)
            enc = AC.load_atst_encoder(p, verbose=False)
            assert enc.meta["layout"] == layout, (layout, enc.meta["layout"])
            with torch.inference_mode():
                assert torch.allclose(enc.net(x, torch.tensor([50])), ref, atol=1e-6), layout
            assert not any(q.requires_grad for q in enc.parameters())
        bad = {k: v for k, v in sd.items() if "blocks.5." not in k}
        p = os.path.join(d, "bad.ckpt"); _write_ckpt(p, "bare", bad)
        try:
            AC.load_atst_encoder(p, verbose=False); raise AssertionError("incomplete checkpoint was accepted")
        except RuntimeError:
            pass
        p = os.path.join(d, "junk.ckpt"); torch.save({"state_dict": {"foo.bar": torch.zeros(2)}}, p)
        try:
            AC.load_atst_encoder(p, verbose=False); raise AssertionError("junk checkpoint accepted")
        except RuntimeError:
            pass
        try:
            AC.load_atst_encoder(os.path.join(d, "bare.ckpt"), expected_sha256="0" * 64, verbose=False)
            raise AssertionError("sha mismatch accepted")
        except RuntimeError:
            pass
    print("ok loader: 5 layouts load strictly and reproduce outputs; incomplete/junk/sha-mismatch raise")


def test_cache_roundtrip_and_resume():
    enc = AC.ATSTEncoder(rand_encoder(), AC.ATSTMel(), {"embed_dim": 768})
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    meta = {"sha256": "x", "mel_variant": "pretrain", "n_last": 1, "chunk": 250, "hop": 125, "embed_dim": 768}
    with tempfile.TemporaryDirectory() as d:
        c = AC.ATSTCache(d, "t1", meta)
        wavs = {i: torch.randn(int((1 + i) * 1.3 * AC.SR)) * 0.05 for i in range(4)}
        s1 = AC.build_cache(list(wavs), lambda i: wavs[i], c, "nat", ex, log=lambda *_: None, max_new=2)
        s2 = AC.build_cache(list(wavs), lambda i: wavs[i], c, "nat", ex, log=lambda *_: None)
        assert s1["new"] == 2 and s2["new"] == 2 and s2["skipped"] == 2
        AC.verify_cache(c, "nat", list(wavs), lambda k: wavs[k].numel(), log=lambda *_: None, full=True)
        try:
            AC.ATSTCache(d, "other_tag", meta); raise AssertionError("mixed tags accepted")
        except RuntimeError:
            pass
        assert c.load("nat", 3).dtype == np.float16
    print("ok cache: resume, shape/finite verification, tag pinning")


# ---------------------------------------------------------------- reference (audiossl) comparisons
def _load_reference(audiossl_dir):
    sys.path.insert(0, audiossl_dir)
    if "fairseq" not in sys.modules:
        for n in ("fairseq", "fairseq.data", "fairseq.data.data_utils"):
            sys.modules[n] = types.ModuleType(n)
        sys.modules["fairseq.data.data_utils"].compute_mask_indices = lambda *a, **k: None
    from audiossl.methods.atstframe.audio_transformer import FrameAST_base
    return FrameAST_base(patch_h=64, patch_w=4, drop_path_rate=0.0).eval()


def _load_into_reference(ref, state):
    miss, unexp = ref.load_state_dict(state, strict=False)
    bad_miss = [k for k in miss if "cls_token" not in k]
    assert not bad_miss and not unexp, f"reference/state mismatch: missing {bad_miss[:5]} unexpected {list(unexp)[:5]}"
    if miss:
        print("  note: reference keys not in our state (ignored):", list(miss))


def _compare_to_reference(ref, mine, mel, tok_len, n_last, atol=1e-4):
    """audiossl takes lengths in mel frames (it floors //4 internally). Try that first, then token units, and report
    which convention reproduces our (token-unit) output. One of them must match."""
    with torch.inference_mode():
        b = mine(mel, tok_len, n_last=n_last)
        tried = {}
        for name, L in (("mel-frames", tok_len * 4), ("tokens", tok_len)):
            a = ref.get_intermediate_layers(mel, L, n=n_last, scene=False)
            ok = all(torch.allclose(a[i, :int(tok_len[i])], b[i, :int(tok_len[i])], atol=atol) for i in range(len(tok_len)))
            tried[name] = float(max((a[i, :int(tok_len[i])] - b[i, :int(tok_len[i])]).abs().max() for i in range(len(tok_len))))
            if ok:
                return name, tried[name]
    raise AssertionError(f"no length convention matches the reference: max diffs {tried}")


def test_audiossl_equivalence(audiossl_dir):
    if not audiossl_dir:
        print("skip audiossl equivalence (no clone)"); return
    ref = _load_reference(audiossl_dir)
    mine = AC.FrameASTMin().eval()
    mine.requires_grad_(False)
    ref_sd = {k: v for k, v in ref.state_dict().items() if "cls_token" not in k}
    mine.load_state_dict(ref_sd, strict=True)
    mel = AC.ATSTMel()(torch.randn(2, 16000 * 8) * 0.1)             # (2,1,64,801)
    tl = torch.tensor([200, 150])
    for n_last in (1, 4, 12):
        conv, err = _compare_to_reference(ref, mine, mel, tl, n_last)
    print(f"ok equivalence with audiossl FrameAST (n_last 1/4/12, padded item). reference length unit = {conv}; max err {err:.2e}")


def test_real_vs_reference(ckpt, audiossl_dir):
    """Same real weights in both implementations, same mel, same masks."""
    if not (ckpt and audiossl_dir):
        print("skip real-vs-reference (need ckpt and audiossl clone)"); return
    enc = AC.load_atst_encoder(ckpt, device="cpu", verbose=False)
    ref = _load_reference(audiossl_dir)
    _load_into_reference(ref, {k: v.clone() for k, v in enc.net.state_dict().items()})
    t = torch.arange(16000 * 8) / 16000.0
    w = torch.stack([0.1 * torch.sin(2 * np.pi * 440 * t) + 0.02 * torch.randn_like(t),
                     0.05 * torch.randn_like(t)])
    mel = enc.mel(w)
    conv, err = _compare_to_reference(ref, enc.net, mel, torch.tensor([200, 150]), 1)
    print(f"ok REAL checkpoint: our encoder == audiossl FrameAST on the same weights (length unit {conv}, max err {err:.2e})")


def test_real_checkpoint(ckpt):
    if not ckpt:
        print("skip real checkpoint"); return
    enc = AC.load_atst_encoder(ckpt, device="cpu", verbose=True)
    ex = AC.ATSTFeatureExtractor(enc, "cpu")
    f = ex(torch.randn(8 * AC.SR) * 0.05)
    assert f.shape == (200, 768) and np.isfinite(f).all()
    print(f"real checkpoint OK: 8 s -> {f.shape}, std {f.std():.3f}, sha256 {enc.meta['sha256']}")


ALL_SYNTHETIC = ("test_rand_encoder_nontrivial", "test_mel_per_example_floor", "test_zscore_shifts_levels",
                 "test_padding_mask_equivalence", "test_chunk_plan_coverage", "test_extractor_shapes_and_chunking",
                 "test_loader_layouts_and_failures", "test_cache_roundtrip_and_resume")

Overwriting C:/Users/231b258/atst_work/test_atst_common.py


In [88]:
# ---- CELL 3: synthetic tests (no real weights). Every line must print "ok". -----------------------------------------
import atst_common as AC; importlib.reload(AC)
import test_atst_common as T; importlib.reload(T)
for _name in T.ALL_SYNTHETIC:
    getattr(T, _name)()
print("SYNTHETIC TESTS PASSED")

ok rand_encoder: finite, std 1.000
ok mel: per-example floor, shape (2, 1, 64, 201)
info: per-clip z-scoring moves the MinMax-normalised mel mean by +0.614 (scale range is 2.0)
ok padded batch == unbatched (max err 2.50e-06)
ok chunk plan: no gaps, no duplicate kept tokens
ok extractor: shapes for 0.4-25.7 s, single-chunk == direct
ok loader: 5 layouts load strictly and reproduce outputs; incomplete/junk/sha-mismatch raise
ok cache: resume, shape/finite verification, tag pinning
SYNTHETIC TESTS PASSED


In [89]:
# ---- CELL 4: reference equivalence with the REAL weights (the check that proves the re-implementation) ---------------
_A = AUDIOSSL if os.path.isdir(AUDIOSSL) else None
if _A is None:
    print("!! no audiossl clone at", AUDIOSSL, "-> reference tests skipped. Strongly recommended: git clone https://github.com/Audio-WestlakeU/audiossl there.")
T.test_audiossl_equivalence(_A)
T.test_real_vs_reference(CKPT, _A)
T.test_real_checkpoint(CKPT)
ATST_PINNED = AC.sha256_file(CKPT)
print("checkpoint sha256 (pinned for all later stages):", ATST_PINNED)

ok equivalence with audiossl FrameAST (n_last 1/4/12, padded item). reference length unit = mel-frames; max err 3.22e-06
ok REAL checkpoint: our encoder == audiossl FrameAST on the same weights (length unit mel-frames, max err 9.78e-06)
[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstrea

In [90]:
# ---- CELL 5: load the integration into THIS kernel, pin the checkpoint, run the V5-side self-tests ------------------
%run -i "$HERE/atst_v5_integration.py"
ATST_EXPECTED_SHA256 = ATST_PINNED
import vaani_track_ensemble as VE; importlib.reload(VE)
atst_selftest_v5()
VE.selftest(official_combined_score, match_events, events_to_frames, decode_v5)
print("V5-SIDE SELF-TESTS PASSED | cache dir:", ATST_CACHE_DIR, "| run tag:", ATST_RUN_TAG)

[2026-10-12 17:33:16] selftest 1/6 dataset parity+alignment OK on 24 rows (15 unshifted/long, 9 shifted)
[2026-10-12 17:33:16] selftest 2/6 zero-init equivalence OK (max |diff| < 1e-5)
[2026-10-12 17:33:17] selftest 3/6 gradient flow OK (atst_proj receives finite non-zero gradient at init)
[2026-10-12 17:33:17] selftest 4/6 tracks_atst parity OK (20.3 s clip, 2034 frames, 3 windows)
[2026-10-12 17:33:17] selftest 5/6 ATSTOnlySED shapes + v5_losses OK
[2026-10-12 17:33:17] selftest 6/6 time grid 801 OK. ALL ATST/V5 SELF-TESTS PASSED
V5-SIDE SELF-TESTS PASSED | cache dir: C:/Users/231b258/atst_cache_v1 | run tag: v5c_atst_fusion_e1


In [91]:
# ---- CELL 6: one real clip through the real encoder, plus a chunked 23.7 s clip -------------------------------------
_enc = AC.load_atst_encoder(CKPT, device=device, mel_variant=ATST_MEL_VARIANT, expected_sha256=ATST_EXPECTED_SHA256)
_ex = AC.ATSTFeatureExtractor(_enc, device)
_clip = _load_waveform(int(VAL_GOLD[0]), full_dataset, SR, WAV_CACHE_DIRS)
_f = _ex(_clip)
print(_clip.numel() / SR, "s ->", _f.shape, "| expected tokens", AC.atst_T_expected(max(_clip.numel(), SR)), "| finite", np.isfinite(_f).all(), "| std %.3f" % _f.std())
_lf = _ex(torch.cat([_clip] * 4)[: int(23.7 * SR)])
print("23.7 s chunked ->", _lf.shape, "| finite", np.isfinite(_lf).all())
print(json.dumps(_enc.meta, indent=1)); print(AC.PROVENANCE["atst_as2M.ckpt"])
del _enc, _ex
if torch.cuda.is_available(): torch.cuda.empty_cache()

[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstream_finetuned': 138, 'pretrainedsed_wrapper': 0}
   hyper_parameters: {'learning_rate': 3.125, 'max_epochs': 11, 'niter_per_epoch': 1208, 'warmup_epochs': 2, 'num_labels': 527, 'multi_label': True, 'layer_wise_lr': 0.75, 'freeze_embed': Fa

In [100]:
# ---- CELL 7: storage estimate + probe cache (resumable; safe to interrupt and re-run) -------------------------------
if GO_PROBE_CACHE:
    atst_estimate_storage()
    atst_build_caches(subset="probe")
else:
    print("skipped (GO_PROBE_CACHE=False)")

[2026-10-12 17:36:06] ATST cache estimate: 194.6 h of train/val/synth audio -> 26.9 GB fp16 at 40 ms (13.5 GB if pooled to 80 ms); test (11 h) ~1.5 GB. Free disk is not checked automatically.
[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstream_finetuned': 138, 'pretrainedsed_wrapper': 0

In [105]:
# ---- PATCH: atst_open_cache defaults to the pinned sha256 -----------------------------------------------------------
_p = f"{HERE}/atst_v5_integration.py"
_s = open(_p, encoding="utf-8").read()
_old = "    meta = dict(_META, sha256=sha256)\n"
_new = "    meta = dict(_META, sha256=sha256 or globals().get('ATST_EXPECTED_SHA256'))\n"
if _old in _s:
    open(_p, "w", encoding="utf-8").write(_s.replace(_old, _new))
    print("file patched")
else:
    print("file already patched or text differs")

def atst_open_cache(sha256=None, create=True):
    meta = dict(_META, sha256=sha256 or globals().get("ATST_EXPECTED_SHA256"))
    return AC.ATSTCache(ATST_CACHE_DIR, ATST_CACHE_TAG, meta, create=create)

assert ATST_EXPECTED_SHA256 == "8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2", \
    f"pinned sha {ATST_EXPECTED_SHA256} differs from the cache's sha (checkpoint file changed?)"
_c = atst_open_cache(create=False)
print("cache opens OK:", ATST_CACHE_DIR)

file patched
cache opens OK: C:/Users/231b258/atst_cache_v1


In [106]:
# ---- CELL 8: go/no-go probe. Continue to cell 10 only if "both - BEATs" has CI90 lower bound > 0 ---------------------
if GO_PROBE:
    PROBE = atst_probe(atst_open_cache(create=False), n_updates=1500, n_boot=100)
else:
    print("skipped (GO_PROBE=False)")

[2026-10-13 17:07:49] probe rows: 31554 (gold 7554, synth 24000)
  sampler target shares: {'gold': 0.6, 'synth': 0.4} | rows: {'gold': 7554, 'synth': 24000}
  sampler target shares: {'gold': 0.6, 'synth': 0.4} | rows: {'gold': 7554, 'synth': 24000}
  sampler target shares: {'gold': 0.6, 'synth': 0.4} | rows: {'gold': 7554, 'synth': 24000}
[2026-10-13 17:31:53] PROBE frame macro-AP on Val-N: BEATs 0.3946 | ATST 0.4610 | both 0.4817 | both-BEATs +0.0871 CI90 (0.073199540139131, 0.12267055911450725) | ATST-BEATs +0.0664 CI90 (0.05222161110026485, 0.10981842641343534)
[2026-10-13 17:31:53] PROBE go/no-go: proceed to the fusion run only if 'both - BEATs' > 0 with the CI90 lower bound > 0 (ATST adds information beyond BEATs); otherwise stop and keep the ensemble route.


In [107]:
# ---- CELL 9: Experiment A, V5-A + V5-C track ensemble (CPU only, reads real v5_work pickles) -----------------------
if GO_ENSEMBLE:
    ENS = atst_ensemble_experiment("v5a", "v5c_smel", write_zip=False, control="best", v5_dir=V5_DIR)
else:
    print("skipped (GO_ENSEMBLE=False)")

[2026-10-13 17:31:53] ENS members A=v5a (hash da24d4767f8c) B=v5c_smel (hash 94f7e049f6a4df41); decoders {'A': 'dd6c19368a69', 'B': 'dd6c19368a69'}; primary decoder=A
[2026-10-13 17:31:58] ENS [A only     |dec A] J 0.9782 | Val-N 0.9208 (F1 0.2771 P 0.294 R 0.262 Dice 0.6436) halves A 0.9986 B 0.9585 | Val-S 1.0718 | ev/clip 1.50 empty 31 short-recall 0.134 | ECE 0.1284
[2026-10-13 17:31:59] ENS [A only     |dec B] J 0.9782 | Val-N 0.9208 (F1 0.2771 P 0.294 R 0.262 Dice 0.6436) halves A 0.9986 B 0.9585 | Val-S 1.0718 | ev/clip 1.50 empty 31 short-recall 0.134 | ECE 0.1284
[2026-10-13 17:32:00] ENS [B only     |dec A] J 0.9741 | Val-N 0.9135 (F1 0.2693 P 0.284 R 0.256 Dice 0.6442) halves A 1.0001 B 0.9485 | Val-S 1.0729 | ev/clip 1.53 empty 26 short-recall 0.128 | ECE 0.1346
[2026-10-13 17:32:01] ENS [B only     |dec B] J 0.9741 | Val-N 0.9135 (F1 0.2693 P 0.284 R 0.256 Dice 0.6442) halves A 1.0001 B 0.9485 | Val-S 1.0729 | ev/clip 1.53 empty 26 short-recall 0.128 | ECE 0.1346
[2026-10-

In [108]:
# ---- CELL 10: full ATST cache (train + val + synthetic + test). Resumable. -----------------------------------------
if GO_FULL_CACHE:
    atst_estimate_storage()
    atst_build_caches(subset="all")
else:
    print("skipped (GO_FULL_CACHE=False)")

[2026-10-13 17:32:07] ATST cache estimate: 194.6 h of train/val/synth audio -> 26.9 GB fp16 at 40 ms (13.5 GB if pooled to 80 ms); test (11 h) ~1.5 GB. Free disk is not checked automatically.
[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstream_finetuned': 138, 'pretrainedsed_wrapper': 0

In [111]:
# ---- DIAGNOSE v2 (read-only, makes no assumptions about the checkpoint layout) ----------------------------------------
import glob, time
for p in sorted(glob.glob(f"{V5_DIR}/{ATST_RUN_TAG}*")):
    print(f"{os.path.basename(p):<46} {os.path.getsize(p)/1e6:8.1f} MB   {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))}")
for name in ("student", "ema"):
    p = f"{V5_DIR}/{ATST_RUN_TAG}_{name}.pt"
    if not os.path.exists(p): continue
    ck = torch.load(p, map_location="cpu", weights_only=False)
    sd = ck.get("model", {})
    atst_keys = [k for k in sd if k.startswith("atst_")]
    print(f"\n{name}: top-level keys = {list(ck.keys())}")
    print(f"   update={ck.get('update')} | score={ {k: ck['score'][k] for k in ('J','N','S') if k in ck.get('score', {})} }")
    print(f"   n model tensors={len(sd)} | atst_* tensors={len(atst_keys)} {atst_keys[:4]}")
    if atst_keys:
        w = sd["atst_proj.weight"]; print(f"   atst_proj.weight |max|={float(w.abs().max()):.5f} (0.0 would mean the ATST path was never trained)")
    print("   cfg total_updates:", ck.get("cfg", {}).get("total_updates"), "| hash present:", "hash" in ck)
hp = f"{V5_DIR}/{ATST_RUN_TAG}_history.json"
print("\nhistory.json exists:", os.path.exists(hp))
if os.path.exists(hp):
    h = json.load(open(hp)); print("   updates:", h.get("updates"), "| skipped:", h.get("skipped"), "| keys:", list(h.keys()))
print("recipe total_updates:", CFG_ATST["total_updates"], "| eval_every:", CFG_ATST["eval_every"])

v5c_atst_fusion_e1_decoder.json                     0.0 MB   2026-10-10 13:52
v5c_atst_fusion_e1_ema.pt                          10.5 MB   2026-10-10 13:44
v5c_atst_fusion_e1_student.pt                      10.5 MB   2026-10-10 13:49
v5c_atst_fusion_e1_val_tracks.pkl                  50.8 MB   2026-10-10 13:52

student: top-level keys = ['model', 'update', 'score', 'hash', 'canonical_tags', 'cfg']
   update=2000 | score={'J': 1.0030596879916016, 'N': 0.9097027730181284, 'S': 1.1553788650535841}
   n model tensors=144 | atst_* tensors=4 ['atst_proj.weight', 'atst_proj.bias', 'atst_gate.weight', 'atst_gate.bias']
   atst_proj.weight |max|=0.01402 (0.0 would mean the ATST path was never trained)
   cfg total_updates: 8000 | hash present: True

ema: top-level keys = ['model', 'update', 'score', 'hash', 'canonical_tags', 'cfg']
   update=1750 | score={'J': 0.9920347642730678, 'N': 0.9053295292280409, 'S': 1.1335012003991642}
   n model tensors=144 | atst_* tensors=4 ['atst_proj.weight', 'at

In [ ]:
import glob
ATST_RUN_TAG = "v5c_atst_fusion_e2"      # cells 12 and 13 read this global, so they follow automatically
assert not glob.glob(f"{V5_DIR}/{ATST_RUN_TAG}*"), "tag already used"
FUSION_PATHS, FUSION_BEST = atst_stage_train_fusion("fuse", ATST_RUN_TAG)

  sampler target shares: {'gold': 0.35, 'synth': 0.3, 'silver': 0.25, 'bronze': 0.1} | rows: {'gold': 7554, 'synth': 24000, 'silver': 55482, 'bronze': 17128}
[2026-10-13 17:36:32] v5c_atst_fusion_e2 start: J=0.7799 N=0.8778 S=0.6202562959867896 thr=0.6
[2026-10-13 17:54:00] v5c_atst_fusion_e2 upd=1000 skipped=0 lr=9.80e-05 frame=0.0574 onset=0.0771 offset=0.0807 dice=0.5101 weak=0.3808 | student: J=0.9511 N=0.8775 S=1.0713 *saved* | ema: J=0.8051 N=0.8498 S=0.7321 *saved*
[2026-10-13 18:11:31] v5c_atst_fusion_e2 upd=2000 skipped=0 lr=8.86e-05 frame=0.0433 onset=0.0692 offset=0.0729 dice=0.3579 weak=0.2601 | student: J=0.9797 N=0.8724 S=1.1548 *saved* | ema: J=0.9510 N=0.8911 S=1.0489 *saved*
[2026-10-13 18:28:53] v5c_atst_fusion_e2 upd=3000 skipped=0 lr=7.29e-05 frame=0.0422 onset=0.0682 offset=0.0716 dice=0.3167 weak=0.2428 | student: J=1.0223 N=0.9122 S=1.2019 *saved* | ema: J=1.0060 N=0.8990 S=1.1806 *saved*


In [ ]:
# ---- CELL 12: B3 evaluation under the FIXED V5-A decoder, paired against V5-C on the same Val-N / Val-S clips --------
if GO_EVAL:
    _c = atst_open_cache(create=False)
    atst_eval_and_decoder("fuse", ATST_RUN_TAG, _c, select=False)
    _d = json.load(open(f"{V5_DIR}/v5a_decoder.json"))["decoder"]
    ENS_B3 = VE.run_ensemble_experiment(dict(globals()), "v5c_smel", ATST_RUN_TAG, ens_tag=ATST_ENS_TAG,
                                        decoders={"A": _d, "B": _d}, primary="A", control="A only", write_zip=False, log=log)
else:
    print("skipped (GO_EVAL=False)")

In [ ]:
# ---- CELL 13: B3 test tracks (float32) + zip + archive verification. Submit ONLY if cell 12 printed ACCEPT. ---------
if GO_TEST_ZIP:
    _c = atst_open_cache(create=False)
    atst_test_tracks32("fuse", ATST_RUN_TAG, _c)
    ZIP_B3 = atst_write_submission(ATST_RUN_TAG)
    atst_verify_zip(ZIP_B3, expected_ids={os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES})
    print("Candidate zip:", ZIP_B3)
else:
    print("skipped (GO_TEST_ZIP=False)")

In [116]:
# =============================================================================
# DOMAIN ROUTING -- CPU only, no training.
#
# Leaderboard evidence: the short-mel submission (its own decoder, 2.78 events/clip)
# beat v5a overall (1.2604 vs 1.2047) ONLY because of the synthetic clips
# (F1 0.579 -> 0.640, Dice 0.784 -> 0.864); on the natural clips it was worse
# (F1 0.406 -> 0.374). So the best submission available today is:
#   natural test clips   <- events from submission_v5a.zip
#   synthetic test clips <- events from submission_v5c_smel.zip
# That needs to know which test clips are synthetic. This cell:
#   1. looks for an explicit cue (file-name pattern, sample rate, format, duration);
#   2. otherwise trains a small classifier on signal statistics -- our own Gold val
#      (natural) vs our own Val-S mixtures (clean speech + pasted noise) -- and
#      labels the TEST_SYN_COUNT most synthetic-looking test clips as synthetic;
#   3. writes submission_route_<NAT>_<SYN>.zip from the two existing zips (events are
#      copied verbatim -- nothing is re-decoded).
# The leaderboard's natural/synthetic breakdown then shows directly whether the
# routing worked: natural should move back toward v5a's, synthetic stay at smel's.
# =============================================================================
RUN_ROUTE = True
NAT_ZIP = "submission_v5c_atst_fusion_e2.zip"  # use your actual ATST ZIP filename
SYN_ZIP = "submission_v5c_smel.zip"     # better on synthetic clips (LB syn F1 0.640 / Dice 0.864)
TEST_SYN_COUNT = 2080                   # organisers' split: 3,437 natural + 2,080 synthetic
import re
from collections import Counter


def _sig_stats(x, sr):
    x = np.asarray(x, dtype=np.float32)
    n = len(x); hop = max(1, int(0.01 * sr))
    fr = x[: n // hop * hop].reshape(-1, hop) if n >= hop else x.reshape(1, -1)
    e = 10 * np.log10((fr ** 2).mean(1) + 1e-12)
    j = np.abs(np.diff(e)) if len(e) > 1 else np.zeros(1)
    nz = np.flatnonzero(np.abs(x) > 1e-4)
    return {"dur": n / sr,
            "zero_frac": float((x == 0).mean()),
            "lead_sil": (nz[0] / sr) if len(nz) else n / sr,
            "trail_sil": ((n - 1 - nz[-1]) / sr) if len(nz) else n / sr,
            "peak_db": float(20 * np.log10(np.abs(x).max() + 1e-9)),
            "rms_db": float(10 * np.log10((x ** 2).mean() + 1e-12)),
            "floor_db": float(np.percentile(e, 5)), "p50_db": float(np.percentile(e, 50)),
            "range_db": float(np.percentile(e, 95) - np.percentile(e, 5)),
            "jump_max": float(j.max()), "jump_p99": float(np.percentile(j, 99)),
            "clip_frac": float((np.abs(x) >= 0.999).mean())}


if RUN_ROUTE:
    # ---- 1. explicit cues --------------------------------------------------------
    info = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        i = sf.info(pth)
        info[cid] = {"pattern": re.sub(r"\d+", "#", cid), "sr": i.samplerate, "ch": i.channels,
                     "fmt": f"{i.format}/{i.subtype}", "ext": os.path.splitext(pth)[1].lower()}
    for key in ("pattern", "sr", "ch", "fmt", "ext"):
        cnt = Counter(v[key] for v in info.values())
        log(f"ROUTE test {key}: {cnt.most_common(8)}")

    # ---- 2. signal statistics: test, natural val, our synthetic val --------------
    t0 = time.time()
    S_test = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        x, sr0 = sf.read(pth, dtype="float32")
        if x.ndim > 1:
            x = x.mean(1)
        S_test[cid] = _sig_stats(x, sr0)
    S_nat = {k: _sig_stats(v["wav"].numpy(), SR) for k, v in VALN.items()}
    S_syn = {k: _sig_stats(v["wav"].numpy(), SR) for k, v in VALS.items()} if VALS else {}
    log(f"ROUTE stats computed for {len(S_test)} test / {len(S_nat)} natural / {len(S_syn)} synthetic clips "
        f"({time.time() - t0:.0f}s)")
    feats = list(next(iter(S_test.values())).keys())
    import pandas as pd
    df_t = pd.DataFrame.from_dict(S_test, orient="index")[feats]
    df_n = pd.DataFrame.from_dict(S_nat, orient="index")[feats]
    print("feature medians  test | natural val | our synthetic val")
    for f_ in feats:
        print(f"  {f_:10s} {df_t[f_].median():9.4f} | {df_n[f_].median():9.4f} | "
              f"{(pd.DataFrame.from_dict(S_syn, orient='index')[f_].median() if S_syn else float('nan')):9.4f}")
    df_t.assign(**{k: [info[c][k] for c in df_t.index] for k in ("pattern", "sr", "fmt")}).to_csv(
        f"{V5_DIR}/test_domain_stats.csv")

    # ---- classifier: natural val (0) vs our synthetic val (1) ----------------------
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import cross_val_score
    df_s = pd.DataFrame.from_dict(S_syn, orient="index")[feats]
    Xtr = np.vstack([df_n.values, df_s.values]); ytr = np.r_[np.zeros(len(df_n)), np.ones(len(df_s))]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    acc = cross_val_score(clf, Xtr, ytr, cv=5).mean()
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(df_t.values)[:, 1]
    k = min(TEST_SYN_COUNT, len(p))
    order = np.argsort(-p)
    syn_ids = set(df_t.index[order[:k]])
    margin = p[order[k - 1]] - p[order[k]] if k < len(p) else float("nan")
    log(f"ROUTE classifier: 5-fold acc natural-vs-our-synthetic {acc:.3f}; test p(syn) quantiles "
        f"{np.percentile(p, [5, 25, 50, 75, 95]).round(3).tolist()}; share p>0.5 {np.mean(p > 0.5):.3f} "
        f"(expected {TEST_SYN_COUNT / len(p):.3f}); share in (0.2,0.8) {np.mean((p > 0.2) & (p < 0.8)):.3f}")
    log("ROUTE note: if a pattern/sr/format group above has ~2,080 clips, prefer that explicit cue over the "
        "classifier (set SYN_IDS_OVERRIDE below to that group).")
    SYN_IDS_OVERRIDE = None     # e.g. {c for c in info if info[c]["pattern"] == "<pattern>"}
    if SYN_IDS_OVERRIDE:
        syn_ids = set(SYN_IDS_OVERRIDE)
    with open(f"{V5_DIR}/test_syn_ids.json", "w") as f:
        json.dump(sorted(syn_ids), f)

    # ---- 3. routed submission from the two existing zips ---------------------------
    def _read_zip(name):
        with zipfile.ZipFile(f"{SUB_DIR}/{name}") as z:
            return {json.loads(l)["clip_id"]: [(e["onset"], e["offset"]) for e in json.loads(l)["events"]]
                    for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()}
    Z_nat, Z_syn = _read_zip(NAT_ZIP), _read_zip(SYN_ZIP)
    assert set(Z_nat) == set(Z_syn) == set(S_test), "the two zips and the test folder cover different clips"
    routed = {c: (Z_syn[c] if c in syn_ids else Z_nat[c]) for c in Z_nat}
    durs = {c: S_test[c]["dur"] for c in routed}
    tag = f"route_{NAT_ZIP.replace('submission_', '').replace('.zip', '')}_{SYN_ZIP.replace('submission_', '').replace('.zip', '')}"
    zp = write_submission(routed, SUB_DIR, f"{tag}{'_SMOKE' if SMOKE else ''}.zip".replace("route_", "submission_route_"),
                          durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    n_syn = sum(c in syn_ids for c in routed)
    log(f"ROUTE wrote {zp}: {n_syn} clips from {SYN_ZIP}, {len(routed) - n_syn} from {NAT_ZIP}; "
        f"events/clip {np.mean([len(v) for v in routed.values()]):.2f}")


[2026-10-14 10:41:55] ROUTE test pattern: [('IISc_VaaniProject_K_WestBengal_Darjeeling_Darjeeling#_#_Darjeeling-SPECIFIC_#_#_#', 216), ('IISc_VaaniProject_K_WestBengal_Darjeeling_Darjeeling#_#_GENERIC_#_#_#', 127), ('IISc_VaaniProject_K_Manipur_ImphalWest_ImphalWest#_#_GENERIC_#_#_#', 89), ('IISc_VaaniProject_K_UttarPradesh_Lalitpur_Lalitpur#_#_Lalitpur-SPECIFIC_#_#_#', 71), ('IISc_VaaniProject_S_AndhraPradesh_Anantpur_#_#_Anantpur-SPECIFIC_#_#_#', 71), ('IISc_VaaniProject_S_AndhraPradesh_Chittoor_#_#_Chittoor-SPECIFIC_#_#_#', 66), ('IISc_VaaniProject_S_AndhraPradesh_Anantpur_#_#_GENERIC_#_#_#', 63), ('IISc_VaaniProject_K_Karnataka_Bangalore_Bangalore#_#_Bangalore-SPECIFIC_#_#_#', 52)]
[2026-10-14 10:41:55] ROUTE test sr: [(16000, 5517)]
[2026-10-14 10:41:55] ROUTE test ch: [(1, 5517)]
[2026-10-14 10:41:55] ROUTE test fmt: [('WAV/PCM_16', 5517)]
[2026-10-14 10:41:55] ROUTE test ext: [('.wav', 5517)]
[2026-10-14 10:42:05] ROUTE stats computed for 5517 test / 1367 natural / 1000 syntheti

In [117]:
import glob, hashlib, pickle, zipfile, time, os, json, torch, numpy as np
_mb = lambda p: os.path.getsize(p) / 1e6
_ts = lambda p: time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))
print("V5_DIR:", V5_DIR, "| SUB_DIR:", SUB_DIR, "| TEST_FILES:", len(TEST_FILES), "| EXPECTED_TEST_CLIPS:", EXPECTED_TEST_CLIPS)

print("\n== ZIPs ==")
for p in sorted(glob.glob(f"{SUB_DIR}/*.zip")):
    n = None
    try:
        with zipfile.ZipFile(p) as z:
            if z.namelist() == ["predictions.jsonl"]:
                n = sum(1 for l in z.read("predictions.jsonl").decode().splitlines() if l.strip())
    except Exception as e:
        n = f"ERR {e}"
    print(f"{os.path.basename(p):<58}{_mb(p):7.2f} MB {_ts(p)} clips={n} sha1={hashlib.sha1(open(p,'rb').read()).hexdigest()[:12]}")

print("\n== router id files ==")
for p in sorted(set(glob.glob(f"{BASE_DIR}/**/*syn_ids*.json", recursive=True) + glob.glob(f"{SUB_DIR}/*syn*.json"))):
    try: print(p, "| n ids =", len(json.load(open(p))), "|", _ts(p))
    except Exception as e: print(p, "ERR", e)

print("\n== run tags in V5_DIR ==")
tags = sorted({os.path.basename(p)[:-len("_decoder.json")] for p in glob.glob(f"{V5_DIR}/*_decoder.json")})
ref_keys = (set(VALN_REFS), set(VALS_REFS))
for t in tags:
    print(f"\n[{t}]")
    for suf in ("val_tracks.pkl", "test_tracks32.pkl", "test_tracks.pkl", "decoder.json", "student.pt", "ema.pt", "history.json"):
        p = f"{V5_DIR}/{t}_{suf}"
        print(f"   {suf:<20}", f"{_mb(p):8.1f} MB {_ts(p)}" if os.path.exists(p) else "-")
    vp = f"{V5_DIR}/{t}_val_tracks.pkl"
    if os.path.exists(vp):
        v = pickle.load(open(vp, "rb")); k0 = next(iter(v["N"]))
        tr = v["N"][k0]
        print(f"   val: fps={v['fps']:.4f} hash={v['hash'][:10]} N={len(v['N'])}/{len(ref_keys[0])} S={len(v['S'])}/{len(ref_keys[1])} "
              f"keys_match={set(v['N'])==ref_keys[0] and set(v['S'])==ref_keys[1]} cat{tuple(tr['cat'].shape)} {tr['cat'].dtype} onset{tuple(tr['onset'].shape)}")
    for nm in ("student", "ema"):
        p = f"{V5_DIR}/{t}_{nm}.pt"
        if os.path.exists(p):
            ck = torch.load(p, map_location="cpu", weights_only=False)
            ks = list(ck["model"])
            print(f"   {nm}: update={ck.get('update')} J={ck.get('score', {}).get('J')} hash={str(ck.get('hash'))[:10]} "
                  f"atst_ln={'atst_ln.weight' in ks} atst_*={sum(k.startswith('atst_') for k in ks)} short_mel={any(k.startswith('short_mel') for k in ks)} "
                  f"total_updates={ck.get('cfg', {}).get('total_updates')} shares={ck.get('cfg', {}).get('shares')}")

V5_DIR: C:/Users/231b258/v5_work | SUB_DIR: C:/Users/231b258/submission | TEST_FILES: 5517 | EXPECTED_TEST_CLIPS: 5517

== ZIPs ==
submission.zip                                               0.19 MB 2026-10-03 12:54 clips=5517 sha1=3fda5e8b3a94
submission_route_v5a_v5c_smel.zip                            0.21 MB 2026-10-12 16:46 clips=5517 sha1=aea4e9ba811b
submission_route_v5c_atst_fusion_e2_v5c_smel.zip             0.21 MB 2026-10-14 10:42 clips=5517 sha1=a34ad5e732b7
submission_v5a.zip                                           0.20 MB 2026-10-07 12:42 clips=5517 sha1=ee96f01243c9
submission_v5a_bnd.zip                                       0.21 MB 2026-10-08 12:44 clips=5517 sha1=db4f8242008d
submission_v5a_dec2.zip                                      0.21 MB 2026-10-08 17:18 clips=5517 sha1=8a9e2e7e794b
submission_v5a_final.zip                                     0.20 MB 2026-10-07 17:56 clips=5517 sha1=b05e9fdd003a
submission_v5a_probe_max1.zip                                0.1

In [119]:
# =============================================================================
# OV-1 (fixed) -- official validation set: BEATs cache, v5a / v5c_smel tracks, per-domain
#         scores, and an independent audit of the router. Paste into your V5 kernel.
#
# FIX vs the first version: v5_from_ckpt asserted state_hash(...) == stored hash and failed for v5a_student.pt.
# This version loads through ov_load_v5(), which keeps the integrity guarantee but does not depend on the
# hash definition: (1) the loaded weights must be bit-identical to the tensors stored in the checkpoint,
# (2) if the stored hash does not match the recomputed one, 3 Val-N clips are re-tracked and must reproduce
# <tag>_val_tracks.pkl (so the weights provably are the ones that produced those validation tracks).
#
# New files it writes (nothing existing is overwritten):
#   <BASE_DIR>/beats_cache_offval/<N_|S_><stem>.npy     BEATs for the 2,501 official clips (already built: skipped)
#   <V5_DIR>/<tag>_offval_tracks32.pkl                   float32 tracks per tag
#   <V5_DIR>/offval_index.json                           clip list, domain, district, half
#   <V5_DIR>/test_syn_prob_ov.json                       p(synthetic) per test clip (audit only)
# =============================================================================
import os, json, pickle, time
import numpy as np, soundfile as sf, torch, torch.nn.functional as F

OV_DIR = f"{BASE_DIR}/validation/validation"     # folder that contains validationMetadata.json
OV_BEATS_DIR = f"{BASE_DIR}/beats_cache_offval"
OV_TAGS_V5 = ["v5a", "v5c_smel"]                  # plain V5 checkpoints (ATST-e2 needs its own inference: OV-2)
FIXED_DEC = json.load(open(f"{V5_DIR}/v5a_decoder.json"))["decoder"]

# ---- clip list ---------------------------------------------------------------------------
with open(os.path.join(OV_DIR, "validationMetadata.json"), encoding="utf-8") as f:
    _meta = json.load(f)
OV = {}
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]
    syn = bool(r["syntheticData"])
    pth = os.path.join(OV_DIR, "syntheticNoiseAudio" if syn else "naturalNoisyAudio", r["segmentFileName"])
    assert os.path.exists(pth), pth
    key = ("S_" if syn else "N_") + stem                   # the same stem can exist in both domains
    assert key not in OV, f"duplicate {key}"
    OV[key] = {"path": pth, "syn": syn, "district": r["district"],
               "ref": [(float(e["start"]), float(e["end"])) for e in r["NoiseSubCategoryTimeStamp"]
                       if float(e["end"]) > float(e["start"])]}
_ds = sorted({v["district"] for v in OV.values()})
_half = {d: ("A" if i % 2 == 0 else "B") for i, d in enumerate(_ds)}  # same rule for both domains
for v in OV.values():
    v["half"] = _half[v["district"]]
OVK = {(dom, h): [k for k, v in OV.items() if v["syn"] == (dom == "S") and (h == "all" or v["half"] == h)]
       for dom in ("N", "S") for h in ("all", "A", "B")}
OV_REFS = {k: v["ref"] for k, v in OV.items()}
_idx_p = f"{V5_DIR}/offval_index.json"
if not os.path.exists(_idx_p):
    json.dump({k: {kk: v[kk] for kk in ("path", "syn", "district", "half")} for k, v in OV.items()}, open(_idx_p, "w"))
log(f"OV: natural {len(OVK[('N', 'all')])} (A {len(OVK[('N', 'A')])} / B {len(OVK[('N', 'B')])}), "
    f"synthetic {len(OVK[('S', 'all')])} (A {len(OVK[('S', 'A')])} / B {len(OVK[('S', 'B')])}), {len(_ds)} districts")

# ---- BEATs cache (same extractor and padding rule as the test cache) ----------------------
os.makedirs(OV_BEATS_DIR, exist_ok=True)
_new = 0; t0 = time.time()
for j, (k, v) in enumerate(OV.items()):
    out = os.path.join(OV_BEATS_DIR, f"{k}.npy")
    if os.path.exists(out):
        continue
    w = load_test_wav(v["path"])
    if w.numel() < 16000:
        w = F.pad(w, (0, 16000 - w.numel()))
    feat, _ = extract_beats_full(get_live_beats(), w, device)
    if not np.isfinite(feat).all():
        raise RuntimeError(f"non-finite BEATs for {k}")
    np.save(out + ".tmp.npy", feat.astype(np.float16)); os.replace(out + ".tmp.npy", out); _new += 1
    if _new % 500 == 0:
        print(f"  BEATs {j + 1}/{len(OV)} ({time.time() - t0:.0f}s)")
log(f"OV BEATs: {_new} new, {len(OV)} total in {OV_BEATS_DIR}")


def ov_beats(k, n):
    bf = np.load(os.path.join(OV_BEATS_DIR, f"{k}.npy")).astype(np.float32)
    check_beats_len(bf, max(n, 16000), k)
    return bf


# ---- checkpoint loader that does not depend on the hash definition -----------------------
def ov_load_v5(path, tag, n_check=3, atol=2e-3):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = new_v5(ck["cfg"]); net.load_state_dict(ck["model"]); net.eval()
    sd = net.state_dict()
    # (1) loaded weights are exactly what the file stores
    bad = [k for k, v in ck["model"].items() if k not in sd or not torch.equal(v.cpu(), sd[k].detach().cpu())]
    if bad:
        raise RuntimeError(f"{tag}: loaded weights differ from the checkpoint file in {len(bad)} tensors, first {bad[:5]}")
    h_now = state_hash(sd)
    if h_now == ck["hash"]:
        log(f"OV load {tag}: stored hash matches ({h_now}); weights bit-identical to the file")
        return net, ck, "hash-ok"
    # (2) stored hash disagrees with the recomputed one: prove the weights by re-scoring stored validation tracks
    vp = f"{V5_DIR}/{tag}_val_tracks.pkl"
    if not os.path.exists(vp):
        raise RuntimeError(f"{tag}: stored hash {ck['hash']} != recomputed {h_now} and no {vp} to verify against")
    vt = pickle.load(open(vp, "rb"))
    T_win, fps = model_time_grid(net, device)
    ks = list(VALN)[:n_check]
    worst = 0.0
    for k in ks:
        tr = tracks_v5(net, VALN[k]["wav"], VALN[k]["beats"].astype(np.float32), device, T_win, fps)
        for name in ("cat", "onset", "offset"):
            a, b = np.asarray(tr[name], np.float32), np.asarray(vt["N"][k][name], np.float32)
            if a.shape != b.shape:
                raise RuntimeError(f"{tag}: re-tracked '{name}' shape {a.shape} != stored {b.shape} for clip {k}")
            worst = max(worst, float(np.abs(a - b).max()))
    if worst > atol:
        raise RuntimeError(f"{tag}: STOP. hash mismatch AND re-tracked Val-N clips differ from {vp} by {worst:.4g} (> {atol}). "
                           f"This checkpoint does not reproduce the validation tracks it is paired with.")
    log(f"OV load {tag}: stored hash {ck['hash']} != recomputed {h_now} (stale hash definition?), but weights are bit-identical to "
        f"the file AND reproduce {n_check} stored Val-N tracks (max |diff| {worst:.2e}) -> accepted")
    return net, ck, f"hash-stale-behaviour-verified(max_diff={worst:.2e})"


# ---- float32 tracks per tag (checkpoint = the one its decoder/val tracks came from) -------
def ov_tracks_v5(tag):
    p = f"{V5_DIR}/{tag}_offval_tracks32.pkl"
    dj = json.load(open(f"{V5_DIR}/{tag}_decoder.json"))
    if os.path.exists(p):
        res = pickle.load(open(p, "rb"))
        assert res["hash"] == dj["hash"] and set(res["tracks"]) == set(OV), f"{p} is stale -- rename it and re-run"
        return res
    net, ck, stamp = ov_load_v5(dj["checkpoint"], tag)
    assert ck["hash"] == dj["hash"], f"{tag}: decoder.json hash {dj['hash']} != checkpoint stored hash {ck['hash']}"
    T_win, fps = model_time_grid(net, device)
    out = {}
    for k, v in OV.items():
        w = load_test_wav(v["path"])
        out[k] = tracks_v5(net, w, ov_beats(k, w.numel()), device, T_win, fps)
    res = {"fps": fps, "tracks": out, "hash": ck["hash"], "checkpoint": dj["checkpoint"], "load_check": stamp}
    pickle.dump(res, open(p, "wb"))
    del net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res


OVT = {t: ov_tracks_v5(t) for t in OV_TAGS_V5}
OWN_DEC = {t: json.load(open(f"{V5_DIR}/{t}_decoder.json"))["decoder"] for t in OVT}


def ov_eval(tracks, fps, dec, dom, h="all"):
    ks = OVK[(dom, h)]
    pred = {k: decode_v5(tracks[k], dec, fps) for k in ks}
    s = official_combined_score({k: OV_REFS[k] for k in ks}, pred)
    ne = np.array([len(pred[k]) for k in ks])
    return dict(s, ev=float(ne.mean()), empty=int((ne == 0).sum()))


def ov_report(name, tracks, fps, dec, dec_name):
    for dom in ("N", "S"):
        a, A, B = (ov_eval(tracks, fps, dec, dom, h) for h in ("all", "A", "B"))
        nref = np.mean([len(OV_REFS[k]) for k in OVK[(dom, "all")]])
        log(f"OV {name:<22} [{dec_name:<9}] {'NAT' if dom == 'N' else 'SYN'} comb {a['combined']:.4f} "
            f"(F1 {a['event_f1']:.4f} P {a['precision']:.3f} R {a['recall']:.3f} Dice {a['segment_dice']:.4f}) "
            f"halves {A['combined']:.4f}/{B['combined']:.4f} ev/clip {a['ev']:.2f} (ref {nref:.2f}) empty {a['empty']}")


for t, r in OVT.items():
    ov_report(t, r["tracks"], r["fps"], FIXED_DEC, "v5a-dec")
    ov_report(t, r["tracks"], r["fps"], OWN_DEC[t], "own-dec")
print("Leaderboard for reference: v5a NAT .4056/.6627 SYN .5789/.7836 | v5c_smel NAT .3735/.6714 SYN .6400/.8638 (F1/Dice)")


# ---- router audit: is a top-2080 quota consistent with an independent classifier? --------
def _sig(x, sr):
    h = max(1, sr // 100); m = max(1, len(x) // h)
    e = 10 * np.log10((x[:m * h].reshape(m, h) ** 2).mean(1) + 1e-12)
    j = np.abs(np.diff(e)) if len(e) > 1 else np.zeros(1)
    return [len(x) / sr, float(np.mean(x == 0)), *np.percentile(e, [1, 5, 25, 50, 95]).tolist(),
            float(np.percentile(e, 95) - np.percentile(e, 5)), float(j.max()), float(np.percentile(j, 99)),
            float(np.percentile(j, 90)), float(np.mean(j > 20)), float(np.mean(e < -60))]


def _mono(p):
    x, sr = sf.read(p, dtype="float32")
    return (x.mean(1) if x.ndim > 1 else x), sr


from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import roc_auc_score
_ok = list(OV)
Xo = np.array([_sig(*_mono(OV[k]["path"])) for k in _ok]); yo = np.array([int(OV[k]["syn"]) for k in _ok])
go = [OV[k]["district"] for k in _ok]
clf = HistGradientBoostingClassifier(max_iter=300, random_state=SEED)
pcv = cross_val_predict(clf, Xo, yo, cv=GroupKFold(5), groups=go, method="predict_proba")[:, 1]
log(f"ROUTER AUDIT (official val, district-grouped 5-fold): AUC {roc_auc_score(yo, pcv):.4f}; at p>0.5: "
    f"synthetic recall {np.mean(pcv[yo == 1] > .5):.3f}, natural false-positive {np.mean(pcv[yo == 0] > .5):.3f}")
clf.fit(Xo, yo)
_tid = [os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES]
pt = clf.predict_proba(np.array([_sig(*_mono(p)) for p in TEST_FILES]))[:, 1]
json.dump(dict(zip(_tid, map(float, pt))), open(f"{V5_DIR}/test_syn_prob_ov.json", "w"))
cur = set(json.load(open(f"{V5_DIR}/test_syn_ids.json")))
top = set(np.array(_tid)[np.argsort(-pt)[:len(cur)]])
log(f"ROUTER AUDIT test: p>0.5 for {int((pt > .5).sum())} clips (current quota {len(cur)}); "
    f"confident p>0.9 {int((pt > .9).sum())}, p<0.1 {int((pt < .1).sum())}, unsure {int(((pt >= .1) & (pt <= .9)).sum())}; "
    f"overlap of current {len(cur)} ids with this classifier's top-{len(cur)}: {len(cur & top)}")

[2026-10-14 16:27:36] OV: natural 1806 (A 851 / B 955), synthetic 695 (A 367 / B 328), 141 districts
[2026-10-14 16:27:36] OV BEATs: 0 new, 2501 total in C:/Users/231b258/beats_cache_offval
[2026-10-14 16:27:36] OV load v5a: stored hash da24d4767f8c != recomputed 2156f4249213f016 (stale hash definition?), but weights are bit-identical to the file AND reproduce 3 stored Val-N tracks (max |diff| 0.00e+00) -> accepted
[2026-10-14 16:28:30] OV load v5c_smel: stored hash matches (94f7e049f6a4df41); weights bit-identical to the file
[2026-10-14 16:29:11] OV v5a                    [v5a-dec  ] NAT comb 0.9926 (F1 0.3470 P 0.369 R 0.327 Dice 0.6456) halves 0.9865/0.9970 ev/clip 1.44 (ref 1.63) empty 65
[2026-10-14 16:29:11] OV v5a                    [v5a-dec  ] SYN comb 1.3678 (F1 0.5705 P 0.670 R 0.497 Dice 0.7973) halves 1.3933/1.3397 ev/clip 2.36 (ref 3.18) empty 8
[2026-10-14 16:29:12] OV v5a                    [own-dec  ] NAT comb 0.9926 (F1 0.3470 P 0.369 R 0.327 Dice 0.6456) halves 0.986

In [120]:
# =============================================================================
# SNAP -- synthetic-clip boundary snapping (CPU only; run after OV-1 in the same kernel).
#
# Why: the organisers' synthetic clips add noise ONLY inside each labelled span, with
# hard edges. Measured on the 695 official synthetic clips: the strongest spectral
# rise/fall within +-150 ms of a true edge lies within the scorer tolerance for 93% of
# onsets and 96% of offsets (median error 5 ms). So model edges can be moved onto
# the audio's own change points. (On natural clips the same probe is near chance:
# 0.67/0.62 -> natural clips are never touched.)
#
# Part A (always): validation on the official synthetic clips, events from SNAP_TAG's
#   own decoder (float32 OV tracks from OV-1). Pre-declared grid of 8 configs, chosen
#   on district half A, confirmed on half B, district-grouped paired bootstrap.
#   ACCEPT: delta >= +0.03 on BOTH halves and bootstrap P(delta > 0) >= 0.95.
# Part B (only if accepted): copies the events of BEST_ZIP, snaps ONLY the clips listed
#   in test_syn_ids.json (natural clips byte-identical), writes a NEW zip. Nothing
#   existing is overwritten.
# =============================================================================
import os, json, pickle, zipfile, hashlib, numpy as np, soundfile as sf

SNAP_TAG = "v5c_smel"                                            # the synthetic-route model
BEST_ZIP = f"{SUB_DIR}/submission_route_v5c_atst_fusion_e2_v5c_smel.zip"   # the 1.2825 zip (read only)
SYN_IDS = f"{V5_DIR}/test_syn_ids.json"                          # the routing used by BEST_ZIP
SNAP_GRID = [dict(W=w, thr=t) for w in (0.08, 0.12, 0.15, 0.20) for t in (0.0, 0.5)]   # fixed in advance
SNAP_HOP, SNAP_NFFT = 80, 512                                    # 5 ms hop, 32 ms window


def snap_curves(x, K=3):
    """rise / fall change curves (5 ms hop): mean positive / negative change of the
    log spectrum between the 15 ms after and the 15 ms before each point"""
    x = np.asarray(x, np.float32)
    n = 1 + len(x) // SNAP_HOP
    xp = np.pad(x, (SNAP_NFFT // 2, SNAP_NFFT // 2 + SNAP_HOP))
    idx = np.arange(SNAP_NFFT)[None, :] + SNAP_HOP * np.arange(n)[:, None]
    L = np.log(np.abs(np.fft.rfft(xp[idx] * np.hanning(SNAP_NFFT).astype(np.float32), axis=1)) ** 2 + 1e-10)
    c = np.cumsum(np.vstack([np.zeros((1, L.shape[1]), L.dtype), L]), 0)
    t = np.arange(n); a = np.clip(t - K, 0, n); b = np.clip(t + K, 0, n)
    d = (c[b] - c[t]) / np.maximum(b - t, 1)[:, None] - (c[t] - c[a]) / np.maximum(t - a, 1)[:, None]
    return np.mean(np.maximum(d, 0), 1), np.mean(np.maximum(-d, 0), 1)


def _best(curve, t, W, thr, lo_t, hi_t):
    i = int(round(t * 16000 / SNAP_HOP)); w = int(round(W * 16000 / SNAP_HOP))
    lo = max(0, i - w, int(np.ceil(lo_t * 16000 / SNAP_HOP)))
    hi = min(len(curve), i + w + 1, int(np.floor(hi_t * 16000 / SNAP_HOP)) + 1)
    if hi <= lo:
        return t
    j = lo + int(np.argmax(curve[lo:hi]))
    return j * SNAP_HOP / 16000 if curve[j] >= thr else t


def snap_events(events, up, dn, dur, W=0.15, thr=0.0, min_len=0.06):
    """onset -> strongest rise, offset -> strongest fall within +-W s; never crosses a
    neighbouring event; edges at the clip start/end are left alone"""
    ev = sorted((float(s), float(e)) for s, e in events)
    out = []
    for k, (s, e) in enumerate(ev):
        prev_end = out[-1][1] if out else 0.0
        nxt = ev[k + 1][0] if k + 1 < len(ev) else dur
        s2 = s if s <= 0.02 else _best(up, s, W, thr, prev_end, min(e - min_len, nxt))
        e2 = e if e >= dur - 0.02 else _best(dn, e, W, thr, max(s2 + min_len, s), nxt)
        if e2 - s2 < min_len:
            s2, e2 = s, e
        out.append((round(s2, 3), round(e2, 3)))
    return [(a, b) for a, b in out if b > a]


# ---- official scorer pieces, per clip (for grouped bootstrap) ----------------------------
def _clip_stats(ref, pred):
    mr, mp, cand = set(), set(), []
    for ri, (ro, rf) in enumerate(ref):
        tol = max(0.2 * (rf - ro), 0.05)
        for pi, (po, pf) in enumerate(pred):
            if abs(po - ro) <= tol and abs(pf - rf) <= tol:
                cand.append((abs(po - ro) + abs(pf - rf), ri, pi))
    for _, ri, pi in sorted(cand):
        if ri not in mr and pi not in mp:
            mr.add(ri); mp.add(pi)
    tp = len(mr)
    al = list(ref) + list(pred)
    if not al:
        dice = 1.0
    else:
        mt = max(o for _, o in al) + 0.5; n = int(mt / 0.01) + 1
        r = np.zeros(n, np.int32); p = np.zeros(n, np.int32)
        for arr, evs in ((r, ref), (p, pred)):
            for on, off in evs:
                i0 = int(on / 0.01); i1 = min(int(off / 0.01) + 1, n)
                if i1 > i0: arr[i0:i1] = 1
        t = r.sum() + p.sum(); dice = 1.0 if t == 0 else 2.0 * (r & p).sum() / t
    return np.array([tp, len(pred) - tp, len(ref) - tp, dice], np.float64)


def _agg(S):
    tp, fp, fn = S[:, 0].sum(), S[:, 1].sum(), S[:, 2].sum()
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"f1": f1, "p": p, "r": r, "dice": S[:, 3].mean(), "comb": f1 + S[:, 3].mean()}


def _boot(Sc, Sx, groups, n=2000, seed=0):
    g = np.array(groups); ug = np.unique(g); idx = {u: np.where(g == u)[0] for u in ug}
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        ii = np.concatenate([idx[u] for u in rng.choice(ug, len(ug))])
        d.append(_agg(Sx[ii])["comb"] - _agg(Sc[ii])["comb"])
    d = np.array(d); return float(np.mean(d > 0)), np.percentile(d, [5, 95])


# ---- Part A: validation on the official synthetic clips -----------------------------------
_idx = json.load(open(f"{V5_DIR}/offval_index.json"))
_ovt = pickle.load(open(f"{V5_DIR}/{SNAP_TAG}_offval_tracks32.pkl", "rb"))
_dec = json.load(open(f"{V5_DIR}/{SNAP_TAG}_decoder.json"))["decoder"]
_ref = {k: v["ref"] for k, v in OV.items()} if "OV" in globals() else None
assert _ref is not None, "run OV-1 first in this kernel (it defines OV)"
KS = sorted(k for k, v in _idx.items() if v["syn"])
_half = {k: _idx[k]["half"] for k in KS}; _grp = [_idx[k]["district"] for k in KS]
_base = {k: decode_v5(_ovt["tracks"][k], _dec, _ovt["fps"]) for k in KS}
_cur = {}
for k in KS:
    x, sr = sf.read(_idx[k]["path"], dtype="float32"); x = x.mean(1) if x.ndim > 1 else x
    assert sr == 16000, (k, sr)
    _cur[k] = snap_curves(x) + (len(x) / sr,)


def _stats(pred):
    return np.stack([_clip_stats(_ref[k], pred[k]) for k in KS])


S0 = _stats(_base)
H = {h: np.array([_half[k] == h for k in KS]) for h in "AB"}
b0 = {h: _agg(S0[H[h]]) for h in "AB"}; ball = _agg(S0)
log(f"SNAP base {SNAP_TAG} own decoder, official SYN: comb {ball['comb']:.4f} (F1 {ball['f1']:.4f} P {ball['p']:.3f} "
    f"R {ball['r']:.3f} Dice {ball['dice']:.4f}) halves {b0['A']['comb']:.4f}/{b0['B']['comb']:.4f}")
cands = []
for cfg in SNAP_GRID:
    Sx = _stats({k: snap_events(_base[k], *_cur[k], **cfg) for k in KS})
    a = {h: _agg(Sx[H[h]]) for h in "AB"}
    cands.append((a["A"]["comb"], cfg, Sx, a))
    log(f"  SNAP {cfg}: half A {a['A']['comb']:.4f} ({a['A']['comb'] - b0['A']['comb']:+.4f})   [half B shown after selection]")
cands.sort(key=lambda z: -z[0])
_, SNAP_CFG, Sx, a = cands[0]                      # selected on half A only
allx = _agg(Sx); pgt, ci = _boot(S0, Sx, _grp)
dA, dB = a["A"]["comb"] - b0["A"]["comb"], a["B"]["comb"] - b0["B"]["comb"]
SNAP_OK = dA >= 0.03 and dB >= 0.03 and pgt >= 0.95
log(f"SNAP selected {SNAP_CFG}: SYN comb {allx['comb']:.4f} (F1 {allx['f1']:.4f} P {allx['p']:.3f} R {allx['r']:.3f} "
    f"Dice {allx['dice']:.4f}) | delta A {dA:+.4f} B {dB:+.4f} | district bootstrap P>0 {pgt:.3f} CI90 [{ci[0]:+.4f},{ci[1]:+.4f}] "
    f"-> {'ACCEPT' if SNAP_OK else 'REJECT'}")

# ---- Part B: test zip (only if accepted) ----------------------------------------------------
if SNAP_OK:
    sha = hashlib.sha256(open(BEST_ZIP, "rb").read()).hexdigest()
    with zipfile.ZipFile(BEST_ZIP) as z:
        rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
    best_ev = {r["clip_id"]: [(e["onset"], e["offset"]) for e in r["events"]] for r in rows}
    syn = set(json.load(open(SYN_IDS)))
    files = {os.path.splitext(os.path.basename(p))[0]: p for p in TEST_FILES}
    assert set(best_ev) == set(files) and len(best_ev) == len(rows) == len(files), "best zip / test clip mismatch"
    assert syn <= set(files)
    out, durs, moved = {}, {}, 0
    for cid, pth in files.items():
        w = load_test_wav(pth).numpy(); durs[cid] = len(w) / 16000
        if cid in syn and best_ev[cid]:
            out[cid] = snap_events(best_ev[cid], *snap_curves(w), durs[cid], **SNAP_CFG)
            moved += int(out[cid] != [tuple(map(float, e)) for e in best_ev[cid]])
        else:
            out[cid] = best_ev[cid]
    name = f"submission_route_e2_smel_snapW{int(SNAP_CFG['W'] * 1000)}.zip"
    assert not os.path.exists(f"{SUB_DIR}/{name}"), f"{name} exists -- not overwriting"
    write_submission(out, SUB_DIR, name, durs, expected=EXPECTED_TEST_CLIPS if "EXPECTED_TEST_CLIPS" in globals() else 5517)
    assert hashlib.sha256(open(BEST_ZIP, "rb").read()).hexdigest() == sha, "best zip changed!"
    nat_same = all(out[c] == best_ev[c] for c in files if c not in syn)
    log(f"SNAP wrote {name}: {len(syn)} synthetic-routed clips, {moved} changed; natural clips identical: {nat_same}; "
        f"source {os.path.basename(BEST_ZIP)} sha256 {sha[:12]} unchanged")


[2026-10-14 16:30:32] SNAP base v5c_smel own decoder, official SYN: comb 1.4989 (F1 0.6401 P 0.643 R 0.638 Dice 0.8588) halves 1.5320/1.4628
[2026-10-14 16:30:32]   SNAP {'W': 0.08, 'thr': 0.0}: half A 1.5492 (+0.0172)   [half B shown after selection]
[2026-10-14 16:30:32]   SNAP {'W': 0.08, 'thr': 0.5}: half A 1.5492 (+0.0172)   [half B shown after selection]
[2026-10-14 16:30:32]   SNAP {'W': 0.12, 'thr': 0.0}: half A 1.5696 (+0.0375)   [half B shown after selection]
[2026-10-14 16:30:32]   SNAP {'W': 0.12, 'thr': 0.5}: half A 1.5696 (+0.0375)   [half B shown after selection]
[2026-10-14 16:30:32]   SNAP {'W': 0.15, 'thr': 0.0}: half A 1.5707 (+0.0387)   [half B shown after selection]
[2026-10-14 16:30:33]   SNAP {'W': 0.15, 'thr': 0.5}: half A 1.5716 (+0.0396)   [half B shown after selection]
[2026-10-14 16:30:33]   SNAP {'W': 0.2, 'thr': 0.0}: half A 1.5739 (+0.0419)   [half B shown after selection]
[2026-10-14 16:30:33]   SNAP {'W': 0.2, 'thr': 0.5}: half A 1.5739 (+0.0419)   [hal

In [121]:
import json, os, numpy as np
p_syn = json.load(open(f"{V5_DIR}/test_syn_prob_ov.json"))
old_ids = set(json.load(open(f"{V5_DIR}/test_syn_ids.json")))
test_ids = {os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES}
assert set(p_syn) == test_ids and len(test_ids) == 5517 and all(0 <= p <= 1 for p in p_syn.values())

K = len(old_ids)                                           # keep the same quota so only the ranking changes
order = sorted(p_syn, key=lambda c: (-p_syn[c], c))        # deterministic tie-break
new_ids = set(order[:K])
entering, leaving = sorted(new_ids - old_ids), sorted(old_ids - new_ids)
print(f"quota {K} | overlap {len(old_ids & new_ids)} | entering {len(entering)} | leaving {len(leaving)}")
print(f"p at the cutoff: rank {K}={p_syn[order[K-1]]:.3f}, rank {K+1}={p_syn[order[K]]:.3f}")
print(f"clips with p>0.5: {sum(p > .5 for p in p_syn.values())} (quota {K})  <- independent estimate of the synthetic count")
for name, ids in (("entering", entering), ("leaving", leaving)):
    pp = np.array([p_syn[c] for c in ids]); print(f"{name}: p mean {pp.mean():.2f}, in [0.2,0.8]: {int(((pp > .2) & (pp < .8)).sum())}/{len(pp)}")
dst = f"{V5_DIR}/test_syn_ids_ovrouter.json"
assert not os.path.exists(dst), "already exists"
json.dump(sorted(new_ids), open(dst, "w")); print("saved", dst)

quota 2080 | overlap 1983 | entering 97 | leaving 97
p at the cutoff: rank 2080=0.612, rank 2081=0.608
clips with p>0.5: 2092 (quota 2080)  <- independent estimate of the synthetic count
entering: p mean 0.95, in [0.2,0.8]: 12/97
leaving: p mean 0.09, in [0.2,0.8]: 15/97
saved C:/Users/231b258/v5_work/test_syn_ids_ovrouter.json


In [123]:
import zipfile, hashlib
SRC_ATST   = f"{SUB_DIR}/submission_v5c_atst_fusion_e2.zip"                         # unsnapped ATST (natural source)
SRC_SMEL   = f"{SUB_DIR}/submission_v5c_smel.zip"                                   # unsnapped short-mel (synthetic source)
SNAPPED_OLD = f"{SUB_DIR}/submission_route_e2_smel_snapW200.zip"                          # the existing SNAP W=0.20 submission
OUT = f"{SUB_DIR}/submission_route_e2_smel_snapW200_ovrouter.zip"
SNAP_HOOK = None   # def hook(cid, smel_events): ... -> [{"onset":..,"offset":..}, ...]  = YOUR SNAP W=0.20 thr=0.0 on that clip

def read_zip(p):
    with zipfile.ZipFile(p) as z:
        assert z.namelist() == ["predictions.jsonl"], z.namelist()
        rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
    return rows, {r["clip_id"]: r["events"] for r in rows}
rows_old, snapped = read_zip(SNAPPED_OLD); _, atst = read_zip(SRC_ATST); _, smel = read_zip(SRC_SMEL)
assert set(snapped) == set(atst) == set(smel) == test_ids
# integrity: the old snapped ZIP must equal the ATST source on every clip outside the old synthetic set
bad = [c for c in test_ids - old_ids if snapped[c] != atst[c]]
assert not bad, f"{len(bad)} non-synthetic clips differ between the snapped ZIP and the ATST source, e.g. {bad[:3]}"
print(f"to snap (entering): {len(entering)} | revert to ATST (leaving): {len(leaving)} | copied unchanged: {5517 - len(entering) - len(leaving)}")

if SNAP_HOOK is None:
    print("dry run only: set SNAP_HOOK to build the ZIP")
else:
    assert not os.path.exists(OUT), f"{OUT} exists, refusing to overwrite"
    new_snapped = {c: SNAP_HOOK(c, smel[c]) for c in entering}
    lines = []
    for r in rows_old:                                   # keep the original clip order
        c = r["clip_id"]
        ev = new_snapped[c] if c in new_snapped else (atst[c] if c in leaving else snapped[c])
        lines.append(json.dumps({"clip_id": c, "events": ev}))
    with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("predictions.jsonl", "\n".join(lines) + "\n")
    atst_verify_zip(OUT, expected_ids=test_ids)
    print(OUT, hashlib.sha1(open(OUT, "rb").read()).hexdigest()[:12])

to snap (entering): 97 | revert to ATST (leaving): 97 | copied unchanged: 5323
dry run only: set SNAP_HOOK to build the ZIP


In [124]:
import json, collections
meta = json.load(open(f"{BASE_DIR}/validation/validation/validationMetadata.json", encoding="utf-8"))
print("entries:", len(meta)); print("top-level keys:", sorted({k for r in meta for k in r}))
for r in meta[:2]: print(json.dumps(r, indent=1)[:900])
ev_keys = collections.Counter(k for r in meta for e in r["NoiseSubCategoryTimeStamp"] for k in e)
print("event keys:", dict(ev_keys))
for dom in (False, True):
    rs = [r for r in meta if bool(r["syntheticData"]) == dom]
    n = [len(r["NoiseSubCategoryTimeStamp"]) for r in rs]
    print(("SYN" if dom else "NAT"), len(rs), "events/clip mean %.2f" % (sum(n) / len(n)), "| clips with 0 events:", sum(x == 0 for x in n),
          "| category fields:", sorted({k for r in rs for k in r if "ategor" in k}))

entries: 2501
top-level keys: ['NoiseCategory', 'NoiseSubCategoryTimeStamp', 'annotationQuality', 'district', 'duration', 'imageFileName', 'isTranscriptionAvailable', 'language', 'segmentFileName', 'state', 'syntheticData', 'transcript']
{
 "segmentFileName": "IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213767_010_LowerDibangvalley-SPECIFIC_01478_3234_6516.wav",
 "imageFileName": "Images/IISc_VaaniProject_LowerDibangvalley-SPECIFIC_01478.jpg",
 "state": "ArunachalPradesh",
 "district": "LowerDibangvalley",
 "duration": 3.247,
 "language": "Assamese",
 "annotationQuality": "verified_timestamps",
 "isTranscriptionAvailable": true,
 "transcript": "\u0987\u09af\u09bc\u09be\u09a4 <bird sound> \u09ae\u0987 \u09a4\u09bf\u09a8\u09bf\u099f\u09be \u09ac\u09c7\u09b2\u09c7\u0997 </bird sound> <horn> \u0995\u09be\u09b2\u09be\u09f0 \u09f0 \u09b6\u09cd\u09ac\u09be\u09f0\u09cd\u099f </horn> <barking> \u09a4\u09bf\u09a8\u09bf\u099f\u09be \u0989\u09b2\u09be\u0987 \u0

In [125]:
# =============================================================================
# OV-2 -- ATST-e2 on the official validation set + Candidate A blend verdict (CPU decode, GPU inference).
# Run AFTER ov1_offval_fixed.py in the same kernel. Needs: OV, OVK, OV_REFS, OVT, ov_beats, FIXED_DEC, ov_eval, ov_report
# (from OV-1) and the integration functions (atst_from_ckpt-style loading, tracks_atst, atst_time_grid, new_atst_model, ...).
# Writes only NEW files:
#   <BASE_DIR>/atst_cache_offval/<N_|S_><stem>.npy      fp16 ATST tokens for the 2,501 clips
#   <V5_DIR>/v5c_atst_fusion_e2_offval_tracks32.pkl     float32 tracks
# =============================================================================
import os, json, pickle, time
import numpy as np, torch

E2_TAG = "v5c_atst_fusion_e2"
OV_ATST_DIR = f"{BASE_DIR}/atst_cache_offval"
_SHA = ATST_EXPECTED_SHA256 or "8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2"
assert AC.sha256_file(ATST_CKPT) == _SHA, "ATST checkpoint file differs from the pinned one"
os.makedirs(OV_ATST_DIR, exist_ok=True)

# ---- 1. ATST features (same extractor / padding rule as the test cache) -----------------
_enc = AC.load_atst_encoder(ATST_CKPT, device=device, mel_variant=ATST_MEL_VARIANT, expected_sha256=_SHA)
_ex = AC.ATSTFeatureExtractor(_enc, device)
_new = 0; t0 = time.time()
for j, (k, v) in enumerate(OV.items()):
    out = os.path.join(OV_ATST_DIR, f"{k}.npy")
    if os.path.exists(out):
        continue
    w = load_test_wav(v["path"])
    feat = _ex(w)
    exp = AC.atst_T_expected(max(w.numel(), SR))
    if feat.shape != (exp, ATST_DIM) or not np.isfinite(feat).all():
        raise RuntimeError(f"{k}: bad ATST features {feat.shape} vs ({exp},{ATST_DIM})")
    np.save(out + ".tmp.npy", feat.astype(np.float16)); os.replace(out + ".tmp.npy", out); _new += 1
    if _new % 500 == 0:
        print(f"  ATST {j + 1}/{len(OV)} ({time.time() - t0:.0f}s)")
log(f"OV ATST: {_new} new, {len(OV)} total in {OV_ATST_DIR}")
del _enc, _ex
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def ov_atst(k):
    return np.load(os.path.join(OV_ATST_DIR, f"{k}.npy")).astype(np.float32)


# ---- 2. e2 checkpoint: bit-identical weights + reproduces its stored Val-N tracks ----------
def ov_load_atst(tag, kind="fuse", n_check=3, atol=2e-3):
    dj = json.load(open(f"{V5_DIR}/{tag}_decoder.json"))
    ck = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    assert ck["hash"] == dj["hash"], f"{tag}: decoder.json hash != checkpoint hash"
    net = new_atst_model(kind, ck["cfg"]); net.load_state_dict(ck["model"]); net.eval()
    sd = net.state_dict()
    bad = [k for k, v in ck["model"].items() if k not in sd or not torch.equal(v.cpu(), sd[k].detach().cpu())]
    assert not bad, f"loaded weights differ from file: {bad[:5]}"
    T_win, fps = atst_time_grid(net)
    vt = pickle.load(open(f"{V5_DIR}/{tag}_val_tracks.pkl", "rb"))
    cache = atst_open_cache(create=False)
    worst = 0.0
    for k in list(VALN)[:n_check]:
        tr = tracks_atst(net, VALN[k]["wav"], VALN[k]["beats"].astype(np.float32), cache.load("nat", int(k)).astype(np.float32), device, T_win, fps)
        for name in ("cat", "onset", "offset"):
            worst = max(worst, float(np.abs(np.asarray(tr[name], np.float32) - np.asarray(vt["N"][k][name], np.float32)).max()))
    assert worst <= atol, f"STOP: re-tracked Val-N differs from stored val tracks by {worst:.4g}"
    log(f"OV load {tag}: weights bit-identical; reproduces {n_check} stored Val-N tracks (max |diff| {worst:.2e}); stored-hash match: "
        f"{state_hash(sd) == ck['hash']}")
    return net, ck, T_win, fps


_p = f"{V5_DIR}/{E2_TAG}_offval_tracks32.pkl"
if os.path.exists(_p):
    E2 = pickle.load(open(_p, "rb")); assert set(E2["tracks"]) == set(OV)
else:
    net, ck, T_win, fps = ov_load_atst(E2_TAG)
    tr = {}
    for k, v in OV.items():
        w = load_test_wav(v["path"])
        tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    E2 = {"fps": fps, "tracks": tr, "hash": ck["hash"], "checkpoint": json.load(open(f"{V5_DIR}/{E2_TAG}_decoder.json"))["checkpoint"]}
    pickle.dump(E2, open(_p, "wb"))
    del net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
OVT[E2_TAG] = E2
OWN_DEC[E2_TAG] = json.load(open(f"{V5_DIR}/{E2_TAG}_decoder.json"))["decoder"]
assert abs(E2["fps"] - OVT["v5c_smel"]["fps"]) < 1e-9

# ---- 3. per-domain scores on the official validation set (fixed decoder, and each model's own) ----
for t in ("v5a", "v5c_smel", E2_TAG):
    ov_report(t, OVT[t]["tracks"], OVT[t]["fps"], FIXED_DEC, "v5a-dec")
for t in ("v5a", "v5c_smel", E2_TAG):
    ov_report(t, OVT[t]["tracks"], OVT[t]["fps"], OWN_DEC[t], "own-dec")
print("Leaderboard ref (F1/Dice): v5a NAT .4056/.6627 SYN .5789/.7836 | smel NAT .3735/.6714 SYN .6400/.8638 | e2 NAT .4053/.6717 SYN .6865/.7909")

# ---- 4. Candidate A on the OFFICIAL validation set (predeclared variants and rule) ------------
fps = E2["fps"]; TA, TB = OVT["v5c_smel"]["tracks"], E2["tracks"]      # A = short-mel (boundaries), B = ATST fusion


def cat_blend(ta, tb, w, bnd="a"):
    src = ta if bnd == "a" else tb
    return {"duration": ta["duration"], "onset": src["onset"], "offset": src["offset"],
            "cat": ((1 - w) * ta["cat"].astype(np.float64) + w * tb["cat"].astype(np.float64)).astype(np.float32)}


VARIANTS = {"smel": lambda a, b: a, "atst": lambda a, b: b,
            "cat25": lambda a, b: cat_blend(a, b, .25), "cat50": lambda a, b: cat_blend(a, b, .50), "cat75": lambda a, b: cat_blend(a, b, .75),
            "cat50_atstbnd": lambda a, b: cat_blend(a, b, .50, "b"), "full50": lambda a, b: VE.blend_one(a, b, .5, "prob")}
for k in OV:
    assert TA[k]["cat"].shape == TB[k]["cat"].shape and abs(TA[k]["duration"] - TB[k]["duration"]) < 1e-6, k
me, ef = match_events, events_to_frames
R = {}
for name, f in VARIANTS.items():
    R[name] = {}
    for dom in ("N", "S"):
        ks = OVK[(dom, "all")]; ix = {k: i for i, k in enumerate(ks)}
        pred = {k: decode_v5(f(TA[k], TB[k]), FIXED_DEC, fps) for k in ks}
        st = VE.stats_for(pred, OV_REFS, ks, me, ef)
        halves = {h: VE.aggregate(st[[ix[k] for k in OVK[(dom, h)]]])["combined"] for h in "AB"}
        R[name][dom] = dict(st=st, agg=VE.aggregate(st), halves=halves, groups=[OV[k]["district"] for k in ks])
        a = R[name][dom]["agg"]
        print(f"{name:<14} {'NAT' if dom == 'N' else 'SYN'} comb {a['combined']:.4f} (F1 {a['event_f1']:.4f} P {a['precision']:.3f} "
              f"R {a['recall']:.3f} Dice {a['segment_dice']:.4f}) halves {halves['A']:.4f}/{halves['B']:.4f}")
print("\nPREDECLARED RULE: delta >= +0.01, both halves delta > 0, district-bootstrap P(>0) >= 0.90")
for dom, ctrl in (("N", "atst"), ("S", "smel")):
    for name in VARIANTS:
        if name in ("smel", "atst"):
            continue
        r, c = R[name][dom], R[ctrl][dom]
        d = r["agg"]["combined"] - c["agg"]["combined"]; dA, dB = (r["halves"][h] - c["halves"][h] for h in "AB")
        bs = VE.paired_cluster_bootstrap(c["st"], r["st"], r["groups"], n_boot=2000)
        ok = d >= .01 and dA > 0 and dB > 0 and bs["p_gt0"] >= .90
        print(f"{'NAT' if dom == 'N' else 'SYN'} {name:<14} vs {ctrl}: delta {d:+.4f} halves {dA:+.4f}/{dB:+.4f} "
              f"CI90 [{bs['ci90'][0]:+.4f},{bs['ci90'][1]:+.4f}] P>0 {bs['p_gt0']:.2f} -> {'PASS' if ok else 'reject'}")

[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram: {'encoder.encoder.blocks': 132, 'encoder.encoder.patch_embed': 2, 'encoder.encoder.norm_frame': 2, 'encoder.encoder.mask_embed': 1, 'encoder.encoder.pos_embed': 1, 'head.norm.running_mean': 1, 'head.norm.running_var': 1, 'head.norm.num_batches_tracked': 1, 'head.linear.weight': 1, 'head.linear.bias': 1}
   layout_counts: {'c2f_frame_encoder': 0, 'lightning_teacher': 0, 'downstream_finetuned': 138, 'pretrainedsed_wrapper': 0}
   hyper_parameters: {'learning_rate': 3.125, 'max_epochs': 11, 'niter_per_epoch': 1208, 'warmup_epochs': 2, 'num_labels': 527, 'multi_label': True, 'layer_wise_lr': 0.75, 'freeze_embed': Fa

In [126]:
# =============================================================================
# OV-4 -- decoder tuning on the OFFICIAL validation set, cross-fitted by district half (CPU only, writes nothing).
# Run AFTER ov1_offval_fixed.py and ov2_atst_offval.py in the same kernel.
# Needs: OV, OVK, OV_REFS, OVT (v5a, v5c_smel, v5c_atst_fusion_e2), FIXED_DEC, OWN_DEC, decode_v5, match_events, events_to_frames, VE.
# Predeclared grid (24 configs): threshold shift x min-duration scale x merge-gap scale. Other decoder fields unchanged.
# Protocol: pick the best config on half A, score it on half B (and the reverse). Baseline = the decoder used in production.
# =============================================================================
import itertools, numpy as np

E2_TAG = "v5c_atst_fusion_e2"
THR_SHIFTS = (-0.20, -0.15, -0.10, -0.05, 0.0, 0.05)
MINDUR_SCALES = (1.0, 0.7)
GAP_SCALES = (1.0, 1.5)
for key in ("thr_short", "thr_long", "mindur_short", "mindur_long", "gap_short", "gap_long"):
    assert key in FIXED_DEC, f"decoder has no '{key}': fields are {sorted(FIXED_DEC)}"


def make_grid(base):
    out = []
    for dt, md, gm in itertools.product(THR_SHIFTS, MINDUR_SCALES, GAP_SCALES):
        d = dict(base)
        d["thr_short"] = float(np.clip(base["thr_short"] + dt, 0.05, 0.95)); d["thr_long"] = float(np.clip(base["thr_long"] + dt, 0.05, 0.95))
        d["mindur_short"] = base["mindur_short"] * md; d["mindur_long"] = base["mindur_long"] * md
        d["gap_short"] = base["gap_short"] * gm; d["gap_long"] = base["gap_long"] * gm
        out.append(((dt, md, gm), d))
    return out


me, ef = match_events, events_to_frames
TASKS = [  # (label, model tag, domain, production decoder)
    ("e2   NAT", E2_TAG, "N", FIXED_DEC),
    ("e2   SYN", E2_TAG, "S", FIXED_DEC),
    ("smel SYN", "v5c_smel", "S", OWN_DEC["v5c_smel"])]
agg = VE.aggregate
FINAL = {}
for label, tag, dom, base in TASKS:
    tr, fps = OVT[tag]["tracks"], OVT[tag]["fps"]
    ks = OVK[(dom, "all")]; ix = {k: i for i, k in enumerate(ks)}
    iA = [ix[k] for k in OVK[(dom, "A")]]; iB = [ix[k] for k in OVK[(dom, "B")]]
    dist = np.array([OV[k]["district"] for k in ks])

    def stats(dec):
        pred = {k: decode_v5(tr[k], dec, fps) for k in ks}
        return VE.stats_for(pred, OV_REFS, ks, me, ef), float(np.mean([len(pred[k]) for k in ks]))

    st_base, ev_base = stats(base)
    cfgs = make_grid(base); S = []
    for (cfg, d) in cfgs:
        S.append(stats(d))
    comb = lambda s, idx: agg(s[idx])["combined"]
    print(f"\n=== {label}: production decoder  all-clips {agg(st_base)['combined']:.4f} (F1 {agg(st_base)['event_f1']:.4f} "
          f"P {agg(st_base)['precision']:.3f} R {agg(st_base)['recall']:.3f} Dice {agg(st_base)['segment_dice']:.4f}) ev/clip {ev_base:.2f}")
    ok_folds = 0
    for fit, test, i_fit, i_test in (("A", "B", iA, iB), ("B", "A", iB, iA)):
        best = int(np.argmax([comb(s[0], i_fit) for s in S])); cfg, d = cfgs[best]; st = S[best][0]
        delta = comb(st, i_test) - comb(st_base, i_test)
        bs = VE.paired_cluster_bootstrap(st_base[i_test], st[i_test], dist[i_test], n_boot=2000)
        a = agg(st[i_test])
        passed = delta >= 0.01 and bs["p_gt0"] >= 0.90; ok_folds += passed
        print(f"  tuned on half {fit} -> cfg (thr {cfg[0]:+.2f}, mindur x{cfg[1]}, gap x{cfg[2]}) | held-out half {test}: "
              f"{comb(st_base, i_test):.4f} -> {comb(st, i_test):.4f} (delta {delta:+.4f}, CI90 [{bs['ci90'][0]:+.4f},{bs['ci90'][1]:+.4f}], "
              f"P>0 {bs['p_gt0']:.2f}) F1 {a['event_f1']:.4f} P {a['precision']:.3f} R {a['recall']:.3f} Dice {a['segment_dice']:.4f} "
              f"ev/clip {S[best][1]:.2f} -> {'PASS' if passed else 'reject'}")
    allbest = int(np.argmax([agg(s[0])["combined"] for s in S])); cfg, d = cfgs[allbest]
    FINAL[label] = {"both_folds_pass": ok_folds == 2, "cfg": cfg, "decoder": d, "all_clips": agg(S[allbest][0])["combined"],
                    "baseline_all": agg(st_base)["combined"]}
    print(f"  best on all clips (for reference only): cfg {cfg} {agg(S[allbest][0])['combined']:.4f} vs production {agg(st_base)['combined']:.4f} "
          f"| accepted by both folds: {ok_folds == 2}")

print("\nSUMMARY (accept a decoder change only where BOTH cross-fit folds pass):")
for label, r in FINAL.items():
    print(f"  {label}: {'ACCEPT' if r['both_folds_pass'] else 'keep production decoder'} | all-clips {r['baseline_all']:.4f} -> {r['all_clips']:.4f}, cfg {r['cfg']}")
print("If 'e2 SYN' beats 'smel SYN' by a clear margin, e2 with its tuned decoder could replace short-mel as the synthetic source (compare the held-out numbers).")


=== e2   NAT: production decoder  all-clips 1.0119 (F1 0.3491 P 0.367 R 0.333 Dice 0.6627) ev/clip 1.48
  tuned on half A -> cfg (thr -0.20, mindur x1.0, gap x1.5) | held-out half B: 1.0176 -> 1.0103 (delta -0.0073, CI90 [-0.0252,+0.0140], P>0 0.27) F1 0.3504 P 0.340 R 0.361 Dice 0.6599 ev/clip 1.74 -> reject
  tuned on half B -> cfg (thr -0.05, mindur x1.0, gap x1.5) | held-out half A: 1.0041 -> 1.0277 (delta +0.0237, CI90 [+0.0144,+0.0329], P>0 1.00) F1 0.3511 P 0.364 R 0.339 Dice 0.6767 ev/clip 1.52 -> PASS
  best on all clips (for reference only): cfg (-0.1, 1.0, 1.5) 1.0279 vs production 1.0119 | accepted by both folds: False

=== e2   SYN: production decoder  all-clips 1.5017 (F1 0.6786 P 0.778 R 0.602 Dice 0.8232) ev/clip 2.46
  tuned on half A -> cfg (thr -0.20, mindur x1.0, gap x1.0) | held-out half B: 1.4958 -> 1.4779 (delta -0.0179, CI90 [-0.0359,+0.0068], P>0 0.10) F1 0.6460 P 0.716 R 0.589 Dice 0.8319 ev/clip 2.65 -> reject
  tuned on half B -> cfg (thr +0.00, mindur x1.0

In [127]:
tr, fps = OVT[E2_TAG]["tracks"], OVT[E2_TAG]["fps"]
ks = OVK[("N", "all")]; ix = {k: i for i, k in enumerate(ks)}
iA = [ix[k] for k in OVK[("N", "A")]]; iB = [ix[k] for k in OVK[("N", "B")]]
def st(dec): return VE.stats_for({k: decode_v5(tr[k], dec, fps) for k in ks}, OV_REFS, ks, me, ef)
b = st(FIXED_DEC); bA, bB = agg(b[iA])["combined"], agg(b[iB])["combined"]
rows = []
for cfg, d in make_grid(FIXED_DEC):
    s = st(d); dA, dB = agg(s[iA])["combined"] - bA, agg(s[iB])["combined"] - bB
    rows.append((min(dA, dB), cfg, dA, dB))
for m, cfg, dA, dB in sorted(rows, reverse=True)[:8]:
    print(f"thr {cfg[0]:+.2f} mindur x{cfg[1]} gap x{cfg[2]} | delta half A {dA:+.4f} half B {dB:+.4f}")

thr -0.05 mindur x1.0 gap x1.5 | delta half A +0.0237 half B +0.0091
thr -0.05 mindur x0.7 gap x1.5 | delta half A +0.0228 half B +0.0086
thr -0.10 mindur x1.0 gap x1.5 | delta half A +0.0306 half B +0.0037
thr -0.05 mindur x1.0 gap x1.0 | delta half A +0.0205 half B +0.0031
thr +0.00 mindur x1.0 gap x1.5 | delta half A +0.0027 half B +0.0033
thr -0.10 mindur x0.7 gap x1.5 | delta half A +0.0291 half B +0.0026
thr -0.05 mindur x0.7 gap x1.0 | delta half A +0.0197 half B +0.0026
thr +0.00 mindur x0.7 gap x1.5 | delta half A +0.0025 half B +0.0027


In [128]:
import pickle, json, numpy as np
vt = pickle.load(open(f"{V5_DIR}/v5c_atst_fusion_e2_val_tracks.pkl", "rb")); fpsN = vt["fps"]; trN = vt["N"]
cand = dict(FIXED_DEC)
for key in ("thr_short", "thr_long"): cand[key] = FIXED_DEC[key] - 0.05
for key in ("gap_short", "gap_long"): cand[key] = FIXED_DEC[key] * 1.5
kN = list(VALN_REFS); ixN = {k: i for i, k in enumerate(kN)}
def stN(dec):
    pred = {k: decode_v5(trN[k], dec, fpsN) for k in kN}
    return VE.stats_for(pred, VALN_REFS, kN, match_events, events_to_frames), float(np.mean([len(pred[k]) for k in kN]))
(sb, eb), (sc, ec) = stN(FIXED_DEC), stN(cand)
hA, hB = [ixN[k] for k in VALN_HALF["A"]], [ixN[k] for k in VALN_HALF["B"]]
c = lambda s, i=slice(None): VE.aggregate(s[i])["combined"]
dist = [VALN[k]["district"] for k in kN]
bs = VE.paired_cluster_bootstrap(sb, sc, dist, n_boot=2000)
print(f"Val-N e2: production {c(sb):.4f} -> candidate {c(sc):.4f} (delta {c(sc) - c(sb):+.4f}) | halves A {c(sc, hA) - c(sb, hA):+.4f} B {c(sc, hB) - c(sb, hB):+.4f} "
      f"| CI90 [{bs['ci90'][0]:+.4f},{bs['ci90'][1]:+.4f}] P>0 {bs['p_gt0']:.2f} | ev/clip {eb:.2f} -> {ec:.2f}")
ok = (c(sc) - c(sb)) >= 0.005 and c(sc, hA) > c(sb, hA) and c(sc, hB) > c(sb, hB)
print("PASS (build a candidate)" if ok else "REJECT (keep the production decoder)")

Val-N e2: production 0.9404 -> candidate 0.9500 (delta +0.0096) | halves A +0.0110 B +0.0083 | CI90 [-0.0002,+0.0207] P>0 0.95 | ev/clip 1.52 -> 1.58
PASS (build a candidate)


In [129]:
import zipfile, hashlib, os
SNAPPED = f"{SUB_DIR}/submission_route_e2_smel_snapW200.zip"
OUT = f"{SUB_DIR}/submission_route_e2_smel_snapW200_natdec.zip"
assert not os.path.exists(OUT), OUT
old_ids = set(json.load(open(f"{V5_DIR}/test_syn_ids.json")))
t32 = pickle.load(open(f"{V5_DIR}/v5c_atst_fusion_e2_test_tracks32.pkl", "rb"))
with zipfile.ZipFile(SNAPPED) as z:
    rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
snap = {r["clip_id"]: [(e["onset"], e["offset"]) for e in r["events"]] for r in rows}
assert set(snap) == set(t32["tracks"]) and len(snap) == 5517
preds, durs, changed = {}, {}, 0
for cid, tr in t32["tracks"].items():
    durs[cid] = tr["duration"]
    if cid in old_ids:
        preds[cid] = snap[cid]                                    # synthetic: untouched (short-mel + SNAP)
    else:
        preds[cid] = decode_v5(tr, cand, t32["fps"])             # natural: e2 with the candidate decoder
        changed += (list(map(tuple, preds[cid])) != list(map(tuple, snap[cid])))
zp = write_submission(preds, SUB_DIR, os.path.basename(OUT), durs, expected=EXPECTED_TEST_CLIPS)
atst_verify_zip(zp, expected_ids=set(snap))
with zipfile.ZipFile(zp) as z:
    new = {json.loads(l)["clip_id"]: [(e["onset"], e["offset"]) for e in json.loads(l)["events"]] for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()}
assert all(np.allclose(np.array(new[c]).ravel(), np.array(snap[c]).ravel(), atol=1e-6) and len(new[c]) == len(snap[c]) for c in old_ids), "synthetic clips changed!"
print(f"{zp} | natural clips with changed events: {changed} | synthetic clips identical to the SNAP ZIP: {len(old_ids)}")

  wrote C:/Users/231b258/submission\submission_route_e2_smel_snapW200_natdec.zip: 5517 clips, 14019 events, 45 empty
zip OK: C:/Users/231b258/submission\submission_route_e2_smel_snapW200_natdec.zip: 5517 clips, 14019 events
C:/Users/231b258/submission\submission_route_e2_smel_snapW200_natdec.zip | natural clips with changed events: 1355 | synthetic clips identical to the SNAP ZIP: 2080


In [131]:
# =============================================================================
# OV-3 -- does training on the OFFICIAL validation clips help? Held-out protocol by district half.
# Run AFTER ov1_offval_fixed.py and ov2_atst_offval.py in the same kernel (needs OV, OVK, _meta, ov_beats, ov_atst, ov_eval,
# FIXED_DEC, E2 and the integration functions). Writes only NEW files tagged TAG. Nothing is overwritten.
# CHECK THE COMPETITION RULES FIRST: training on the official validation clips may be disallowed.
# =============================================================================
import os, json, math, time, random
import numpy as np, torch
from collections import defaultdict

RUN_FT = True                  # flip to True to actually train (the False run only prints the plan and runs a 6-step smoke test)
FOLD = "A"                      # train on official half FOLD, evaluate on the other half
UPDATES = 2000                  # fine-tune length (~40 min). The e2 run used 8000.
EVAL_EVERY = 500
FT_LR_MULT = 0.3                # fraction of the original peak lr
OV_SHARE = 0.35                 # fraction of each batch drawn from the official clips of the training half
BASE_TAG = "v5c_atst_fusion_e2"
TAG = f"v5c_atst_fusion_ovft_{FOLD}"
EVAL_HALF = "B" if FOLD == "A" else "A"

# ---- official clips as training rows -------------------------------------------------------
OV_EVS, OV_TIER, dropped, q_count = {}, {}, 0, defaultdict(int)
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]; syn = bool(r["syntheticData"])
    key = ("S_" if syn else "N_") + stem
    raw = r["NoiseSubCategoryTimeStamp"]
    evs = parse_events(raw, None)
    dropped += len(raw) - len(evs)
    OV_EVS[key] = evs
    q_count[(syn, r.get("annotationQuality"))] += 1
    OV_TIER[key] = T_SYNTH if syn else T_GOLD
print("annotationQuality by (synthetic, quality):", dict(q_count))
print(f"events dropped by parse_events (category not among the 7): {dropped}")
ov_train_keys = [k for k in OV if OV[k]["half"] == FOLD and OV_EVS[k]]          # clips with no usable event are skipped
print(f"training half {FOLD}: {len(ov_train_keys)} clips | evaluation half {EVAL_HALF}: "
      f"{len(OVK[('N', EVAL_HALF)])} natural + {len(OVK[('S', EVAL_HALF)])} synthetic")
assert not set(ov_train_keys) & set(OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]), "train/eval overlap"
assert {OV[k]["district"] for k in ov_train_keys}.isdisjoint({OV[k]["district"] for k in OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]})


class OVMixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("ov", key, tier) served from the official validation clips (same crop/label/weak-target logic)."""
    def _load(self, kind, key):
        if kind != "ov":
            return super()._load(kind, key)
        wav = load_test_wav(OV[key]["path"]); bf = ov_beats(key, wav.numel())
        evs = OV_EVS[key]
        return wav, bf, evs, sorted({c for _, _, c in evs})

    def _full(self, kind, key):
        return ov_atst(key) if kind == "ov" else super()._full(kind, key)


def make_mix_sampler(base_rows, ov_rows, shares, ov_share, n):
    from torch.utils.data import WeightedRandomSampler
    from collections import Counter
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    w = [(1 - ov_share) * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows] + [ov_share / len(ov_rows)] * len(ov_rows)
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_half_scores(net, T_win, fps, half):
    ks = OVK[("N", half)] + OVK[("S", half)]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, half) for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    rows = list(TRAIN_ROWS) + [("ov", k, OV_TIER[k]) for k in ov_train_keys]
    ds = OVMixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                      shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    ov_rows = [r for r in rows if r[0] == "ov"]
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler(list(TRAIN_ROWS), ov_rows, cfg["shares"], OV_SHARE, n_draw)
    # the sampler indexes `rows`: base rows first, official rows last (same order as built above)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = {h: None for h in (EVAL_HALF,)}
    if run:
        base[EVAL_HALF] = {dom: ov_eval(E2["tracks"], E2["fps"], FIXED_DEC, dom, EVAL_HALF) for dom in ("N", "S")}
        base[EVAL_HALF]["J"] = 0.62 * base[EVAL_HALF]["N"]["combined"] + 0.38 * base[EVAL_HALF]["S"]["combined"]
        log(f"BASELINE {BASE_TAG} on held-out half {EVAL_HALF}: J {base[EVAL_HALF]['J']:.4f} | {fmt(base[EVAL_HALF])}")
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if run and (upd % EVAL_EVERY == 0 or upd >= updates):
            tr = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            line = f"{TAG} upd={upd} skipped={skipped} {tr}"
            for name, net in (("student", model), ("ema", ema)):
                net.eval(); sc = ov_half_scores(net, T_win, fps, EVAL_HALF); hist.append({"update": upd, name: sc["J"]})
                line += f"\n    {name}: J {sc['J']:.4f} (baseline {base[EVAL_HALF]['J']:.4f}, delta {sc['J'] - base[EVAL_HALF]['J']:+.4f}) | {fmt(sc)}"
            model.train(); log(line)
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sc = ov_half_scores(net, T_win, fps, EVAL_HALF); sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        torch.save({"model": sd, "update": upd, "score": {"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]}, "hash": state_hash(sd),
                    "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "ov_train_half": FOLD, "ov_eval_half": EVAL_HALF, "ov_share": OV_SHARE, "updates": upd}}, p)
        paths[name] = p
    json.dump({"hist": hist, "baseline_J": base[EVAL_HALF]["J"], "fold": FOLD}, open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


# smoke test (6 updates, no evaluation, nothing saved) always runs; the real run needs RUN_FT = True
finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OVFT_PATHS = finetune(UPDATES, run=True)
    print(OVFT_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG} ({UPDATES} updates, ~40 min).")

annotationQuality by (synthetic, quality): {(True, 'verified_timestamps'): 695, (False, 'verified_timestamps'): 1806}
events dropped by parse_events (category not among the 7): 0
training half A: 1216 clips | evaluation half B: 955 natural + 328 synthetic
[2026-10-14 17:01:45] v5c_atst_fusion_ovft_A: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-14 17:01:45] BASELINE v5c_atst_fusion_e2 on held-out half B: J 1.1993 | NAT 1.0176 (F1 0.3592 Dice 0.6584 ev/clip 1.55) | SYN 1.4958 (F1 0.6809 Dice 0.8149 ev/clip 2.50)
[2026-10-14 17:09:48] v5c_atst_fusion_ovft_A upd=500 skipped=0 frame=0.0405 onset=0.0723 offset=0.0739 dice=0.2878 weak=0.2165
    student: J 1.2384 (baseline 1.1993, delta +0.0391) | NAT 1.0273 (F1 0.3628 Dice 0.6644 ev/clip 1.63) | SYN 1.5828 (F1 0.7384 Dice 0.8444 ev/clip 2.57)
    ema: J 1.2124 (baseline 1.1993, delta +0.0131) | NAT 1.0182 (F1 0.3568 Dice 0.6614 ev/clip 1.57) | SYN 1.5294 (F1 0.7018 Dice 0.8275 ev/clip 2.55)
[2026-10-1

In [134]:
# =============================================================================
# OV-3 -- does training on the OFFICIAL validation clips help? Held-out protocol by district half.
# Run AFTER ov1_offval_fixed.py and ov2_atst_offval.py in the same kernel (needs OV, OVK, _meta, ov_beats, ov_atst, ov_eval,
# FIXED_DEC, E2 and the integration functions). Writes only NEW files tagged TAG. Nothing is overwritten.
# CHECK THE COMPETITION RULES FIRST: training on the official validation clips may be disallowed.
# =============================================================================
import os, json, math, time, random
import numpy as np, torch
from collections import defaultdict

RUN_FT = True                  # flip to True to actually train (the False run only prints the plan and runs a 6-step smoke test)
FOLD = "B"                      # train on official half FOLD, evaluate on the other half
UPDATES = 2000                  # fine-tune length (~40 min). The e2 run used 8000.
EVAL_EVERY = 500
FT_LR_MULT = 0.3                # fraction of the original peak lr
OV_SHARE = 0.35                 # fraction of each batch drawn from the official clips of the training half
BASE_TAG = "v5c_atst_fusion_e2"
TAG = f"v5c_atst_fusion_ovft_{FOLD}"
EVAL_HALF = "B" if FOLD == "A" else "A"

# ---- official clips as training rows -------------------------------------------------------
OV_EVS, OV_TIER, dropped, q_count = {}, {}, 0, defaultdict(int)
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]; syn = bool(r["syntheticData"])
    key = ("S_" if syn else "N_") + stem
    raw = r["NoiseSubCategoryTimeStamp"]
    evs = parse_events(raw, None)
    dropped += len(raw) - len(evs)
    OV_EVS[key] = evs
    q_count[(syn, r.get("annotationQuality"))] += 1
    OV_TIER[key] = T_SYNTH if syn else T_GOLD
print("annotationQuality by (synthetic, quality):", dict(q_count))
print(f"events dropped by parse_events (category not among the 7): {dropped}")
ov_train_keys = [k for k in OV if OV[k]["half"] == FOLD and OV_EVS[k]]          # clips with no usable event are skipped
print(f"training half {FOLD}: {len(ov_train_keys)} clips | evaluation half {EVAL_HALF}: "
      f"{len(OVK[('N', EVAL_HALF)])} natural + {len(OVK[('S', EVAL_HALF)])} synthetic")
assert not set(ov_train_keys) & set(OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]), "train/eval overlap"
assert {OV[k]["district"] for k in ov_train_keys}.isdisjoint({OV[k]["district"] for k in OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]})


class OVMixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("ov", key, tier) served from the official validation clips (same crop/label/weak-target logic)."""
    def _load(self, kind, key):
        if kind != "ov":
            return super()._load(kind, key)
        wav = load_test_wav(OV[key]["path"]); bf = ov_beats(key, wav.numel())
        evs = OV_EVS[key]
        return wav, bf, evs, sorted({c for _, _, c in evs})

    def _full(self, kind, key):
        return ov_atst(key) if kind == "ov" else super()._full(kind, key)


def make_mix_sampler(base_rows, ov_rows, shares, ov_share, n):
    from torch.utils.data import WeightedRandomSampler
    from collections import Counter
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    w = [(1 - ov_share) * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows] + [ov_share / len(ov_rows)] * len(ov_rows)
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_half_scores(net, T_win, fps, half):
    ks = OVK[("N", half)] + OVK[("S", half)]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, half) for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    rows = list(TRAIN_ROWS) + [("ov", k, OV_TIER[k]) for k in ov_train_keys]
    ds = OVMixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                      shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    ov_rows = [r for r in rows if r[0] == "ov"]
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler(list(TRAIN_ROWS), ov_rows, cfg["shares"], OV_SHARE, n_draw)
    # the sampler indexes `rows`: base rows first, official rows last (same order as built above)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = {h: None for h in (EVAL_HALF,)}
    if run:
        base[EVAL_HALF] = {dom: ov_eval(E2["tracks"], E2["fps"], FIXED_DEC, dom, EVAL_HALF) for dom in ("N", "S")}
        base[EVAL_HALF]["J"] = 0.62 * base[EVAL_HALF]["N"]["combined"] + 0.38 * base[EVAL_HALF]["S"]["combined"]
        log(f"BASELINE {BASE_TAG} on held-out half {EVAL_HALF}: J {base[EVAL_HALF]['J']:.4f} | {fmt(base[EVAL_HALF])}")
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if run and (upd % EVAL_EVERY == 0 or upd >= updates):
            tr = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            line = f"{TAG} upd={upd} skipped={skipped} {tr}"
            for name, net in (("student", model), ("ema", ema)):
                net.eval(); sc = ov_half_scores(net, T_win, fps, EVAL_HALF); hist.append({"update": upd, name: sc["J"]})
                line += f"\n    {name}: J {sc['J']:.4f} (baseline {base[EVAL_HALF]['J']:.4f}, delta {sc['J'] - base[EVAL_HALF]['J']:+.4f}) | {fmt(sc)}"
            model.train(); log(line)
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sc = ov_half_scores(net, T_win, fps, EVAL_HALF); sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        torch.save({"model": sd, "update": upd, "score": {"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]}, "hash": state_hash(sd),
                    "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "ov_train_half": FOLD, "ov_eval_half": EVAL_HALF, "ov_share": OV_SHARE, "updates": upd}}, p)
        paths[name] = p
    json.dump({"hist": hist, "baseline_J": base[EVAL_HALF]["J"], "fold": FOLD}, open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


# smoke test (6 updates, no evaluation, nothing saved) always runs; the real run needs RUN_FT = True
finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OVFT_PATHS = finetune(UPDATES, run=True)
    print(OVFT_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG} ({UPDATES} updates, ~40 min).")

annotationQuality by (synthetic, quality): {(True, 'verified_timestamps'): 695, (False, 'verified_timestamps'): 1806}
events dropped by parse_events (category not among the 7): 0
training half B: 1283 clips | evaluation half A: 851 natural + 367 synthetic
[2026-10-15 11:50:50] v5c_atst_fusion_ovft_B: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-15 11:50:51] BASELINE v5c_atst_fusion_e2 on held-out half A: J 1.1952 | NAT 1.0041 (F1 0.3364 Dice 0.6676 ev/clip 1.39) | SYN 1.5070 (F1 0.6764 Dice 0.8305 ev/clip 2.43)
[2026-10-15 11:58:59] v5c_atst_fusion_ovft_B upd=500 skipped=0 frame=0.0414 onset=0.0700 offset=0.0727 dice=0.3004 weak=0.2134
    student: J 1.2449 (baseline 1.1952, delta +0.0497) | NAT 1.0163 (F1 0.3414 Dice 0.6749 ev/clip 1.44) | SYN 1.6179 (F1 0.7449 Dice 0.8730 ev/clip 2.57)
    ema: J 1.2131 (baseline 1.1952, delta +0.0179) | NAT 1.0082 (F1 0.3396 Dice 0.6686 ev/clip 1.40) | SYN 1.5474 (F1 0.7048 Dice 0.8426 ev/clip 2.48)
[2026-10-1

In [135]:
import json
for f in ("A", "B"):
    p = f"{V5_DIR}/v5c_atst_fusion_ovft_{f}_history.json"
    if not os.path.exists(p): print(f"fold {f}: not finished"); continue
    h = json.load(open(p)); base = h["baseline_J"]
    last = max(e["update"] for e in h["hist"])
    fin = {k: e[k] for e in h["hist"] if e["update"] == last for k in ("student", "ema") if k in e}
    print(f"fold {f} (tested on half {'B' if f == 'A' else 'A'}): baseline J {base:.4f} | final update {last}: "
          + " | ".join(f"{k} J {v:.4f} (delta {v - base:+.4f})" for k, v in fin.items()))

fold A (tested on half B): baseline J 1.1993 | final update 2000: student J 1.2400 (delta +0.0407) | ema J 1.2367 (delta +0.0374)
fold B (tested on half A): baseline J 1.1952 | final update 2000: student J 1.2552 (delta +0.0600) | ema J 1.2457 (delta +0.0505)


In [136]:
# =============================================================================
# OV-5 -- final model: ATST-e2 fine-tuned on ALL official validation clips (same recipe as the OV-3 folds). No held-out clips.
# Run AFTER ov1_offval_fixed.py and ov2_atst_offval.py in the same kernel (needs OV, OVK, _meta, ov_beats, ov_atst, ov_eval,
# FIXED_DEC, E2 and the integration functions). Writes only NEW files tagged TAG. Nothing is overwritten.
# CHECK THE COMPETITION RULES FIRST: training on the official validation clips may be disallowed.
# =============================================================================
import os, json, math, time, random
import numpy as np, torch
from collections import defaultdict

RUN_FT = True                   # trains for real (the 6-step smoke test still runs first)
FOLD = "ALL"                    # train on ALL 2,501 official clips: NO held-out evaluation exists for this run
UPDATES = 2000                  # fine-tune length (~40 min). The e2 run used 8000.
EVAL_EVERY = 500
FT_LR_MULT = 0.3                # fraction of the original peak lr
OV_SHARE = 0.35                 # fraction of each batch drawn from the official clips of the training half
BASE_TAG = "v5c_atst_fusion_e2"
TAG = f"v5c_atst_fusion_ovft_{FOLD}"
EVAL_HALF = None

# ---- official clips as training rows -------------------------------------------------------
OV_EVS, OV_TIER, dropped, q_count = {}, {}, 0, defaultdict(int)
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]; syn = bool(r["syntheticData"])
    key = ("S_" if syn else "N_") + stem
    raw = r["NoiseSubCategoryTimeStamp"]
    evs = parse_events(raw, None)
    dropped += len(raw) - len(evs)
    OV_EVS[key] = evs
    q_count[(syn, r.get("annotationQuality"))] += 1
    OV_TIER[key] = T_SYNTH if syn else T_GOLD
print("annotationQuality by (synthetic, quality):", dict(q_count))
print(f"events dropped by parse_events (category not among the 7): {dropped}")
ov_train_keys = [k for k in OV if OV_EVS[k]]          # clips with no usable event are skipped
print(f"training on ALL official clips: {len(ov_train_keys)} (no held-out clips; judge this model only by the fold results and the leaderboard)")


class OVMixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("ov", key, tier) served from the official validation clips (same crop/label/weak-target logic)."""
    def _load(self, kind, key):
        if kind != "ov":
            return super()._load(kind, key)
        wav = load_test_wav(OV[key]["path"]); bf = ov_beats(key, wav.numel())
        evs = OV_EVS[key]
        return wav, bf, evs, sorted({c for _, _, c in evs})

    def _full(self, kind, key):
        return ov_atst(key) if kind == "ov" else super()._full(kind, key)


def make_mix_sampler(base_rows, ov_rows, shares, ov_share, n):
    from torch.utils.data import WeightedRandomSampler
    from collections import Counter
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    w = [(1 - ov_share) * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows] + [ov_share / len(ov_rows)] * len(ov_rows)
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_half_scores(net, T_win, fps, half):
    ks = OVK[("N", half)] + OVK[("S", half)]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, half) for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    rows = list(TRAIN_ROWS) + [("ov", k, OV_TIER[k]) for k in ov_train_keys]
    ds = OVMixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                      shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    ov_rows = [r for r in rows if r[0] == "ov"]
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler(list(TRAIN_ROWS), ov_rows, cfg["shares"], OV_SHARE, n_draw)
    # the sampler indexes `rows`: base rows first, official rows last (same order as built above)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = {}
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if upd % 250 == 0:
            log(f"{TAG} upd={upd}/{updates} loss={float(loss):.4f}")
        if False:   # no held-out clips in the ALL run
            tr = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            line = f"{TAG} upd={upd} skipped={skipped} {tr}"
            for name, net in (("student", model), ("ema", ema)):
                net.eval(); sc = ov_half_scores(net, T_win, fps, EVAL_HALF); hist.append({"update": upd, name: sc["J"]})
                line += f"\n    {name}: J {sc['J']:.4f} (baseline {base[EVAL_HALF]['J']:.4f}, delta {sc['J'] - base[EVAL_HALF]['J']:+.4f}) | {fmt(sc)}"
            model.train(); log(line)
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sc = {"J": 0.0, "N": {"combined": 0.0}, "S": {"combined": 0.0}}; sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        torch.save({"model": sd, "update": upd, "score": {"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]}, "hash": state_hash(sd),
                    "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "ov_train_half": FOLD, "ov_eval_half": EVAL_HALF, "ov_share": OV_SHARE, "updates": upd}}, p)
        paths[name] = p
    json.dump({"hist": hist, "fold": FOLD, "updates": upd}, open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


# smoke test (6 updates, no evaluation, nothing saved) always runs; the real run needs RUN_FT = True
finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OVFT_PATHS = finetune(UPDATES, run=True)
    print(OVFT_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG} ({UPDATES} updates, ~40 min).")

annotationQuality by (synthetic, quality): {(True, 'verified_timestamps'): 695, (False, 'verified_timestamps'): 1806}
events dropped by parse_events (category not among the 7): 0
training on ALL official clips: 2499 (no held-out clips; judge this model only by the fold results and the leaderboard)
[2026-10-15 12:23:55] v5c_atst_fusion_ovft_ALL: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-15 12:27:30] v5c_atst_fusion_ovft_ALL upd=250/2000 loss=0.3351
[2026-10-15 12:31:08] v5c_atst_fusion_ovft_ALL upd=500/2000 loss=0.4047
[2026-10-15 12:34:45] v5c_atst_fusion_ovft_ALL upd=750/2000 loss=0.4900
[2026-10-15 12:38:23] v5c_atst_fusion_ovft_ALL upd=1000/2000 loss=0.4848
[2026-10-15 12:42:05] v5c_atst_fusion_ovft_ALL upd=1250/2000 loss=0.4493
[2026-10-15 12:45:51] v5c_atst_fusion_ovft_ALL upd=1500/2000 loss=0.5358
[2026-10-15 12:49:29] v5c_atst_fusion_ovft_ALL upd=1750/2000 loss=0.4564
[2026-10-15 12:53:09] v5c_atst_fusion_ovft_ALL upd=2000/2000 loss=0.4

In [137]:
TAG_ALL = "v5c_atst_fusion_ovft_ALL"
ck_all = f"{V5_DIR}/{TAG_ALL}_student.pt"
assert os.path.exists(ck_all), "ov5 has not finished: " + ck_all
p32 = atst_test_tracks32("fuse", TAG_ALL, atst_open_cache(create=False), ck_path=ck_all)
print("test tracks:", p32)

test tracks: C:/Users/231b258/v5_work/v5c_atst_fusion_ovft_ALL_test_tracks32.pkl


In [138]:
import zipfile, hashlib, pickle, json, os, numpy as np
BASE_ZIP = f"{SUB_DIR}/submission_route_e2_smel_snapW200_natdec.zip"     # the 1.2885 incumbent
OUT = f"{SUB_DIR}/submission_route_natdec_synFT_all.zip"
assert not os.path.exists(OUT), OUT
old_ids = set(json.load(open(f"{V5_DIR}/test_syn_ids.json")))
t = pickle.load(open(p32, "rb")); fps = t["fps"]
with zipfile.ZipFile(BASE_ZIP) as z:
    rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
base = {r["clip_id"]: [(e["onset"], e["offset"]) for e in r["events"]] for r in rows}
assert set(base) == set(t["tracks"]) and len(base) == 5517
preds, durs = {}, {}
for cid, tr in t["tracks"].items():
    durs[cid] = tr["duration"]
    preds[cid] = decode_v5(tr, FIXED_DEC, fps) if cid in old_ids else base[cid]
zp = write_submission(preds, SUB_DIR, os.path.basename(OUT), durs, expected=EXPECTED_TEST_CLIPS)
atst_verify_zip(zp, expected_ids=set(base))
nat_same = all(len(preds[c]) == len(base[c]) and np.allclose(np.array(preds[c], float).ravel(), np.array(base[c], float).ravel(), atol=1e-6)
               for c in set(base) - old_ids)
ev_new = np.mean([len(preds[c]) for c in old_ids]); ev_old = np.mean([len(base[c]) for c in old_ids])
print(zp, hashlib.sha1(open(zp, "rb").read()).hexdigest()[:12])
print(f"natural clips identical to the 1.2885 ZIP: {nat_same} | synthetic events/clip: new {ev_new:.2f} vs old {ev_old:.2f} "
      f"| synthetic empty clips: new {sum(len(preds[c]) == 0 for c in old_ids)} vs old {sum(len(base[c]) == 0 for c in old_ids)}")
assert nat_same, "natural clips changed: do not submit"

  wrote C:/Users/231b258/submission\submission_route_natdec_synFT_all.zip: 5517 clips, 11963 events, 47 empty
zip OK: C:/Users/231b258/submission\submission_route_natdec_synFT_all.zip: 5517 clips, 11963 events
C:/Users/231b258/submission\submission_route_natdec_synFT_all.zip 31717e1fc406
natural clips identical to the 1.2885 ZIP: True | synthetic events/clip: new 2.83 vs old 3.81 | synthetic empty clips: new 3 vs old 1


In [139]:
import numpy as np, collections
def summarize(name, items):
    n, ov, cov, ed = [], [], [], []
    for D, evs in items:
        evs = sorted(evs); n.append(len(evs)); ed += [e - s for s, e in evs]
        ov.append(any(evs[i][1] > evs[i + 1][0] for i in range(len(evs) - 1)))
        u, cur = 0.0, -1.0
        for s, e in evs:
            s2 = max(s, cur)
            if e > s2: u += e - s2; cur = max(cur, e)
        cov.append(u / max(D, 1e-6))
    c = collections.Counter(min(x, 5) for x in n)
    print(f"{name:<18} clips {len(n):>5} | events/clip {np.mean(n):.2f} | "
          + " ".join(f"{k}{'+' if k == 5 else ''}:{100 * c[k] / len(n):.0f}%" for k in range(1, 6))
          + f" | clips with overlap {100 * np.mean(ov):.0f}% | coverage {100 * np.mean(cov):.0f}% | event dur p10/med/p90 "
          + "/".join(f"{np.percentile(ed, q):.2f}" for q in (10, 50, 90)) + f" | clip dur med {np.median([D for D, _ in items]):.1f}s")
mk = lambda syn: [(float(r["duration"]), [(float(e["start"]), float(e["end"])) for e in r["NoiseSubCategoryTimeStamp"]])
                  for r in _meta if bool(r["syntheticData"]) == syn]
summarize("official synthetic", mk(True)); summarize("official natural", mk(False))
ids = sorted(SYNTH_TRAIN.items)[:3000]
summarize("our synthetic", [(float(SYNTH_TRAIN.items[s].get("duration", 8.0)), [(a, b) for a, b, _ in SYNTH_TRAIN.items[s]["events"]]) for s in ids])

official synthetic clips   695 | events/clip 3.18 | 1:16% 2:11% 3:18% 4:49% 5+:6% | clips with overlap 0% | coverage 43% | event dur p10/med/p90 0.20/0.52/1.06 | clip dur med 4.6s
official natural   clips  1806 | events/clip 1.63 | 1:57% 2:28% 3:11% 4:3% 5+:1% | clips with overlap 17% | coverage 30% | event dur p10/med/p90 0.23/0.44/2.76 | clip dur med 4.4s
our synthetic      clips  3000 | events/clip 2.15 | 1:37% 2:30% 3:21% 4:7% 5+:5% | clips with overlap 36% | coverage 32% | event dur p10/med/p90 0.27/0.54/2.80 | clip dur med 6.2s


In [140]:
# =============================================================================
# OV-6 -- build synthetic mixtures in the OFFICIAL style (statistics sampled from the 695 official synthetic validation clips).
# Differences from your build_synth: events/clip, categories, clip duration and event duration are sampled from the official
# empirical distributions; foregrounds NEVER overlap; hard edges (no fade); clean speech beds only (no natural events kept).
# Same sources as before (training Gold clips only, never test), same SNR range. Writes only NEW folders:
#   <BASE_DIR>/synth_official_style/{wav,beats,manifest.jsonl}      (same layout as your SynthStore)
#   <BASE_DIR>/atst_cache_osyn/<id>.npy                              (fp16 ATST tokens per mixture)
# Run in the V5 kernel after ov1/ov2 (needs the V5 objects + AC, ATST_CKPT, ATST_EXPECTED_SHA256). Resumable.
# =============================================================================
import os, json, time, numpy as np, soundfile as sf, torch
from collections import defaultdict, Counter

OS_ROOT = f"{BASE_DIR}/synth_official_style"
OS_ATST_DIR = f"{BASE_DIR}/atst_cache_osyn"
OS_N = 8000
OS_SEED = 20261015
OS_PREFIX = "o"
MIN_GAP = int(0.05 * SR)

# ---- official statistics -----------------------------------------------------------------------
_off = [r for r in _meta if bool(r["syntheticData"])]
OFF_D = np.array([float(r["duration"]) for r in _off])
OFF_K = np.array([len(r["NoiseSubCategoryTimeStamp"]) for r in _off])
OFF_EVD = np.array([float(e["end"]) - float(e["start"]) for r in _off for e in r["NoiseSubCategoryTimeStamp"]])
_cc = Counter(e["category"] for r in _off for e in r["NoiseSubCategoryTimeStamp"])
print("official synthetic: clips", len(_off), "| events/clip", OFF_K.mean().round(2), "| category counts", dict(_cc))

# ---- beds and foreground bank (same rules as build_synth, Gold training clips only) ---------------
gold_indices = list(tiers["gold"])
for gi in gold_indices:
    assert AUDIO_STEM[gi] not in TEST_IDS
lru = _WavLRU()
beds, fg = [], defaultdict(list)
for gi in gold_indices:
    raw = label_table["NoiseSubCategoryTimeStamp"][gi] or []
    evs = parse_events(raw, label_table["NoiseCategory"][gi])
    d = float(DURATION_COL[gi]) if DURATION_COL is not None else len(lru.get(gi)) / SR
    if len(evs) == len([1 for e in raw if e]):
        t = 0.0
        for s, e, _ in sorted(evs):
            if s - 0.1 - t >= 1.0:
                beds.append((gi, t, s - 0.1))
            t = max(t, e + 0.1)
        if d - t >= 1.0:
            beds.append((gi, t, d))
    for s, e, c in evs:
        if FG_DUR_RANGE[0] <= e - s <= FG_DUR_RANGE[1]:
            fg[c].append(("gold", gi, s, e))
fgs = {c: sorted(v, key=lambda t: t[3] - t[2]) for c, v in fg.items() if v}
fdur = {c: np.array([t[3] - t[2] for t in v]) for c, v in fgs.items()}
cats_avail = [c for c in range(NC) if c in fgs]
cp = np.array([_cc.get(CATS[c], 0) + 1e-6 for c in cats_avail], float); cp /= cp.sum()      # class balance of the official mixtures
print(f"beds {len(beds)} | foreground clips per category { {CATS[c]: len(fgs[c]) for c in cats_avail} }")
print("category prob used:", {CATS[c]: round(float(p), 3) for c, p in zip(cats_avail, cp)})

# ---- ATST extractor (pinned checkpoint) ----------------------------------------------------------
_SHA = ATST_EXPECTED_SHA256 or "8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2"
assert AC.sha256_file(ATST_CKPT) == _SHA
_enc = AC.load_atst_encoder(ATST_CKPT, device=device, mel_variant=ATST_MEL_VARIANT, expected_sha256=_SHA)
_ex = AC.ATSTFeatureExtractor(_enc, device)
os.makedirs(os.path.join(OS_ROOT, "wav"), exist_ok=True); os.makedirs(os.path.join(OS_ROOT, "beats"), exist_ok=True); os.makedirs(OS_ATST_DIR, exist_ok=True)
store = SynthStore(OS_ROOT); done = set(store.items)
print(f"already built: {len(done)}/{OS_N}")
bm = None; t0 = time.time(); n_new = 0; n_dropped = 0
mf = open(os.path.join(OS_ROOT, "manifest.jsonl"), "a", encoding="utf-8")
try:
    for n in range(OS_N):
        sid = f"{OS_PREFIX}{n:06d}"
        if sid in done:
            continue
        rng = np.random.default_rng([OS_SEED, n])
        # clean bed with an official clip duration
        D = float(np.clip(OFF_D[int(rng.integers(len(OFF_D)))], 1.5, 14.0))
        parts, src, L = [], [], 0
        while L < D * SR:
            gi, a, b = beds[int(rng.integers(len(beds)))]
            seg = _fade(lru.get(gi)[int(a * SR):int(b * SR)])
            parts.append(seg); src.append([int(gi), round(a, 3), round(b, 3)]); L += len(seg)
        mix = np.concatenate(parts)[: int(D * SR)].astype(np.float32)
        speech_rms = _active_rms(mix)
        # foreground specs: count, category and duration all drawn from the official empirical distributions
        K = int(OFF_K[int(rng.integers(len(OFF_K)))])
        specs = []
        for _ in range(K):
            c = cats_avail[int(rng.choice(len(cats_avail), p=cp))]
            dt = float(OFF_EVD[int(rng.integers(len(OFF_EVD)))])
            ii = int(np.searchsorted(fdur[c], dt))
            kind, a, s, e = fgs[c][int(rng.integers(ii, len(fgs[c])))] if ii < len(fgs[c]) else fgs[c][-1]
            Ln = min(dt, e - s); o = float(rng.uniform(0, max(0.0, (e - s) - Ln)))
            specs.append((c, a, s + o, s + o + Ln))
        specs.sort(key=lambda t: -(t[3] - t[2]))
        placed, pasted = [], []
        for c, a, ss, ee in specs:
            seg = lru.get(a)[int(ss * SR):int(ee * SR)]
            if len(seg) < int(0.05 * SR):
                continue
            seg = seg[: int(0.8 * len(mix))]
            ok = False
            for _try in range(60):                                   # NO overlap, with a minimum gap
                pos = int(rng.integers(0, len(mix) - len(seg) + 1))
                if all(pos + len(seg) + MIN_GAP <= p0 or pos >= p1 + MIN_GAP for p0, p1 in placed):
                    ok = True; break
            if not ok:
                n_dropped += 1; continue
            placed.append((pos, pos + len(seg)))
            snr = float(rng.uniform(*SNR_RANGE))
            gain = speech_rms / (_active_rms(seg) * 10 ** (snr / 20))
            mix[pos:pos + len(seg)] += seg.astype(np.float32) * gain       # hard edges: no fade
            pasted.append({"cat": CATS[c], "src": ["gold", int(a), round(ss, 4), round(ee, 4)], "pos": pos, "len": int(len(seg)),
                           "snr": round(snr, 2), "gain": round(float(gain), 5)})
        if not pasted:
            continue
        pk = float(np.abs(mix).max())
        if pk > 0.99:
            mix *= 0.99 / pk
        events = sorted((round(p["pos"] / SR, 4), round((p["pos"] + p["len"]) / SR, 4), p["cat"]) for p in pasted)
        sf.write(os.path.join(OS_ROOT, "wav", f"{sid}.flac"), mix, SR, format="FLAC", subtype="PCM_16")
        w_back, _ = sf.read(os.path.join(OS_ROOT, "wav", f"{sid}.flac"), dtype="float32")
        if bm is None:
            bm = get_live_beats()
        feat, _ = extract_beats_full(bm, torch.from_numpy(w_back), device)
        if not np.isfinite(feat).all():
            raise RuntimeError(f"non-finite BEATs output for {sid}")
        np.save(os.path.join(OS_ROOT, "beats", f"{sid}.npy"), feat.astype(np.float16))
        af = _ex(torch.from_numpy(w_back))
        if af.shape != (AC.atst_T_expected(max(len(w_back), SR)), ATST_DIM) or not np.isfinite(af).all():
            raise RuntimeError(f"bad ATST features for {sid}: {af.shape}")
        np.save(os.path.join(OS_ATST_DIR, f"{sid}.npy"), af.astype(np.float16))
        mf.write(json.dumps({"id": sid, "duration": round(len(w_back) / SR, 4), "mode": "os_clean", "bed_src": src,
                             "district": DISTRICT[src[0][0]], "events": events, "pasted": pasted}) + "\n"); mf.flush()   # manifest last: done == complete
        n_new += 1
        if n_new % 500 == 0:
            print(f"  {n_new} new ({time.time() - t0:.0f}s)")
finally:
    mf.close()
del _enc, _ex
if torch.cuda.is_available():
    torch.cuda.empty_cache()
log(f"official-style synthetic: {n_new} new mixtures in {time.time() - t0:.0f}s | events dropped for lack of room: {n_dropped}")

# ---- compare against the official synthetic clips (the numbers you pasted earlier) -----------------
OS_STORE = SynthStore(OS_ROOT)


def _summ(name, items):
    n, ov, cov, ed = [], [], [], []
    for D_, evs in items:
        evs = sorted(evs); n.append(len(evs)); ed += [e - s for s, e in evs]
        ov.append(any(evs[i][1] > evs[i + 1][0] for i in range(len(evs) - 1)))
        u, cur = 0.0, -1.0
        for s, e in evs:
            s2 = max(s, cur)
            if e > s2:
                u += e - s2; cur = max(cur, e)
        cov.append(u / max(D_, 1e-6))
    c = Counter(min(x, 5) for x in n)
    print(f"{name:<20} clips {len(n):>5} | events/clip {np.mean(n):.2f} | " + " ".join(f"{k}{'+' if k == 5 else ''}:{100 * c[k] / len(n):.0f}%" for k in range(1, 6))
          + f" | overlap {100 * np.mean(ov):.0f}% | coverage {100 * np.mean(cov):.0f}% | event dur p10/med/p90 "
          + "/".join(f"{np.percentile(ed, q):.2f}" for q in (10, 50, 90)) + f" | clip dur med {np.median([d for d, _ in items]):.1f}s")


_summ("official synthetic", [(float(r["duration"]), [(float(e["start"]), float(e["end"])) for e in r["NoiseSubCategoryTimeStamp"]]) for r in _off])
_summ("new official-style", [(r["duration"], [(a, b) for a, b, _ in r["events"]]) for r in OS_STORE.items.values()])
OS_IDS = sorted(OS_STORE.items)
print("mixtures available:", len(OS_IDS), "(the two lines above should now look alike)")

official synthetic: clips 695 | events/clip 3.18 | category counts {'animal': 548, 'vehicle_traffic': 626, 'human_non_speech': 670, 'baby_child': 231, 'singing_music': 28, 'appliance_machine': 32, 'phone_signal_alarm': 78}
beds 12190 | foreground clips per category {'animal': 2178, 'appliance_machine': 44, 'baby_child': 1107, 'human_non_speech': 5657, 'phone_signal_alarm': 453, 'singing_music': 106, 'vehicle_traffic': 3213}
category prob used: {'animal': 0.248, 'appliance_machine': 0.014, 'baby_child': 0.104, 'human_non_speech': 0.303, 'phone_signal_alarm': 0.035, 'singing_music': 0.013, 'vehicle_traffic': 0.283}
[ATST] checkpoint survey
   path: C:/Users/231b258/atst_as2M.ckpt
   size_mb: 686.7
   sha256: 8739265cdc6f5990aba209785ccac07009508813f0c1be0757dc05de6345e4d2
   top_level_keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters']
   n_tensors: 143
   prefix_histogram

In [141]:
# =============================================================================
# OV-7 -- fine-tune ATST-e2 on the OV-6 official-STYLE synthetic mixtures. NO official validation clip is used for training,
# so ALL 2,501 official validation clips (1,806 natural + 695 synthetic) are a clean held-out test.
# Same recipe as OV-3/OV-5 (init from e2, 2,000 updates, lr x0.3, 35% of each batch from the new mixtures), so the numbers are comparable.
# Run in the V5 kernel AFTER ov1/ov2 (OV, OVK, ov_beats, ov_atst, ov_eval, FIXED_DEC, E2, integration functions) and AFTER OV-6 finished.
# Writes only NEW files tagged TAG. Nothing is overwritten. A 6-step smoke test runs first and writes nothing.
# What it uses from the official validation set: only the METADATA statistics OV-6 sampled (events per clip, durations, category mix).
# =============================================================================
import os, json, math, time
import numpy as np, torch
from collections import defaultdict, Counter

RUN_FT = True                   # the 6-step smoke test always runs first; if it errors nothing is trained
UPDATES = 2000                  # ~30 min of training
MID_EVAL_AT = 1000              # one mid-run check (student only); the final check scores student AND ema
FT_LR_MULT = 0.3
OS_SHARE = 0.35                 # fraction of each batch drawn from the new mixtures (the other 65% is the original e2 training mix)
BASE_TAG = "v5c_atst_fusion_e2"
TAG = "v5c_atst_fusion_osft"

OS_ROOT = f"{BASE_DIR}/synth_official_style"
OS_ATST_DIR = f"{BASE_DIR}/atst_cache_osyn"
OS_STORE = SynthStore(OS_ROOT)
OS_IDS = sorted(OS_STORE.items)
assert len(OS_IDS) >= 7000, f"only {len(OS_IDS)} mixtures: run OV-6 to completion first"

# ---- preflight (read-only) ---------------------------------------------------------------------
_gold = set(int(g) for g in tiers["gold"])
missing = [s for s in OS_IDS if not os.path.exists(os.path.join(OS_ATST_DIR, f"{s}.npy"))]
assert not missing, f"{len(missing)} mixtures lack ATST features, e.g. {missing[:3]}"
bad_src = 0
for s in OS_IDS:
    r = OS_STORE.items[s]
    ok = all(int(b[0]) in _gold for b in r["bed_src"]) and all(p["src"][0] == "gold" and int(p["src"][1]) in _gold for p in r["pasted"])
    bad_src += (not ok)
assert bad_src == 0, f"{bad_src} mixtures use sources outside the Gold training clips"
assert not any(AUDIO_STEM[g] in TEST_IDS for g in _gold), "a test clip is among the sources"
print(f"preflight OK: {len(OS_IDS)} mixtures, all sources are Gold training clips, ATST features present, no official validation audio used")


class OSMixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("os", sid, tier) served from the OV-6 mixtures (same crop/label/weak-target logic as the other rows)."""
    def _load(self, kind, key):
        if kind != "os":
            return super()._load(kind, key)
        r = OS_STORE.items[key]
        wav = OS_STORE.wav(key); bf = OS_STORE.beats(key)
        evs = [(float(s), float(e), CAT_IDX[c]) for s, e, c in r["events"]]
        check_beats_len(bf, wav.numel(), f"os:{key}")
        return wav, bf, evs, sorted({c for _, _, c in evs})

    def _full(self, kind, key):
        if kind != "os":
            return super()._full(kind, key)
        return np.load(os.path.join(OS_ATST_DIR, f"{key}.npy")).astype(np.float32)


def make_mix_sampler(base_rows, extra_rows, shares, extra_share, n):
    from torch.utils.data import WeightedRandomSampler
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    w = [(1 - extra_share) * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows] + [extra_share / len(extra_rows)] * len(extra_rows)
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_all_scores(net, T_win, fps):
    """Score on ALL official validation clips under the FIXED decoder (none of them was trained on)."""
    ks = OVK[("N", "all")] + OVK[("S", "all")]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, "all") for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    rows = list(TRAIN_ROWS) + [("os", s, T_SYNTH) for s in OS_IDS]
    ds = OSMixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                      shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    os_rows = [r for r in rows if r[0] == "os"]
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler(list(TRAIN_ROWS), os_rows, cfg["shares"], OS_SHARE, n_draw)
    # the sampler indexes `rows`: original rows first, new mixtures last (same order as built above)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = None
    if run:
        base = {dom: ov_eval(E2["tracks"], E2["fps"], FIXED_DEC, dom, "all") for dom in ("N", "S")}
        base["J"] = 0.62 * base["N"]["combined"] + 0.38 * base["S"]["combined"]
        log(f"BASELINE {BASE_TAG} on ALL official clips: J {base['J']:.4f} | {fmt(base)}")
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if upd % 250 == 0:
            tr_ = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            log(f"{TAG} upd={upd}/{updates} skipped={skipped} {tr_}")
        if run and upd == MID_EVAL_AT and upd < updates:
            model.eval(); sc = ov_all_scores(model, T_win, fps); hist.append({"update": upd, "student": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]})
            log(f"{TAG} upd={upd} student: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
            model.train()
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sc = ov_all_scores(net, T_win, fps); hist.append({"update": upd, name: sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]})
        log(f"{TAG} FINAL {name}: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
        sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        torch.save({"model": sd, "update": upd, "score": {"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]}, "hash": state_hash(sd),
                    "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "os_share": OS_SHARE, "os_mixtures": len(OS_IDS), "updates": upd,
                             "official_val_clips_trained_on": 0}}, p)
        paths[name] = p
    json.dump({"hist": hist, "baseline": {"J": base["J"], "N": base["N"]["combined"], "S": base["S"]["combined"]}, "updates": upd},
              open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OSFT_PATHS = finetune(UPDATES, run=True)
    print(OSFT_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG} ({UPDATES} updates).")

preflight OK: 8000 mixtures, all sources are Gold training clips, ATST features present, no official validation audio used
[2026-10-15 16:11:16] v5c_atst_fusion_osft: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-15 16:11:17] BASELINE v5c_atst_fusion_e2 on ALL official clips: J 1.1980 | NAT 1.0119 (F1 0.3491 Dice 0.6627 ev/clip 1.48) | SYN 1.5017 (F1 0.6786 Dice 0.8232 ev/clip 2.46)
[2026-10-15 16:15:06] v5c_atst_fusion_osft upd=250/2000 skipped=0 frame=0.0466 onset=0.0870 offset=0.0897 dice=0.2554 weak=0.2998
[2026-10-15 16:18:57] v5c_atst_fusion_osft upd=500/2000 skipped=0 frame=0.0459 onset=0.0860 offset=0.0887 dice=0.2608 weak=0.3009
[2026-10-15 16:22:46] v5c_atst_fusion_osft upd=750/2000 skipped=0 frame=0.0446 onset=0.0823 offset=0.0852 dice=0.2554 weak=0.2949
[2026-10-15 16:26:33] v5c_atst_fusion_osft upd=1000/2000 skipped=0 frame=0.0447 onset=0.0835 offset=0.0858 dice=0.2581 weak=0.2912
[2026-10-15 16:28:10] v5c_atst_fusion_osft upd=1000 st

In [142]:
# =============================================================================
# OV-10 -- restore SNAP (your cell-75 boundary snapping) and a synthetic-decoder shift for the FINE-TUNED model, validated on held-out folds.
#
# Why: the fine-tuned model predicts ~2.57 synthetic events/clip vs 3.18 in the reference (under-detection), and its synthetic Dice (0.8232)
# is below the SNAP version's (0.8786) because the 1.3201 ZIP dropped SNAP.
#
# Part A (validation, clean): fold-A model scores official half B, fold-B model scores official half A (neither model saw its half).
#   12 pre-declared configs = thr shift {0,-0.05,-0.10} x gap mult {1.0,1.5} x SNAP {off, W=0.20 thr=0}.
#   Config is SELECTED on fold A -> half B and CONFIRMED on fold B -> half A.  ACCEPT only if the confirm delta >= +0.01 AND the select delta >= +0.01.
# Part B (only if accepted): natural clips copied unchanged from the 1.3201 ZIP; the 2,080 synthetic clips are re-decoded from FINAL_TAG's
#   float32 test tracks with the accepted decoder, then snapped. Writes a NEW zip. Nothing existing is overwritten.
# Run in the V5 kernel (needs OV, OVK, OV_REFS, ov_beats, ov_atst, FIXED_DEC, decode_v5, snap_curves/snap_events/_clip_stats/_agg from the SNAP cell).
# =============================================================================
import os, json, pickle, zipfile, hashlib, itertools
import numpy as np, torch

FOLD_TAGS = {"A": "v5c_atst_fusion_ovft_A", "B": "v5c_atst_fusion_ovft_B"}   # OV-3 folds (use v5c_atst_fusion_ovos_A/B after OV-9)
FINAL_TAG = "v5c_atst_fusion_ovft_ALL"                                          # model whose synthetic test clips get re-decoded (1.3201 model)
BASE_ZIP = f"{SUB_DIR}/submission_route_natdec_synFT_all.zip"                   # the 1.3201 ZIP (natural clips come from here)
OUT_NAME = "submission_route_natdec_synFT_all_snapdec.zip"
ACCEPT_MIN = 0.01
GRID = [dict(dthr=a, gm=b, snap=c) for a in (0.0, -0.05, -0.10) for b in (1.0, 1.5) for c in (False, True)]   # fixed in advance
SNAP_CFG = dict(W=0.20, thr=0.0)                                                # the config validated in the SNAP cell

_ref = {k: v["ref"] for k, v in OV.items()}


def dec_variant(dthr, gm):
    d = dict(FIXED_DEC)
    for key in ("thr_short", "thr_long"):
        d[key] = FIXED_DEC[key] + dthr
    for key in ("gap_short", "gap_long"):
        d[key] = FIXED_DEC[key] * gm
    return d


def fold_tracks(tag, half):
    ck = torch.load(f"{V5_DIR}/{tag}_student.pt", map_location="cpu", weights_only=False)
    net = new_atst_model("fuse", ck["cfg"]); net.load_state_dict(ck["model"]); net.eval()
    T_win, fps = atst_time_grid(net)
    tr = {}
    with torch.no_grad():
        for k in OVK[("S", half)]:
            w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    del net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return tr, fps


def evaluate(tr, fps, half):
    ks = OVK[("S", half)]
    cur = {}
    for k in ks:
        x = load_test_wav(OV[k]["path"]).numpy(); cur[k] = snap_curves(x) + (len(x) / 16000,)
    res = {}
    for i, g in enumerate(GRID):
        d = dec_variant(g["dthr"], g["gm"])
        pred = {}
        for k in ks:
            ev = decode_v5(tr[k], d, fps)
            pred[k] = snap_events(ev, *cur[k], **SNAP_CFG) if g["snap"] else ev
        S = np.stack([_clip_stats(_ref[k], pred[k]) for k in ks]); a = _agg(S)
        res[i] = dict(a, ev=float(np.mean([len(pred[k]) for k in ks])))
    return res


half_of = {"A": "B", "B": "A"}                     # fold X model is tested on the OTHER half
R = {}
for f, tag in FOLD_TAGS.items():
    if not os.path.exists(f"{V5_DIR}/{tag}_student.pt"):
        print(f"fold {f}: {tag} missing, skipping"); continue
    tr, fps_ = fold_tracks(tag, half_of[f]); R[f] = evaluate(tr, fps_, half_of[f])
    b = R[f][0]
    print(f"fold {f} model on held-out half {half_of[f]} | baseline (FIXED_DEC, no snap): comb {b['comb']:.4f} (F1 {b['f1']:.4f} Dice {b['dice']:.4f}) ev/clip {b['ev']:.2f}")
    for i, g in enumerate(GRID):
        r = R[f][i]
        print(f"   thr {g['dthr']:+.2f} gap x{g['gm']} snap {'on ' if g['snap'] else 'off'} | comb {r['comb']:.4f} ({r['comb'] - b['comb']:+.4f}) F1 {r['f1']:.4f} Dice {r['dice']:.4f} ev/clip {r['ev']:.2f}")

CHOSEN = None
if "A" in R and "B" in R:
    sel = max(range(1, len(GRID)), key=lambda i: R["A"][i]["comb"] - R["A"][0]["comb"])         # selected on fold A -> half B
    dS = R["A"][sel]["comb"] - R["A"][0]["comb"]; dC = R["B"][sel]["comb"] - R["B"][0]["comb"]     # confirmed on fold B -> half A
    ok = dS >= ACCEPT_MIN and dC >= ACCEPT_MIN
    print(f"SELECTED {GRID[sel]} | select delta {dS:+.4f} | confirm delta {dC:+.4f} -> {'ACCEPT' if ok else 'REJECT'}")
    if ok:
        CHOSEN = GRID[sel]
else:
    print("need BOTH fold models for an honest select/confirm split; nothing chosen")

# ---- Part B ------------------------------------------------------------------------------------
if CHOSEN is not None:
    OUT = f"{SUB_DIR}/{OUT_NAME}"
    assert not os.path.exists(OUT), OUT
    p32 = f"{V5_DIR}/{FINAL_TAG}_test_tracks32.pkl"
    if not os.path.exists(p32):
        p32 = atst_test_tracks32("fuse", FINAL_TAG, atst_open_cache(create=False), ck_path=f"{V5_DIR}/{FINAL_TAG}_student.pt")
    old_ids = set(json.load(open(f"{V5_DIR}/test_syn_ids.json")))
    t = pickle.load(open(p32, "rb")); fps = t["fps"]
    with zipfile.ZipFile(BASE_ZIP) as z:
        rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
    base = {r["clip_id"]: [(e["onset"], e["offset"]) for e in r["events"]] for r in rows}
    assert set(base) == set(t["tracks"]) and len(base) == 5517
    files = {os.path.splitext(os.path.basename(p))[0]: p for p in TEST_FILES}
    d = dec_variant(CHOSEN["dthr"], CHOSEN["gm"])
    preds, durs = {}, {}
    for cid, tr_ in t["tracks"].items():
        durs[cid] = tr_["duration"]
        if cid in old_ids:
            ev = decode_v5(tr_, d, fps)
            if CHOSEN["snap"] and ev:
                ev = snap_events(ev, *snap_curves(load_test_wav(files[cid]).numpy()), durs[cid], **SNAP_CFG)
            preds[cid] = ev
        else:
            preds[cid] = base[cid]
    zp = write_submission(preds, SUB_DIR, OUT_NAME, durs, expected=EXPECTED_TEST_CLIPS)
    atst_verify_zip(zp, expected_ids=set(base))
    nat_same = all(len(preds[c]) == len(base[c]) and np.allclose(np.array(preds[c], float).ravel(), np.array(base[c], float).ravel(), atol=1e-6)
                   for c in set(base) - old_ids)
    ev_new = np.mean([len(preds[c]) for c in old_ids]); ev_old = np.mean([len(base[c]) for c in old_ids])
    print(zp, hashlib.sha1(open(zp, "rb").read()).hexdigest()[:12])
    print(f"natural clips identical to the 1.3201 ZIP: {nat_same} | synthetic events/clip: new {ev_new:.2f} vs {ev_old:.2f} "
          f"| synthetic empty clips: new {sum(len(preds[c]) == 0 for c in old_ids)} vs {sum(len(base[c]) == 0 for c in old_ids)}")
    assert nat_same, "natural clips changed: do not submit"

fold A model on held-out half B | baseline (FIXED_DEC, no snap): comb 1.5996 (F1 0.7541 Dice 0.8455) ev/clip 2.56
   thr +0.00 gap x1.0 snap off | comb 1.5996 (+0.0000) F1 0.7541 Dice 0.8455 ev/clip 2.56
   thr +0.00 gap x1.0 snap on  | comb 1.4975 (-0.1021) F1 0.6646 Dice 0.8329 ev/clip 2.56
   thr +0.00 gap x1.5 snap off | comb 1.5704 (-0.0292) F1 0.7241 Dice 0.8463 ev/clip 2.50
   thr +0.00 gap x1.5 snap on  | comb 1.4786 (-0.1210) F1 0.6443 Dice 0.8342 ev/clip 2.50
   thr -0.05 gap x1.0 snap off | comb 1.5999 (+0.0004) F1 0.7493 Dice 0.8506 ev/clip 2.59
   thr -0.05 gap x1.0 snap on  | comb 1.4978 (-0.1018) F1 0.6593 Dice 0.8385 ev/clip 2.59
   thr -0.05 gap x1.5 snap off | comb 1.5687 (-0.0308) F1 0.7188 Dice 0.8499 ev/clip 2.54
   thr -0.05 gap x1.5 snap on  | comb 1.4787 (-0.1209) F1 0.6395 Dice 0.8392 ev/clip 2.54
   thr -0.10 gap x1.0 snap off | comb 1.5938 (-0.0058) F1 0.7401 Dice 0.8537 ev/clip 2.63
   thr -0.10 gap x1.0 snap on  | comb 1.4942 (-0.1053) F1 0.6528 Dice 0.8414

In [143]:
# =============================================================================
# OV-8 -- fine-tune ATST-e2 on [original mix + OV-6 official-style mixtures + official validation clips].
# FOLD = "A": train on official half A, test on half B | "B": train on half B, test on half A | "ALL": train on all 2,501 clips, NO held-out.
# Same recipe as OV-3/OV-5/OV-7 (init from e2, 2,000 updates, lr x0.3). Batch mix: 40% original rows, 30% new mixtures, 30% official clips.
# Run in the V5 kernel after ov1/ov2/ov6 (and after OV-7 has finished). Writes only NEW files tagged TAG. A 6-step smoke test runs first.
# =============================================================================
import os, json, math, time
import numpy as np, torch
from collections import defaultdict, Counter

FOLD = "A"                      # "A" | "B" | "ALL"
UPDATES = 2000
MID_EVAL_AT = 1000
FT_LR_MULT = 0.3
OS_SHARE = 0.30
OV_SHARE = 0.30
RUN_FT = True
BASE_TAG = "v5c_atst_fusion_e2"
TAG = f"v5c_atst_fusion_osov_{FOLD}"
EVAL_HALF = None if FOLD == "ALL" else ("B" if FOLD == "A" else "A")
assert FOLD in ("A", "B", "ALL")

OS_ROOT = f"{BASE_DIR}/synth_official_style"
OS_ATST_DIR = f"{BASE_DIR}/atst_cache_osyn"
OS_STORE = SynthStore(OS_ROOT)
OS_IDS = sorted(OS_STORE.items)
assert len(OS_IDS) >= 7000, f"only {len(OS_IDS)} mixtures: run OV-6 to completion first"
_gold = set(int(g) for g in tiers["gold"])
assert not [s for s in OS_IDS if not os.path.exists(os.path.join(OS_ATST_DIR, f"{s}.npy"))], "mixtures lack ATST features"
assert all(int(b[0]) in _gold for s in OS_IDS for b in OS_STORE.items[s]["bed_src"]), "mixture beds outside Gold training clips"

# ---- official clips as training rows ----------------------------------------------------------------
OV_EVS, OV_TIER = {}, {}
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]; syn = bool(r["syntheticData"]); key = ("S_" if syn else "N_") + stem
    OV_EVS[key] = parse_events(r["NoiseSubCategoryTimeStamp"], None)
    OV_TIER[key] = T_SYNTH if syn else T_GOLD
ov_train_keys = [k for k in OV if OV_EVS[k] and (FOLD == "ALL" or OV[k]["half"] == FOLD)]
if EVAL_HALF:
    ev_keys = OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]
    assert not set(ov_train_keys) & set(ev_keys), "train/eval overlap"
    assert {OV[k]["district"] for k in ov_train_keys}.isdisjoint({OV[k]["district"] for k in ev_keys}), "district overlap"
    print(f"training: {len(ov_train_keys)} official clips (half {FOLD}) + {len(OS_IDS)} mixtures | held-out half {EVAL_HALF}: "
          f"{len(OVK[('N', EVAL_HALF)])} natural + {len(OVK[('S', EVAL_HALF)])} synthetic")
else:
    print(f"training: ALL {len(ov_train_keys)} official clips + {len(OS_IDS)} mixtures | NO held-out clips (judge by the fold runs / leaderboard)")


class MixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("os", sid, tier) from OV-6 and ("ov", key, tier) from the official clips (same crop/label logic)."""
    def _load(self, kind, key):
        if kind == "os":
            r = OS_STORE.items[key]; wav = OS_STORE.wav(key); bf = OS_STORE.beats(key)
            evs = [(float(s), float(e), CAT_IDX[c]) for s, e, c in r["events"]]
            check_beats_len(bf, wav.numel(), f"os:{key}")
            return wav, bf, evs, sorted({c for _, _, c in evs})
        if kind == "ov":
            wav = load_test_wav(OV[key]["path"]); bf = ov_beats(key, wav.numel()); evs = OV_EVS[key]
            return wav, bf, evs, sorted({c for _, _, c in evs})
        return super()._load(kind, key)

    def _full(self, kind, key):
        if kind == "os":
            return np.load(os.path.join(OS_ATST_DIR, f"{key}.npy")).astype(np.float32)
        if kind == "ov":
            return ov_atst(key)
        return super()._full(kind, key)


def make_mix_sampler(base_rows, groups, shares, n):
    from torch.utils.data import WeightedRandomSampler
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    base_share = 1.0 - sum(sh for _, sh in groups); assert base_share > 0
    w = [base_share * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows]
    for rws, sh in groups:
        w += [sh / len(rws)] * len(rws)
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_half_scores(net, T_win, fps, half):
    ks = OVK[("N", half)] + OVK[("S", half)]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, half) for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    os_rows = [("os", s, T_SYNTH) for s in OS_IDS]; ov_rows = [("ov", k, OV_TIER[k]) for k in ov_train_keys]
    rows = list(TRAIN_ROWS) + os_rows + ov_rows
    ds = MixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                    shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler(list(TRAIN_ROWS), [(os_rows, OS_SHARE), (ov_rows, OV_SHARE)], cfg["shares"], n_draw)   # index order = rows order
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = None
    if run and EVAL_HALF:
        base = {dom: ov_eval(E2["tracks"], E2["fps"], FIXED_DEC, dom, EVAL_HALF) for dom in ("N", "S")}
        base["J"] = 0.62 * base["N"]["combined"] + 0.38 * base["S"]["combined"]
        log(f"BASELINE {BASE_TAG} on held-out half {EVAL_HALF}: J {base['J']:.4f} | {fmt(base)}")
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if upd % 250 == 0:
            tr_ = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            log(f"{TAG} upd={upd}/{updates} skipped={skipped} {tr_}")
        if run and EVAL_HALF and upd == MID_EVAL_AT and upd < updates:
            model.eval(); sc = ov_half_scores(model, T_win, fps, EVAL_HALF); hist.append({"update": upd, "student": sc["J"]})
            log(f"{TAG} upd={upd} student: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
            model.train()
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        score = {"J": 0.0, "N": 0.0, "S": 0.0}
        if EVAL_HALF:
            sc = ov_half_scores(net, T_win, fps, EVAL_HALF); hist.append({"update": upd, name: sc["J"]})
            score = {"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]}
            log(f"{TAG} FINAL {name}: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
        torch.save({"model": sd, "update": upd, "score": score, "hash": state_hash(sd), "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "fold": FOLD, "os_share": OS_SHARE, "ov_share": OV_SHARE, "os_mixtures": len(OS_IDS),
                             "official_clips_trained_on": len(ov_train_keys), "updates": upd}}, p)
        paths[name] = p
    json.dump({"hist": hist, "fold": FOLD, "updates": upd, "baseline_J": (base or {}).get("J")}, open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OSOV_PATHS = finetune(UPDATES, run=True)
    print(OSOV_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG}.")

training: 1216 official clips (half A) + 8000 mixtures | held-out half B: 955 natural + 328 synthetic
[2026-10-15 16:58:29] v5c_atst_fusion_osov_A: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-15 16:58:29] BASELINE v5c_atst_fusion_e2 on held-out half B: J 1.1993 | NAT 1.0176 (F1 0.3592 Dice 0.6584 ev/clip 1.55) | SYN 1.4958 (F1 0.6809 Dice 0.8149 ev/clip 2.50)
[2026-10-15 17:01:58] v5c_atst_fusion_osov_A upd=250/2000 skipped=0 frame=0.0473 onset=0.0872 offset=0.0902 dice=0.2728 weak=0.2835
[2026-10-15 17:05:28] v5c_atst_fusion_osov_A upd=500/2000 skipped=0 frame=0.0449 onset=0.0849 offset=0.0875 dice=0.2688 weak=0.2810
[2026-10-15 17:08:57] v5c_atst_fusion_osov_A upd=750/2000 skipped=0 frame=0.0447 onset=0.0857 offset=0.0880 dice=0.2600 weak=0.2782
[2026-10-15 17:12:27] v5c_atst_fusion_osov_A upd=1000/2000 skipped=0 frame=0.0421 onset=0.0816 offset=0.0842 dice=0.2630 weak=0.2684
[2026-10-15 17:13:16] v5c_atst_fusion_osov_A upd=1000 student: J 1.2

In [144]:
import os, re, collections
d = os.path.dirname(TEST_FILES[0])
for root in {d, os.path.dirname(d)}:
    names = os.listdir(root)
    print(root, "| extensions:", sorted({os.path.splitext(f)[1].lower() for f in names}), "| non-audio files:", [f for f in names if not f.lower().endswith(('.wav', '.flac', '.mp3'))][:15])
# how well would transcript tags predict the number of events, IF the test clips had transcripts? (official validation only)
tg = lambda t: len(re.findall(r"<(?!/)[^<>]+>", t or ""))
for syn in (False, True):
    rs = [r for r in _meta if bool(r["syntheticData"]) == syn]
    ok = sum(tg(r.get("transcript")) == len(r["NoiseSubCategoryTimeStamp"]) for r in rs)
    print("SYN" if syn else "NAT", len(rs), "clips | transcript available:", sum(bool(r.get("isTranscriptionAvailable")) for r in rs),
          "| tag count == event count:", f"{100 * ok / len(rs):.0f}%", "| mean tags", round(sum(tg(r.get('transcript')) for r in rs) / len(rs), 2),
          "vs mean events", round(sum(len(r['NoiseSubCategoryTimeStamp']) for r in rs) / len(rs), 2))

C:/Users/231b258/test | extensions: ['', '.txt'] | non-audio files: ['audio', 'dataset_info.txt']
C:/Users/231b258/test/audio | extensions: ['.wav'] | non-audio files: []
NAT 1806 clips | transcript available: 1806 | tag count == event count: 33% | mean tags 2.24 vs mean events 1.63
SYN 695 clips | transcript available: 695 | tag count == event count: 28% | mean tags 2.22 vs mean events 3.18


In [145]:
from collections import Counter
tr, fps = OVT["v5c_atst_fusion_e2"]["tracks"], OVT["v5c_atst_fusion_e2"]["fps"]
def classify(ref, pred):
    pairs = VE._greedy_pairs(ref, pred); mr = {r for r, _ in pairs}; mp = {p for _, p in pairs}
    out = Counter()
    for ri, (rs, re_) in enumerate(ref):
        if ri in mr: out["ref matched"] += 1; continue
        hits = [(pi, min(re_, pe) - max(rs, ps)) for pi, (ps, pe) in enumerate(pred) if min(re_, pe) > max(rs, ps)]
        if not hits: out["ref MISSED (no overlapping prediction)"] += 1; continue
        pi, ov = max(hits, key=lambda h: h[1]); ps, pe = pred[pi]
        merged = sum(1 for qs, qe in ref if min(qe, pe) > max(qs, ps)) >= 2
        out["ref in a MERGED prediction" if merged else ("ref BOUNDARY miss (overlap >= 50%)" if ov / (re_ - rs) >= 0.5 else "ref partial overlap (< 50%)")] += 1
    for pi, (ps, pe) in enumerate(pred):
        if pi not in mp:
            out["pred FP, overlaps a reference" if any(min(re_, pe) > max(rs, ps) for rs, re_ in ref) else "pred FP, no reference nearby"] += 1
    return out
for dom in ("N", "S"):
    tot = Counter()
    for k in OVK[(dom, "all")]:
        tot += classify(OV_REFS[k], decode_v5(tr[k], FIXED_DEC, fps))
    nref = sum(v for k, v in tot.items() if k.startswith("ref")); npred = sum(v for k, v in tot.items() if k.startswith("pred"))
    print("NATURAL" if dom == "N" else "SYNTHETIC", f"| {nref} reference events, {npred} unmatched predictions")
    for k, v in sorted(tot.items()): print(f"   {k:<44}{v:>6}  ({100 * v / (nref if k.startswith('ref') else npred):.0f}%)")

NATURAL | 2944 reference events, 1685 unmatched predictions
   pred FP, no reference nearby                   223  (13%)
   pred FP, overlaps a reference                 1462  (87%)
   ref BOUNDARY miss (overlap >= 50%)             757  (26%)
   ref MISSED (no overlapping prediction)         390  (13%)
   ref in a MERGED prediction                     623  (21%)
   ref matched                                    979  (33%)
   ref partial overlap (< 50%)                    195  (7%)
SYNTHETIC | 2213 reference events, 381 unmatched predictions
   pred FP, no reference nearby                    16  (4%)
   pred FP, overlaps a reference                  365  (96%)
   ref BOUNDARY miss (overlap >= 50%)              99  (4%)
   ref MISSED (no overlapping prediction)         444  (20%)
   ref in a MERGED prediction                     302  (14%)
   ref matched                                   1332  (60%)
   ref partial overlap (< 50%)                     36  (2%)


In [146]:
import itertools, numpy as np
E2t = OVT["v5c_atst_fusion_e2"]; tr, fps = E2t["tracks"], E2t["fps"]
ks = OVK[("N", "all")]; ix = {k: i for i, k in enumerate(ks)}
iA = [ix[k] for k in OVK[("N", "A")]]; iB = [ix[k] for k in OVK[("N", "B")]]
dist = np.array([OV[k]["district"] for k in ks]); me, ef = match_events, events_to_frames
pred0 = {k: decode_v5(tr[k], FIXED_DEC, fps) for k in ks}
# 1) systematic boundary bias for events that DO overlap a prediction well (IoU >= 0.5)
on_e, of_e = [], []
for k in ks:
    for rs, re_ in OV_REFS[k]:
        best, bi = None, 0.0
        for ps, pe in pred0[k]:
            inter = min(re_, pe) - max(rs, ps)
            if inter > 0 and inter / (max(re_, pe) - min(rs, ps)) > bi: bi, best = inter / (max(re_, pe) - min(rs, ps)), (ps, pe)
        if best and bi >= 0.5: on_e.append(best[0] - rs); of_e.append(best[1] - re_)
pc = lambda a: " / ".join(f"{1000 * np.percentile(a, q):+.0f}" for q in (10, 25, 50, 75, 90))
print(f"{len(on_e)} overlapping pairs | signed onset error ms (p10/25/50/75/90): {pc(on_e)} | offset error: {pc(of_e)}  (+ = prediction later than the label)")
# 2) predeclared grid
def dil(ev, a, b, dur):
    out = [(max(0.0, s - a), min(dur, e + b)) for s, e in ev]
    return sorted((round(s, 3), round(e, 3)) for s, e in out if e > s)
CFG = []
for nms, gm, dt in itertools.product((FIXED_DEC["nms_iou"], 0.8, 0.6, 0.4), (1.0, 0.5), (0.0, 0.05)):
    d = dict(FIXED_DEC); d["nms_iou"] = nms; d["thr_short"] += dt; d["thr_long"] += dt; d["gap_short"] *= gm; d["gap_long"] *= gm
    CFG.append((f"nms {nms} gap x{gm} thr +{dt}", d, (0.0, 0.0)))
for a, b in itertools.product((-0.04, 0.0, 0.04), repeat=2):
    if (a, b) != (0.0, 0.0): CFG.append((f"boundary shift onset-{a:+.2f} offset{b:+.2f}", dict(FIXED_DEC), (a, b)))
def stats(d, ab):
    pred = {k: (decode_v5(tr[k], d, fps) if ab == (0.0, 0.0) else dil(decode_v5(tr[k], d, fps), ab[0], ab[1], tr[k]["duration"])) for k in ks}
    return VE.stats_for(pred, OV_REFS, ks, me, ef), float(np.mean([len(pred[k]) for k in ks]))
sb, evb = stats(FIXED_DEC, (0.0, 0.0)); agg = VE.aggregate; c = lambda s, i=slice(None): agg(s[i])["combined"]
print(f"baseline natural: {c(sb):.4f} (F1 {agg(sb)['event_f1']:.4f} P {agg(sb)['precision']:.3f} R {agg(sb)['recall']:.3f} Dice {agg(sb)['segment_dice']:.4f}) ev/clip {evb:.2f}")
S = [stats(d, ab) for _, d, ab in CFG]
for j in sorted(range(len(CFG)), key=lambda j: -c(S[j][0]))[:6]:
    print(f"   {CFG[j][0]:<40} all {c(S[j][0]):.4f} ({c(S[j][0]) - c(sb):+.4f}) halves A {c(S[j][0], iA) - c(sb, iA):+.4f} B {c(S[j][0], iB) - c(sb, iB):+.4f} ev/clip {S[j][1]:.2f}")
for fit, test, i_fit, i_test in (("A", "B", iA, iB), ("B", "A", iB, iA)):
    j = int(np.argmax([c(s[0], i_fit) for s in S])); st = S[j][0]; dlt = c(st, i_test) - c(sb, i_test)
    bs = VE.paired_cluster_bootstrap(sb[i_test], st[i_test], dist[i_test], n_boot=2000)
    print(f"tuned on half {fit}: {CFG[j][0]} -> held-out half {test} delta {dlt:+.4f} CI90 [{bs['ci90'][0]:+.4f},{bs['ci90'][1]:+.4f}] P>0 {bs['p_gt0']:.2f} -> {'PASS' if dlt >= 0.01 and bs['p_gt0'] >= 0.9 else 'reject'}")

1733 overlapping pairs | signed onset error ms (p10/25/50/75/90): -117 / -42 / -9 / +23 / +107 | offset error: -114 / -29 / +3 / +65 / +150  (+ = prediction later than the label)
baseline natural: 1.0119 (F1 0.3491 P 0.367 R 0.333 Dice 0.6627) ev/clip 1.48
   nms 0.7 gap x1.0 thr +0.0                all 1.0119 (+0.0000) halves A +0.0000 B +0.0000 ev/clip 1.48
   nms 0.6 gap x1.0 thr +0.0                all 1.0117 (-0.0002) halves A +0.0006 B -0.0008 ev/clip 1.47
   nms 0.8 gap x1.0 thr +0.0                all 1.0115 (-0.0004) halves A +0.0000 B -0.0007 ev/clip 1.48
   nms 0.7 gap x0.5 thr +0.0                all 1.0106 (-0.0012) halves A -0.0008 B -0.0016 ev/clip 1.48
   nms 0.6 gap x0.5 thr +0.0                all 1.0105 (-0.0014) halves A -0.0002 B -0.0023 ev/clip 1.47
   nms 0.8 gap x0.5 thr +0.0                all 1.0102 (-0.0017) halves A -0.0008 B -0.0023 ev/clip 1.49
tuned on half A: nms 0.6 gap x1.0 thr +0.0 -> held-out half B delta -0.0008 CI90 [-0.0020,+0.0003] P>0 0.14 -> re

In [147]:
import random
E2_TAG = "v5c_atst_fusion_e2"
dj = json.load(open(f"{V5_DIR}/{E2_TAG}_decoder.json")); ck_ = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
net = new_atst_model("fuse", ck_["cfg"]); net.load_state_dict(ck_["model"]); net.eval()
T_win, fps = atst_time_grid(net); cache = atst_open_cache(create=False)
def score_set(idxs, label):
    refs, pred, n_ev, durs = {}, {}, [], []
    for i in idxs:
        w = _load_waveform(i, full_dataset, SR, WAV_CACHE_DIRS)
        cats = [c for c in (label_table["NoiseCategory"][i] or []) if c in CAT_IDX]
        evs = parse_events(label_table["NoiseSubCategoryTimeStamp"][i], cats)
        tr = tracks_atst(net, w, load_nat_beats(i), cache.load("nat", int(i)).astype(np.float32), device, T_win, fps)
        refs[i] = [(s, e) for s, e, _ in evs]; pred[i] = decode_v5(tr, FIXED_DEC, fps); n_ev.append(len(evs)); durs += [e - s for s, e, _ in evs]
    sc = official_combined_score(refs, pred)
    print(f"{label:<9} clips {len(idxs)} | events/clip {np.mean(n_ev):.2f} | event dur p10/50/90 {'/'.join(f'{np.percentile(durs, q):.2f}' for q in (10, 50, 90))} "
          f"| F1 {sc['event_f1']:.4f} P {sc['precision']:.3f} R {sc['recall']:.3f} Dice {sc['segment_dice']:.4f}")
rng = random.Random(0)
score_set(rng.sample(VAL_GOLD, min(1200, len(VAL_GOLD))), "Gold val"); score_set(rng.sample(tiers["silver"], 1200), "Silver")
print("official natural for reference: events/clip 1.63, event dur p10/50/90 0.23/0.44/2.76")

Gold val  clips 1200 | events/clip 1.69 | event dur p10/50/90 0.22/0.49/2.65 | F1 0.2820 P 0.298 R 0.268 Dice 0.6515
Silver    clips 1200 | events/clip 1.43 | event dur p10/50/90 0.30/0.89/6.33 | F1 0.3026 P 0.291 R 0.315 Dice 0.6079
official natural for reference: events/clip 1.63, event dur p10/50/90 0.23/0.44/2.76


In [148]:
rng = random.Random(0)
gold_s = rng.sample(VAL_GOLD, min(1200, len(VAL_GOLD))); silver_s = rng.sample(tiers["silver"], 1200)    # same samples as before
def bias_set(idxs, label):
    on_e, of_e, ratio = [], [], []
    for i in idxs:
        w = _load_waveform(i, full_dataset, SR, WAV_CACHE_DIRS)
        cats = [c for c in (label_table["NoiseCategory"][i] or []) if c in CAT_IDX]
        refs = [(s, e) for s, e, _ in parse_events(label_table["NoiseSubCategoryTimeStamp"][i], cats)]
        tr = tracks_atst(net, w, load_nat_beats(i), cache.load("nat", int(i)).astype(np.float32), device, T_win, fps)
        pred = decode_v5(tr, FIXED_DEC, fps)
        for rs, re_ in refs:
            best, bi = None, 0.0
            for ps, pe in pred:
                inter = min(re_, pe) - max(rs, ps)
                if inter > 0 and inter / (max(re_, pe) - min(rs, ps)) > bi: bi, best = inter / (max(re_, pe) - min(rs, ps)), (ps, pe)
            if best and bi >= 0.3:
                on_e.append(best[0] - rs); of_e.append(best[1] - re_); ratio.append((re_ - rs) / max(best[1] - best[0], 1e-3))
    pc = lambda a: " / ".join(f"{1000 * np.percentile(a, q):+.0f}" for q in (10, 25, 50, 75, 90))
    print(f"{label:<9} {len(on_e)} overlapping pairs | prediction minus label, ms (p10/25/50/75/90): onset {pc(on_e)} | offset {pc(of_e)} "
          f"| label/pred duration ratio median {np.median(ratio):.2f}")
bias_set(gold_s, "Gold val"); bias_set(silver_s, "Silver")

Gold val  1368 overlapping pairs | prediction minus label, ms (p10/25/50/75/90): onset -237 / -75 / -3 / +55 / +208 | offset -196 / -30 / +19 / +135 / +346 | label/pred duration ratio median 0.93
Silver    1166 overlapping pairs | prediction minus label, ms (p10/25/50/75/90): onset -167 / -32 / +10 / +165 / +854 | offset -525 / -97 / -9 / +28 / +165 | label/pred duration ratio median 1.05


In [149]:
_v5_losses_orig = globals().get("_v5_losses_orig", v5_losses)
SILVER_W, SILVER_MAXDUR = 0.3, 2.5
def v5_losses_silver(out, batch, cfg, device):
    T = out["T"]; per_cat = out["onset_logits"].ndim == 3
    frame_t, on_t, off_t, valid, bmask = build_targets_v5(batch["events"], batch["vspan"], T, device, per_cat)
    tier = batch["tier"].to(device)
    ok = torch.tensor([1.0 if (ev and max(e - s for s, e, _ in ev) <= SILVER_MAXDUR) else 0.0 for ev in batch["events"]], device=device)
    strong = ((tier == T_GOLD) | (tier == T_SYNTH)).float() + SILVER_W * (tier == T_SILVER).float() * ok     # the ONLY change
    sm = valid * strong[:, None]
    cl = out["cat_logits"].float(); parts = {}
    parts["frame"] = _ASL(cl, frame_t, mask=sm)
    bm = bmask * strong[:, None]
    parts["onset"] = masked_bce(out["onset_logits"].float(), on_t, mask=bm)
    parts["offset"] = masked_bce(out["offset_logits"].float(), off_t, mask=bm)
    p = torch.sigmoid(cl); pu = (1.0 - torch.prod(1.0 - p, dim=-1)) * valid; yu = frame_t.max(-1).values * valid
    d = 1.0 - (2.0 * (pu * yu).sum(1) + 1.0) / (pu.sum(1) + yu.sum(1) + 1.0)
    parts["dice"] = (d * strong).sum() / strong.sum().clamp_min(1.0)
    q = mil_pool(cl, out["attn_logits"], valid); z = batch["z"].to(device); zm = batch["zmask"].to(device)
    roww = torch.ones_like(strong); roww[tier == T_BRONZE] = cfg["bronze_weak_w"]; wz = zm * roww[:, None]
    bce = -(z * torch.log(q) + (1 - z) * torch.log(1 - q)); parts["weak"] = (bce * wz).sum() / wz.sum().clamp_min(1.0)
    loss = (cfg["w_frame"] * parts["frame"] + cfg["w_on"] * parts["onset"] + cfg["w_off"] * parts["offset"] + cfg["w_dice"] * parts["dice"] + cfg["w_weak"] * parts["weak"])
    return loss, parts
v5_losses = v5_losses_silver          # to undo later: v5_losses = _v5_losses_orig

In [ ]:
# =============================================================================
# OV-9 -- fine-tune ATST-e2 on the official validation clips (OV-3/OV-5 recipe) PLUS the OV-6 official-style mixtures.
#   FOLD = "A" : train on official half A + mixtures, evaluate on half B   (compare with OV-3 fold A: J 1.2400, NAT 1.0196, SYN 1.5996)
#   FOLD = "B" : the mirror image                                           (compare with OV-3 fold B: J 1.2552)
#   FOLD = "ALL": train on all 2,499 usable official clips + mixtures; no held-out clips exist, so judge it by the fold result + one leaderboard probe
# Run in the V5 kernel after ov1/ov2 and OV-6. Writes only NEW files tagged TAG. A 6-step smoke test (nothing written) always runs first.
# Batch mix per update: OV_SHARE official clips, OS_SHARE mixtures, the rest the original e2 training mix.
# =============================================================================
import os, json, math, time
import numpy as np, torch
from collections import defaultdict, Counter

FOLD = "A"                      # "A", "B" or "ALL"
RUN_FT = True
UPDATES = 2000
MID_EVAL_AT = 1000              # folds only: one student-only check mid-run
FT_LR_MULT = 0.3
OV_SHARE = 0.35                 # official validation clips (same as OV-3/OV-5)
OS_SHARE = 0.25                 # OV-6 mixtures; the remaining 0.40 is the original mix
BASE_TAG = "v5c_atst_fusion_e2"
TAG = f"v5c_atst_fusion_ovsil_{FOLD}"
EVAL_HALF = None if FOLD == "ALL" else ("B" if FOLD == "A" else "A")

# ---- OV-6 mixtures ---------------------------------------------------------------------------------
OS_ROOT = f"{BASE_DIR}/synth_official_style"
OS_ATST_DIR = f"{BASE_DIR}/atst_cache_osyn"
OS_STORE = SynthStore(OS_ROOT)
OS_IDS = sorted(OS_STORE.items)
assert len(OS_IDS) >= 7000, f"only {len(OS_IDS)} mixtures: finish OV-6 first"
_gold = set(int(g) for g in tiers["gold"])
assert not [s for s in OS_IDS if not os.path.exists(os.path.join(OS_ATST_DIR, f"{s}.npy"))], "mixtures without ATST features"
assert all(int(b[0]) in _gold for s in OS_IDS for b in OS_STORE.items[s]["bed_src"]), "mixture bed outside Gold training clips"

# ---- official validation clips as training rows (same as OV-3/OV-5) ---------------------------------
OV_EVS, OV_TIER = {}, {}
for r in _meta:
    stem = os.path.splitext(r["segmentFileName"])[0]; syn = bool(r["syntheticData"])
    key = ("S_" if syn else "N_") + stem
    OV_EVS[key] = parse_events(r["NoiseSubCategoryTimeStamp"], None)
    OV_TIER[key] = T_SYNTH if syn else T_GOLD
if FOLD == "ALL":
    ov_train_keys = [k for k in OV if OV_EVS[k]]
else:
    ov_train_keys = [k for k in OV if OV[k]["half"] == FOLD and OV_EVS[k]]
    ev_keys = OVK[("N", EVAL_HALF)] + OVK[("S", EVAL_HALF)]
    assert not set(ov_train_keys) & set(ev_keys), "train/eval overlap"
    assert {OV[k]["district"] for k in ov_train_keys}.isdisjoint({OV[k]["district"] for k in ev_keys}), "district overlap"
print(f"FOLD {FOLD}: {len(ov_train_keys)} official clips + {len(OS_IDS)} mixtures" + (f" | held-out half {EVAL_HALF}" if EVAL_HALF else " | NO held-out clips"))


class OVOSMixDataset(V5ATSTDataset):
    """V5ATSTDataset + rows ("ov", key, tier) from the official validation clips + rows ("os", sid, tier) from the OV-6 mixtures."""
    def _load(self, kind, key):
        if kind == "ov":
            wav = load_test_wav(OV[key]["path"]); bf = ov_beats(key, wav.numel()); evs = OV_EVS[key]
            return wav, bf, evs, sorted({c for _, _, c in evs})
        if kind == "os":
            r = OS_STORE.items[key]; wav = OS_STORE.wav(key); bf = OS_STORE.beats(key)
            evs = [(float(s), float(e), CAT_IDX[c]) for s, e, c in r["events"]]
            check_beats_len(bf, wav.numel(), f"os:{key}")
            return wav, bf, evs, sorted({c for _, _, c in evs})
        return super()._load(kind, key)

    def _full(self, kind, key):
        if kind == "ov":
            return ov_atst(key)
        if kind == "os":
            return np.load(os.path.join(OS_ATST_DIR, f"{key}.npy")).astype(np.float32)
        return super()._full(kind, key)


def make_mix_sampler3(base_rows, ov_rows, os_rows, shares, ov_share, os_share, n):
    from torch.utils.data import WeightedRandomSampler
    cnt = Counter(r[2] for r in base_rows); live = {t: s for t, s in shares.items() if cnt.get(t, 0) > 0 and s > 0}; tot = sum(live.values())
    w = ([(1 - ov_share - os_share) * live.get(r[2], 0.0) / tot / cnt[r[2]] if r[2] in live else 0.0 for r in base_rows]
         + [ov_share / len(ov_rows)] * len(ov_rows) + [os_share / len(os_rows)] * len(os_rows))
    return WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=int(n), replacement=True)


def ov_half_scores(net, T_win, fps, half):
    ks = OVK[("N", half)] + OVK[("S", half)]; tr = {}
    for k in ks:
        w = load_test_wav(OV[k]["path"]); tr[k] = tracks_atst(net, w, ov_beats(k, w.numel()), ov_atst(k), device, T_win, fps)
    res = {dom: ov_eval(tr, fps, FIXED_DEC, dom, half) for dom in ("N", "S")}
    res["J"] = 0.62 * res["N"]["combined"] + 0.38 * res["S"]["combined"]
    return res


def fmt(res):
    return " | ".join(f"{'NAT' if d == 'N' else 'SYN'} {res[d]['combined']:.4f} (F1 {res[d]['event_f1']:.4f} Dice {res[d]['segment_dice']:.4f} "
                      f"ev/clip {res[d]['ev']:.2f})" for d in ("N", "S"))


def finetune(updates, run=True):
    set_seed(SEED)
    dj = json.load(open(f"{V5_DIR}/{BASE_TAG}_decoder.json"))
    ck0 = torch.load(dj["checkpoint"], map_location="cpu", weights_only=False)
    model = new_atst_model("fuse", ck0["cfg"]); model.load_state_dict(ck0["model"])
    ema = new_atst_model("fuse", ck0["cfg"]); ema.load_state_dict(ck0["model"]); freeze_model(ema)
    T_win, fps = atst_time_grid(model)
    cache = atst_open_cache(create=False)
    cfg = CFG_ATST
    base_rows = list(TRAIN_ROWS)
    ov_rows = [("ov", k, OV_TIER[k]) for k in ov_train_keys]
    os_rows = [("os", s, T_SYNTH) for s in OS_IDS]
    rows = base_rows + ov_rows + os_rows                      # sampler order: base, official, mixtures
    ds = OVOSMixDataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True, atst_cache=cache, atst_mask_prob=ATST_ROW_MASK_PROB,
                        shift_prob=cfg["shift_prob"], beats_mask_prob=cfg["beats_mask_prob"])
    n_draw = (updates + 20) * cfg["batch_size"] * cfg["grad_accum_steps"]
    sampler = make_mix_sampler3(base_rows, ov_rows, os_rows, cfg["shares"], OV_SHARE, OS_SHARE, n_draw)
    loader = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0, collate_fn=collate_v5_atst,
                                         drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    lr0 = cfg["lr"] * FT_LR_MULT; floor = cfg["min_lr"] / cfg["lr"]
    opt = torch.optim.AdamW(params, lr=lr0, weight_decay=cfg["weight_decay"])
    warm = min(100, max(1, updates // 10))

    def lr_at(u):
        if u < warm:
            return 0.1 + 0.9 * u / warm
        pr = min(1.0, (u - warm) / max(1, updates - warm))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = lr0 * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda")); accum = cfg["grad_accum_steps"]
    base = None
    if run and EVAL_HALF:
        base = {dom: ov_eval(E2["tracks"], E2["fps"], FIXED_DEC, dom, EVAL_HALF) for dom in ("N", "S")}
        base["J"] = 0.62 * base["N"]["combined"] + 0.38 * base["S"]["combined"]
        log(f"BASELINE {BASE_TAG} on held-out half {EVAL_HALF}: J {base['J']:.4f} | {fmt(base)}")
    upd = micro = skipped = 0; runl = defaultdict(float); t0 = time.time(); hist = []
    model.train(); opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = weak_augment(model.mel_frontend(wav.float()))
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device), mel=mel,
                        atst=batch["atst"].to(device, non_blocking=True))
        loss, parts = v5_losses(out, batch, cfg, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            runl[k] += float(v.detach())
        runl["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:
            skipped += 1; continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = lr0 * lr_at(upd)
        update_teacher(model, ema, decay=cfg["ema_decay"])
        if upd % 250 == 0:
            tr_ = " ".join(f"{k}={v / runl['n']:.4f}" for k, v in runl.items() if k != "n"); runl = defaultdict(float)
            log(f"{TAG} upd={upd}/{updates} skipped={skipped} {tr_}")
        if run and EVAL_HALF and upd == MID_EVAL_AT and upd < updates:
            model.eval(); sc = ov_half_scores(model, T_win, fps, EVAL_HALF); hist.append({"update": upd, "student": sc["J"]})
            log(f"{TAG} upd={upd} student: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
            model.train()
        if upd >= updates:
            break
    log(f"{TAG}: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min")
    if not run:
        return None
    paths = {}
    for name, net in (("student", model), ("ema", ema)):
        net.eval(); sd = net.state_dict(); p = f"{V5_DIR}/{TAG}_{name}.pt"
        assert not os.path.exists(p), f"{p} exists"
        sc = None
        if EVAL_HALF:
            sc = ov_half_scores(net, T_win, fps, EVAL_HALF); hist.append({"update": upd, name: sc["J"]})
            log(f"{TAG} FINAL {name}: J {sc['J']:.4f} (baseline {base['J']:.4f}, delta {sc['J'] - base['J']:+.4f}) | {fmt(sc)}")
        torch.save({"model": sd, "update": upd, "score": ({"J": sc["J"], "N": sc["N"]["combined"], "S": sc["S"]["combined"]} if sc else None),
                    "hash": state_hash(sd), "canonical_tags": canonical_tags, "cfg": cfg,
                    "atst": {"kind": "fuse", "cache_tag": ATST_CACHE_TAG, "atst_sha256": ATST_EXPECTED_SHA256, "mel_variant": ATST_MEL_VARIANT,
                             "init_from": BASE_TAG, "ov_train_half": FOLD, "ov_eval_half": EVAL_HALF, "ov_share": OV_SHARE, "os_share": OS_SHARE,
                             "os_mixtures": len(OS_IDS), "updates": upd}}, p)
        paths[name] = p
    json.dump({"hist": hist, "fold": FOLD, "baseline_J": (base["J"] if base else None), "updates": upd},
              open(f"{V5_DIR}/{TAG}_history.json", "w"), default=float)
    return paths


finetune(6, run=False)
print("SMOKE OK (6 updates, nothing written).")
if RUN_FT:
    assert not os.path.exists(f"{V5_DIR}/{TAG}_student.pt"), f"tag {TAG} already used"
    OVOS_PATHS = finetune(UPDATES, run=True)
    print(OVOS_PATHS)
else:
    print(f"RUN_FT is False: set RUN_FT = True to train {TAG}.")

FOLD A: 1216 official clips + 8000 mixtures | held-out half B
[2026-10-16 17:11:48] v5c_atst_fusion_ovsil_A: 6 updates (0 AMP-skipped) in 0.1 min
SMOKE OK (6 updates, nothing written).
[2026-10-16 17:11:48] BASELINE v5c_atst_fusion_e2 on held-out half B: J 1.1993 | NAT 1.0176 (F1 0.3592 Dice 0.6584 ev/clip 1.55) | SYN 1.4958 (F1 0.6809 Dice 0.8149 ev/clip 2.50)
